# LLM-RAG Healthcare Literature Search — Analysis Notebook

Preprocessing, statistical analysis, and figure generation for *Blind Spots in AI-Assisted Healthcare Evidence Search: Retrieval Gaps and Risk of Bias*.

Run on the University of Florida HiperGator cluster. Run the **environment report** cell (below) first to record software versions.


## Cell 00 — ENVIRONMENT REPORT

*Records Python and package versions; run this first.*


In [ ]:
# ============================================================================
# ENVIRONMENT REPORT  —  run this as the FIRST cell of the notebook
# Records the exact Python and package versions used, prints a version table,
# writes environment.txt, and prints a ready-to-paste Code Availability
# sentence for the manuscript.
# ============================================================================
import sys, platform
from importlib import metadata

# Packages actually imported by the analysis notebook
PACKAGES = [
    "pandas", "numpy", "scipy", "statsmodels",
    "scikit-learn", "matplotlib", "seaborn", "plotly",
]
# import name -> distribution name (only where they differ)
DIST = {"scikit-learn": "scikit-learn"}

def _ver(pkg):
    try:
        return metadata.version(DIST.get(pkg, pkg))
    except Exception:
        return "not installed"

py = platform.python_version()
rows = [(p, _ver(p)) for p in PACKAGES]

print("=" * 60)
print("ENVIRONMENT REPORT")
print("=" * 60)
print(f"Python           {py}")
print(f"Platform         {platform.platform()}")
print("-" * 60)
for name, ver in rows:
    print(f"{name:<16} {ver}")
print("=" * 60)

# Write a pinned environment file for the repository
with open("environment.txt", "w") as fh:
    fh.write(f"python=={py}\n")
    for name, ver in rows:
        fh.write(f"{name}=={ver}\n")
print("Wrote environment.txt")

# Ready-to-paste Code Availability sentence (versions filled in automatically)
pkg_str = ", ".join(f"{n} {v}" for n, v in rows)
print("\n--- paste into the Code Availability statement ---\n")
print(
    f"All analyses were performed in Python {py} using {pkg_str}. "
    "The complete analysis code is openly available at "
    "https://github.com/<your-username>/rag-llm-evidence-search."
)


## Cell 01 — SETUP

*Save outputs to ~/Desktop + one-time cleanup*


In [ ]:
# ============================================================================
# The original code wrote generated files to the current working directory;
# launching Jupyter from the Desktop put them on ~/Desktop. This cell makes
# that explicit without creating any new folder.
# ============================================================================
import os
from pathlib import Path

os.chdir(Path('~/Desktop').expanduser())   # outputs land here, as before
print(f"Outputs will be saved to: {os.getcwd()}")

for _f in ['ai2_topk_pooled_recall.csv',
           'query_strategy_platform_blocked_ranks.csv',
           'Accepted_SubThreshold_Matches.csv',
           'query_strategy_recall_report.txt',
           'threshold_sensitivity_report.txt',
           'threshold_sweep_band_digest.csv',
           'Threshold_Sensitivity_Trend.png',
           'Threshold_Sensitivity_Trend.pdf',
           'df_variations_with_nearmatch_foldin.csv',
           'platform_recall_with_nearmatch_foldin.csv',
           'cochran_foldin_adjusted.txt',
           'formulation_recall_foldin_recompute.csv',
           'friedman_foldin_adjusted.txt',
           'primary_recompute_foldin_all_metrics.csv',
           'primary_foldin_adjusted.txt',
           'adopted_primary_platform_recall.csv',
           'appendix_c_formulation_table.md',
           'appendix_c_formulation_table.csv']:
    try:
        os.remove(_f)
        print(f"  removed stray file: {_f}")
    except FileNotFoundError:
        pass

## Cell 02 — BASE CLASS + NEAR-MATCH ADOPTION

*Data loading, matching, AND the near-match adoption — folds accepted near-matches into matched sets at the source*


In [ ]:
# ===== PART 1A: COMMON FUNCTIONS AND BASE ANALYSIS =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from collections import defaultdict
import re
import os
from pathlib import Path
import zipfile
import glob
from sklearn.metrics import jaccard_score
import warnings
warnings.filterwarnings('ignore')

class LLMAnalysisBase:
    def __init__(self):
        # Updated LLM mapping with all lowercase keys for robust parsing
        self.llm_mapping = {
            'ai2': 'Ai2 Finder',        # Handles AI2, Ai2 Finder, ai2
            'con': 'Consensus',  # Handles Con, CON, con
            'cons': 'Consensus', # Handles Cons, CONS, cons
            'gpt': 'ChatGPT',    # Handles GPT, gpt, Gpt
            'cld': 'Claude',     # Handles Cld, CLD, cld
            'gem': 'Gemini'      # Handles Gem, GEM, gem, Gemini -> gem
        }
        self.strategy_mapping = {
            '1': 'Zero-Shot',
            '2': 'Few-Shot',
            '3': 'Persona-based',
            '4': 'Chain-of-Thought',
            '5': 'PICO Framework'
        }
        self.standardized_files = {}  # For standardized CSV files
        self.recalled_files = {}      # For recalled CSV files
        self.gold_standard_df = None  # Gold standard data
        self.gold_standard_count = 83
        # Venue categorization based on your reference CSV
        self.journal_names = {
            'JAMIA', 'Journal of Biomedical Informatics', 'Informatics in Medicine Unlocked',
            'JMIR Medical Informatics', 'BMC Medical Informatics and Decision Making',
            'Open Forum Infectious Diseases', 'Anesthesia & Analgesia', 'Health Informatics Journal',
            'Journal of Pain and Symptom Management', 'PLOS ONE', 'Healthcare',
            'Alcohol: Clinical and Experimental Research', 'Alcohol', 'npj Digital Medicine',
            'International Journal of Medical Informatics', 'Regional Anesthesia & Pain Medicine',
            'JCO Clinical Cancer Informatics', 'Frontiers in Public Health',
            'Methods of Information in Medicine', 'Journal of Personalized Medicine',
            'Journal of Pain Research', 'International Journal of Environmental Research and Public Health',
            'Addiction', 'IEEE Access', 'Technology and Health Care',
            'Pharmacoepidemiology and Drug Safety', 'Pain Medicine', 'Journal of Clinical Epidemiology',
            'Applied Clinical Informatics', 'BMC Medical Research Methodology',
            'European Heart Journal – Digital Health', 'Substance Abuse',
            'Upsala Journal of Medical Sciences', 'Applied Sciences', 'Molecular Psychiatry',
            'Sensors', 'The Lancet Digital Health', 'Addiction Science & Clinical Practice'
        }
        self.conference_names = {
            '2020 IEEE ICHI', '2023 IEEE BigData', '2018 IEEE iCCECE', 'AMIA Symposium Proceedings',
            '2020 IEEE BIBM', 'Studies in Health Technology and Informatics', '2022 ICMHI',
            '2022 DataSciTech', 'BioNLP', '2024 ACM Conference', '2022 IEEE ICHI', '2014 IEEE EMBC'
        }
        self.publisher_names = {
            'IEEE', 'Springer Nature', 'AMIA', 'OUP', 'Elsevier', 'JMIR', 'Wolters Kluwer',
            'IOS Press', 'ACM', 'SCITEPRESS', 'ACL', 'SAGE', 'PLOS', 'MDPI', 'Wiley',
            'BMJ', 'Frontiers', 'Thieme', 'Taylor & Francis'
        }

    # ========================= FILE PROCESSING METHODS =========================
    def normalize_identifier_consistent(self, identifier):
        """Consistent normalization used throughout all functions"""
        if not identifier or pd.isna(identifier):
            return ""
        identifier = str(identifier).strip().lower()
        # Handle DOI formats comprehensively
        if identifier.startswith('doi:'):
            identifier = identifier.replace('doi:', '').strip()
        elif identifier.startswith('https://doi.org/'):
            identifier = identifier.replace('https://doi.org/', '').strip()
        elif identifier.startswith('http://dx.doi.org/'):
            identifier = identifier.replace('http://dx.doi.org/', '').strip()
        elif identifier.startswith('http://doi.org/'):
            identifier = identifier.replace('http://doi.org/', '').strip()
        # Remove any remaining doi: prefix
        identifier = identifier.replace('doi:', '').strip()
        return identifier

    def clean_venue_name(self, venue_name):
        """Remove suffixes and standardize venue names for display"""
        venue = str(venue_name).strip()
        # Remove classification suffixes
        venue = re.sub(r'\s*\(Journal\)\s*$', '', venue, flags=re.IGNORECASE)
        venue = re.sub(r'\s*\(Conference/proceeding\)\s*$', '', venue, flags=re.IGNORECASE)
        venue = re.sub(r'\s*\(Conference\)\s*$', '', venue, flags=re.IGNORECASE)
        venue = re.sub(r'\s*\(Proceeding\)\s*$', '', venue, flags=re.IGNORECASE)
        # Apply standardization after removing suffixes
        return self.standardize_venue_name(venue)

    def standardize_venue_name(self, venue_name):
        """Standardize long venue names"""
        venue = str(venue_name)
        # Conference name standardization
        if 'IEEE' in venue and 'Conference' in venue:
            # Extract year and main conference name
            year_match = re.search(r'(\d{4})', venue)
            year = year_match.group(1) if year_match else ''
            if 'Healthcare Informatics' in venue or 'ICHI' in venue:
                return f'IEEE ICHI {year}'
            elif 'EMBC' in venue or 'Engineering in Medicine' in venue:
                return f'IEEE EMBC {year}'
            elif 'BigData' in venue:
                return f'IEEE BigData {year}'
            elif 'Data Science' in venue:
                return f'IEEE Data Sci {year}'
            else:
                return f'IEEE Conf {year}'
        # Other conference standardizations
        if 'AMIA' in venue:
            year_match = re.search(r'(\d{4})', venue)
            year = year_match.group(1) if year_match else ''
            return f'AMIA {year}'
        # Journal standardizations
        if 'Journal of Medical Internet Research' in venue:
            return 'JMIR'
        elif 'JMIR' in venue and 'Medical Informatics' in venue:
            return 'JMIR Med Inform'
        elif 'JMIR' in venue and 'Public Health' in venue:
            return 'JMIR Public Health'
        # Truncate if still too long
        if len(venue) > 25:
            return venue[:22] + "..."
        return venue

    def parse_filename(self, filename):
        """Enhanced filename parsing that handles both hyphens and underscores"""
        # Remove file extension
        base_name = filename.split('.')[0]
        # Check if it's a recalled file (ends with -R or _R)
        is_recalled = base_name.endswith('-R') or base_name.endswith('_R') or base_name.endswith('R')
        if is_recalled:
            if base_name.endswith('-R') or base_name.endswith('_R'):
                base_name = base_name[:-2]  # Remove -R or _R suffix
            elif base_name.endswith('R'):
                base_name = base_name[:-1]  # Remove R suffix
        # Pattern: LLM-XYZ or LLM_XYZ where X=strategy, Y=variation, Z=run
        patterns = [
            r'([a-zA-Z0-9]+)-(\d)(\d)(\d)',  # Original hyphen pattern
            r'([a-zA-Z0-9]+)_(\d)(\d)(\d)',  # Underscore pattern
            r'([a-zA-Z0-9]+)(\d)(\d)(\d)'    # No separator pattern
        ]
        match = None
        for pattern in patterns:
            match = re.match(pattern, base_name, re.IGNORECASE)
            if match:
                break
        if match:
            # Convert LLM code to lowercase for consistent matching
            raw_llm_code = match.group(1)
            llm_code = raw_llm_code.lower()
            strategy = match.group(2)
            variation = match.group(3)
            run = match.group(4)
            # Handle special cases
            if llm_code == 'gemini':  # Handle full "Gemini" name
                llm_code = 'gem'
            # Check against mapping
            if llm_code in self.llm_mapping:
                return {
                    'llm': self.llm_mapping[llm_code],
                    'strategy': strategy,
                    'variation': variation,
                    'run': run,
                    'is_recalled': is_recalled
                }
            else:
                print(f"Debug: LLM code '{llm_code}' not found in mapping. Available codes: {list(self.llm_mapping.keys())}")
        else:
            print(f"Debug: Filename {filename} does not match expected pattern")
        return None

    def load_files_from_path(self, base_path="~/Desktop/Pooled Std Recall Runs"):
        """Load and organize CSV files from the specified directory path - CURRENT DIRECTORY ONLY"""
        print(f"Loading CSV files from: {base_path}")
        # Convert to Path object for better handling
        base_dir = Path(base_path).expanduser()
        if not base_dir.exists():
            print(f"Error: Directory {base_dir} does not exist!")
            print("Please check the path and try again.")
            return False
        print(f"Directory found: {base_dir}")
        # Look for gold standard file - ONLY in current directory
        gold_standard_files = []
        for pattern in ['*gold*standard*.csv', '*Gold*Standard*.csv', 'gold_standard.csv', 'Gold_Standard.csv']:
            gold_standard_files.extend(list(base_dir.glob(pattern)))
        if gold_standard_files:
            gold_file = gold_standard_files[0]  # Take the first match
            print(f"Loading Gold Standard from: {gold_file}")
            try:
                self.gold_standard_df = pd.read_csv(gold_file)
                self.gold_standard_count = len(self.gold_standard_df)
                print(f"✓ Loaded Gold Standard with {self.gold_standard_count} studies")
                print("Gold Standard columns:", list(self.gold_standard_df.columns))
            except Exception as e:
                print(f"Error loading Gold Standard file: {e}")
                return False
        else:
            print("Warning: Gold Standard file not found!")
        # Find all CSV files - ONLY in current directory, NOT in subdirectories
        print("\nScanning for CSV files...")
        all_csv_files = []
        # Search ONLY in current directory (no subdirectories)
        all_csv_files.extend(list(base_dir.glob('*.csv')))
        print(f"Found {len(all_csv_files)} CSV files in current directory only")
        # Use sets to track unique files and detect duplicates
        processed_files = set()
        duplicate_count = 0
        processed_count = 0
        for filepath in all_csv_files:
            filename = filepath.name
            # Skip gold standard file
            if any(term in filename.lower() for term in ['gold', 'standard']):
                continue
            # Skip hidden files and system files
            if filename.startswith('.') or filename.startswith('~'):
                continue
            # Check for duplicates by filename
            if filename in processed_files:
                duplicate_count += 1
                print(f"⚠ Duplicate file skipped: {filename}")
                continue
            processed_files.add(filename)
            # Parse filename
            parsed = self.parse_filename(filename)
            if parsed:
                processed_count += 1
                # Create unique key for this variation
                file_key = f"{parsed['llm']}-{parsed['strategy']}{parsed['variation']}-{parsed['run']}"
                if parsed['is_recalled']:
                    # Recalled files (processed with metadata)
                    existing_keys = [info.get('file_key') for info in self.recalled_files.values()]
                    if file_key not in existing_keys:
                        self.recalled_files[filename] = {
                            'path': str(filepath),
                            'llm': parsed['llm'],
                            'strategy': parsed['strategy'],
                            'variation': parsed['variation'],
                            'run': parsed['run'],
                            'file_key': file_key
                        }
                        print(f"✓ Recalled file: {filename} -> {file_key}")
                    else:
                        print(f"⚠ Duplicate variation skipped: {filename} -> {file_key}")
                else:
                    # Standardized files
                    existing_keys = [info.get('file_key') for info in self.standardized_files.values()]
                    if file_key not in existing_keys:
                        self.standardized_files[filename] = {
                            'path': str(filepath),
                            'llm': parsed['llm'],
                            'strategy': parsed['strategy'],
                            'variation': parsed['variation'],
                            'run': parsed['run'],
                            'file_key': file_key
                        }
                        print(f"✓ Standardized file: {filename} -> {file_key}")
                    else:
                        print(f"⚠ Duplicate variation skipped: {filename} -> {file_key}")
            else:
                if not any(term in filename.lower() for term in ['readme', 'info', 'notes']):
                    print(f"⚠ Skipped (no pattern match): {filename}")
        print(f"\n=== SUMMARY ===")
        print(f"Total CSV files found in current directory: {len(all_csv_files)}")
        print(f"Duplicate files skipped: {duplicate_count}")
        print(f"Files processed: {processed_count}")
        print(f"Standardized files: {len(self.standardized_files)}")
        print(f"Recalled files: {len(self.recalled_files)}")
        if self.standardized_files:
            standardized_llms = set([info['llm'] for info in self.standardized_files.values()])
            print(f"LLMs found in standardized files: {standardized_llms}")
        if self.recalled_files:
            recalled_llms = set([info['llm'] for info in self.recalled_files.values()])
            print(f"LLMs found in recalled files: {recalled_llms}")
        # Analyze file distribution
        print(f"\n=== FILE DISTRIBUTION ANALYSIS ===")
        self._analyze_file_distribution()
        # Check if we have enough files to proceed
        if len(self.standardized_files) == 0 and len(self.recalled_files) == 0:
            print("\n❌ No valid CSV files found!")
            print("Please check that your files follow the expected naming pattern:")
            print("  Standardized files: LLM-XYZ.csv (e.g., GPT-111.csv, CON-422.csv)")
            print("  Recalled files: LLM-XYZR.csv (e.g., GPT-111R.csv, CON-422R.csv)")
            return False
        return True

    def _analyze_file_distribution(self):
        """Analyze the distribution of files across LLMs, strategies, and variations"""
        print("Analyzing file distribution...")
        # Analyze standardized files
        standardized_groups = defaultdict(list)
        for filename, info in self.standardized_files.items():
            key = f"{info['llm']}-{info['strategy']}{info['variation']}"
            standardized_groups[key].append(info['run'])
        print(f"\nStandardized files distribution:")
        missing_standardized = []
        for group_key, runs in standardized_groups.items():
            expected_runs = {'1', '2', '3'}
            actual_runs = set(runs)
            missing = expected_runs - actual_runs
            if missing:
                missing_standardized.append((group_key, missing))
                print(f"  ⚠ {group_key}: has runs {sorted(actual_runs)}, missing {sorted(missing)}")
            else:
                print(f"  ✓ {group_key}: complete (runs 1,2,3)")
        # Analyze recalled files
        recalled_groups = defaultdict(list)
        for filename, info in self.recalled_files.items():
            key = f"{info['llm']}-{info['strategy']}{info['variation']}"
            recalled_groups[key].append(info['run'])
        print(f"\nRecalled files distribution:")
        missing_recalled = []
        for group_key, runs in recalled_groups.items():
            expected_runs = {'1', '2', '3'}
            actual_runs = set(runs)
            missing = expected_runs - actual_runs
            if missing:
                missing_recalled.append((group_key, missing))
                print(f"  ⚠ {group_key}: has runs {sorted(actual_runs)}, missing {sorted(missing)}")
            else:
                print(f"  ✓ {group_key}: complete (runs 1,2,3)")
        if missing_standardized:
            print(f"\n⚠ Warning: {len(missing_standardized)} standardized file groups are incomplete")
        if missing_recalled:
            print(f"⚠ Warning: {len(missing_recalled)} recalled file groups are incomplete")
        if not missing_standardized and not missing_recalled:
            print(f"\n✓ All file groups are complete!")

    def load_csv_file_content(self, filepath):
        """Load content from CSV files"""
        try:
            return pd.read_csv(filepath)
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            return pd.DataFrame()

    # ========================= DATA EXTRACTION METHODS =========================
    def extract_titles_from_standardized_files(self, df):
        """Extract titles from standardized CSV files - ENHANCED VERSION"""
        if df.empty:
            return set()
        # Priority order for title columns
        title_columns = ['Title', 'title', 'Study Title', 'Article Title', 'Study name', 'study_title', 'article_title']
        for col in title_columns:
            if col in df.columns:
                titles = df[col].dropna().astype(str).str.strip()
                # Remove empty strings and normalize
                normalized_titles = set()
                for title in titles:
                    if title and title.lower() not in ['nan', 'none', '']:
                        # Basic normalization: lowercase, strip whitespace
                        normalized_title = title.lower().strip()
                        normalized_titles.add(normalized_title)
                print(f"    Extracted {len(normalized_titles)} titles from column '{col}'")
                return normalized_titles
        # If no title column found, try first column as fallback
        if len(df.columns) > 0:
            first_col_titles = df.iloc[:, 0].dropna().astype(str).str.strip()
            normalized_titles = set()
            for title in first_col_titles:
                if title and title.lower() not in ['nan', 'none', '']:
                    normalized_title = title.lower().strip()
                    normalized_titles.add(normalized_title)
            print(f"    Extracted {len(normalized_titles)} titles from first column (fallback)")
            return normalized_titles
        return set()

    def extract_studies_from_recalled_files(self, df):
        """Extract study identifiers from recalled CSV files with metadata"""
        if df.empty:
            return set(), df
        # Priority order for study identification in recalled files
        identifier_columns = ['DOI', 'Study name', 'Title', 'LLM Title']
        for col in identifier_columns:
            if col in df.columns:
                studies = df[col].dropna().astype(str).str.strip()
                if col == 'DOI':
                    return set(studies), df  # DOIs as-is
                else:
                    return set(studies.str.lower()), df  # Titles in lowercase
        return set(), df

    def calculate_jaccard_similarity(self, set1, set2):
        """Calculate Jaccard similarity between two sets"""
        if not set1 and not set2:
            return 1.0
        if not set1 or not set2:
            return 0.0
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        return intersection / union if union > 0 else 0.0

    # ========================= CORE ANALYSIS METHODS =========================
    def calculate_consistency_scores(self):
        """Calculate consistency scores at both variation and strategy levels"""
        print("Calculating consistency scores using standardized CSV files...")
        # Two dictionaries for different granularities
        variation_consistency_scores = {}  # LLM-Strategy-Variation level
        strategy_consistency_scores = {}   # LLM-Strategy level (averaged across variations)
        # Group standardized files by LLM and strategy
        groups = defaultdict(lambda: defaultdict(list))
        for filename, info in self.standardized_files.items():
            llm_strategy_key = f"{info['llm']}-{info['strategy']}"
            variation = info['variation']
            groups[llm_strategy_key][variation].append((filename, info))
        print(f"Debug: Consistency analysis - found {len(groups)} LLM-Strategy groups")
        for llm_strategy_key, variations in groups.items():
            print(f"Debug: Processing {llm_strategy_key} with {len(variations)} variations")
            # Calculate consistency for each variation separately (for detailed plots)
            variation_consistencies = []
            for variation, files in variations.items():
                variation_key = f"{llm_strategy_key}{variation}"  # e.g., "Ai2 Finder-11"
                print(f"  Analyzing variation {variation} with {len(files)} runs")
                if len(files) < 2:
                    print(f"  Warning: Variation {variation} has only {len(files)} files, using consistency = 1.0")
                    variation_consistency_scores[variation_key] = 1.0
                    variation_consistencies.append(1.0)
                    continue
                # Load titles for each run from standardized files
                titles_by_run = {}
                for filename, info in files:
                    df = self.load_csv_file_content(info['path'])
                    titles = self.extract_titles_from_standardized_files(df)
                    titles_by_run[info['run']] = titles
                    print(f"    Run {info['run']}: {len(titles)} titles")
                # Calculate pairwise Jaccard similarities between runs
                runs = list(titles_by_run.keys())
                similarities = []
                for i in range(len(runs)):
                    for j in range(i + 1, len(runs)):
                        sim = self.calculate_jaccard_similarity(
                            titles_by_run[runs[i]],
                            titles_by_run[runs[j]]
                        )
                        similarities.append(sim)
                        print(f"    Jaccard similarity between run {runs[i]} and {runs[j]}: {sim:.3f}")
                # Average consistency score for this specific variation
                if similarities:
                    variation_consistency = np.mean(similarities)
                    variation_consistency_scores[variation_key] = variation_consistency
                    variation_consistencies.append(variation_consistency)
                    print(f"  Variation {variation} consistency: {variation_consistency:.3f}")
                else:
                    variation_consistency_scores[variation_key] = 1.0
                    variation_consistencies.append(1.0)
                    print(f"  Variation {variation} consistency: 1.0 (single file)")
            # Calculate overall LLM-Strategy consistency by averaging across variations
            if variation_consistencies:
                overall_consistency = np.mean(variation_consistencies)
                strategy_consistency_scores[llm_strategy_key] = overall_consistency
                print(f"Overall {llm_strategy_key} consistency: {overall_consistency:.3f}")
        # Return both dictionaries
        return variation_consistency_scores, strategy_consistency_scores

    def calculate_recall_scores(self):
        """Calculate recall scores using recalled CSV files with metadata"""
        print("Calculating recall scores using recalled CSV files...")
        recall_scores = {}
        variation_metadata = {}  # Store metadata for each variation
        # Group recalled files by LLM, strategy, and variation
        groups = defaultdict(list)
        for filename, info in self.recalled_files.items():
            key = f"{info['llm']}-{info['strategy']}{info['variation']}"
            groups[key].append((filename, info))
        print(f"Debug: Recall analysis - found {len(groups)} groups")
        for group_key in groups.keys():
            print(f"Debug: Group {group_key} has {len(groups[group_key])} files")
        for group_key, files in groups.items():
            # Combine all matched studies across runs
            all_matches = set()
            combined_metadata = []
            for filename, info in files:
                df = self.load_csv_file_content(info['path'])
                if not df.empty:
                    matches, metadata_df = self.extract_studies_from_recalled_files(df)
                    all_matches.update(matches)
                    combined_metadata.append(metadata_df)
            # Combine metadata from all runs
            if combined_metadata:
                full_metadata = pd.concat(combined_metadata, ignore_index=True).drop_duplicates()
            else:
                full_metadata = pd.DataFrame()
            # Calculate recall against gold standard
            recall = len(all_matches) / self.gold_standard_count if self.gold_standard_count > 0 else 0.0
            recall_scores[group_key] = {
                'recall': recall,
                'matched_count': len(all_matches),
                'matches': all_matches
            }
            variation_metadata[group_key] = full_metadata
        return recall_scores, variation_metadata

    # ========================= KNOWLEDGE GAP ANALYSIS METHODS =========================
    def create_study_identifier_mapping(self):
        """Create a comprehensive mapping of study identifiers with consistent normalization"""
        if self.gold_standard_df is None:
            return {}
        study_mapping = {}
        for idx, row in self.gold_standard_df.iterrows():
            study_id = row.get('Study ID', f'Study_{idx}')
            identifiers = set()
            # Add normalized title
            if 'Title' in row and pd.notna(row['Title']):
                title_normalized = self.normalize_identifier_consistent(row['Title'])
                if title_normalized:
                    identifiers.add(title_normalized)
            # Add normalized DOI
            if 'DOI' in row and pd.notna(row['DOI']):
                doi_normalized = self.normalize_identifier_consistent(row['DOI'])
                if doi_normalized:
                    identifiers.add(doi_normalized)
            # Add normalized Study ID
            study_id_normalized = self.normalize_identifier_consistent(study_id)
            if study_id_normalized:
                identifiers.add(study_id_normalized)
            study_mapping[study_id] = {
                'identifiers': identifiers,
                'row_data': row
            }
        return study_mapping

    def identify_never_retrieved_studies(self, top_combinations):
        """Identify studies that are never retrieved by any combination with improved matching"""
        if self.gold_standard_df is None:
            return []
        # Get all matches from all top combinations and normalize them consistently
        all_retrieved = set()
        for combo in top_combinations:
            for match in combo['matches']:
                normalized_match = self.normalize_identifier_consistent(match)
                if normalized_match:
                    all_retrieved.add(normalized_match)
        print(f"Debug: Total unique normalized identifiers retrieved: {len(all_retrieved)}")
        # Create study mapping with consistent normalization
        study_mapping = {}
        for idx, row in self.gold_standard_df.iterrows():
            study_id = row.get('Study ID', f'Study_{idx}')
            identifiers = set()
            # Add normalized title
            if 'Title' in row and pd.notna(row['Title']):
                title_normalized = self.normalize_identifier_consistent(row['Title'])
                if title_normalized:
                    identifiers.add(title_normalized)
            # Add normalized DOI
            if 'DOI' in row and pd.notna(row['DOI']):
                doi_normalized = self.normalize_identifier_consistent(row['DOI'])
                if doi_normalized:
                    identifiers.add(doi_normalized)
            # Add normalized Study ID
            study_id_normalized = self.normalize_identifier_consistent(study_id)
            if study_id_normalized:
                identifiers.add(study_id_normalized)
            study_mapping[study_id] = {
                'identifiers': identifiers,
                'row_data': row
            }
        # Find never retrieved studies with consistent matching
        never_retrieved = []
        retrieved_studies = set()
        for study_id, study_info in study_mapping.items():
            study_retrieved = False
            # Check if any identifier for this study was retrieved
            for identifier in study_info['identifiers']:
                if identifier in all_retrieved:
                    study_retrieved = True
                    retrieved_studies.add(study_id)
                    break
            if not study_retrieved:
                never_retrieved.append(study_info['row_data'])
        print(f"Debug: Studies retrieved: {len(retrieved_studies)} out of {len(study_mapping)} total studies")
        print(f"Debug: Studies never retrieved: {len(never_retrieved)} out of {len(study_mapping)} total studies")
        print(f"Debug: Retrieval rate: {len(retrieved_studies)/len(study_mapping)*100:.1f}%")
        return never_retrieved

    def preprocess_metadata_columns(self):
        """Preprocess NLP Methodology and Substances columns to extract individual methods/substances"""
        if self.gold_standard_df is None:
            return {}
        # Process substances
        substances_by_study = {}
        all_substances = set()
        for idx, row in self.gold_standard_df.iterrows():
            study_id = row.get('Study ID', f'Study_{idx}')
            substances = []
            if 'Substances covered' in row and pd.notna(row['Substances covered']):
                substances_str = str(row['Substances covered'])
                # Split by semicolon and clean
                substances = [s.strip() for s in substances_str.split(';') if s.strip()]
                all_substances.update(substances)
            substances_by_study[study_id] = substances
        # Process NLP methodologies
        methodologies_by_study = {}
        all_methodologies = set()
        for idx, row in self.gold_standard_df.iterrows():
            study_id = row.get('Study ID', f'Study_{idx}')
            methodologies = []
            if 'NLP Methodology' in row and pd.notna(row['NLP Methodology']):
                method_str = str(row['NLP Methodology'])
                # Split by semicolon and clean
                methodologies = [m.strip() for m in method_str.split(';') if m.strip()]
                all_methodologies.update(methodologies)
            methodologies_by_study[study_id] = methodologies
        return {
            'substances_by_study': substances_by_study,
            'all_substances': list(all_substances),
            'methodologies_by_study': methodologies_by_study,
            'all_methodologies': list(all_methodologies)
        }

    def categorize_substances_final(self, substance_list):
        """Categorize substances based on the manuscript Results section"""
        # Handle single substances first
        if len(substance_list) == 1:
            substance = substance_list[0].lower()
            # Tobacco category (most studied, n=41)
            if 'tobacco' in substance or 'smoking' in substance:
                return 'Tobacco', 'Tobacco'
            # Alcohol category (n=23)
            elif 'alcohol' in substance:
                return 'Alcohol', 'Alcohol'
            # Non-prescription Opioids category (n=27)
            elif 'opioid' in substance:
                return 'Opioids', 'Opioids'
            # Cannabis category (n=8)
            elif 'cannabis' in substance:
                return 'Cannabis', 'Cannabis'
            # Stimulants category (cocaine, amphetamine - n=3)
            elif any(stim in substance for stim in ['cocaine', 'amphetamine']):
                if 'cocaine' in substance:
                    return 'Stimulants', 'Cocaine'
                elif 'amphetamine' in substance:
                    return 'Stimulants', 'Amphetamines'
                else:
                    return 'Stimulants', 'Stimulants'
            # Other substances
            elif 'mdma' in substance:
                return 'Other Substances', 'MDMA'
            elif 'electronic cigarette' in substance:
                return 'Other Substances', 'E-Cigarettes'
            elif 'intravenous' in substance or 'iv drug' in substance:
                return 'Other Substances', 'IV Drugs'
            elif any(term in substance for term in ['substance not otherwise', 'drug', 'substance']):
                return 'Other Substances', 'Unspecified Drugs'
            else:
                return 'Other Substances', 'Other'
        # Handle multiple substances
        substance_str = '; '.join(substance_list).lower()
        if 'polysubstance' in substance_str:
            return 'Other Substances', 'Polysubstance Use'
        # Count major substance categories
        has_tobacco = any('tobacco' in s.lower() or 'smoking' in s.lower() for s in substance_list)
        has_alcohol = any('alcohol' in s.lower() for s in substance_list)
        has_opioids = any('opioid' in s.lower() for s in substance_list)
        has_cannabis = any('cannabis' in s.lower() for s in substance_list)
        has_stimulants = any(any(stim in s.lower() for stim in ['cocaine', 'amphetamine']) for s in substance_list)
        major_categories = sum([has_tobacco, has_alcohol, has_opioids, has_cannabis, has_stimulants])
        # Create display text
        display_parts = []
        if has_tobacco: display_parts.append('Tobacco')
        if has_alcohol: display_parts.append('Alcohol')
        if has_opioids: display_parts.append('Opioids')
        if has_cannabis: display_parts.append('Cannabis')
        if has_stimulants: display_parts.append('Stimulants')
        # Determine primary category
        if major_categories >= 2:
            return 'Other Substances', '/'.join(display_parts[:4])
        elif major_categories == 1:
            if has_tobacco: return 'Tobacco', '/'.join(display_parts)
            elif has_alcohol: return 'Alcohol', '/'.join(display_parts)
            elif has_opioids: return 'Opioids', '/'.join(display_parts)
            elif has_cannabis: return 'Cannabis', '/'.join(display_parts)
            elif has_stimulants: return 'Stimulants', '/'.join(display_parts)
        return 'Other Substances', '/'.join(display_parts) if display_parts else 'Unspecified'

    def categorize_nlp_methodologies(self, methodology_list):
        """Categorize NLP methodologies into 4 main categories based on the research field"""
        methodology_str = '; '.join(methodology_list).lower()
        # Define the 4 main categories based on your specification
        if any(keyword in methodology_str for keyword in [
            'rule-based', 'expert system', 'heuristic', 'pattern matching',
            'regular expression', 'dictionary-based', 'keyword', 'lexicon',
            'ontology', 'grammar', 'parsing', 'regex']):
            return 'Rule-based', '; '.join(methodology_list[:2])
        elif any(keyword in methodology_str for keyword in [
            'transformer', 'bert', 'gpt', 'llm', 'large language model',
            'chatgpt', 'claude', 'gemini', 'openai', 'generative ai',
            'pre-trained', 'fine-tuning', 'prompt', 'llama', 't5']):
            return 'Large Language Models (LLMs)', '; '.join(methodology_list[:2])
        elif any(keyword in methodology_str for keyword in [
            'deep learning', 'neural network', 'cnn', 'rnn', 'lstm', 'gru',
            'convolutional', 'recurrent', 'embedding', 'word2vec', 'glove',
            'attention', 'encoder', 'decoder', 'bilstm', 'seq2seq']):
            return 'Deep Learning (Non-Transformer)', '; '.join(methodology_list[:2])
        elif any(keyword in methodology_str for keyword in [
            'machine learning', 'svm', 'support vector', 'random forest',
            'naive bayes', 'logistic regression', 'decision tree',
            'k-means', 'clustering', 'classification', 'supervised',
            'unsupervised', 'feature extraction', 'bag of words',
            'tf-idf', 'n-gram', 'statistical', 'frequency']):
            return 'Conventional Machine Learning', '; '.join(methodology_list[:2])
        else:
            return 'Other Methods', '; '.join(methodology_list[:2])

    def get_text_color_for_background(self, background_intensity):
        """Determine text color (black or white) based on background intensity"""
        # If background is light (low intensity), use black text
        # If background is dark (high intensity), use white text
        return 'white' if background_intensity > 0.5 else 'black'

    # ========================= BASE ANALYSIS WORKFLOW =========================
    def run_base_analysis(self, files_path="~/Desktop/Pooled Std Recall Runs"):
        """Run the base analysis steps common to both methods"""
        print("=== LLM Literature Search Base Analysis ===")
        print("Standardized CSV files: Used for Jaccard similarity (consistency analysis)")
        print("Recalled CSV files (with metadata): Used for all other analyses")
        print("Gold Standard CSV: Used for gap analysis and heatmaps\n")
        
        # Step 1: Load files from path
        if not self.load_files_from_path(files_path):
            print("❌ Failed to load files. Analysis cannot proceed.")
            return None
        
        # Step 2: Calculate consistency scores at both granularities
        print("\n=== STEP 2: Consistency Analysis (Standardized CSV Files) ===")
        variation_consistency_scores, strategy_consistency_scores = self.calculate_consistency_scores()
        
        # Step 3: Calculate recall scores using recalled CSV files
        print("\n=== STEP 3: Recall Analysis (Recalled CSV Files) ===")
        recall_scores, variation_metadata = self.calculate_recall_scores()
        
        # Store all computed variables needed for further analysis
        return {
            'analyzer': self,
            'variation_consistency_scores': variation_consistency_scores,
            'strategy_consistency_scores': strategy_consistency_scores,
            'recall_scores': recall_scores,
            'variation_metadata': variation_metadata
        }

# ============================================================================
# ADOPT ACCEPTED NEAR-MATCHES INTO THE CORE ANALYSIS  [reviewer robustness]
# the matches they are, folded into the matched sets at the SOURCE. The ENTIRE
# existing kit (recall, precision, consistency, F1, Friedman, Cochran's Q,
# post-hocs, diminishing returns, every figure) then runs on the corrected data
# with no other change -- i.e. the analysis redone with these studies included,
# ============================================================================
ADOPT_NEARMATCHES = True

if ADOPT_NEARMATCHES and not getattr(LLMAnalysisBase, '_nearmatch_patched', False):
    import os as _os
    import pandas as _pd
    from pathlib import Path as _P

    def _nm_load_acc():
        _dirs = [_P.cwd(), _P('~/Desktop').expanduser(),
                 _P('~/Desktop/Pooled Std Recall Runs').expanduser()]
        _names = ['02_Sensitivity_Analysis_20to60.csv', '02_Sensitivity_Analysis_40to60.csv']
        p = next((str(d / n) for d in _dirs for n in _names if (d / n).exists()), None)
        if p is None:
            return {}, None
        sa = _pd.read_csv(p)
        acc = sa[sa['Sensitivity_Class'] == 'Confirmed false negative']
        p2llm = {'Ai2 Paper Finder': 'Ai2 Finder', 'ChatGPT-4.1': 'ChatGPT',
                 'Claude-Sonnet-4.0': 'Claude', 'Consensus': 'Consensus',
                 'Gemini-2.5-Pro': 'Gemini'}
        cell = {}
        for _, r in acc.iterrows():
            llm = p2llm.get(r['Platform'], r['Platform'])
            # The sensitivity file numbers variations 1-3, but the recalled-file names
            # (and therefore recalled_files['variation']) use the 0-2 filename digit.
            # Map SA variation v -> filename digit v-1 so studies land in the CORRECT cell.
            try:
                _vfile = str(int(r['Variation']) - 1)
            except (TypeError, ValueError):
                _vfile = str(r['Variation'])
            cell.setdefault((llm, str(r['Strategy']), _vfile, str(r['Run'])),
                            set()).add(str(r['Gold_Title']).strip().lower())
        return cell, p

    _orig_load = LLMAnalysisBase.load_csv_file_content
    def _load_tracking(self, path, *a, **k):
        self._current_path = str(path)
        return _orig_load(self, path, *a, **k)
    LLMAnalysisBase.load_csv_file_content = _load_tracking

    _orig_extract = LLMAnalysisBase.extract_studies_from_recalled_files
    def _extract_aug(self, df):
        studies, meta = _orig_extract(self, df)
        if not hasattr(self, '_nm_path_adds'):
            acc_cell, _sa_used = _nm_load_acc()
            self._nm_path_adds = {}
            _unmatched_keys = []
            for fn, info in getattr(self, 'recalled_files', {}).items():
                key = (info['llm'], str(info['strategy']), str(info['variation']), str(info['run']))
                if key in acc_cell:
                    self._nm_path_adds[str(info['path'])] = acc_cell[key]
            self._nm_total = sum(len(v) for v in acc_cell.values())
            self._nm_files = len(self._nm_path_adds)
            if not acc_cell:
                print("  [near-match fold-in WARNING] sensitivity file NOT found or had no "
                      "confirmed false negatives -> NO studies folded in. Searched cwd, ~/Desktop, "
                      "and ~/Desktop/Pooled Std Recall Runs. Place 02_Sensitivity_Analysis_20to60.csv "
                      "in one of these and re-run, or the analysis stays at strict-60%.")
            elif self._nm_files == 0:
                _ac = list(acc_cell.keys())[:3]
                _rf = [(i['llm'], str(i['strategy']), str(i['variation']), str(i['run']))
                       for i in list(self.recalled_files.values())[:3]]
                print("  [near-match fold-in WARNING] sensitivity file found (" + str(_sa_used) +
                      ") with " + str(len(acc_cell)) + " cells, but ZERO recalled files matched "
                      "its (llm,strategy,variation,run) keys -> key-format mismatch, nothing folded in.")
                print("    acc_cell sample keys:    " + str(_ac))
                print("    recalled_files sample:   " + str(_rf))
                print("    Fix: align the key formats (these must be identical strings).")
            else:
                print(f"  [near-match fold-in ACTIVE] sensitivity file: {_sa_used}")
                print(f"  {self._nm_total} accepted occurrences across {self._nm_files} recalled "
                      f"files folded into matched sets.")
        adds = self._nm_path_adds.get(getattr(self, '_current_path', None))
        if adds:
            studies = set(studies) | set(adds)
        return studies, meta
    LLMAnalysisBase.extract_studies_from_recalled_files = _extract_aug
    LLMAnalysisBase._nearmatch_patched = True
    print("✓ LLMAnalysisBase patched: accepted near-matches will be adopted into the "
          "core analysis (recall, precision, consistency, F1, and all downstream stats/figures).")
    print("  Validation target after analysis: ChatGPT pooled 41->43 (51.8%), "
          "Claude 37->38 (45.8%); Ai2/Consensus/Gemini unchanged.")
elif not ADOPT_NEARMATCHES:
    print("ADOPT_NEARMATCHES=False -> strict-60% analysis (near-matches NOT folded in).")

# Initialize and run the base analysis
print("=== Running Base Analysis (Common to Both Methods) ===")
base_analyzer = LLMAnalysisBase()
base_results = base_analyzer.run_base_analysis("~/Desktop/Pooled Std Recall Runs")

## Cell 03 — COMBINATORIAL

*Pairwise/sequential coverage*


In [ ]:
import shutil
import gc
try:
    if os.path.exists("traditional_analysis_results"):
        shutil.rmtree("traditional_analysis_results")
        print("🧹 Removed previous traditional_analysis_results directory")
    for var_name in ['traditional_results', 'traditional_analyzer', 'combinations', 'representatives', 'df_variations']:
        if var_name in globals():
            del globals()[var_name]
    gc.collect()
    print("🧹 Cleared previous analysis variables from memory")
except Exception as e:
    print(f"Note: Cleanup completed with minor issues: {e}")

# ===== PART 1B: TRADITIONAL COMBINATORIAL ANALYSIS WITH FILE OUTPUT =====
# ===== IMPORTS =====
import pickle
import json
from datetime import datetime
import os
import shutil
import gc
from itertools import combinations
from collections import defaultdict
import pandas as pd

class TraditionalCombinatorialAnalysis(LLMAnalysisBase):
    def __init__(self, base_results):
        # Initialize from base class
        super().__init__()
        # Copy over the loaded data from base analysis
        self.standardized_files = base_results['analyzer'].standardized_files
        self.recalled_files = base_results['analyzer'].recalled_files
        self.gold_standard_df = base_results['analyzer'].gold_standard_df
        self.gold_standard_count = base_results['analyzer'].gold_standard_count
        # Store base results
        self.base_results = base_results

    def select_top_representatives(self, recall_scores, df_variations):
        """Select top-performing variation per strategy based on recall"""
        print("Selecting top representatives per strategy based on recall...")
        representatives = {}
        
        # Group by LLM and strategy
        strategy_groups = defaultdict(list)
        for key, data in recall_scores.items():
            llm, strategy_var = key.split('-', 1)
            strategy = strategy_var[0]
            strategy_groups[(llm, strategy)].append((key, data))
        
        # Create summary table
        summary_data = []
        for (llm, strategy), variations in strategy_groups.items():
            # Find variation with highest recall
            best_variation = max(variations, key=lambda x: x[1]['recall'])
            strategy_name = self.strategy_mapping.get(strategy, f"Strategy-{strategy}")
            rep_key = f"{llm}-{strategy_name}"
            representatives[rep_key] = {
                'key': best_variation[0],
                'recall': best_variation[1]['recall'],
                'matched_count': best_variation[1]['matched_count'],
                'matches': best_variation[1]['matches']
            }
            
            # Add to summary
            variation_num = best_variation[0].split('-')[1][1]  # Extract variation number
            consistency = df_variations[df_variations['Full_Key'] == best_variation[0]]['Consistency'].iloc[0]
            summary_data.append({
                'Representative': rep_key,
                'Original_Key': best_variation[0],
                'Variation_Number': variation_num,
                'Recall': best_variation[1]['recall'],
                'Consistency': consistency,
                'Efficiency': best_variation[1]['recall'] * consistency,
                'Studies_Found': best_variation[1]['matched_count']
            })
        
        # Display selection summary
        summary_df = pd.DataFrame(summary_data)
        print("\n=== Selected Representatives Summary ===")
        print(summary_df.to_string(index=False))
        return representatives

    def generate_all_combinations(self, representatives, consistency_scores):
        """Generate all possible combinations using traditional combinatorial calculation"""
        print("=== USING TRADITIONAL COMBINATORIAL CALCULATION ===")
        
        if representatives is None or consistency_scores is None:
            raise ValueError("Representatives and consistency_scores must be provided for traditional method")
        
        rep_keys = list(representatives.keys())
        combination_results = []
        
        print(f"Generating combinations for {len(rep_keys)} representatives...")
        total_combinations = 2**len(rep_keys) - 1  # All possible non-empty combinations
        print(f"Total combinations to evaluate: {total_combinations}")
        
        # Generate all possible combinations
        for r in range(1, len(rep_keys) + 1):
            print(f"Processing combinations of size {r}...")
            for combo in combinations(rep_keys, r):
                # Calculate combined recall using recalled file data
                all_matches = set()
                total_consistency = 0
                valid_consistency_count = 0
                
                for rep_key in combo:
                    # Add matches from recalled files
                    all_matches.update(representatives[rep_key]['matches'])
                    # Get consistency score from standardized files analysis
                    original_key = representatives[rep_key]['key']
                    if original_key in consistency_scores:
                        total_consistency += consistency_scores[original_key]
                        valid_consistency_count += 1
                
                combined_recall = len(all_matches) / self.gold_standard_count
                avg_consistency = total_consistency / valid_consistency_count if valid_consistency_count > 0 else 0
                efficiency_score = combined_recall * avg_consistency
                
                combination_results.append({
                    'combination': combo,
                    'combined_recall': combined_recall,
                    'avg_consistency': avg_consistency,
                    'efficiency_score': efficiency_score,
                    'matched_count': len(all_matches),
                    'matches': all_matches
                })
        
        # Sort by efficiency score
        combination_results.sort(key=lambda x: x['efficiency_score'], reverse=True)
        print(f"✓ Generated and sorted {len(combination_results)} combinations")
        return combination_results

    def save_results_to_files(self, results, output_dir="traditional_analysis_results"):
        """Save all results to files for later use"""
        print(f"\n=== SAVING RESULTS TO FILES ===")
        
        # Create output directory if it doesn't exist
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"Created directory: {output_dir}")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # 1. Save the complete results as pickle (preserves all Python objects)
        pickle_file = os.path.join(output_dir, f"traditional_results_complete_{timestamp}.pkl")
        with open(pickle_file, 'wb') as f:
            pickle.dump(results, f)
        print(f"✓ Saved complete results (pickle): {pickle_file}")
        
        # 2. Save combinations data as CSV for easy viewing
        combinations_data = []
        for i, combo in enumerate(results['combinations'][:100]):  # Top 100 combinations
            combinations_data.append({
                'Rank': i + 1,
                'Combination': ' + '.join(combo['combination']),
                'Size': len(combo['combination']),
                'Recall': combo['combined_recall'],
                'Consistency': combo['avg_consistency'],
                'Efficiency': combo['efficiency_score'],
                'Studies_Matched': combo['matched_count']
            })
        
        combinations_df = pd.DataFrame(combinations_data)
        csv_file = os.path.join(output_dir, f"top_100_combinations_{timestamp}.csv")
        combinations_df.to_csv(csv_file, index=False)
        print(f"✓ Saved top 100 combinations (CSV): {csv_file}")
        
        # 3. Save representatives data as CSV
        representatives_data = []
        for rep_name, rep_data in results['representatives'].items():
            representatives_data.append({
                'Representative': rep_name,
                'Original_Key': rep_data['key'],
                'Recall': rep_data['recall'],
                'Studies_Matched': rep_data['matched_count']
            })
        
        representatives_df = pd.DataFrame(representatives_data)
        rep_csv_file = os.path.join(output_dir, f"representatives_{timestamp}.csv")
        representatives_df.to_csv(rep_csv_file, index=False)
        print(f"✓ Saved representatives (CSV): {rep_csv_file}")
        
        # 4. Save df_variations as CSV
        variations_csv_file = os.path.join(output_dir, f"all_variations_{timestamp}.csv")
        results['df_variations'].to_csv(variations_csv_file, index=False)
        print(f"✓ Saved all variations data (CSV): {variations_csv_file}")
        
        # 5. Save summary statistics as JSON
        summary_stats = {
            'analysis_type': 'traditional_combinatorial',
            'timestamp': timestamp,
            'total_representatives': len(results['representatives']),
            'total_combinations_evaluated': len(results['combinations']),
            'gold_standard_count': self.gold_standard_count,
            'top_10_combinations': [
                {
                    'rank': i + 1,
                    'combination': list(combo['combination']),
                    'recall': combo['combined_recall'],
                    'consistency': combo['avg_consistency'],
                    'efficiency': combo['efficiency_score'],
                    'studies_matched': combo['matched_count']
                }
                for i, combo in enumerate(results['combinations'][:10])
            ],
            'best_individual_representatives': [
                {
                    'name': rep_name,
                    'recall': rep_data['recall'],
                    'studies_matched': rep_data['matched_count']
                }
                for rep_name, rep_data in sorted(results['representatives'].items(),
                                               key=lambda x: x[1]['recall'], reverse=True)[:5]
            ]
        }
        
        json_file = os.path.join(output_dir, f"analysis_summary_{timestamp}.json")
        with open(json_file, 'w') as f:
            json.dump(summary_stats, f, indent=2, default=str)
        print(f"✓ Saved analysis summary (JSON): {json_file}")
        
        # 6. Create a loader script for easy reuse
        loader_script = f'''#!/usr/bin/env python3
"""
Auto-generated loader script for Traditional Combinatorial Analysis Results
Generated on: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
import pickle
import pandas as pd
import json
from pathlib import Path

def load_traditional_results(results_dir="{output_dir}"):
    """
    Load all traditional analysis results from files
    Returns:
        dict: Complete results dictionary with all analysis data
    """
    results_dir = Path(results_dir)
    # Find the most recent pickle file
    pickle_files = list(results_dir.glob("traditional_results_complete_*.pkl"))
    if not pickle_files:
        raise FileNotFoundError("No traditional results pickle file found!")
    latest_pickle = max(pickle_files, key=lambda p: p.stat().st_mtime)
    print(f"Loading traditional analysis results from: {{latest_pickle}}")
    with open(latest_pickle, 'rb') as f:
        results = pickle.load(f)
    print(f"✓ Loaded {{len(results['combinations'])}} combinations")
    print(f"✓ Loaded {{len(results['representatives'])}} representatives")
    print(f"✓ Loaded variations data with {{len(results['df_variations'])}} entries")
    return results

def load_combinations_csv(results_dir="{output_dir}"):
    """Load top combinations as DataFrame"""
    results_dir = Path(results_dir)
    csv_files = list(results_dir.glob("top_100_combinations_*.csv"))
    if not csv_files:
        raise FileNotFoundError("No combinations CSV file found!")
    latest_csv = max(csv_files, key=lambda p: p.stat().st_mtime)
    return pd.read_csv(latest_csv)

def load_representatives_csv(results_dir="{output_dir}"):
    """Load representatives as DataFrame"""
    results_dir = Path(results_dir)
    csv_files = list(results_dir.glob("representatives_*.csv"))
    if not csv_files:
        raise FileNotFoundError("No representatives CSV file found!")
    latest_csv = max(csv_files, key=lambda p: p.stat().st_mtime)
    return pd.read_csv(latest_csv)

def load_summary_json(results_dir="{output_dir}"):
    """Load analysis summary"""
    results_dir = Path(results_dir)
    json_files = list(results_dir.glob("analysis_summary_*.json"))
    if not json_files:
        raise FileNotFoundError("No summary JSON file found!")
    latest_json = max(json_files, key=lambda p: p.stat().st_mtime)
    with open(latest_json, 'r') as f:
        return json.load(f)

# Example usage:
if __name__ == "__main__":
    # Load complete results
    traditional_results = load_traditional_results()
    # Access components
    combinations = traditional_results['combinations']
    representatives = traditional_results['representatives']
    df_variations = traditional_results['df_variations']
    print(f"Top combination: {{' + '.join(combinations[0]['combination'])}}")
    print(f"Efficiency: {{combinations[0]['efficiency_score']:.3f}}")
    # Or load specific components as DataFrames
    combinations_df = load_combinations_csv()
    representatives_df = load_representatives_csv()
    summary = load_summary_json()
    print("\\nFiles loaded successfully!")
'''
        
        loader_file = os.path.join(output_dir, "load_traditional_results.py")
        with open(loader_file, 'w') as f:
            f.write(loader_script)
        print(f"✓ Created loader script: {loader_file}")
        
        # 7. Save metadata about file structure
        metadata = {
            'files_created': {
                'complete_results_pickle': f"traditional_results_complete_{timestamp}.pkl",
                'top_combinations_csv': f"top_100_combinations_{timestamp}.csv",
                'representatives_csv': f"representatives_{timestamp}.csv",
                'variations_csv': f"all_variations_{timestamp}.csv",
                'summary_json': f"analysis_summary_{timestamp}.json",
                'loader_script': "load_traditional_results.py"
            },
            'description': {
                'complete_results_pickle': 'Complete Python objects - use for full analysis continuation',
                'top_combinations_csv': 'Top 100 combinations in readable format',
                'representatives_csv': 'Selected representatives data',
                'variations_csv': 'All LLM-strategy-variation performance data',
                'summary_json': 'Analysis summary and top results',
                'loader_script': 'Python script to easily reload all data'
            },
            'usage_instructions': [
                'To reload complete results: exec(open("load_traditional_results.py").read()); results = load_traditional_results()',
                'To view top combinations: pd.read_csv("top_100_combinations_*.csv")',
                'To continue with visualizations: run the loader script and use the returned results'
            ]
        }
        
        metadata_file = os.path.join(output_dir, "README_file_structure.json")
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
        print(f"✓ Saved file structure metadata: {metadata_file}")
        
        print(f"\n{'='*60}")
        print(f"  ALL RESULTS SAVED TO: {output_dir}")
        print(f"{'='*60}")
        print(f"To reload these results in a new session:")
        print(f"1. Run: exec(open('{loader_file}').read())")
        print(f"2. Load: traditional_results = load_traditional_results()")
        print(f"3. Use: combinations = traditional_results['combinations']")
        print(f"\nOr import the loader functions and use them directly.")
        
        return output_dir

    def run_traditional_analysis(self):
        """Run traditional combinatorial analysis"""
        print("\n=== TRADITIONAL COMBINATORIAL ANALYSIS ===")
        
        # Step 4: Visualize ALL variations before selection (uses variation-level consistency)
        print("\n=== STEP 4: Comprehensive Visualization of All Variations ===")
        df_variations = self.visualize_all_variations(
            self.base_results['variation_consistency_scores'],
            self.base_results['recall_scores'],
            self.base_results['variation_metadata']
        )
        
        # Step 5: Select top representatives based on recall
        print("\n=== STEP 5: Representative Selection (Based on Recall) ===")
        representatives = self.select_top_representatives(
            self.base_results['recall_scores'],
            df_variations
        )
        
        # Step 6: Generate combinations using traditional method
        print("\n=== STEP 6: Traditional Combination Analysis ===")
        combinations = self.generate_all_combinations(
            representatives,
            self.base_results['variation_consistency_scores']
        )
        
        # Display top 10 combinations
        print("\n=== Top 10 Combinations by Efficiency Score ===")
        for i, combo in enumerate(combinations[:10]):
            print(f"{i+1}. {' + '.join(combo['combination'])}")
            print(f"   Recall: {combo['combined_recall']:.3f}, "
                  f"Consistency: {combo['avg_consistency']:.3f}, "
                  f"Efficiency: {combo['efficiency_score']:.3f}")
        
        # Package results
        results = {
            'combinations': combinations,
            'representatives': representatives,
            'df_variations': df_variations,
            'method': 'traditional',
            'base_results': self.base_results,  # Include base results for complete context
            'analysis_metadata': {
                'total_combinations_evaluated': len(combinations),
                'total_representatives': len(representatives),
                'gold_standard_count': self.gold_standard_count,
                'top_efficiency_score': combinations[0]['efficiency_score'] if combinations else 0,
                'analysis_timestamp': datetime.now().isoformat()
            }
        }
        
        # Save results to files
        output_dir = self.save_results_to_files(results)
        results['output_directory'] = output_dir
        
        return results

    def visualize_all_variations(self, variation_consistency_scores, recall_scores, variation_metadata):
        """Create comprehensive visualizations for all variations before selection"""
        print("Creating visualizations for all variations...")
        
        # Prepare data for visualization
        variation_data = []
        
        # Collect all variation data using VARIATION-LEVEL consistency
        for var_key in recall_scores.keys():
            llm, strategy_var = var_key.split('-', 1)
            strategy = strategy_var[0]
            variation = strategy_var[1]
            
            if strategy == '4':  # Chain-of-Thought strategy
                variation_display = str(int(variation) - 1) if int(variation) > 0 else '0'
            else:
                variation_display = variation
            
            # Use variation-level consistency score
            consistency = variation_consistency_scores.get(var_key, 0.65)  # Use variation-specific consistency
            recall = recall_scores[var_key]['recall']
            efficiency = recall * consistency
            
            variation_data.append({
                'LLM': llm,
                'Strategy': self.strategy_mapping.get(strategy, f"Strategy-{strategy}"),
                'Variation': variation_display,
                'Full_Key': var_key,
                'Consistency': consistency,
                'Recall': recall,
                'Efficiency': efficiency,
                'Matched_Count': recall_scores[var_key]['matched_count']
            })
        
        df_variations = pd.DataFrame(variation_data)
        
        print(f"Debug: LLMs in variation data: {df_variations['LLM'].unique()}")
        print(f"Debug: Total variations: {len(df_variations)}")
        
        return df_variations

# Run traditional analysis if selected
if base_results:
    print("\n" + "="*50)
    print("TRADITIONAL COMBINATORIAL ANALYSIS")
    print("="*50)
    traditional_analyzer = TraditionalCombinatorialAnalysis(base_results)
    traditional_results = traditional_analyzer.run_traditional_analysis()
    print("✓ Traditional analysis completed!")
    print(f"✓ Results saved to: {traditional_results.get('output_directory', 'traditional_analysis_results')}")
else:
    print("❌ Base analysis failed - cannot proceed with traditional analysis")

## Cell 04 — DATA PREP

*Precision (matched/total_retrieved) + variation prep*


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# ===== PRECISION CALCULATION MODULE =====
def calculate_precision_from_standardized_files(analyzer, recall_scores):
    """
    Calculate precision properly using:
    - Matched studies from recalled files (True Positives)
    - Total retrieved from standardized files (True Positives + False Positives)
    Precision = Matched / Total Retrieved
    """
    print("\n=== Calculating Precision from Standardized Files ===")
    precision_data = {}
    
    for var_key, recall_info in recall_scores.items():
        # Parse variation key (e.g., "ChatGPT-11" or "Ai2 Finder-23")
        llm, strategy_var = var_key.split('-', 1)
        strategy = strategy_var[0]
        variation = strategy_var[1]
        
        # Get matched count (True Positives) from recalled files
        matched_count = recall_info['matched_count']
        
        # Find corresponding standardized files for this variation
        standardized_key_pattern = f"{llm}-{strategy}{variation}"
        
        # Collect all unique studies retrieved across all runs
        retrieved_studies = set()
        
        for std_filename, std_info in analyzer.standardized_files.items():
            std_key = f"{std_info['llm']}-{std_info['strategy']}{std_info['variation']}"
            
            if std_key == standardized_key_pattern:
                # Load standardized file and extract titles
                df = analyzer.load_csv_file_content(std_info['path'])
                titles = analyzer.extract_titles_from_standardized_files(df)
                retrieved_studies.update(titles)
        
        total_retrieved = len(retrieved_studies)
        
        # Calculate precision
        precision = min(matched_count / total_retrieved, 1.0) if total_retrieved > 0 else 0.0
        
        # Store results
        precision_data[var_key] = {
            'matched_count': matched_count,
            'total_retrieved': total_retrieved,
            'precision': precision
        }
        
        print(f"  {var_key}: Retrieved={total_retrieved}, Matched={matched_count}, Precision={precision:.3f}")
    
    return precision_data

# ===== HELPER FUNCTION: CALCULATE ADAPTIVE Y-AXIS LIMITS =====
def calculate_adaptive_ylim(data_series, buffer_percent=0.1):
    """
    Calculate y-axis limits based on actual data range with buffer
    
    Args:
        data_series: pandas Series with metric values
        buffer_percent: percentage of range to add as buffer (default 10%)
    
    Returns:
        tuple: (y_min, y_max) for axis limits
    """
    min_val = data_series.min()
    max_val = data_series.max()
    data_range = max_val - min_val
    
    # Add buffer
    buffer = data_range * buffer_percent
    
    # Set limits with buffer, but constrain to [0, 1] since these are normalized metrics
    y_min = max(0, min_val - buffer)
    y_max = min(1, max_val + buffer)
    
    # Ensure minimum range of 0.1 for visibility
    if y_max - y_min < 0.1:
        midpoint = (y_min + y_max) / 2
        y_min = max(0, midpoint - 0.05)
        y_max = min(1, midpoint + 0.05)
    
    return (y_min, y_max)

# ===== VISUALIZATION: COMPREHENSIVE VARIATION ANALYSIS =====
def visualize_all_variations(analyzer, variation_consistency_scores, recall_scores, variation_metadata,
                              make_plot=False):
    """Create comprehensive visualizations for all variations before selection"""
    print("Creating visualizations for all variations...")
    
    # STEP 1: Calculate precision for all variations using the precision module
    print("\n=== Step 1: Calculating Precision ===")
    precision_data = calculate_precision_from_standardized_files(analyzer, recall_scores)
    
    # STEP 2: Prepare data for visualization
    print("\n=== Step 2: Preparing Visualization Data ===")
    variation_data = []
    
    # Collect all variation data WITHOUT modifying variation numbers
    for var_key in recall_scores.keys():
        llm, strategy_var = var_key.split('-', 1)
        strategy = strategy_var[0]
        variation = strategy_var[1]
        
        # ✓ KEEP ORIGINAL VARIATION NUMBER - DO NOT RENUMBER
        variation_display = variation
        
        # Use variation-level consistency score
        consistency = variation_consistency_scores.get(var_key, 0.65)
        recall = recall_scores[var_key]['recall']
        matched_count = recall_scores[var_key]['matched_count']
        
        # Get precision from calculated precision_data
        precision_info = precision_data.get(var_key, {})
        precision = precision_info.get('precision', 0.0)
        total_retrieved = precision_info.get('total_retrieved', 0)
        
        variation_data.append({
            'LLM': llm,
            'Strategy': analyzer.strategy_mapping.get(strategy, f"Strategy-{strategy}"),
            'Strategy_Code': strategy,
            'Variation': variation_display,
            'Full_Key': var_key,
            'Consistency': consistency,
            'Recall': recall,
            'Precision': precision,
            'Matched_Count': matched_count,
            'Total_Retrieved': total_retrieved
        })
    
    df_variations = pd.DataFrame(variation_data)
    
    # STEP 3: Display statistics and check for zeros
    print(f"\n=== Step 3: Data Summary and Diagnostics ===")
    print(f"LLMs in variation data: {df_variations['LLM'].unique()}")
    print(f"Total variations: {len(df_variations)}")
    print(f"\nMetric Ranges:")
    print(f"  Recall:      {df_variations['Recall'].min():.3f} to {df_variations['Recall'].max():.3f}")
    print(f"  Precision:   {df_variations['Precision'].min():.3f} to {df_variations['Precision'].max():.3f}")
    print(f"  Consistency: {df_variations['Consistency'].min():.3f} to {df_variations['Consistency'].max():.3f}")
    
    # Calculate adaptive y-limits for each metric
    recall_ylim = calculate_adaptive_ylim(df_variations['Recall'])
    precision_ylim = calculate_adaptive_ylim(df_variations['Precision'])
    consistency_ylim = calculate_adaptive_ylim(df_variations['Consistency'])
    
    print(f"\nAdaptive Y-axis Limits:")
    print(f"  Recall:      [{recall_ylim[0]:.3f}, {recall_ylim[1]:.3f}]")
    print(f"  Precision:   [{precision_ylim[0]:.3f}, {precision_ylim[1]:.3f}]")
    print(f"  Consistency: [{consistency_ylim[0]:.3f}, {consistency_ylim[1]:.3f}]")
    
    # Check for zero values
    print("\n=== Zero Value Diagnostic ===")
    zero_recalls = df_variations[df_variations['Recall'] == 0]
    if len(zero_recalls) > 0:
        print(f"⚠ Found {len(zero_recalls)} variations with Recall = 0:")
        print(zero_recalls[['LLM', 'Strategy', 'Variation', 'Full_Key', 'Recall', 'Matched_Count']])
    else:
        print("✓ No zero Recall values found")
    
    zero_precisions = df_variations[df_variations['Precision'] == 0]
    if len(zero_precisions) > 0:
        print(f"⚠ Found {len(zero_precisions)} variations with Precision = 0:")
        print(zero_precisions[['LLM', 'Strategy', 'Variation', 'Full_Key', 'Precision', 'Total_Retrieved']])
    else:
        print("✓ No zero Precision values found")
    
    # Verify precision differs from recall
    same_values = (df_variations['Precision'] == df_variations['Recall']).sum()
    if same_values == 0:
        print(f"✓ Precision differs from Recall (calculation correct)")
    else:
        print(f"⚠ Warning: Precision equals Recall for {same_values} variations")
    
    print("\n=== Chain-of-Thought Strategy Diagnostic ===")
    cot_data = df_variations[df_variations['Strategy'] == 'Chain-of-Thought']
    if len(cot_data) > 0:
        print(f"Total Chain-of-Thought variations: {len(cot_data)}")
        print("\nChain-of-Thought values:")
        print(cot_data[['LLM', 'Variation', 'Full_Key', 'Recall', 'Precision', 'Consistency']].sort_values(['LLM', 'Variation']).to_string())
    else:
        print("⚠ No Chain-of-Thought variations found!")
    
    # STEP 4: Create visualizations
    print("\n=== Step 4: Creating Visualizations ===")
    
    if make_plot:
        fig, axes = plt.subplots(3, 3, figsize=(20, 18))
        plt.style.use('default')
    
        # ===============================================================
        # ROW 1: HEATMAPS - Recall, Precision, Consistency
        # ===============================================================
    
        # A. Recall Scores by LLM and Strategy
        print("  Creating Recall heatmap...")
        pivot_recall = df_variations.pivot_table(
            values='Recall',
            index=['LLM', 'Strategy'],
            columns='Variation',
            aggfunc='first'  # Use first value if duplicates exist
        )
    
        # Replace any remaining NaN with 0 only for display
        pivot_recall_display = pivot_recall.fillna(0)
    
        # ORIGINAL HEATMAP STYLE - NO gridlines, NO vmin/vmax constraints
        sns.heatmap(pivot_recall_display, annot=True, fmt='.3f', cmap='Greens',
                    ax=axes[0,0], cbar_kws={'label': 'Recall Score'})
        axes[0,0].set_title('A. Recall Scores Across All Variations', fontsize=18, fontweight='bold')
        axes[0,0].set_xlabel('Variation Number')
        axes[0,0].set_ylabel('LLM - Strategy')
    
        # B. Precision Scores by LLM and Strategy
        print("  Creating Precision heatmap...")
        pivot_precision = df_variations.pivot_table(
            values='Precision',
            index=['LLM', 'Strategy'],
            columns='Variation',
            aggfunc='first'
        )
    
        pivot_precision_display = pivot_precision.fillna(0)
    
        # ORIGINAL HEATMAP STYLE
        sns.heatmap(pivot_precision_display, annot=True, fmt='.3f', cmap='Oranges',
                    ax=axes[0,1], cbar_kws={'label': 'Precision Score'})
        axes[0,1].set_title('B. Precision Scores Across All Variations', fontsize=18, fontweight='bold')
        axes[0,1].set_xlabel('Variation Number')
        axes[0,1].set_ylabel('LLM - Strategy')
    
        # C. Consistency Scores by LLM and Strategy
        print("  Creating Consistency heatmap...")
        pivot_consistency = df_variations.pivot_table(
            values='Consistency',
            index=['LLM', 'Strategy'],
            columns='Variation',
            aggfunc='first'
        )
    
        pivot_consistency_display = pivot_consistency.fillna(0)
    
        # ORIGINAL HEATMAP STYLE
        sns.heatmap(pivot_consistency_display, annot=True, fmt='.3f', cmap='Blues',
                    ax=axes[0,2], cbar_kws={'label': 'Consistency Score'})
        axes[0,2].set_title('C. Consistency Scores Across All Variations', fontsize=18, fontweight='bold')
        axes[0,2].set_xlabel('Variation Number')
        axes[0,2].set_ylabel('LLM - Strategy')
    
        # ===============================================================
        # ROW 2: BOX PLOTS BY LLM - Recall, Precision, Consistency
        # ===============================================================
    
        # D. Box plots for Recall by LLM (ADAPTIVE y-axis)
        print("  Creating Recall by LLM boxplot...")
        sns.boxplot(data=df_variations, x='LLM', y='Recall', ax=axes[1,0], palette='Greens')
        axes[1,0].set_title('D. Recall Distribution by LLM', fontsize=18, fontweight='bold')
        axes[1,0].tick_params(axis='x', rotation=45)
        axes[1,0].set_xlabel('LLM', fontsize=16, fontweight='bold')
        axes[1,0].set_ylabel('Recall Score', fontsize=16, fontweight='bold')
        axes[1,0].set_ylim(recall_ylim)  # ✓ ADAPTIVE: Based on actual data range
        axes[1,0].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
        # E. Box plots for Precision by LLM (ADAPTIVE y-axis)
        print("  Creating Precision by LLM boxplot...")
        sns.boxplot(data=df_variations, x='LLM', y='Precision', ax=axes[1,1], palette='Oranges')
        axes[1,1].set_title('E. Precision Distribution by LLM', fontsize=18, fontweight='bold')
        axes[1,1].tick_params(axis='x', rotation=45)
        axes[1,1].set_xlabel('LLM', fontsize=16, fontweight='bold')
        axes[1,1].set_ylabel('Precision Score', fontsize=16, fontweight='bold')
        axes[1,1].set_ylim(precision_ylim)  # ✓ ADAPTIVE: Based on actual data range
        axes[1,1].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
        # F. Box plots for Consistency by LLM (ADAPTIVE y-axis)
        print("  Creating Consistency by LLM boxplot...")
        sns.boxplot(data=df_variations, x='LLM', y='Consistency', ax=axes[1,2], palette='Blues')
        axes[1,2].set_title('F. Consistency Distribution by LLM', fontsize=18, fontweight='bold')
        axes[1,2].tick_params(axis='x', rotation=45)
        axes[1,2].set_xlabel('LLM', fontsize=16, fontweight='bold')
        axes[1,2].set_ylabel('Consistency Score', fontsize=16, fontweight='bold')
        axes[1,2].set_ylim(consistency_ylim)  # ✓ ADAPTIVE: Based on actual data range
        axes[1,2].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
        # ===============================================================
        # ROW 3: BOX PLOTS BY STRATEGY - Recall, Precision, Consistency
        # ===============================================================
    
        # G. Box plots for Recall by Strategy (ADAPTIVE y-axis)
        print("  Creating Recall by Strategy boxplot...")
        sns.boxplot(data=df_variations, x='Strategy', y='Recall', ax=axes[2,0], palette='Greens')
        axes[2,0].set_title('G. Recall Distribution by Strategy', fontsize=18, fontweight='bold')
        axes[2,0].tick_params(axis='x', rotation=45)
        axes[2,0].set_xlabel('Strategy', fontsize=16, fontweight='bold')
        axes[2,0].set_ylabel('Recall Score', fontsize=16, fontweight='bold')
        axes[2,0].set_ylim(recall_ylim)  # ✓ ADAPTIVE: Same as row 2 for consistency
        axes[2,0].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
        # H. Box plots for Precision by Strategy (ADAPTIVE y-axis)
        print("  Creating Precision by Strategy boxplot...")
        sns.boxplot(data=df_variations, x='Strategy', y='Precision', ax=axes[2,1], palette='Oranges')
        axes[2,1].set_title('H. Precision Distribution by Strategy', fontsize=18, fontweight='bold')
        axes[2,1].tick_params(axis='x', rotation=45)
        axes[2,1].set_xlabel('Strategy', fontsize=16, fontweight='bold')
        axes[2,1].set_ylabel('Precision Score', fontsize=16, fontweight='bold')
        axes[2,1].set_ylim(precision_ylim)  # ✓ ADAPTIVE: Same as row 2 for consistency
        axes[2,1].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
        # I. Box plots for Consistency by Strategy (ADAPTIVE y-axis)
        print("  Creating Consistency by Strategy boxplot...")
        sns.boxplot(data=df_variations, x='Strategy', y='Consistency', ax=axes[2,2], palette='Blues')
        axes[2,2].set_title('I. Consistency Distribution by Strategy', fontsize=18, fontweight='bold')
        axes[2,2].tick_params(axis='x', rotation=45)
        axes[2,2].set_xlabel('Strategy', fontsize=16, fontweight='bold')
        axes[2,2].set_ylabel('Consistency Score', fontsize=16, fontweight='bold')
        axes[2,2].set_ylim(consistency_ylim)  # ✓ ADAPTIVE: Same as row 2 for consistency
        axes[2,2].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
        plt.tight_layout()
    
        # Save figure
        output_filename = 'Overall_Performance_With_Precision.png'
        plt.savefig(output_filename, dpi=300, bbox_inches='tight',
                    facecolor='white', edgecolor='none', format='png',
                    transparent=False, pad_inches=0.2)
        print(f"\n✓ Visualization saved to: {output_filename}")
        plt.show()
    else:
        plt.close('all')
    
    # STEP 5: Save results
    output_file = 'df_variations_with_precision.csv'
    df_variations.to_csv(output_file, index=False)
    print(f"✓ Results saved to: {output_file}")
    
    # STEP 6: Display summaries
    print("\n=== Precision Summary by LLM ===")
    summary_llm = df_variations.groupby('LLM')['Precision'].agg(['count', 'mean', 'std', 'min', 'max']).round(3)
    print(summary_llm)
    
    print("\n=== Precision Summary by Strategy ===")
    summary_strategy = df_variations.groupby('Strategy')['Precision'].agg(['count', 'mean', 'std', 'min', 'max']).round(3)
    print(summary_strategy)
    
    print("\n=== Recall Summary by LLM ===")
    recall_llm = df_variations.groupby('LLM')['Recall'].agg(['count', 'mean', 'std', 'min', 'max']).round(3)
    print(recall_llm)
    
    print("\n=== Recall Summary by Strategy ===")
    recall_strategy = df_variations.groupby('Strategy')['Recall'].agg(['count', 'mean', 'std', 'min', 'max']).round(3)
    print(recall_strategy)
    
    return df_variations

# ===== EXECUTION =====
print("\n" + "="*70)
print("COMPREHENSIVE VARIATION ANALYSIS WITH PRECISION CALCULATION")
print("="*70)

# Run the visualization
if 'base_results' in globals() and base_results:
    print("✓ Using Base Analysis Results for Visualization")
    df_variations = visualize_all_variations(
        base_results['analyzer'],
        base_results['variation_consistency_scores'],
        base_results['recall_scores'],
        base_results['variation_metadata'],
        make_plot=False
    )
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE!")
    print("="*70)
    print("✓ df_variations is ready with Recall, Precision, and Consistency")
    print("✓ You can now use df_variations for further analyses")
    
else:
    print("❌ No analysis results available. Please run Part 1A (Base Analysis) first.")
    df_variations = None

## Cell 05 — PRECISION GLOBAL

*precision_results global*


In [ ]:
# ============================================================================
#  cell 5's precision computation for df_variations — that part of the removal
#  was correct (cell 5 already computes and caps precision for df_variations).
#  However, cell 8 (scatter plots) depends on a SEPARATELY-NAMED global,
#  `precision_results = {'precision_df': ..., 'precision_by_llm': ..., 
#  here in trimmed form: computes precision_results only — no CSV resave, no
#  df_variations rebuild (cell 5 already owns and caps that). Precision capped
#  at 1.0 for consistency with cells 5/24 (A-5).
# ============================================================================
import pandas as pd

def calculate_precision_from_standardized_files(analyzer, recall_scores):
    """
    Calculate precision using:
    - Matched studies from recalled files (True Positives)
    - Total retrieved from standardized files (True Positives + False Positives)
    Precision = min(Matched / Total Retrieved, 1.0)
    """
    print("\n=== Calculating Precision from Standardized Files ===")
    precision_data = []

    for var_key, recall_info in recall_scores.items():
        llm, strategy_var = var_key.split('-', 1)
        strategy, variation = strategy_var[0], strategy_var[1]
        matched_count = recall_info['matched_count']
        standardized_key_pattern = f"{llm}-{strategy}{variation}"

        retrieved_studies = set()
        runs_found = []
        for std_filename, std_info in analyzer.standardized_files.items():
            std_key = f"{std_info['llm']}-{std_info['strategy']}{std_info['variation']}"
            if std_key == standardized_key_pattern:
                runs_found.append(std_info['run'])
                df = analyzer.load_csv_file_content(std_info['path'])
                titles = analyzer.extract_titles_from_standardized_files(df)
                retrieved_studies.update(titles)

        total_retrieved = len(retrieved_studies)
        if total_retrieved > 0:
            precision = min(matched_count / total_retrieved, 1.0)
        else:
            precision = 0.0
            print(f"  Warning: {var_key} has 0 retrieved studies!")

        precision_data.append({
            'Variation_Key': var_key, 'LLM': llm, 'Strategy': strategy, 'Variation': variation,
            'Matched_Count': matched_count, 'Total_Retrieved': total_retrieved,
            'Precision': precision, 'Runs_Found': len(runs_found)
        })

    precision_df = pd.DataFrame(precision_data)
    print(f"\n=== Precision Calculation Summary ===")
    print(f"Total variations analyzed: {len(precision_df)}")
    print(f"Precision range: {precision_df['Precision'].min():.3f} to {precision_df['Precision'].max():.3f}")
    print(f"Mean precision: {precision_df['Precision'].mean():.3f}")

    incomplete = precision_df[precision_df['Runs_Found'] < 3]
    if len(incomplete) > 0:
        print(f"\n⚠ Warning: {len(incomplete)} variations have incomplete runs:")
        for _, row in incomplete.iterrows():
            print(f"  {row['Variation_Key']}: only {row['Runs_Found']} runs found")

    return {
        'precision_df': precision_df,
        'precision_by_llm': precision_df.groupby('LLM')['Precision'].agg(['mean', 'std', 'min', 'max']),
        'precision_by_strategy': precision_df.groupby('Strategy')['Precision'].agg(['mean', 'std', 'min', 'max'])
    }

# ===== EXECUTION =====
if 'base_results' in globals() and base_results:
    precision_results = calculate_precision_from_standardized_files(
        base_results['analyzer'], base_results['recall_scores'])
    print(f"\n✓ precision_results ready: {len(precision_results['precision_df'])} precision calculations")
else:
    print("❌ Error: base_results not found. Please run base analysis first.")
    precision_results = None

## Cell 06 — CIRCULAR BAR PLOTS

*Figure 2 (from df_variations)*


In [ ]:
# ===== VISUALIZATION CELL 3: CIRCULAR BAR PLOTS WITH PRECISION VERIFICATION =====
def create_circular_bar_plots(df_variations):
    """Create enhanced academic circular bar plots with faded vertical lines from bar tops"""
    print("\n" + "="*70)
    print("CREATING CIRCULAR BAR PLOTS")
    print("="*70)
    
    # STEP 1: Verify precision column exists and is correct
    print("\n=== Step 1: Verifying Precision Data ===")
    if 'Precision' not in df_variations.columns:
        print("❌ ERROR: Precision column not found in df_variations!")
        print("Available columns:", df_variations.columns.tolist())
        print("Please run the visualization cell 1 first to calculate precision.")
        return
    
    # Check if precision differs from recall (it should!)
    same_values = (df_variations['Precision'] == df_variations['Recall']).sum()
    if same_values == len(df_variations):
        print("❌ ERROR: Precision equals Recall for all rows!")
        print("Precision calculation may be incorrect. Please recalculate.")
        return
    elif same_values > 0:
        print(f"⚠ Warning: Precision equals Recall for {same_values} out of {len(df_variations)} rows")
    else:
        print(f"✓ Precision differs from Recall (calculation verified)")
    
    # Display metric ranges
    print(f"\nMetric Ranges:")
    print(f"  Recall:      {df_variations['Recall'].min():.3f} to {df_variations['Recall'].max():.3f}")
    print(f"  Precision:   {df_variations['Precision'].min():.3f} to {df_variations['Precision'].max():.3f}")
    print(f"  Consistency: {df_variations['Consistency'].min():.3f} to {df_variations['Consistency'].max():.3f}")
    
    # Check for missing values
    missing_precision = df_variations['Precision'].isna().sum()
    if missing_precision > 0:
        print(f"⚠ Warning: {missing_precision} rows have missing precision values")
        print("Filling missing values with 0...")
        df_variations['Precision'] = df_variations['Precision'].fillna(0)
    
    print("✓ Precision data verified and ready for plotting")
    
    # STEP 2: Prepare data
    print("\n=== Step 2: Preparing Plot Data ===")
    
    # Apply standard abbreviations
    df_variations = df_variations.copy()
    df_variations['Strategy'] = df_variations['Strategy'].replace({
        'Chain-of-Thought': 'CoT',
        'ChatGPT': 'ChatGPT',
        'PICO Framework': 'PICO',
        'Persona-based': 'Persona'
    })
    
    # Define LLM categories
    specialized_llms = ['Consensus', 'Ai2 Finder']
    general_llms = ['Claude', 'ChatGPT', 'Gemini']
    
    # Organize LLMs by category
    unique_llms = sorted(df_variations['LLM'].unique())
    ordered_llms = []
    for llm in unique_llms:
        if llm in specialized_llms:
            ordered_llms.append(llm)
    for llm in unique_llms:
        if llm in general_llms:
            ordered_llms.append(llm)
    
    n_llms = len(ordered_llms)
    specialized_count = len([llm for llm in ordered_llms if llm in specialized_llms])
    
    print(f"LLMs to plot: {ordered_llms}")
    print(f"Specialized: {specialized_count}, General-purpose: {n_llms - specialized_count}")
    
    # STEP 3: Create figure
    print("\n=== Step 3: Creating Figure ===")
    
    # Create figure with enhanced styling
    fig, axes = plt.subplots(3, 5, figsize=(22, 14), subplot_kw=dict(projection='polar'))
    fig.patch.set_facecolor('white')
    
    # Hide unused subplots
    if n_llms < 5:
        for row in range(3):
            for col in range(n_llms, 5):
                axes[row, col].set_visible(False)
    
    # Enhanced color scheme
    strategies = sorted(df_variations['Strategy'].unique())
    colors = plt.cm.Set3(np.linspace(0, 1, len(strategies)))
    strategy_color_map = dict(zip(strategies, colors))
    
    # Calculate category-specific scales
    specialized_data = df_variations[df_variations['LLM'].isin(specialized_llms)]
    general_data = df_variations[df_variations['LLM'].isin(general_llms)]
    
    scales = {}
    for metric in ['Recall', 'Precision', 'Consistency']:
        specialized_max = specialized_data[metric].max() if not specialized_data.empty else 0
        general_max = general_data[metric].max() if not general_data.empty else 0
        scales[f'{metric}_specialized'] = specialized_max * 1.15
        scales[f'{metric}_general'] = general_max * 1.15
        print(f"{metric} scales - Specialized: {scales[f'{metric}_specialized']:.3f}, "
              f"General: {scales[f'{metric}_general']:.3f}")
    
    def get_enhanced_rticks(max_scale):
        """Generate enhanced radial ticks for better readability"""
        if max_scale <= 0.1:
            return [0.02, 0.04, 0.06, 0.08, 0.1]
        elif max_scale <= 0.3:
            return [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
        elif max_scale <= 0.5:
            return [0.1, 0.2, 0.3, 0.4, 0.5]
        elif max_scale <= 0.8:
            return [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
        elif max_scale <= 1.0:
            return [0.2, 0.4, 0.6, 0.8, 1.0]
        else:
            step = max_scale / 5
            return [step * i for i in range(1, 6)]
    
    def create_enhanced_circular_plot(ax, llm_data, metric, title, llm_name, category_scale):
        """Create enhanced circular plot with faded vertical lines from bar tops"""
        if llm_data.empty:
            ax.set_visible(False)
            return
        
        llm_data = llm_data.sort_values(['Strategy', 'Variation'])
        values = llm_data[metric].values
        N = len(values)
        
        if N == 0:
            ax.set_visible(False)
            return
        
        # Calculate angles and bar width
        theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
        width = (2 * np.pi / N) * 0.9
        
        # FIRST: Draw faded vertical lines from bar tops to edge (behind bars)
        spoke_angles = np.linspace(0, 2*np.pi, N, endpoint=False)
        for i, (spoke_angle, value) in enumerate(zip(spoke_angles, values)):
            # Faded vertical line from top of bar to outer edge
            ax.plot([spoke_angle, spoke_angle], [value, category_scale],
                    linestyle='-', linewidth=1.2, color='gray',
                    alpha=0.3, zorder=0)
        
        # THEN: Create bars with slight transparency so lines show through
        bars = ax.bar(theta, values, width=width, bottom=0.0, alpha=0.75)
        
        # Apply color scheme with uniform black edges
        for i, (bar, (_, row)) in enumerate(zip(bars, llm_data.iterrows())):
            strategy = row['Strategy']
            bar.set_facecolor(strategy_color_map[strategy])
            bar.set_edgecolor('black')
            bar.set_linewidth(0.8)
            bar.set_zorder(2)
        
        # Create enhanced angle labels
        angle_labels = []
        for _, row in llm_data.iterrows():
            strategy = row['Strategy']
            variation = row['Variation']
            if 'PICO' in strategy:
                strategy = 'PICO'
            elif 'persona-based' in strategy.lower() or 'Persona-based' in strategy:
                strategy = 'Persona'
            angle_labels.append(f"{strategy}-V{variation}")
        
        # Set enhanced scale and labels
        ax.set_ylim(0, category_scale)
        theta_degrees = np.degrees(theta)
        
        # Enhanced radial grid system with level-specific styling
        rticks = get_enhanced_rticks(category_scale)
        rticks = [r for r in rticks if r <= category_scale]
        
        # Turn off default grid for custom enhancement
        ax.grid(False)
        
        # Create level-specific horizontal grid lines
        theta_grid = np.linspace(0, 2*np.pi, 120)
        for i, radius in enumerate(rticks):
            if i == 0:
                linestyle, alpha, linewidth, color = ':', 0.4, 0.8, 'gray'
            elif i == 1:
                linestyle, alpha, linewidth, color = '--', 0.5, 1.0, 'gray'
            elif i == 2:
                linestyle, alpha, linewidth, color = '-.', 0.6, 1.2, 'dimgray'
            elif i == len(rticks) - 1:
                linestyle, alpha, linewidth, color = '-', 0.8, 1.8, 'black'
            else:
                linestyle = '-'
                alpha = 0.5 + (i * 0.1)
                linewidth = 0.8 + (i * 0.2)
                color = 'gray'
            
            ax.plot(theta_grid, [radius] * len(theta_grid),
                    linestyle=linestyle, linewidth=linewidth,
                    color=color, alpha=alpha, zorder=1)
        
        # Enhanced radial labels
        ax.set_rticks(rticks)
        ax.set_rlabel_position(15)
        for i, label in enumerate(ax.get_yticklabels()):
            label.set_fontsize(8)
            if i == len(rticks) - 1:
                label.set_fontweight('bold')
                label.set_alpha(0.9)
            elif i >= len(rticks) - 2:
                label.set_fontweight('semibold')
                label.set_alpha(0.8)
            else:
                label.set_fontweight('normal')
                label.set_alpha(0.7)
        
        # Two-colored titles
        title_parts = title.split('\n')
        llm_part = title_parts[0] if len(title_parts) > 0 else llm_name
        metric_part = title_parts[1] if len(title_parts) > 1 else metric
        
        ax.text(0.53, 1.28, llm_part, transform=ax.transAxes,
                ha='center', va='center', fontsize=20, fontweight='bold',
                color='black')
        ax.text(0.55, 1.19, metric_part, transform=ax.transAxes,
                ha='center', va='center', fontsize=12, fontweight='bold',
                color='darkred')
        
        # Clean angular positioning
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)
        
        # Enhanced angular labels
        ax.tick_params(axis='x', labelsize=12, pad=12, colors='black')
        ax.set_thetagrids(theta_degrees, angle_labels, fontsize=8, alpha=0.9)
    
    # STEP 4: Create plots
    print("\n=== Step 4: Creating Individual Plots ===")
    
    # Metrics and titles
    metrics = ['Recall', 'Precision', 'Consistency']
    metric_titles = ['Recall Scores', 'Precision Scores', 'Consistency Scores']
    
    # Create enhanced plots
    for row, (metric, metric_title) in enumerate(zip(metrics, metric_titles)):
        print(f"Plotting row {row+1}: {metric_title}")
        for col, llm in enumerate(ordered_llms):
            if col < 5:
                llm_data = df_variations[df_variations['LLM'] == llm]
                title = f"{llm}\n{metric_title}"
                
                if llm in specialized_llms:
                    category_scale = scales[f'{metric}_specialized']
                else:
                    category_scale = scales[f'{metric}_general']
                
                create_enhanced_circular_plot(axes[row, col], llm_data, metric, title, llm, category_scale)
    
    # STEP 5: Add separators and labels
    print("\n=== Step 5: Adding Separators and Labels ===")
    
    # Vertical separators between LLM columns
    separator_x_positions = [0.23, 0.61, 0.80]
    for x_pos in separator_x_positions[:n_llms-1]:
        for row in range(3):
            fig.add_artist(plt.Line2D([x_pos, x_pos],
                                      [0.07 + (2-row) * 0.31, 0.07 + (2-row) * 0.31 + 0.25],
                                      color='lightgray', linewidth=1.2, alpha=0.7,
                                      linestyle=':', transform=fig.transFigure))
    
    # Enhanced category separator
    if specialized_count > 0 and specialized_count < len(ordered_llms):
        separator_x = specialized_count - 0.5
        for row in range(3):
            fig.add_artist(plt.Line2D([0.18 + separator_x * 0.16, 0.18 + separator_x * 0.16],
                                      [0.07 + (2 - row) * 0.31, 0.07 + (2 - row) * 0.31 + 0.25],
                                      color='black', linewidth=2, linestyle='-', alpha=0.7,
                                      transform=fig.transFigure))
    
    # Horizontal separators between metric rows
    for row in range(1, 3):
        y_pos = 0.07 + (3-row) * 0.31 - 0.02
        fig.add_artist(plt.Line2D([0.05, 0.95], [y_pos, y_pos],
                                  color='gray', linewidth=1.0, alpha=0.3,
                                  linestyle='--', transform=fig.transFigure))
    
    # Row labels
    row_labels = ['D. Recall\nby Prompt Variation', 'E. Precision\nby Prompt Variation', 'F. Consistency\nby Prompt Variation']
    for row, label in enumerate(row_labels):
        fig.text(0.02, 0.75 - row * 0.28, label, fontsize=22, fontweight='bold',
                 rotation=90, va='center', ha='center', color='darkred')
    
    # Category headers
    if specialized_count > 0:
        specialized_center = (specialized_count - 1) / 2
        fig.text(0.16 + specialized_center * 0.16, 1.02,
                 'Specialized Search LLMs',
                 fontsize=26, fontweight='bold', ha='center', va='center',
                 color='darkblue')
    
    general_count = len([llm for llm in ordered_llms if llm in general_llms])
    if general_count > 0:
        general_start = specialized_count
        general_center = general_start + (general_count - 1) / 2
        fig.text(0.20 + general_center * 0.16, 1.02,
                 'Conversational LLMs',
                 fontsize=26, fontweight='bold', ha='center', va='center',
                 color='darkblue')
    
    # Legend
    legend_elements = []
    for strategy in strategies:
        legend_elements.append(plt.Rectangle((0,0),1,1,
                                             facecolor=strategy_color_map[strategy],
                                             edgecolor='black', linewidth=0.8,
                                             alpha=0.85, label=strategy))
    
    legend = fig.legend(handles=legend_elements, loc='center', bbox_to_anchor=(0.5, 0.02),
                        ncol=len(legend_elements), fontsize=18, frameon=True,
                        fancybox=False, edgecolor='black', facecolor='white',
                        columnspacing=1.2, handlelength=1.5)
    
    # STEP 6: Save and display
    print("\n=== Step 6: Finalizing Plot ===")
    plt.tight_layout(pad=3.0)
    plt.subplots_adjust(left=0.05, bottom=0.08, top=0.91, right=0.98,
                        hspace=0.50, wspace=0.15)
    
    output_file = 'circular_bar_plots_llm_analysis.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none', format='png',
                transparent=False, pad_inches=0.2)
    print(f"✓ Plot saved to: {output_file}")
    
    plt.show()
    
    print("\n" + "="*70)
    print("CIRCULAR BAR PLOTS COMPLETE!")
    print("="*70)

# ===== GENERATE COMPLETE TEXT REPORT FOR CIRCULAR BAR PLOTS =====
def generate_circular_bar_report(df_variations, output_file='report_circular_bar_plots.txt'):
    """Generate comprehensive text report for circular bar plot results with COMPLETE DATA"""
    
    with open(output_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("CIRCULAR BAR PLOTS ANALYSIS REPORT - COMPLETE DATA\n")
        f.write("=" * 80 + "\n\n")
        
        f.write(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total Variations Analyzed: {len(df_variations)}\n\n")
        
        # === SECTION 1: COMPLETE DATA TABLE ===
        f.write("=" * 80 + "\n")
        f.write("SECTION 1: COMPLETE VARIATION DATA TABLE\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("All Variations (Sorted by LLM, Strategy, Variation):\n")
        f.write("-" * 120 + "\n")
        f.write(f"{'LLM':<12} | {'Strategy':<15} | {'Var':<3} | {'Recall':<8} | {'Precision':<10} | {'Consistency':<11} | {'Efficiency':<10} | {'Matched':<7}\n")
        f.write("-" * 120 + "\n")
        
        # Sort by LLM, Strategy, Variation
        df_sorted = df_variations.sort_values(['LLM', 'Strategy', 'Variation'])
        
        for idx, row in df_sorted.iterrows():
            efficiency = row['Recall'] * row['Consistency']
            f.write(f"{row['LLM']:<12} | {row['Strategy']:<15} | {row['Variation']:<3} | "
                   f"{row['Recall']:<8.4f} | {row['Precision']:<10.4f} | "
                   f"{row['Consistency']:<11.4f} | {efficiency:<10.4f} | "
                   f"{row['Matched_Count']:<7}\n")
        
        # === SECTION 2: OVERALL STATISTICS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 2: OVERALL PERFORMANCE STATISTICS\n")
        f.write("=" * 80 + "\n\n")
        
        for metric in ['Recall', 'Precision', 'Consistency']:
            f.write(f"\n{metric} Statistics:\n")
            f.write(f"  Mean:     {df_variations[metric].mean():.4f}\n")
            f.write(f"  Median:   {df_variations[metric].median():.4f}\n")
            f.write(f"  Std Dev:  {df_variations[metric].std():.4f}\n")
            f.write(f"  Min:      {df_variations[metric].min():.4f}\n")
            f.write(f"  Max:      {df_variations[metric].max():.4f}\n")
            f.write(f"  Q1:       {df_variations[metric].quantile(0.25):.4f}\n")
            f.write(f"  Q3:       {df_variations[metric].quantile(0.75):.4f}\n")
        
        # === SECTION 3: COMPLETE LLM-LEVEL ANALYSIS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 3: COMPLETE LLM-LEVEL ANALYSIS\n")
        f.write("=" * 80 + "\n\n")
        
        for llm in sorted(df_variations['LLM'].unique()):
            llm_data = df_variations[df_variations['LLM'] == llm]
            
            f.write(f"\n{llm}:\n")
            f.write("-" * 80 + "\n")
            f.write(f"Total Variations: {len(llm_data)}\n\n")
            
            # Complete data for this LLM
            f.write(f"{'Strategy':<15} | {'Var':<3} | {'Recall':<8} | {'Precision':<10} | {'Consistency':<11} | {'Efficiency':<10}\n")
            f.write("-" * 80 + "\n")
            
            llm_sorted = llm_data.sort_values(['Strategy', 'Variation'])
            for idx, row in llm_sorted.iterrows():
                efficiency = row['Recall'] * row['Consistency']
                f.write(f"{row['Strategy']:<15} | {row['Variation']:<3} | "
                       f"{row['Recall']:<8.4f} | {row['Precision']:<10.4f} | "
                       f"{row['Consistency']:<11.4f} | {efficiency:<10.4f}\n")
            
            # Statistics for this LLM
            f.write(f"\n{llm} Statistics:\n")
            for metric in ['Recall', 'Precision', 'Consistency']:
                f.write(f"  {metric}:\n")
                f.write(f"    Mean:   {llm_data[metric].mean():.4f}\n")
                f.write(f"    Median: {llm_data[metric].median():.4f}\n")
                f.write(f"    Std:    {llm_data[metric].std():.4f}\n")
                f.write(f"    Range:  [{llm_data[metric].min():.4f}, {llm_data[metric].max():.4f}]\n")
        
        # === SECTION 4: COMPLETE STRATEGY-LEVEL ANALYSIS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 4: COMPLETE STRATEGY-LEVEL ANALYSIS\n")
        f.write("=" * 80 + "\n\n")
        
        for strategy in sorted(df_variations['Strategy'].unique()):
            strategy_data = df_variations[df_variations['Strategy'] == strategy]
            
            f.write(f"\n{strategy}:\n")
            f.write("-" * 80 + "\n")
            f.write(f"Total Variations: {len(strategy_data)}\n\n")
            
            # Complete data for this strategy
            f.write(f"{'LLM':<12} | {'Var':<3} | {'Recall':<8} | {'Precision':<10} | {'Consistency':<11} | {'Efficiency':<10}\n")
            f.write("-" * 80 + "\n")
            
            strategy_sorted = strategy_data.sort_values(['LLM', 'Variation'])
            for idx, row in strategy_sorted.iterrows():
                efficiency = row['Recall'] * row['Consistency']
                f.write(f"{row['LLM']:<12} | {row['Variation']:<3} | "
                       f"{row['Recall']:<8.4f} | {row['Precision']:<10.4f} | "
                       f"{row['Consistency']:<11.4f} | {efficiency:<10.4f}\n")
            
            # Statistics for this strategy
            f.write(f"\n{strategy} Statistics:\n")
            for metric in ['Recall', 'Precision', 'Consistency']:
                f.write(f"  {metric}:\n")
                f.write(f"    Mean:   {strategy_data[metric].mean():.4f}\n")
                f.write(f"    Median: {strategy_data[metric].median():.4f}\n")
                f.write(f"    Std:    {strategy_data[metric].std():.4f}\n")
                f.write(f"    Range:  [{strategy_data[metric].min():.4f}, {strategy_data[metric].max():.4f}]\n")
        
        # === SECTION 5: COMPLETE RANKINGS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 5: COMPLETE RANKINGS BY METRIC\n")
        f.write("=" * 80 + "\n\n")
        
        for metric in ['Recall', 'Precision', 'Consistency']:
            f.write(f"\nCOMPLETE RANKING BY {metric.upper()}:\n")
            f.write("-" * 100 + "\n")
            f.write(f"{'Rank':<5} | {'LLM':<12} | {'Strategy':<15} | {'Var':<3} | {metric:<10} | {'Matched':<7}\n")
            f.write("-" * 100 + "\n")
            
            ranked = df_variations.sort_values(metric, ascending=False)
            for rank, row in enumerate(ranked.itertuples(), 1):
                f.write(f"{rank:<5} | {row.LLM:<12} | {row.Strategy:<15} | {row.Variation:<3} | "
                       f"{getattr(row, metric):<10.4f} | {row.Matched_Count:<7}\n")
        
        # Efficiency ranking
        df_variations['Efficiency'] = df_variations['Recall'] * df_variations['Consistency']
        f.write(f"\nCOMPLETE RANKING BY EFFICIENCY (Recall × Consistency):\n")
        f.write("-" * 100 + "\n")
        f.write(f"{'Rank':<5} | {'LLM':<12} | {'Strategy':<15} | {'Var':<3} | {'Efficiency':<10} | {'Recall':<8} | {'Consistency':<11}\n")
        f.write("-" * 100 + "\n")
        
        ranked_eff = df_variations.sort_values('Efficiency', ascending=False)
        for rank, row in enumerate(ranked_eff.itertuples(), 1):
            f.write(f"{rank:<5} | {row.LLM:<12} | {row.Strategy:<15} | {row.Variation:<3} | "
                   f"{row.Efficiency:<10.4f} | {row.Recall:<8.4f} | {row.Consistency:<11.4f}\n")
        
        # === SECTION 6: VARIATION CONSISTENCY WITHIN STRATEGIES ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 6: VARIATION CONSISTENCY WITHIN STRATEGIES (COMPLETE)\n")
        f.write("=" * 80 + "\n\n")
        
        for llm in sorted(df_variations['LLM'].unique()):
            f.write(f"\n{llm}:\n")
            f.write("-" * 80 + "\n")
            llm_data = df_variations[df_variations['LLM'] == llm]
            
            for strategy in sorted(llm_data['Strategy'].unique()):
                strategy_data = llm_data[llm_data['Strategy'] == strategy]
                
                if len(strategy_data) > 1:
                    f.write(f"\n  {strategy}:\n")
                    
                    # Show all variations
                    for idx, row in strategy_data.sort_values('Variation').iterrows():
                        f.write(f"    V{row['Variation']}: Recall={row['Recall']:.4f}, "
                               f"Precision={row['Precision']:.4f}, "
                               f"Consistency={row['Consistency']:.4f}\n")
                    
                    # Variability metrics
                    f.write(f"    Variability:\n")
                    for metric in ['Recall', 'Precision', 'Consistency']:
                        std = strategy_data[metric].std()
                        range_val = strategy_data[metric].max() - strategy_data[metric].min()
                        f.write(f"      {metric}: Std={std:.4f}, Range={range_val:.4f}\n")
        
        # === SECTION 7: CORRELATION ANALYSIS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 7: CORRELATION ANALYSIS\n")
        f.write("=" * 80 + "\n\n")
        
        correlation_matrix = df_variations[['Recall', 'Precision', 'Consistency']].corr()
        
        f.write("Correlation Matrix:\n")
        f.write("-" * 60 + "\n")
        f.write(f"{'':15} | {'Recall':<12} | {'Precision':<12} | {'Consistency':<12}\n")
        f.write("-" * 60 + "\n")
        
        for metric in ['Recall', 'Precision', 'Consistency']:
            f.write(f"{metric:<15} | ")
            for other_metric in ['Recall', 'Precision', 'Consistency']:
                corr = correlation_matrix.loc[metric, other_metric]
                f.write(f"{corr:<12.4f} | ")
            f.write("\n")
        
        # === SECTION 8: PERCENTILE ANALYSIS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 8: PERCENTILE ANALYSIS (COMPLETE)\n")
        f.write("=" * 80 + "\n\n")
        
        percentiles = [0, 10, 25, 50, 75, 90, 100]
        
        for metric in ['Recall', 'Precision', 'Consistency']:
            f.write(f"\n{metric} Percentiles:\n")
            f.write("-" * 60 + "\n")
            
            for p in percentiles:
                value = df_variations[metric].quantile(p/100)
                f.write(f"  {p:3d}th percentile: {value:.4f}\n")
                
                # List variations at this percentile
                if p in [0, 25, 50, 75, 100]:
                    variations_at_percentile = df_variations[
                        abs(df_variations[metric] - value) < 0.001
                    ]
                    if len(variations_at_percentile) > 0 and len(variations_at_percentile) <= 5:
                        f.write(f"      Variations: ")
                        for _, row in variations_at_percentile.iterrows():
                            f.write(f"{row['LLM']}-{row['Strategy']}-V{row['Variation']} ")
                        f.write("\n")
        
        # === SECTION 9: KEY INSIGHTS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 9: KEY INSIGHTS\n")
        f.write("=" * 80 + "\n\n")
        
        # Best overall LLM
        best_llm_recall = df_variations.groupby('LLM')['Recall'].mean().idxmax()
        best_llm_precision = df_variations.groupby('LLM')['Precision'].mean().idxmax()
        best_llm_consistency = df_variations.groupby('LLM')['Consistency'].mean().idxmax()
        
        f.write(f"Best LLM by Average Recall:      {best_llm_recall} "
               f"({df_variations[df_variations['LLM']==best_llm_recall]['Recall'].mean():.4f})\n")
        f.write(f"Best LLM by Average Precision:   {best_llm_precision} "
               f"({df_variations[df_variations['LLM']==best_llm_precision]['Precision'].mean():.4f})\n")
        f.write(f"Best LLM by Average Consistency: {best_llm_consistency} "
               f"({df_variations[df_variations['LLM']==best_llm_consistency]['Consistency'].mean():.4f})\n\n")
        
        # Best overall strategy
        best_strategy_recall = df_variations.groupby('Strategy')['Recall'].mean().idxmax()
        best_strategy_precision = df_variations.groupby('Strategy')['Precision'].mean().idxmax()
        best_strategy_consistency = df_variations.groupby('Strategy')['Consistency'].mean().idxmax()
        
        f.write(f"Best Strategy by Average Recall:      {best_strategy_recall} "
               f"({df_variations[df_variations['Strategy']==best_strategy_recall]['Recall'].mean():.4f})\n")
        f.write(f"Best Strategy by Average Precision:   {best_strategy_precision} "
               f"({df_variations[df_variations['Strategy']==best_strategy_precision]['Precision'].mean():.4f})\n")
        f.write(f"Best Strategy by Average Consistency: {best_strategy_consistency} "
               f"({df_variations[df_variations['Strategy']==best_strategy_consistency]['Consistency'].mean():.4f})\n\n")
        
        # Most consistent LLMs
        llm_variability = df_variations.groupby('LLM')['Recall'].std().sort_values()
        f.write(f"\nLLMs by Recall Consistency (lowest std = most consistent):\n")
        for rank, (llm, std) in enumerate(llm_variability.items(), 1):
            mean_recall = df_variations[df_variations['LLM']==llm]['Recall'].mean()
            f.write(f"  {rank}. {llm:12s}: Std={std:.4f}, Mean={mean_recall:.4f}\n")
        
        # Best individual variations
        f.write(f"\nBest Overall Variation by Efficiency:\n")
        best_var = df_variations.loc[df_variations['Efficiency'].idxmax()]
        f.write(f"  {best_var['LLM']}-{best_var['Strategy']}-V{best_var['Variation']}\n")
        f.write(f"  Recall={best_var['Recall']:.4f}, Precision={best_var['Precision']:.4f}, "
               f"Consistency={best_var['Consistency']:.4f}, Efficiency={best_var['Efficiency']:.4f}\n")
        
        # Balanced performers (high in all three metrics)
        threshold = 0.7
        balanced = df_variations[
            (df_variations['Recall'] > threshold) &
            (df_variations['Precision'] > threshold) &
            (df_variations['Consistency'] > threshold)
        ]
        f.write(f"\nBalanced High Performers (all metrics > {threshold}):\n")
        if len(balanced) > 0:
            for idx, row in balanced.iterrows():
                f.write(f"  {row['LLM']}-{row['Strategy']}-V{row['Variation']}: "
                       f"R={row['Recall']:.4f}, P={row['Precision']:.4f}, C={row['Consistency']:.4f}\n")
        else:
            f.write(f"  No variations meet all criteria\n")
        
        f.write("\n" + "=" * 80 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 80 + "\n")
    
    print(f"✓ Complete circular bar plots report saved to: {output_file}")
    return output_file

# Call the report generation function
report_file = generate_circular_bar_report(df_variations)
    
# ===== EXECUTION =====
print("\n" + "="*70)
print("CIRCULAR BAR PLOTS VISUALIZATION")
print("="*70)

if 'df_variations' in globals() and df_variations is not None:
    print("✓ df_variations found")
    create_circular_bar_plots(df_variations)
else:
    print("❌ Error: df_variations not found!")
    print("Please run Visualization Cell 1 first to create df_variations with precision.")

## Cell 07 — SCATTER PLOTS

*Figure 3 (from df_variations)*


In [ ]:
# ===== SCATTER PLOTS VISUALIZATION WITH PRECISION RESULTS (INDIVIDUAL PLOTS) =====
def create_scatter_plots_individual(df_variations, precision_results, recall_scores, plot_width=12, plot_height=10):
    """
    Create three separate scatter plots with threshold curves
    
    Parameters:
    -----------
    df_variations : DataFrame
        Variations data with Recall, Consistency, etc.
    precision_results : dict
        Results from precision analysis (contains precision_df)
    recall_scores : dict
        Recall scores from base_results
    plot_width : float
        Width of each plot in inches (default: 12)
    plot_height : float
        Height of each plot in inches (default: 10)
    """
    print("\n" + "="*70)
    print("CREATING INDIVIDUAL SCATTER ANALYSIS PLOTS")
    print("="*70)
    
    # Verify inputs
    if df_variations is None or df_variations.empty:
        print("❌ ERROR: df_variations is required!")
        return
    
    if precision_results is None or 'precision_df' not in precision_results:
        print("❌ ERROR: precision_results with precision_df is required!")
        return
        
    print(f"✓ Input data verified")
    print(f"Individual plot dimensions: {plot_width}W x {plot_height}H inches")
    
    # Get precision dataframe and add recall data
    precision_df = precision_results['precision_df'].copy()
    
    # Add recall data to precision_df
    recall_data = []
    for _, row in precision_df.iterrows():
        var_key = row['Variation_Key']
        if var_key in recall_scores:
            recall_data.append(recall_scores[var_key]['recall'])
        else:
            recall_data.append(0.0)
    
    precision_df['Recall'] = recall_data
    
    print(f"Plotting {len(precision_df)} data points from precision analysis")
    
    # Define LLM colors
    llm_colors = {
        'Ai2 Finder': '#27AE60',
        'Consensus': '#E67E22',
        'ChatGPT': '#2980B9',
        'Claude': '#8E44AD',
        'Gemini': '#C0392B'
    }
    
    # Strategy colors
    strategy_names = ['Zero-Shot', 'Few-Shot', 'Persona-based', 'Chain-of-Thought', 'PICO Framework']
    strategy_display = ['Zero-Shot', 'Few-Shot', 'Persona', 'CoT', 'PICO']
    strategy_color_list = plt.cm.Set3(np.linspace(0, 1, len(strategy_names)))
    strategy_color_map = {}
    for i, (name, display) in enumerate(zip(strategy_names, strategy_display)):
        strategy_color_map[name] = strategy_color_list[i]
        strategy_color_map[display] = strategy_color_list[i]
    
    # Map strategy numbers to names for precision_df
    strategy_num_to_name = {
        '1': 'Zero-Shot',
        '2': 'Few-Shot',
        '3': 'Persona-based',
        '4': 'Chain-of-Thought',
        '5': 'PICO Framework'
    }
    
    # Add strategy names to precision_df if not present
    if 'Strategy_Name' not in precision_df.columns:
        precision_df['Strategy_Name'] = precision_df['Strategy'].map(strategy_num_to_name)
    
    # Scale font sizes based on plot size
    avg_size = (plot_width + plot_height) / 2
    scale_factor = avg_size / 11
    
    title_size = int(36 * scale_factor)
    label_size = int(34 * scale_factor)
    legend_size = int(26 * scale_factor)
    curve_label_size = int(18 * scale_factor)
    marker_size = int(320 * scale_factor)
    tick_size = int(22 * scale_factor)
    
    # ===== PLOT A: Precision-Recall by LLM with F1 Threshold Curves =====
    print("\nCreating Plot A: Precision-Recall by LLM...")
    
    fig_a, ax_a = plt.subplots(figsize=(plot_width, plot_height))
    fig_a.patch.set_facecolor('white')
    
    for llm in precision_df['LLM'].unique():
        llm_data = precision_df[precision_df['LLM'] == llm]
        ax_a.scatter(llm_data['Recall'], llm_data['Precision'],
                     alpha=0.9, s=marker_size, label=llm,
                     color=llm_colors.get(llm, 'gray'),
                     edgecolor='black', linewidth=0.8, zorder=3)
    
    # Add F1 threshold curves
    recall_range = np.linspace(0.001, 1.0, 300)
    f1_thresholds = [0.3, 0.5, 0.7, 0.9]
    
    for f1_val in f1_thresholds:
        precision_range = []
        for r in recall_range:
            if r > 0.001:
                p = f1_val * r / (2 * r - f1_val)
                if 0 <= p <= 1.05:
                    precision_range.append(p)
                else:
                    precision_range.append(np.nan)
            else:
                precision_range.append(np.nan)
        
        # Plot threshold curve
        ax_a.plot(recall_range, precision_range, 'k--',
                  alpha=0.9, linewidth=1.5, zorder=1)
        
        # Label the curve
        valid_indices = [i for i, p in enumerate(precision_range)
                         if not np.isnan(p) and 0.3 < p < 0.95]
        if valid_indices:
            mid_idx = valid_indices[len(valid_indices)//3]
            ax_a.text(recall_range[mid_idx], precision_range[mid_idx],
                      f'F1={f1_val}', fontsize=curve_label_size, alpha=0.7,
                      bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                edgecolor='gray', alpha=0.7))
    
    ax_a.set_xlabel('Recall', fontsize=label_size, fontweight='bold')
    ax_a.set_ylabel('Precision', fontsize=label_size, fontweight='bold')
    ax_a.set_title('A. Precision-Recall by LLM',
                   fontsize=title_size, fontweight='bold', pad=15)
    ax_a.legend(fontsize=legend_size, loc='lower right', ncol=1, frameon=True,
                fancybox=False, edgecolor='black', columnspacing=1.5)
    ax_a.grid(True, alpha=0.9, zorder=0)
    ax_a.set_xlim(-0.02, 1.02)
    ax_a.set_ylim(-0.02, 1.02)
    ax_a.tick_params(labelsize=tick_size)
    
    plt.tight_layout()
    output_file_a = 'scatter_plot_A_precision_recall_by_llm.png'
    plt.savefig(output_file_a, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none', format='png',
                transparent=False, pad_inches=0.2)
    print(f"✓ Plot A saved as: {output_file_a}")
    plt.show()
    plt.close()
    
    # ===== PLOT B: Precision-Recall by Strategy with F1 Threshold Curves =====
    print("\nCreating Plot B: Precision-Recall by Strategy...")
    
    fig_b, ax_b = plt.subplots(figsize=(plot_width, plot_height))
    fig_b.patch.set_facecolor('white')
    
    for strategy_name in strategy_names:
        strat_data = precision_df[precision_df['Strategy_Name'] == strategy_name]
        if len(strat_data) > 0:
            display_name = strategy_name.replace('Chain-of-Thought', 'CoT').replace('Persona-based', 'Persona').replace('PICO Framework', 'PICO')
            ax_b.scatter(strat_data['Recall'], strat_data['Precision'],
                         alpha=0.9, s=marker_size, label=display_name,
                         color=strategy_color_map[strategy_name],
                         edgecolor='black', linewidth=0.8, zorder=3)
    
    # Add F1 threshold curves
    for f1_val in f1_thresholds:
        precision_range = []
        for r in recall_range:
            if r > 0.001:
                p = f1_val * r / (2 * r - f1_val)
                if 0 <= p <= 1.05:
                    precision_range.append(p)
                else:
                    precision_range.append(np.nan)
            else:
                precision_range.append(np.nan)
        
        # Plot threshold curve
        ax_b.plot(recall_range, precision_range, 'k--',
                  alpha=0.9, linewidth=1.5, zorder=1)
        
        # Label the curve
        valid_indices = [i for i, p in enumerate(precision_range)
                         if not np.isnan(p) and 0.3 < p < 0.95]
        if valid_indices:
            mid_idx = valid_indices[len(valid_indices)//3]
            ax_b.text(recall_range[mid_idx], precision_range[mid_idx],
                      f'F1={f1_val}', fontsize=curve_label_size, alpha=0.7,
                      bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                edgecolor='gray', alpha=0.7))
    
    ax_b.set_xlabel('Recall', fontsize=label_size, fontweight='bold')
    ax_b.set_ylabel('Precision', fontsize=label_size, fontweight='bold')
    ax_b.set_title('B. Precision-Recall by Strategy',
                   fontsize=title_size, fontweight='bold', pad=15)
    ax_b.legend(fontsize=legend_size, loc='lower right', ncol=1, frameon=True,
                fancybox=False, edgecolor='black', columnspacing=1.5)
    ax_b.grid(True, alpha=0.7, zorder=0)
    ax_b.set_xlim(-0.02, 1.02)
    ax_b.set_ylim(-0.02, 1.02)
    ax_b.tick_params(labelsize=tick_size)
    
    plt.tight_layout()
    output_file_b = 'scatter_plot_B_precision_recall_by_strategy.png'
    plt.savefig(output_file_b, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none', format='png',
                transparent=False, pad_inches=0.2)
    print(f"✓ Plot B saved as: {output_file_b}")
    plt.show()
    plt.close()
    
    # ===== PLOT C: Consistency vs Recall by LLM with Reliability Threshold Curves =====
    print("\nCreating Plot C: Recall vs Consistency by LLM...")
    
    fig_c, ax_c = plt.subplots(figsize=(plot_width, plot_height))
    fig_c.patch.set_facecolor('white')
    
    for llm in df_variations['LLM'].unique():
        llm_data = df_variations[df_variations['LLM'] == llm]
        ax_c.scatter(llm_data['Recall'], llm_data['Consistency'],
                     alpha=0.9, s=marker_size, label=llm,
                     color=llm_colors.get(llm, 'gray'),
                     edgecolor='black', linewidth=0.8, zorder=3)
    
    ax_c.set_xlabel('Recall Score', fontsize=label_size, fontweight='bold')
    ax_c.set_ylabel('Consistency Score', fontsize=label_size, fontweight='bold')
    ax_c.set_title('C. Recall vs Consistency by LLM',
                   fontsize=title_size, fontweight='bold', pad=15)
    ax_c.legend(fontsize=legend_size, loc='lower right', ncol=1, frameon=True,
                fancybox=False, edgecolor='black', columnspacing=1.5)
    ax_c.grid(True, alpha=0.9, zorder=0)
    ax_c.set_xlim(-0.02, 1.02)
    ax_c.set_ylim(-0.02, 1.02)
    ax_c.tick_params(labelsize=tick_size)
    
    plt.tight_layout()
    output_file_c = 'scatter_plot_C_recall_vs_consistency_by_llm.png'
    plt.savefig(output_file_c, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none', format='png',
                transparent=False, pad_inches=0.2)
    print(f"✓ Plot C saved as: {output_file_c}")
    plt.show()
    plt.close()
    
    print("\n" + "="*70)
    print("INDIVIDUAL SCATTER PLOTS COMPLETE!")
    print("="*70)
    print(f"Three separate files created:")
    print(f"  1. {output_file_a}")
    print(f"  2. {output_file_b}")
    print(f"  3. {output_file_c}")

# ===== GENERATE COMPLETE TEXT REPORT FOR SCATTER PLOTS =====
def generate_scatter_plots_report(df_variations, precision_results, output_file='report_scatter_plots.txt'):
    """Generate comprehensive text report for scatter plot results with COMPLETE DATA"""
    
    precision_df = precision_results['precision_df'].copy()
    
    with open(output_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("SCATTER PLOTS ANALYSIS REPORT - COMPLETE DATA\n")
        f.write("=" * 80 + "\n\n")
        
        f.write(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total Variations Analyzed: {len(precision_df)}\n\n")
        
        # === SECTION 1: COMPLETE PRECISION-RECALL DATA TABLE ===
        f.write("=" * 80 + "\n")
        f.write("SECTION 1: COMPLETE PRECISION-RECALL DATA TABLE\n")
        f.write("=" * 80 + "\n\n")
        
        # Merge with recall data
        recall_map = dict(zip(df_variations['Full_Key'], df_variations['Recall']))
        precision_df['Recall'] = precision_df['Variation_Key'].map(recall_map)
        
        # Calculate F1
        precision_df['F1_Score'] = 2 * (precision_df['Precision'] * precision_df['Recall']) / \
                                   (precision_df['Precision'] + precision_df['Recall'] + 1e-10)
        
        f.write("All Variations (Sorted by F1 Score):\n")
        f.write("-" * 130 + "\n")
        f.write(f"{'Rank':<5} | {'Variation Key':<25} | {'LLM':<12} | {'Precision':<10} | {'Recall':<8} | {'F1 Score':<9} | {'Matched':<7} | {'Retrieved':<9}\n")
        f.write("-" * 130 + "\n")
        
        sorted_by_f1 = precision_df.sort_values('F1_Score', ascending=False)
        for rank, row in enumerate(sorted_by_f1.itertuples(), 1):
            f.write(f"{rank:<5} | {row.Variation_Key:<25} | {row.LLM:<12} | "
                   f"{row.Precision:<10.4f} | {row.Recall:<8.4f} | {row.F1_Score:<9.4f} | "
                   f"{row.Matched_Count:<7} | {row.Total_Retrieved:<9}\n")
        
        # === SECTION 2: F1 SCORE ANALYSIS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 2: F1 SCORE ANALYSIS (PRECISION-RECALL TRADE-OFF)\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("F1 Score Statistics:\n")
        f.write(f"  Mean:     {precision_df['F1_Score'].mean():.4f}\n")
        f.write(f"  Median:   {precision_df['F1_Score'].median():.4f}\n")
        f.write(f"  Std Dev:  {precision_df['F1_Score'].std():.4f}\n")
        f.write(f"  Min:      {precision_df['F1_Score'].min():.4f}\n")
        f.write(f"  Max:      {precision_df['F1_Score'].max():.4f}\n")
        f.write(f"  Q1:       {precision_df['F1_Score'].quantile(0.25):.4f}\n")
        f.write(f"  Q3:       {precision_df['F1_Score'].quantile(0.75):.4f}\n\n")
        
        # F1 Distribution
        f.write("F1 Score Distribution:\n")
        f.write("-" * 60 + "\n")
        f1_ranges = [(0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]
        for low, high in f1_ranges:
            count = len(precision_df[(precision_df['F1_Score'] >= low) & (precision_df['F1_Score'] < high)])
            pct = (count / len(precision_df)) * 100
            f.write(f"  {low:.1f} - {high:.1f}: {count:3d} variations ({pct:5.1f}%)\n")
        
        # === SECTION 3: COMPLETE LLM COMPARISON ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 3: COMPLETE PRECISION-RECALL ANALYSIS BY LLM\n")
        f.write("=" * 80 + "\n\n")
        
        for llm in sorted(precision_df['LLM'].unique()):
            llm_data = precision_df[precision_df['LLM'] == llm]
            
            f.write(f"\n{llm}:\n")
            f.write("-" * 100 + "\n")
            f.write(f"Total Variations: {len(llm_data)}\n\n")
            
            # Complete data table for this LLM
            f.write(f"{'Variation':<25} | {'Precision':<10} | {'Recall':<8} | {'F1':<9} | {'Matched':<7} | {'Retrieved':<9}\n")
            f.write("-" * 100 + "\n")
            
            for idx, row in llm_data.sort_values('F1_Score', ascending=False).iterrows():
                f.write(f"{row['Variation_Key']:<25} | {row['Precision']:<10.4f} | "
                       f"{row['Recall']:<8.4f} | {row['F1_Score']:<9.4f} | "
                       f"{row['Matched_Count']:<7} | {row['Total_Retrieved']:<9}\n")
            
            # Statistics
            f.write(f"\n{llm} Statistics:\n")
            f.write(f"  Precision:  Mean={llm_data['Precision'].mean():.4f}, Std={llm_data['Precision'].std():.4f}, "
                   f"Range=[{llm_data['Precision'].min():.4f}, {llm_data['Precision'].max():.4f}]\n")
            f.write(f"  Recall:     Mean={llm_data['Recall'].mean():.4f}, Std={llm_data['Recall'].std():.4f}, "
                   f"Range=[{llm_data['Recall'].min():.4f}, {llm_data['Recall'].max():.4f}]\n")
            f.write(f"  F1 Score:   Mean={llm_data['F1_Score'].mean():.4f}, Std={llm_data['F1_Score'].std():.4f}, "
                   f"Range=[{llm_data['F1_Score'].min():.4f}, {llm_data['F1_Score'].max():.4f}]\n")
        
        # === SECTION 4: COMPLETE STRATEGY COMPARISON ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 4: COMPLETE PRECISION-RECALL ANALYSIS BY STRATEGY\n")
        f.write("=" * 80 + "\n\n")
        
        strategy_map = {'1': 'Zero-Shot', '2': 'Few-Shot', '3': 'Persona', '4': 'CoT', '5': 'PICO'}
        
        for strategy_num, strategy_name in strategy_map.items():
            strategy_data = precision_df[precision_df['Strategy'] == strategy_num]
            
            if len(strategy_data) > 0:
                f.write(f"\n{strategy_name}:\n")
                f.write("-" * 100 + "\n")
                f.write(f"Total Variations: {len(strategy_data)}\n\n")
                
                # Complete data table for this strategy
                f.write(f"{'LLM':<12} | {'Variation':<13} | {'Precision':<10} | {'Recall':<8} | {'F1':<9} | {'Matched':<7}\n")
                f.write("-" * 100 + "\n")
                
                for idx, row in strategy_data.sort_values('F1_Score', ascending=False).iterrows():
                    f.write(f"{row['LLM']:<12} | {row['Variation_Key']:<13} | "
                           f"{row['Precision']:<10.4f} | {row['Recall']:<8.4f} | "
                           f"{row['F1_Score']:<9.4f} | {row['Matched_Count']:<7}\n")
                
                # Statistics
                f.write(f"\n{strategy_name} Statistics:\n")
                f.write(f"  Precision:  Mean={strategy_data['Precision'].mean():.4f}, Std={strategy_data['Precision'].std():.4f}\n")
                f.write(f"  Recall:     Mean={strategy_data['Recall'].mean():.4f}, Std={strategy_data['Recall'].std():.4f}\n")
                f.write(f"  F1 Score:   Mean={strategy_data['F1_Score'].mean():.4f}, Std={strategy_data['F1_Score'].std():.4f}\n")
        
        # === SECTION 5: RELIABILITY ANALYSIS (RECALL VS CONSISTENCY) ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 5: COMPLETE RELIABILITY ANALYSIS (RECALL VS CONSISTENCY)\n")
        f.write("=" * 80 + "\n\n")
        
        # Add consistency data
        consistency_map = dict(zip(df_variations['Full_Key'], df_variations['Consistency']))
        precision_df['Consistency'] = precision_df['Variation_Key'].map(consistency_map)
        
        # Calculate Reliability Score
        precision_df['Reliability'] = 2 * (precision_df['Recall'] * precision_df['Consistency']) / \
                                       (precision_df['Recall'] + precision_df['Consistency'] + 1e-10)
        
        f.write("All Variations (Sorted by Reliability):\n")
        f.write("-" * 120 + "\n")
        f.write(f"{'Rank':<5} | {'Variation Key':<25} | {'LLM':<12} | {'Recall':<8} | {'Consistency':<11} | {'Reliability':<11}\n")
        f.write("-" * 120 + "\n")
        
        sorted_by_reliability = precision_df.sort_values('Reliability', ascending=False)
        for rank, row in enumerate(sorted_by_reliability.itertuples(), 1):
            f.write(f"{rank:<5} | {row.Variation_Key:<25} | {row.LLM:<12} | "
                   f"{row.Recall:<8.4f} | {row.Consistency:<11.4f} | {row.Reliability:<11.4f}\n")
        
        # === SECTION 6: RETRIEVAL VOLUME ANALYSIS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 6: COMPLETE RETRIEVAL VOLUME ANALYSIS\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("All Variations (Sorted by Total Retrieved):\n")
        f.write("-" * 120 + "\n")
        f.write(f"{'Rank':<5} | {'Variation Key':<25} | {'LLM':<12} | {'Retrieved':<9} | {'Matched':<7} | {'Precision':<10}\n")
        f.write("-" * 120 + "\n")
        
        sorted_by_retrieved = precision_df.sort_values('Total_Retrieved', ascending=False)
        for rank, row in enumerate(sorted_by_retrieved.itertuples(), 1):
            f.write(f"{rank:<5} | {row.Variation_Key:<25} | {row.LLM:<12} | "
                   f"{row.Total_Retrieved:<9} | {row.Matched_Count:<7} | {row.Precision:<10.4f}\n")
        
        # Volume by LLM
        f.write("\n\nRetrieval Volume by LLM:\n")
        f.write("-" * 80 + "\n")
        for llm in precision_df['LLM'].unique():
            llm_data = precision_df[precision_df['LLM'] == llm]
            f.write(f"\n{llm}:\n")
            f.write(f"  Average Retrieved: {llm_data['Total_Retrieved'].mean():.1f} (Std: {llm_data['Total_Retrieved'].std():.1f})\n")
            f.write(f"  Average Matched:   {llm_data['Matched_Count'].mean():.1f} (Std: {llm_data['Matched_Count'].std():.1f})\n")
            f.write(f"  Average Precision: {llm_data['Precision'].mean():.4f}\n")
            f.write(f"  Range Retrieved:   [{llm_data['Total_Retrieved'].min()}, {llm_data['Total_Retrieved'].max()}]\n")
            f.write(f"  Range Matched:     [{llm_data['Matched_Count'].min()}, {llm_data['Matched_Count'].max()}]\n")
        
        # === SECTION 7: THRESHOLD ANALYSIS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 7: THRESHOLD ANALYSIS\n")
        f.write("=" * 80 + "\n\n")
        
        thresholds = [0.3, 0.5, 0.7, 0.9]
        
        # F1 Score thresholds
        f.write("Variations Meeting F1 Score Thresholds:\n")
        f.write("-" * 60 + "\n")
        for threshold in thresholds:
            meets_threshold = precision_df[precision_df['F1_Score'] >= threshold]
            f.write(f"\nF1 Score ≥ {threshold}:\n")
            f.write(f"  Count: {len(meets_threshold)} ({len(meets_threshold)/len(precision_df)*100:.1f}%)\n")
            if len(meets_threshold) > 0 and len(meets_threshold) <= 10:
                f.write(f"  Variations:\n")
                for _, row in meets_threshold.iterrows():
                    f.write(f"    {row['Variation_Key']}: F1={row['F1_Score']:.4f}\n")
        
        # Precision thresholds
        f.write("\n\nVariations Meeting Precision Thresholds:\n")
        f.write("-" * 60 + "\n")
        for threshold in thresholds:
            meets_threshold = precision_df[precision_df['Precision'] >= threshold]
            f.write(f"\nPrecision ≥ {threshold}:\n")
            f.write(f"  Count: {len(meets_threshold)} ({len(meets_threshold)/len(precision_df)*100:.1f}%)\n")
        
        # Recall thresholds
        f.write("\n\nVariations Meeting Recall Thresholds:\n")
        f.write("-" * 60 + "\n")
        for threshold in thresholds:
            meets_threshold = precision_df[precision_df['Recall'] >= threshold]
            f.write(f"\nRecall ≥ {threshold}:\n")
            f.write(f"  Count: {len(meets_threshold)} ({len(meets_threshold)/len(precision_df)*100:.1f}%)\n")
        
        # Balanced performers
        f.write("\n\nBalanced Performers (Meeting Multiple Thresholds):\n")
        f.write("-" * 60 + "\n")
        for threshold in [0.5, 0.7]:
            balanced = precision_df[
                (precision_df['Precision'] >= threshold) &
                (precision_df['Recall'] >= threshold) &
                (precision_df['Consistency'] >= threshold)
            ]
            f.write(f"\nAll Metrics ≥ {threshold}:\n")
            f.write(f"  Count: {len(balanced)} ({len(balanced)/len(precision_df)*100:.1f}%)\n")
            if len(balanced) > 0:
                for _, row in balanced.iterrows():
                    f.write(f"    {row['Variation_Key']}: P={row['Precision']:.4f}, "
                           f"R={row['Recall']:.4f}, C={row['Consistency']:.4f}\n")
        
        # === SECTION 8: KEY INSIGHTS ===
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("SECTION 8: KEY INSIGHTS\n")
        f.write("=" * 80 + "\n\n")
        
        best_f1 = precision_df.loc[precision_df['F1_Score'].idxmax()]
        f.write(f"Best F1 Score: {best_f1['Variation_Key']} (F1={best_f1['F1_Score']:.4f})\n")
        f.write(f"  Precision={best_f1['Precision']:.4f}, Recall={best_f1['Recall']:.4f}\n\n")
        
        best_reliability = precision_df.loc[precision_df['Reliability'].idxmax()]
        f.write(f"Best Reliability: {best_reliability['Variation_Key']} (Reliability={best_reliability['Reliability']:.4f})\n")
        f.write(f"  Recall={best_reliability['Recall']:.4f}, Consistency={best_reliability['Consistency']:.4f}\n\n")
        
        # Best by LLM
        f.write("Best Variation per LLM (by F1 Score):\n")
        for llm in precision_df['LLM'].unique():
            llm_data = precision_df[precision_df['LLM'] == llm]
            best = llm_data.loc[llm_data['F1_Score'].idxmax()]
            f.write(f"  {llm}: {best['Variation_Key']} (F1={best['F1_Score']:.4f})\n")
        
        # Trade-off analysis
        f.write("\nPrecision-Recall Trade-off Analysis:\n")
        high_precision_low_recall = precision_df[(precision_df['Precision'] > 0.7) & (precision_df['Recall'] < 0.5)]
        high_recall_low_precision = precision_df[(precision_df['Recall'] > 0.5) & (precision_df['Precision'] < 0.7)]
        
        f.write(f"  High Precision (>0.7), Low Recall (<0.5): {len(high_precision_low_recall)}\n")
        f.write(f"  High Recall (>0.5), Low Precision (<0.7): {len(high_recall_low_precision)}\n")
        
        f.write("\n" + "=" * 80 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 80 + "\n")
    
    print(f"✓ Complete scatter plots report saved to: {output_file}")
    return output_file

# Call the report generation function after creating all plots
report_file = generate_scatter_plots_report(df_variations, precision_results)

# ===== EXECUTION =====
print("\n" + "="*70)
print("INDIVIDUAL SCATTER PLOTS VISUALIZATION")
print("="*70)

# Verify all required data exists
if 'df_variations' not in globals() or df_variations is None:
    print("❌ ERROR: df_variations not found. Please run visualization cell 1 first.")
elif 'precision_results' not in globals() or precision_results is None:
    print("❌ ERROR: precision_results not found. Please run precision analysis first.")
elif 'base_results' not in globals() or base_results is None:
    print("❌ ERROR: base_results not found. Please run base analysis first.")
else:
    print("✓ All required data found")
    print(f"✓ df_variations: {len(df_variations)} variations")
    print(f"✓ precision_results: {len(precision_results['precision_df'])} precision calculations")
    
    # Create individual scatter plots with your preferred size
    # Each plot: 10 inches wide x 10 inches tall (square format)
    create_scatter_plots_individual(df_variations, precision_results,
                                   base_results['recall_scores'],
                                   plot_width=10, plot_height=10)

## Cell 08 — SUNBURST

*Figure 4 (from derived data)*


In [ ]:
# Static Plotly Sunburst Chart Visualization (Non-Interactive)
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from collections import defaultdict
import re
from pathlib import Path
import math
from datetime import datetime
from itertools import combinations as iter_combinations

# Configure Plotly for Jupyter notebook
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

class StaticSunburstChartVisualizer:
    """
    A standalone class to generate static Plotly sunburst charts
    for LLM literature retrieval analysis in Jupyter notebook.
    """
    def __init__(self, gold_standard_df, recalled_files, top_combinations):
        """
        Initializes the visualizer with the necessary data.
        Args:
            gold_standard_df (pd.DataFrame): DataFrame containing the gold standard studies.
            recalled_files (dict): Dictionary mapping recalled filenames to their metadata and paths.
            top_combinations (list): A list of dictionaries, where each dictionary represents
                                     a combination and its performance metrics.
        """
        if gold_standard_df is None or recalled_files is None or top_combinations is None:
            raise ValueError("All data inputs (gold_standard_df, recalled_files, top_combinations) are required.")
        
        self.gold_standard_df = gold_standard_df
        self.recalled_files = recalled_files
        self.top_combinations = top_combinations
        
        # Internal data structures
        self.study_mapping = self._create_study_identifier_mapping()
        self.llm_matches = self._calculate_llm_matches()
        self.all_normalized_matches = self._get_overall_matches()
        
        # COORDINATED COLOR PALETTE - Excludes green/red to avoid confusion with retrieval status
        self.publisher_colors = [
            '#2980B9',  # Deep Blue
            '#8E44AD',  # Deep Purple
            '#E67E22',  # Deep Orange
            '#006064',  # Deep Teal
            '#3E2723',  # Deep Brown
            '#263238',  # Blue Grey
            '#AD1457',  # Deep Pink
            '#4E342E',  # Brown variant
            '#880E4F',  # Deep Magenta
            '#1A237E',  # Deep Indigo
            '#BF360C',  # Deep Red-Orange
            '#5D4037',  # Brown variant
            '#34495E',  # Dark Grey-Blue
            '#7B1FA2',  # Purple variant
            '#455A64'   # Blue Grey variant
        ]
        
        # RETRIEVAL STATUS COLORS - MATCHING PUBLISHER PLOT
        # Green gradient for retrieved (matching 1-5 LLMs retrieval colors)
        self.retrieval_colors = {
            0: '#E74C3C',  # Red - Never Retrieved by Any LLM
            1: '#EB984E',  # Orange - Retrieved by 1 LLM
            2: '#F7DC6F',  # Yellow - Retrieved by 2 LLMs
            3: '#52BE80',  # Light Green - Retrieved by 3 LLMs
            4: '#27AE60',  # Green - Retrieved by 4 LLMs
            5: '#1E8449'   # Dark Green - Retrieved by 5 LLMs (All)
        }
        
        # White for not retrieved by this specific LLM
        self.not_retrieved_color = '#FFFFFF'
        
        # Enhanced visibility variables
        self.PUBLISHER_OPACITY_LEVELS = [0.7, 0.3]
        self.JOURNAL_OPACITY_LEVELS = [0.40, 0.15]
        self.STUDY_OPACITY = 0.9
        self.LINE_WIDTH = 2
        self.CENTER_TITLE_SIZE = 64

    def normalize_identifier_consistent(self, identifier):
        """Consistent normalization for study identifiers (titles, DOIs)."""
        if not identifier or pd.isna(identifier):
            return ""
        identifier = str(identifier).strip().lower()
        if identifier.startswith('doi:'):
            identifier = identifier.replace('doi:', '').strip()
        elif identifier.startswith('https://doi.org/'):
            identifier = identifier.replace('https://doi.org/', '').strip()
        return identifier

    def standardize_publisher_name(self, publisher_name):
        """Standardize publisher names for consistency."""
        publisher = str(publisher_name)
        if 'Wolters Kluwer' in publisher: return 'Wolters Kluwer'
        if 'Taylor & Francis' in publisher or 'Dove Medical Press' in publisher: return 'Taylor & Francis'
        if 'Springer' in publisher: return 'Springer Nature'
        if 'Elsevier' in publisher: return 'Elsevier'
        if 'IEEE' in publisher: return 'IEEE'
        if 'JMIR' in publisher: return 'JMIR Publications'
        return publisher[:25] + "..." if len(publisher) > 25 else publisher

    def _create_study_identifier_mapping(self):
        """Create a comprehensive mapping of study identifiers."""
        study_mapping = {}
        for idx, row in self.gold_standard_df.iterrows():
            study_id = row.get('Study ID', f'Study_{idx}')
            identifiers = set()
            if 'Title' in row and pd.notna(row['Title']):
                identifiers.add(self.normalize_identifier_consistent(row['Title']))
            if 'DOI' in row and pd.notna(row['DOI']):
                identifiers.add(self.normalize_identifier_consistent(row['DOI']))
            identifiers.add(self.normalize_identifier_consistent(study_id))
            study_mapping[study_id] = {'identifiers': identifiers, 'row_data': row}
        return study_mapping

    def _calculate_llm_matches(self):
        """Calculate the set of matched studies for each individual LLM."""
        llm_matches = defaultdict(set)
        llm_names = ['Ai2 Finder', 'Consensus', 'Claude', 'ChatGPT', 'Gemini']
        
        for filename, info in self.recalled_files.items():
            llm_name = info['llm']
            if llm_name in llm_names:
                try:
                    df = pd.read_csv(info['path'])
                    identifier_cols = ['DOI', 'Study name', 'Title', 'LLM Title']
                    for col in identifier_cols:
                        if col in df.columns:
                            matches = df[col].dropna().astype(str).apply(self.normalize_identifier_consistent)
                            llm_matches[llm_name].update(matches)
                            break
                except Exception as e:
                    print(f"Warning: Could not load {filename}: {e}")
        return llm_matches

    def _get_overall_matches(self):
        """Get the combined set of matches from the top combinations."""
        all_matches = set()
        for combo in self.top_combinations:
            for match in combo['matches']:
                normalized_match = self.normalize_identifier_consistent(match)
                if normalized_match:
                    all_matches.add(normalized_match)
        return all_matches

    def _count_llms_retrieved_study(self, study_id):
        """Count how many LLMs retrieved a specific study."""
        if study_id not in self.study_mapping:
            return 0
        llm_names = ['Ai2 Finder', 'Consensus', 'Claude', 'ChatGPT', 'Gemini']
        count = 0
        for llm_name in llm_names:
            if llm_name in self.llm_matches:
                for identifier in self.study_mapping[study_id]['identifiers']:
                    if identifier in self.llm_matches[llm_name]:
                        count += 1
                        break
        return count

    def _is_retrieved_by_specific_llm(self, study_id, target_matches):
        """Check if a study was retrieved by the specific LLM being visualized."""
        if study_id not in self.study_mapping:
            return False
        for identifier in self.study_mapping[study_id]['identifiers']:
            if identifier in target_matches:
                return True
        return False

    def _apply_opacity_to_color(self, hex_color, opacity):
        """Convert hex color to rgba with specified opacity."""
        hex_color = hex_color.lstrip('#')
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        return f'rgba({r}, {g}, {b}, {opacity})'

    def _create_faded_journal_color(self, base_color, fade_factor=0.15):
        """Create a slightly faded version of the base color for journals."""
        hex_color = base_color.lstrip('#')
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        r = int(r + (255 - r) * fade_factor)
        g = int(g + (255 - g) * fade_factor)
        b = int(b + (255 - b) * fade_factor)
        return f'#{r:02x}{g:02x}{b:02x}'

    def _clean_journal_name(self, journal_name):
        """Clean and standardize journal names."""
        journal = str(journal_name).strip()
        journal = re.sub(r'\s*\((Journal|Conference.*?)\)\s*$', '', journal)
        journal = re.sub(r'\s*(Journal|Conference)\s*$', '', journal)
        journal = journal.replace('Proceedings', 'Proc.')
        journal = journal.replace('International', 'Int.')
        journal = journal.replace('Symposium', 'Symp.')
        journal = journal.replace('Workshop', 'Work.')
        if len(journal) > 35:
            journal = journal[:32] + "..."
        return journal

    def _clean_study_name(self, study_id, title=None):
        """Create clean study labels - EMPTY STRING TO HIDE TEXT."""
        return ""

    def _get_text_color_for_background(self, background_color, opacity):
        """Determine appropriate text color based on background color and opacity."""
        hex_color = background_color.lstrip('#')
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        effective_r = r * opacity + 255 * (1 - opacity)
        effective_g = g * opacity + 255 * (1 - opacity)
        effective_b = b * opacity + 255 * (1 - opacity)
        brightness = (effective_r * 0.299 + effective_g * 0.587 + effective_b * 0.114)
        return '#FFFFFF' if brightness < 140 else '#000000'

    def _prepare_sunburst_data(self, target_matches, chart_title):
        """Prepare hierarchical data for static Plotly sunburst chart."""
        ids = []
        labels = []
        parents = []
        values = []
        colors = []
        text_colors = []
        
        # Root level
        root_id = "root"
        ids.append(root_id)
        labels.append(chart_title)
        parents.append("")
        values.append(len(self.gold_standard_df))
        colors.append(self._apply_opacity_to_color('#F5F5F5', 0.6))
        text_colors.append('#000000')
        
        # Group studies by publisher and journal
        publisher_journals = defaultdict(lambda: defaultdict(list))
        for idx, row in self.gold_standard_df.iterrows():
            study_id = row.get('Study ID', f'Study_{idx}')
            journal = row.get('Journal/Conference', 'Unclassified Journal')
            publisher = row.get('Publisher', 'Unclassified Publisher')
            
            publisher_display = self.standardize_publisher_name(str(publisher).strip())
            journal_display = self._clean_journal_name(journal)
            
            retrieved_by_this_llm = self._is_retrieved_by_specific_llm(study_id, target_matches)
            llm_count = self._count_llms_retrieved_study(study_id)
            
            study_info = {
                'study_id': study_id,
                'title': row.get('Title', ''),
                'retrieved_by_this_llm': retrieved_by_this_llm,
                'llm_count': llm_count,
                'year': row.get('Year', '')
            }
            publisher_journals[publisher_display][journal_display].append(study_info)
        
        # Publisher level
        pub_color_idx = 0
        for publisher, journals in publisher_journals.items():
            pub_id = f"pub_{publisher}".replace(" ", "_").replace("&", "and")
            pub_total = sum(len(studies) for studies in journals.values())
            ids.append(pub_id)
            
            if pub_total > 1:
                labels.append(f"{publisher}\n({pub_total})")
            else:
                labels.append(publisher)
            parents.append(root_id)
            values.append(pub_total)
            
            base_pub_color = self.publisher_colors[pub_color_idx % len(self.publisher_colors)]
            publisher_opacity = self.PUBLISHER_OPACITY_LEVELS[pub_color_idx % len(self.PUBLISHER_OPACITY_LEVELS)]
            journal_opacity = self.JOURNAL_OPACITY_LEVELS[pub_color_idx % len(self.JOURNAL_OPACITY_LEVELS)]
            
            publisher_color = self._apply_opacity_to_color(base_pub_color, publisher_opacity)
            colors.append(publisher_color)
            text_colors.append(self._get_text_color_for_background(base_pub_color, publisher_opacity))
            
            # Journal level
            journal_idx = 0
            for journal, studies in journals.items():
                journal_id = f"jour_{publisher}_{journal}_{journal_idx}".replace(" ", "_").replace("/", "_").replace("&", "and")
                journal_total = len(studies)
                ids.append(journal_id)
                
                if journal_total > 1:
                    labels.append(f"{journal}\n({journal_total})")
                else:
                    labels.append(journal)
                parents.append(pub_id)
                values.append(journal_total)
                
                faded_journal_color = self._create_faded_journal_color(base_pub_color, 0.1)
                journal_color = self._apply_opacity_to_color(faded_journal_color, journal_opacity)
                colors.append(journal_color)
                text_colors.append(self._get_text_color_for_background(faded_journal_color, journal_opacity))
                
                journal_idx += 1
                
                # Individual study level
                for i, study in enumerate(studies):
                    study_id = f"study_{publisher}_{journal}_{i}_{study['study_id']}".replace(" ", "_").replace("/", "_").replace("&", "and")
                    study_label = ""
                    ids.append(study_id)
                    labels.append(study_label)
                    parents.append(journal_id)
                    values.append(1)
                    
                    if study['retrieved_by_this_llm']:
                        llm_count = study['llm_count']
                        study_color = self.retrieval_colors.get(llm_count, self.retrieval_colors[0])
                        colors.append(self._apply_opacity_to_color(study_color, self.STUDY_OPACITY))
                    else:
                        colors.append(self._apply_opacity_to_color(self.not_retrieved_color, self.STUDY_OPACITY))
                    text_colors.append('rgba(0, 0, 0, 0)')
            
            pub_color_idx += 1
        
        return {
            'ids': ids,
            'labels': labels,
            'parents': parents,
            'values': values,
            'colors': colors,
            'text_colors': text_colors
        }

    def generate_single_static_chart(self, llm_name=None, save_image=True):
        """Generate a single academic publication chart."""
        if llm_name and llm_name not in self.llm_matches:
            print(f"Error: LLM '{llm_name}' not found. Available: {list(self.llm_matches.keys())}")
            return None
        
        if llm_name:
            target_matches = self.llm_matches[llm_name]
            chart_title = f"{llm_name}"
        else:
            target_matches = self.all_normalized_matches
            chart_title = "Overall"
        
        print(f"Generating chart for {chart_title}: White (not retrieved), Gradient (retrieved)")
        
        sunburst_data = self._prepare_sunburst_data(target_matches, chart_title)
        
        fig = go.Figure(go.Sunburst(
            ids=sunburst_data['ids'],
            labels=sunburst_data['labels'],
            parents=sunburst_data['parents'],
            values=sunburst_data['values'],
            branchvalues="total",
            hoverinfo='text',
            hovertemplate='<b>%{label}</b><extra></extra>',
            marker=dict(
                colors=sunburst_data['colors'],
                line=dict(color='#FFFFFF', width=self.LINE_WIDTH)
            ),
            textinfo="label",
            textfont=dict(
                size=self.CENTER_TITLE_SIZE,
                color=sunburst_data['text_colors'],
                family="Arial, sans-serif"
            ),
            rotation=90,
            maxdepth=4,
            sort=False,
            leaf=dict(opacity=0.95),
            insidetextorientation='radial'
        ))
        
        fig.update_layout(
            width=1000,
            height=1000,
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)',
            margin=dict(t=20, b=20, l=20, r=20),
            showlegend=False
        )
        
        fig.show()
        
        if save_image:
            filename_base = f"retrieval_{llm_name.lower() if llm_name else 'overall'}_white_gradient"
            try:
                fig.write_html(f"{filename_base}.html")
                print(f"✓ Chart saved as: {filename_base}.html")
            except Exception as e:
                print(f"Note: Could not save HTML: {e}")
        
        return fig

    def _print_summary_statistics(self):
        """Print academic summary statistics for all LLMs."""
        print("\n" + "="*80)
        print("  SUNBURST CHART ANALYSIS SUMMARY")
        print("="*80)
        total_studies = len(self.gold_standard_df)
        
        print(f"Color Scheme:")
        print(f"  White: Study NOT retrieved by this specific LLM")
        print(f"  Colored: Study retrieved by this LLM (color shows total retrieval count)")
        print(f"\nRetrieval Color Gradient (for retrieved studies):")
        
        retrieval_labels = {
            0: "Never Retrieved by Any LLM (shouldn't appear if LLM retrieved it)",
            1: "Retrieved by 1 LLM",
            2: "Retrieved by 2 LLMs",
            3: "Retrieved by 3 LLMs",
            4: "Retrieved by 4 LLMs",
            5: "Retrieved by All 5 LLMs"
        }
        
        for count in range(6):
            color = self.retrieval_colors[count]
            print(f"  {color}: {retrieval_labels[count]}")
        
        print(f"\nIndividual LLM Retrieval Statistics:")
        for llm_name in ['Ai2 Finder', 'Consensus', 'Claude', 'ChatGPT', 'Gemini']:
            if llm_name in self.llm_matches:
                llm_retrieved = len([
                    study_id for study_id in self.study_mapping.keys()
                    if self._is_retrieved_by_specific_llm(study_id, self.llm_matches[llm_name])
                ])
                print(f"  {llm_name:10s}: Retrieved {llm_retrieved:3d}/{total_studies} studies ({llm_retrieved/total_studies*100:5.1f}%)")
        
        print("\n" + "="*80)

    def generate_sunburst_report(self, output_file='report_sunburst_charts.txt'):
        """Generate comprehensive text report for sunburst chart results with COMPLETE DATA"""
        with open(output_file, 'w') as f:
            f.write("=" * 80 + "\n")
            f.write("SUNBURST CHARTS ANALYSIS REPORT - COMPLETE DATA\n")
            f.write("=" * 80 + "\n\n")
            f.write(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total Gold Standard Studies: {len(self.gold_standard_df)}\n\n")
            
            # === SECTION 1: COMPLETE STUDY-BY-STUDY RETRIEVAL DATA ===
            f.write("=" * 80 + "\n")
            f.write("SECTION 1: COMPLETE STUDY-BY-STUDY RETRIEVAL DATA\n")
            f.write("=" * 80 + "\n\n")
            f.write("All Studies with Retrieval Status:\n")
            f.write("-" * 150 + "\n")
            f.write(f"{'Study ID':<40} | {'Publisher':<20} | {'Year':<6} | {'Ai2 Finder':<3} | {'Cons':<4} | {'GPT':<3} | {'Cld':<3} | {'Gem':<3} | {'Total':<5}\n")
            f.write("-" * 150 + "\n")
            
            for idx, row in self.gold_standard_df.iterrows():
                study_id = row.get('Study ID', f'Study_{idx}')
                publisher = row.get('Publisher', 'Unknown')
                publisher = self.standardize_publisher_name(str(publisher))[:20]
                year = row.get('Year', 'N/A')
                
                ai2_retrieved = 'Yes' if self._is_retrieved_by_specific_llm(study_id, self.llm_matches.get('Ai2 Finder', set())) else 'No'
                cons_retrieved = 'Yes' if self._is_retrieved_by_specific_llm(study_id, self.llm_matches.get('Consensus', set())) else 'No'
                gpt_retrieved = 'Yes' if self._is_retrieved_by_specific_llm(study_id, self.llm_matches.get('ChatGPT', set())) else 'No'
                cld_retrieved = 'Yes' if self._is_retrieved_by_specific_llm(study_id, self.llm_matches.get('Claude', set())) else 'No'
                gem_retrieved = 'Yes' if self._is_retrieved_by_specific_llm(study_id, self.llm_matches.get('Gemini', set())) else 'No'
                total_llms = self._count_llms_retrieved_study(study_id)
                
                f.write(f"{study_id:<40} | {publisher:<20} | {str(year):<6} | {ai2_retrieved:<3} | "
                       f"{cons_retrieved:<4} | {gpt_retrieved:<3} | {cld_retrieved:<3} | {gem_retrieved:<3} | {total_llms:<5}\n")
            
            # === SECTION 2: OVERALL RETRIEVAL STATISTICS ===
            f.write("\n\n" + "=" * 80 + "\n")
            f.write("SECTION 2: OVERALL RETRIEVAL STATISTICS\n")
            f.write("=" * 80 + "\n\n")
            
            retrieval_by_count = defaultdict(list)
            for study_id in self.study_mapping.keys():
                count = self._count_llms_retrieved_study(study_id)
                retrieval_by_count[count].append(study_id)
            
            f.write("Studies Retrieved by Number of LLMs:\n")
            f.write("-" * 60 + "\n")
            for count in range(6):
                num_studies = len(retrieval_by_count[count])
                percentage = (num_studies / len(self.study_mapping)) * 100
                label = "Never retrieved" if count == 0 else f"Retrieved by {count} LLM(s)"
                f.write(f"  {label:30s}: {num_studies:3d} studies ({percentage:5.1f}%)\n")
                
                if num_studies > 0 and num_studies <= 10:
                    f.write(f"    Studies:\n")
                    for study_id in retrieval_by_count[count]:
                        f.write(f"      {study_id}\n")
                elif num_studies > 10:
                    f.write(f"    First 10 studies:\n")
                    for study_id in retrieval_by_count[count][:10]:
                        f.write(f"      {study_id}\n")
                    f.write(f"    ... and {num_studies - 10} more\n")
            
            total_retrieved = sum(len(retrieval_by_count[i]) for i in range(1, 6))
            total_pct = (total_retrieved / len(self.study_mapping)) * 100
            f.write(f"\n  Total Retrieved by At Least 1 LLM: {total_retrieved:3d} studies ({total_pct:5.1f}%)\n")
            
            # === SECTION 3: COMPLETE INDIVIDUAL LLM PERFORMANCE ===
            f.write("\n\n" + "=" * 80 + "\n")
            f.write("SECTION 3: COMPLETE INDIVIDUAL LLM RETRIEVAL PERFORMANCE\n")
            f.write("=" * 80 + "\n\n")
            
            llm_names = ['Ai2Finder', 'Consensus', 'ChatGPT', 'Claude', 'Gemini']
            for llm_name in llm_names:
                if llm_name in self.llm_matches:
                    f.write(f"\n{llm_name}:\n")
                    f.write("-" * 80 + "\n")
                    
                    retrieved_studies = []
                    missed_studies = []
                    
                    for study_id in self.study_mapping.keys():
                        if self._is_retrieved_by_specific_llm(study_id, self.llm_matches[llm_name]):
                            retrieved_studies.append(study_id)
                        else:
                            missed_studies.append(study_id)
                    
                    f.write(f"Retrieved: {len(retrieved_studies)}/{len(self.study_mapping)} "
                           f"({len(retrieved_studies)/len(self.study_mapping)*100:.1f}%)\n")
                    f.write(f"Missed:    {len(missed_studies)}/{len(self.study_mapping)} "
                           f"({len(missed_studies)/len(self.study_mapping)*100:.1f}%)\n\n")
                    
                    f.write(f"Retrieved Studies:\n")
                    for study_id in retrieved_studies:
                        f.write(f"  {study_id}\n")
                    
                    f.write(f"\nMissed Studies:\n")
                    for study_id in missed_studies:
                        f.write(f"  {study_id}\n")
            
            # === SECTION 4: PAIRWISE LLM OVERLAP ANALYSIS ===
            f.write("\n\n" + "=" * 80 + "\n")
            f.write("SECTION 4: COMPLETE PAIRWISE LLM OVERLAP ANALYSIS\n")
            f.write("=" * 80 + "\n\n")
            
            for llm1, llm2 in iter_combinations(llm_names, 2):
                if llm1 in self.llm_matches and llm2 in self.llm_matches:
                    f.write(f"\n{llm1} vs {llm2}:\n")
                    f.write("-" * 80 + "\n")
                    
                    studies1 = set([sid for sid in self.study_mapping.keys()
                                   if self._is_retrieved_by_specific_llm(sid, self.llm_matches[llm1])])
                    studies2 = set([sid for sid in self.study_mapping.keys()
                                   if self._is_retrieved_by_specific_llm(sid, self.llm_matches[llm2])])
                    
                    both = studies1 & studies2
                    only1 = studies1 - studies2
                    only2 = studies2 - studies1
                    union = studies1 | studies2
                    jaccard = len(both) / len(union) if len(union) > 0 else 0
                    
                    f.write(f"Both retrieved:      {len(both):3d} studies\n")
                    f.write(f"Only {llm1:12s}: {len(only1):3d} studies\n")
                    f.write(f"Only {llm2:12s}: {len(only2):3d} studies\n")
                    f.write(f"Union:               {len(union):3d} studies\n")
                    f.write(f"Jaccard Similarity:  {jaccard:.4f}\n\n")
                    
                    if len(both) > 0:
                        f.write(f"Studies Retrieved by Both:\n")
                        for study_id in sorted(both):
                            f.write(f"  {study_id}\n")
                    
                    if len(only1) > 0:
                        f.write(f"\nStudies Retrieved Only by {llm1}:\n")
                        for study_id in sorted(only1):
                            f.write(f"  {study_id}\n")
                    
                    if len(only2) > 0:
                        f.write(f"\nStudies Retrieved Only by {llm2}:\n")
                        for study_id in sorted(only2):
                            f.write(f"  {study_id}\n")
            
            # === SECTION 5: COMPLETE PUBLISHER-LEVEL ANALYSIS ===
            f.write("\n\n" + "=" * 80 + "\n")
            f.write("SECTION 5: COMPLETE RETRIEVAL BY PUBLISHER\n")
            f.write("=" * 80 + "\n\n")
            
            publisher_stats = defaultdict(lambda: {'studies': [], 'retrieved': [], 'missed': []})
            
            for idx, row in self.gold_standard_df.iterrows():
                study_id = row.get('Study ID', f'Study_{idx}')
                publisher = row.get('Publisher', 'Unknown')
                publisher = self.standardize_publisher_name(str(publisher))
                
                publisher_stats[publisher]['studies'].append(study_id)
                count = self._count_llms_retrieved_study(study_id)
                if count > 0:
                    publisher_stats[publisher]['retrieved'].append(study_id)
                else:
                    publisher_stats[publisher]['missed'].append(study_id)
            
            sorted_publishers = sorted(publisher_stats.items(),
                                       key=lambda x: len(x[1]['studies']),
                                       reverse=True)
            
            f.write("Complete Publisher Breakdown:\n")
            f.write("-" * 80 + "\n")
            for publisher, stats in sorted_publishers:
                total = len(stats['studies'])
                retrieved = len(stats['retrieved'])
                missed = len(stats['missed'])
                rate = (retrieved / total * 100) if total > 0 else 0
                
                f.write(f"\n{publisher}:\n")
                f.write(f"  Total Studies:     {total:3d}\n")
                f.write(f"  Retrieved:         {retrieved:3d} ({rate:5.1f}%)\n")
                f.write(f"  Missed:            {missed:3d} ({100-rate:5.1f}%)\n")
                f.write(f"  All Studies:\n")
                
                for study_id in stats['studies']:
                    retrieval_status = "Retrieved" if study_id in stats['retrieved'] else "Missed"
                    llm_count = self._count_llms_retrieved_study(study_id)
                    f.write(f"    {study_id:<40} | {retrieval_status:<10} | {llm_count} LLMs\n")
            
            # === SECTION 6: COMPLETE TEMPORAL ANALYSIS ===
            f.write("\n\n" + "=" * 80 + "\n")
            f.write("SECTION 6: COMPLETE RETRIEVAL BY PUBLICATION YEAR\n")
            f.write("=" * 80 + "\n\n")
            
            year_stats = defaultdict(lambda: {'studies': [], 'retrieved': [], 'missed': []})
            
            for idx, row in self.gold_standard_df.iterrows():
                study_id = row.get('Study ID', f'Study_{idx}')
                if 'Year' in row and pd.notna(row['Year']):
                    try:
                        year = int(row['Year'])
                        year_stats[year]['studies'].append(study_id)
                        count = self._count_llms_retrieved_study(study_id)
                        if count > 0:
                            year_stats[year]['retrieved'].append(study_id)
                        else:
                            year_stats[year]['missed'].append(study_id)
                    except (ValueError, TypeError):
                        pass
            
            sorted_years = sorted(year_stats.items())
            
            f.write("Complete Year-by-Year Breakdown:\n")
            f.write("-" * 80 + "\n")
            for year, stats in sorted_years:
                total = len(stats['studies'])
                retrieved = len(stats['retrieved'])
                missed = len(stats['missed'])
                rate = (retrieved / total * 100) if total > 0 else 0
                
                f.write(f"\n{year}:\n")
                f.write(f"  Total Studies:     {total:3d}\n")
                f.write(f"  Retrieved:         {retrieved:3d} ({rate:5.1f}%)\n")
                f.write(f"  Missed:            {missed:3d} ({100-rate:5.1f}%)\n")
                f.write(f"  All Studies:\n")
                
                for study_id in stats['studies']:
                    retrieval_status = "Retrieved" if study_id in stats['retrieved'] else "Missed"
                    llm_count = self._count_llms_retrieved_study(study_id)
                    f.write(f"    {study_id:<40} | {retrieval_status:<10} | {llm_count} LLMs\n")
            
            # === SECTION 7: COMPLETE UNIQUE CONTRIBUTIONS ===
            f.write("\n\n" + "=" * 80 + "\n")
            f.write("SECTION 7: COMPLETE UNIQUE LLM CONTRIBUTIONS\n")
            f.write("=" * 80 + "\n\n")
            
            for llm_name in llm_names:
                if llm_name in self.llm_matches:
                    f.write(f"\n{llm_name} Unique Contributions:\n")
                    f.write("-" * 80 + "\n")
                    
                    studies_this_llm = set([
                        sid for sid in self.study_mapping.keys()
                        if self._is_retrieved_by_specific_llm(sid, self.llm_matches[llm_name])
                    ])
                    
                    studies_other_llms = set()
                    for other_llm in llm_names:
                        if other_llm != llm_name and other_llm in self.llm_matches:
                            studies_other_llms.update([
                                sid for sid in self.study_mapping.keys()
                                if self._is_retrieved_by_specific_llm(sid, self.llm_matches[other_llm])
                            ])
                    
                    unique_studies = studies_this_llm - studies_other_llms
                    shared_studies = studies_this_llm & studies_other_llms
                    
                    unique_count = len(unique_studies)
                    shared_count = len(shared_studies)
                    total_count = len(studies_this_llm)
                    
                    unique_pct = (unique_count / total_count * 100) if total_count > 0 else 0
                    shared_pct = (shared_count / total_count * 100) if total_count > 0 else 0
                    
                    f.write(f"Total Retrieved:     {total_count:3d}\n")
                    f.write(f"Unique:              {unique_count:3d} ({unique_pct:5.1f}%)\n")
                    f.write(f"Shared with Others:  {shared_count:3d} ({shared_pct:5.1f}%)\n\n")
                    
                    if unique_count > 0:
                        f.write(f"Unique Studies (Retrieved ONLY by {llm_name}):\n")
                        for study_id in sorted(unique_studies):
                            f.write(f"  {study_id}\n")
                    
                    if shared_count > 0:
                        f.write(f"\nShared Studies (Retrieved by {llm_name} and at least one other LLM):\n")
                        for study_id in sorted(shared_studies):
                            other_llms_retrieved = []
                            for other_llm in llm_names:
                                if other_llm != llm_name and other_llm in self.llm_matches:
                                    if self._is_retrieved_by_specific_llm(study_id, self.llm_matches[other_llm]):
                                        other_llms_retrieved.append(other_llm)
                            f.write(f"  {study_id:<40} | Also by: {', '.join(other_llms_retrieved)}\n")
            
            # === SECTION 8: JOURNAL/CONFERENCE ANALYSIS ===
            f.write("\n\n" + "=" * 80 + "\n")
            f.write("SECTION 8: COMPLETE JOURNAL/CONFERENCE ANALYSIS\n")
            f.write("=" * 80 + "\n\n")
            
            journal_stats = defaultdict(lambda: {'studies': [], 'retrieved': [], 'missed': []})
            
            for idx, row in self.gold_standard_df.iterrows():
                study_id = row.get('Study ID', f'Study_{idx}')
                journal = row.get('Journal/Conference', 'Unknown')
                journal = str(journal).strip()
                
                journal_stats[journal]['studies'].append(study_id)
                count = self._count_llms_retrieved_study(study_id)
                if count > 0:
                    journal_stats[journal]['retrieved'].append(study_id)
                else:
                    journal_stats[journal]['missed'].append(study_id)
            
            sorted_journals = sorted(journal_stats.items(),
                                    key=lambda x: len(x[1]['studies']),
                                    reverse=True)
            
            f.write("Complete Journal/Conference Breakdown:\n")
            f.write("-" * 80 + "\n")
            for journal, stats in sorted_journals:
                total = len(stats['studies'])
                retrieved = len(stats['retrieved'])
                missed = len(stats['missed'])
                rate = (retrieved / total * 100) if total > 0 else 0
                
                f.write(f"\n{journal}:\n")
                f.write(f"  Total Studies:     {total:3d}\n")
                f.write(f"  Retrieved:         {retrieved:3d} ({rate:5.1f}%)\n")
                f.write(f"  Missed:            {missed:3d} ({100-rate:5.1f}%)\n")
                f.write(f"  All Studies:\n")
                
                for study_id in stats['studies']:
                    retrieval_status = "Retrieved" if study_id in stats['retrieved'] else "Missed"
                    llm_count = self._count_llms_retrieved_study(study_id)
                    f.write(f"    {study_id:<40} | {retrieval_status:<10} | {llm_count} LLMs\n")
            
            # === SECTION 9: CONSISTENCY ANALYSIS ===
            f.write("\n\n" + "=" * 80 + "\n")
            f.write("SECTION 9: RETRIEVAL CONSISTENCY ANALYSIS\n")
            f.write("=" * 80 + "\n\n")
            
            consistent_studies = [sid for sid in self.study_mapping.keys()
                                 if self._count_llms_retrieved_study(sid) == 5]
            f.write(f"Studies Retrieved by ALL 5 LLMs ({len(consistent_studies)} studies):\n")
            f.write("-" * 80 + "\n")
            for study_id in consistent_studies:
                f.write(f"  {study_id}\n")
            
            never_retrieved = [sid for sid in self.study_mapping.keys()
                              if self._count_llms_retrieved_study(sid) == 0]
            f.write(f"\n\nStudies NEVER Retrieved by Any LLM ({len(never_retrieved)} studies):\n")
            f.write("-" * 80 + "\n")
            for study_id in never_retrieved:
                f.write(f"  {study_id}\n")
            
            for count in range(1, 5):
                partial_studies = [sid for sid in self.study_mapping.keys()
                                  if self._count_llms_retrieved_study(sid) == count]
                f.write(f"\n\nStudies Retrieved by Exactly {count} LLM(s) ({len(partial_studies)} studies):\n")
                f.write("-" * 80 + "\n")
                for study_id in partial_studies:
                    llms_retrieved = []
                    for llm_name in llm_names:
                        if llm_name in self.llm_matches:
                            if self._is_retrieved_by_specific_llm(study_id, self.llm_matches[llm_name]):
                                llms_retrieved.append(llm_name)
                    f.write(f"  {study_id:<40} | Retrieved by: {', '.join(llms_retrieved)}\n")
            
            # === SECTION 10: KEY INSIGHTS ===
            f.write("\n\n" + "=" * 80 + "\n")
            f.write("SECTION 10: KEY INSIGHTS\n")
            f.write("=" * 80 + "\n\n")
            
            llm_performance = []
            for llm_name in llm_names:
                if llm_name in self.llm_matches:
                    retrieved_count = len([
                        study_id for study_id in self.study_mapping.keys()
                        if self._is_retrieved_by_specific_llm(study_id, self.llm_matches[llm_name])
                    ])
                    llm_performance.append({
                        'llm': llm_name,
                        'retrieved': retrieved_count,
                        'percentage': (retrieved_count / len(self.study_mapping)) * 100
                    })
            
            llm_performance.sort(key=lambda x: x['retrieved'], reverse=True)
            
            f.write("LLM Performance Ranking:\n")
            f.write("-" * 60 + "\n")
            for rank, perf in enumerate(llm_performance, 1):
                f.write(f"  {rank}. {perf['llm']:12s}: {perf['retrieved']:3d}/{len(self.study_mapping)} "
                       f"({perf['percentage']:5.1f}%)\n")
            
            all_retrieved = set()
            for llm_name in llm_names:
                if llm_name in self.llm_matches:
                    all_retrieved.update([
                        study_id for study_id in self.study_mapping.keys()
                        if self._is_retrieved_by_specific_llm(study_id, self.llm_matches[llm_name])
                    ])
            
            total_coverage = len(all_retrieved)
            coverage_pct = (total_coverage / len(self.study_mapping)) * 100
            f.write(f"\nOverall Coverage (At Least 1 LLM): {total_coverage}/{len(self.study_mapping)} "
                   f"({coverage_pct:.1f}%)\n")
            
            max_jaccard = 0
            most_similar_pair = None
            for llm1, llm2 in iter_combinations(llm_names, 2):
                if llm1 in self.llm_matches and llm2 in self.llm_matches:
                    studies1 = set([sid for sid in self.study_mapping.keys()
                                   if self._is_retrieved_by_specific_llm(sid, self.llm_matches[llm1])])
                    studies2 = set([sid for sid in self.study_mapping.keys()
                                   if self._is_retrieved_by_specific_llm(sid, self.llm_matches[llm2])])
                    intersection = len(studies1 & studies2)
                    union = len(studies1 | studies2)
                    jaccard = intersection / union if union > 0 else 0
                    if jaccard > max_jaccard:
                        max_jaccard = jaccard
                        most_similar_pair = (llm1, llm2)
            
            if most_similar_pair:
                f.write(f"\nMost Similar LLM Pair: {most_similar_pair[0]} & {most_similar_pair[1]} "
                       f"(Jaccard = {max_jaccard:.4f})\n")
            
            best_publisher = None
            best_rate = 0
            for publisher, stats in publisher_stats.items():
                total = len(stats['studies'])
                if total >= 3:
                    retrieved = len(stats['retrieved'])
                    rate = retrieved / total if total > 0 else 0
                    if rate > best_rate:
                        best_rate = rate
                        best_publisher = publisher
            
            if best_publisher:
                f.write(f"\nBest Retrieved Publisher (min 3 studies): {best_publisher} "
                       f"({best_rate*100:.1f}% retrieval rate)\n")
            
            worst_year = None
            worst_rate = 1.0
            for year, stats in year_stats.items():
                total = len(stats['studies'])
                if total >= 3:
                    retrieved = len(stats['retrieved'])
                    rate = retrieved / total if total > 0 else 0
                    if rate < worst_rate:
                        worst_rate = rate
                        worst_year = year
            
            if worst_year:
                f.write(f"\nMost Challenging Year (min 3 studies): {worst_year} "
                       f"({worst_rate*100:.1f}% retrieval rate)\n")
            
            f.write("\n" + "=" * 80 + "\n")
            f.write("END OF REPORT\n")
            f.write("=" * 80 + "\n")
        
        print(f"✓ Complete sunburst charts report saved to: {output_file}")
        return output_file

class LLMAnalysisBase:
    def __init__(self):
        self.llm_mapping = {
            'ai2': 'Ai2 Finder', 'con': 'Consensus', 'cons': 'Consensus',
            'gpt': 'ChatGPT', 'cld': 'Claude', 'gem': 'Gemini'
        }
        self.recalled_files = {}
        self.gold_standard_df = None

    def normalize_identifier_consistent(self, identifier):
        if not identifier or pd.isna(identifier):
            return ""
        identifier = str(identifier).strip().lower()
        if identifier.startswith('doi:'):
            identifier = identifier.replace('doi:', '').strip()
        elif identifier.startswith('https://doi.org/'):
            identifier = identifier.replace('https://doi.org/', '').strip()
        return identifier

    def parse_filename(self, filename):
        base_name = filename.split('.')[0]
        is_recalled = base_name.endswith(('-R', '_R', 'R'))
        if is_recalled:
            base_name = re.sub(r'(_R|-R|R)$', '', base_name)
        
        patterns = [
            r'([a-zA-Z0-9]+)-(\d)(\d)(\d)',
            r'([a-zA-Z0-9]+)_(\d)(\d)(\d)',
            r'([a-zA-Z0-9]+)(\d)(\d)(\d)'
        ]
        
        for pattern in patterns:
            match = re.match(pattern, base_name, re.IGNORECASE)
            if match:
                llm_code = match.group(1).lower()
                if llm_code == 'gemini':
                    llm_code = 'gem'
                if llm_code in self.llm_mapping:
                    return {'llm': self.llm_mapping[llm_code], 'is_recalled': is_recalled}
        return None

    def load_files_from_path(self, base_path="."):
        base_dir = Path(base_path).expanduser()
        if not base_dir.exists():
            return False
        
        all_csvs = list(base_dir.glob('*.csv'))
        
        gold_standard_files = [
            f for f in all_csvs
            if 'gold' in f.name.lower() and 'standard' in f.name.lower()
        ]
        
        if gold_standard_files:
            print(f"Found gold standard file: {gold_standard_files[0].name}")
            self.gold_standard_df = pd.read_csv(gold_standard_files[0])
        else:
            print("Warning: Gold standard file not found.")
        
        for filepath in all_csvs:
            if 'gold' in filepath.name.lower() and 'standard' in filepath.name.lower():
                continue
            parsed = self.parse_filename(filepath.name)
            if parsed and parsed['is_recalled']:
                self.recalled_files[filepath.name] = {
                    'path': str(filepath),
                    'llm': parsed['llm']
                }
        
        print(f"Loaded {len(self.recalled_files)} recalled files")
        return True

class TraditionalCombinatorialAnalysis:
    def __init__(self, base_analyzer):
        self.analyzer = base_analyzer
        self.gold_standard_count = len(base_analyzer.gold_standard_df) if base_analyzer.gold_standard_df is not None else 0

    def generate_combinations(self):
        print("Generating simplified combinations for visualization example...")
        all_matches = set()
        
        for combo in self.analyzer.recalled_files.values():
            try:
                df = pd.read_csv(combo['path'])
                identifier_cols = ['DOI', 'Study name', 'Title', 'LLM Title']
                for col in identifier_cols:
                    if col in df.columns:
                        matches = df[col].dropna().astype(str).apply(
                            self.analyzer.normalize_identifier_consistent
                        )
                        all_matches.update(matches)
                        break
            except Exception:
                pass
        
        return [{'matches': all_matches}]

if __name__ == "__main__":
    print("=== Sunburst Charts: White (Not Retrieved) + Gradient (Retrieved) ===")
    
    data_path = "~/Desktop/Pooled Std Recall Runs"
    
    print(f"\n[1/3] Loading data from '{data_path}'...")
    base_analyzer = LLMAnalysisBase()
    
    if not base_analyzer.load_files_from_path(data_path):
        print(f"ERROR: Data path not found. Please check the path: {data_path}")
    else:
        if base_analyzer.gold_standard_df is None:
            print("ERROR: Gold standard file was not loaded. Cannot proceed.")
        else:
            print("✓ Data loaded successfully.")
            
            print("\n[2/3] Generating combination data...")
            combinatorial_analyzer = TraditionalCombinatorialAnalysis(base_analyzer)
            top_combinations_placeholder = combinatorial_analyzer.generate_combinations()
            print("✓ Combination data prepared.")
            
            print("\n[3/3] Initializing visualizer...")
            try:
                visualizer = StaticSunburstChartVisualizer(
                    gold_standard_df=base_analyzer.gold_standard_df,
                    recalled_files=base_analyzer.recalled_files,
                    top_combinations=top_combinations_placeholder
                )
                
                # Generate charts
                print("\n--- Creating sunburst charts ---")
                
                # Overall chart
                visualizer.generate_single_static_chart(llm_name=None, save_image=True)
                
                # Individual LLM charts
                for llm in ['Ai2 Finder', 'Consensus', 'Claude', 'ChatGPT', 'Gemini']:
                    if llm in visualizer.llm_matches and len(visualizer.llm_matches[llm]) > 0:
                        visualizer.generate_single_static_chart(llm_name=llm, save_image=True)
                
                print("\n✓ Visualizations complete!")
                print("✓ White: NOT retrieved by this LLM")
                print("✓ Gradient (Red→Green): Retrieved by this LLM (colored by total retrieval count)")
                
                # Print summary statistics
                visualizer._print_summary_statistics()
                
                # Generate comprehensive report
                print("\n[4/4] Generating comprehensive report...")
                visualizer.generate_sunburst_report('report_sunburst_charts.txt')
                print("✓ Report generation complete!")
                
            except ValueError as e:
                print(f"ERROR: Could not initialize visualizer. {e}")
            except Exception as e:
                print(f"An unexpected error occurred: {e}")

## Cell 09 — DENDROGRAM HEATMAP

*Figure 5 + never-retrieved list + binary_retrieval_matrix.csv*


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch, Rectangle
import matplotlib.patches as mpatches
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from typing import Dict, List, Tuple, Set
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def create_dendrogram_heatmap():
    """Create a dendrogram-based hierarchical heatmap with asymmetric color palette"""
    # ========== DENDROGRAM CONTROL CONFIGURATION ==========
    DENDROGRAM_CONFIG = {
        # Row dendrogram (left side - LLM hierarchy)
        'show_row_dendrogram': True,
        'dendrogram_ratio': 0.10,
        'linkage_method': 'complete',
        'llm_distance': 6.2,
        'strategy_distance': 4.0,
        'variation_distance': 1.0,
        'tree_linewidth': 1.5,
        'tree_color': 'black',
        'force_perfect_hierarchy': True,
        'variation_fontsize': 11,
        'strategy_fontsize': 14,
        'llm_fontsize': 20,
        'strategy_inter_y_spacing': -5.0,
        'llm_inter_y_spacing': -4.0,
        'variation_inter_y_spacing': -0.4,
        'variation_x_offset': 0.1,
        'strategy_x_offset': 1.3,
        'llm_x_offset': 1.1,
        'variation_y_offset': 10,
        'strategy_y_offset': 0.0,
        'llm_y_offset': 25,
        'merge_point_tolerance': 1.0,
        # Column dendrogram (top - Publisher-Journal-Study hierarchy)
        'show_col_dendrogram': True,
        'col_dendrogram_ratio': 0.22,
        'publisher_height': 6.5,
        'journal_height': 4.0,
        'study_height': 0.4,
        # Column dendrogram label controls
        'study_fontsize': 0,
        'journal_fontsize': 12,
        'publisher_fontsize': 20,
        # Column dendrogram spacing controls
        'journal_inter_x_spacing': 0,
        'publisher_inter_x_spacing': 0,
        'study_inter_x_spacing': 0,
        # INCREMENTAL: Journal-level offset
        'journal_multi_study_x_offset': 0.7,
        # MANUAL: Journal-level x-offsets
        'journal_manual_x_offsets': {
            'BMC Medical Informat...': 14,
            'AMIA Symposium Proc.': 10,
            'JAMIA': 45,
            'default': None
        },
        # MANUAL: Publisher-level x-offsets
        'publisher_manual_x_offsets': {
            'Wolters': 0, 'Taylor': 0, 'Springer': 32, 'Elsevier': 30,
            'IEEE': 18, 'JMIR': 0, 'OUP': 60, 'MDPI': 10, 'Wiley': 10,
            'Sage': 0, 'BMJ': 0, 'AMIA': 10, 'Frontiers': 0, 'Nature': 0,
            'PLOS': 5, 'Cambridge': 0, 'Lippincott': 0, 'Karger': 0,
            'Thieme': 0, 'Mary Ann Liebert': 0,
            'American Medical Association': 0, 'IOS Pre.': -2,
            'SciTePre': 0, 'Unknown': 0, 'default': 0
        },
        # Column dendrogram offset controls
        'study_x_offset': 0, 'study_y_offset': 0,
        'journal_x_offset': 1, 'journal_y_offset': -3.5,
        'publisher_x_offset': 0, 'publisher_y_offset': -2.2,
        # Column dendrogram label rotation
        'journal_label_rotation': 90,
        'publisher_label_rotation': 90,
        # Column dendrogram merge point tolerance
        'col_merge_point_tolerance': 0.5,
    }
    
    # ========== SEPARATOR LINE CONFIGURATION - MUCH THINNER ==========
    SEPARATOR_CONFIG = {
        # Horizontal separators (between LLM groups) - REDUCED
        'llm_separator_width': 0.4,           # Was 1.5, now 0.8
        'strategy_separator_width': 0,      # Was 0.5, now 0.3
        'variation_separator_width': 0,
        'separator_color': 'white',
        'extend_llm_separator_to_colors': True,
        'extend_strategy_separator_to_colors': False,
        'extend_variation_separator_to_colors': False,
        'extend_llm_separator_to_sidebar': True,
        
        # Vertical separators (between study groups) - REDUCED
        'show_vertical_separators': True,
        
        # Three-level vertical separator hierarchy - MUCH THINNER
        'publisher_separator_width': 0.2,     # Was 1.5, now 0.8
        'journal_separator_width': 0.1,       # Was 0.7, now 0.4
        'study_separator_width': 0.01,        # Was 0.25, now 0.15
        
        'publisher_separator_color': 'white',
        'journal_separator_color': 'white',
        'study_separator_color': 'white',
        'vertical_separator_color': 'white',
        
        # Alpha transparency for visual hierarchy - INCREASED TRANSPARENCY
        'publisher_separator_alpha': 0.3,     # Was 1.0, now 0.9
        'journal_separator_alpha': 0.2,      # Was 0.85, now 0.75
        'study_separator_alpha': 0.01,         # Was 0.6, now 0.5
    }
    
    # ========== SIDEBAR ADJUSTMENT CONTROLS ==========
    SIDEBAR_CONFIG = {
        'sidebar_width': 0.15,
        'sidebar_gap': 0.00,
        'bar_alpha': 1,
        'use_llm_colors': True,
        'show_vertical_grid': True,
        'vertical_grid_positions': [25, 50, 75, 100],
        'vertical_grid_color': 'white',
        'vertical_grid_linewidth': 1.2,
        'vertical_grid_alpha': 0.5,
        'show_minor_vertical_grid': True,
        'minor_vertical_grid_positions': [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95],
        'minor_vertical_grid_color': 'white',
        'minor_vertical_grid_linewidth': 0.8,
        'minor_vertical_grid_alpha': 0.35,
        'show_labels': True,
        'label_fontsize': 12,
        'label_x_position': 6,
        'label_rotation': 0,
        'label_color': 'white',
        'label_fontweight': 'bold',
    }
    
    # ========== ROW COLORS GAP CONFIGURATION ==========
    ROW_COLORS_CONFIG = {
        'gap_after_colors': 0.00,
    }
    
    # ========== LEGEND CONFIGURATION ==========
    LEGEND_CONFIG = {
        'show_legend': True,
        'title_fontsize': 22,
        'subtitle_fontsize': 18,
        'item_fontsize': 16,
        'box_facecolor': 'white',
        'box_edgecolor': 'black',
        'box_linewidth': 1.5,
        'box_alpha': 0.95,
        'columnspacing': 0.8,
        'handlelength': 1.2,
        'handleheight': 0.8,
    }
    
    # ========== MAIN CONFIGURATION ==========
    CONFIG = {
        'llms': ['Ai2 Finder', 'Consensus', 'Claude', 'ChatGPT', 'Gemini'],
        'strategies': ['1', '2', '3', '4', '5'],
        'variations': ['0', '1', '2'],
        'strategy_names': {
            '1': 'Zero-Shot', '2': 'Few-Shot', '3': 'Persona',
            '4': 'CoT', '5': 'PICO'
        },
    }
    
    def create_strategic_gradient_colormap(config=None):
        """Create ASYMMETRIC gradient"""
        from matplotlib.colors import LinearSegmentedColormap
        
        color_map = {
            0: '#0a4660',
            1: '#1074a1',
            2: '#17a3e2',
            3: '#54bded',
            4: '#94d6f3',
            5: '#f2b307',
            6: '#f9ca48',
            7: '#fbde8d',
        }
        
        positions = np.linspace(0, 1, 8)
        
        def hex_to_rgb(hex_color):
            hex_color = hex_color.lstrip('#')
            return tuple(int(hex_color[i:i+2], 16)/255.0 for i in (0, 2, 4))
        
        def rgb_to_hex(rgb):
            return '#{:02x}{:02x}{:02x}'.format(
                int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255)
            )
        
        colors_list = [hex_to_rgb(color_map[i]) for i in range(8)]
        
        cmap = LinearSegmentedColormap.from_list(
            "asymmetric_retrieval",
            list(zip(positions, colors_list))
        )
        
        print("\n🎨 Asymmetric Retrieval Gradient (Distinct → Smooth):")
        print("=" * 70)
        labels = [
            "Never retrieved",
            "1 other LLM",
            "2 other LLMs",
            "3 other LLMs",
            "4 other LLMs",
            "Other strategy",
            "Other variation",
            "This prompt"
        ]
        for i, (pos, color) in enumerate(zip(positions, colors_list)):
            hex_color = rgb_to_hex(color)
            contrast = "DISTINCT" if i <= 4 else "SMOOTH & BRIGHT"
            print(f"  Value {i}: {hex_color}  {labels[i]:25s} [{contrast}]")
        print("=" * 70)
        
        return cmap, colors_list
    
    def get_text_color_for_background(hex_color):
        """Determine if white or black text should be used based on background color."""
        hex_color = hex_color.lstrip('#')
        r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
        r, g, b = r/255.0, g/255.0, b/255.0
        
        def adjust(color):
            if color <= 0.03928:
                return color / 12.92
            return ((color + 0.055) / 1.055) ** 2.4
        
        r, g, b = adjust(r), adjust(g), adjust(b)
        luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        return 'white' if luminance < 0.5 else 'black'
    
    def standardize_publisher_name(publisher_name):
        """Standardize publisher names for consistency"""
        publisher = str(publisher_name)
        if 'Wolters Kluwer' in publisher: return 'Wolters'
        if 'Taylor & Francis' in publisher or 'Dove Medical Press' in publisher: return 'Taylor'
        if 'Springer' in publisher: return 'Springer'
        if 'Elsevier' in publisher: return 'Elsevier'
        if 'IEEE' in publisher: return 'IEEE'
        if 'JMIR' in publisher: return 'JMIR'
        if 'SCITEPRESS' in publisher: return 'SciTePre'
        if 'IOS Press' in publisher: return 'IOS Pre.'
        if 'Oxford University Press' in publisher or 'OUP' in publisher: return 'OUP'
        if 'MDPI' in publisher: return 'MDPI'
        if 'Wiley' in publisher: return 'Wiley'
        if 'Sage' in publisher: return 'Sage'
        if 'BMJ' in publisher: return 'BMJ'
        if 'AMIA' in publisher: return 'AMIA'
        if 'Frontiers' in publisher: return 'Frontiers'
        if 'Nature Publishing' in publisher: return 'Nature'
        if 'PLOS' in publisher or 'Public Library of Science' in publisher: return 'PLOS'
        if 'Cambridge University Press' in publisher: return 'Cambridge'
        if 'Lippincott' in publisher: return 'Lippincott'
        if 'Karger' in publisher: return 'Karger'
        if 'Thieme' in publisher: return 'Thieme'
        if 'Mary Ann Liebert' in publisher: return 'Mary Ann Liebert'
        if 'American Medical Association' in publisher or 'AMA' in publisher: return 'American Medical Association'
        return publisher
    
    def clean_journal_name(journal_name):
        """Clean and shorten journal names for display"""
        import re
        journal = str(journal_name).strip()
        journal = re.sub(r'\s*\((Journal|Conference.*?)\)\s*$', '', journal)
        journal = re.sub(r'\s*(Journal|Conference)\s*$', '', journal)
        journal = journal.replace('International', 'Int.')
        journal = journal.replace('Proceedings', 'Proc.')
        if len(journal) > 20:
            journal = journal[:20] + "..."
        return journal
    
    def get_study_year(study_row):
        """Extract year from study row, with fallback handling"""
        if 'Year' in study_row.columns:
            year = study_row['Year'].iloc[0]
            if pd.notna(year):
                try:
                    return int(year)
                except (ValueError, TypeError):
                    pass
        return 9999
    
    def extract_study_data():
        """Extract and process study data WITH CHRONOLOGICAL HIERARCHICAL ORDERING"""
        try:
            analyzer = base_results['analyzer']
            study_mapping = analyzer.create_study_identifier_mapping()
            
            print("🔍 Building Publisher-Journal-Study hierarchy (chronological ordering)...")
            
            publisher_journals = defaultdict(lambda: defaultdict(list))
            
            for study_id in study_mapping.keys():
                study_row = analyzer.gold_standard_df[
                    analyzer.gold_standard_df['Study ID'] == study_id
                ]
                
                if not study_row.empty:
                    publisher = study_row['Publisher'].iloc[0] if 'Publisher' in study_row.columns else 'Unknown Publisher'
                    journal = study_row['Journal/Conference'].iloc[0] if 'Journal/Conference' in study_row.columns else 'Unknown Journal'
                    year = get_study_year(study_row)
                    
                    publisher = standardize_publisher_name(str(publisher))
                    
                    publisher_journals[publisher][journal].append({
                        'study_id': study_id,
                        'year': year
                    })
            
            sorted_studies = []
            publisher_years = {}
            for publisher, journals in publisher_journals.items():
                all_years = []
                for journal, studies in journals.items():
                    all_years.extend([s['year'] for s in studies if s['year'] != 9999])
                if all_years:
                    publisher_years[publisher] = min(all_years)
                else:
                    publisher_years[publisher] = 9999
            
            sorted_publishers = sorted(publisher_journals.keys(), key=lambda p: publisher_years[p])
            
            for publisher in sorted_publishers:
                journals = publisher_journals[publisher]
                
                journal_years = {}
                for journal, studies in journals.items():
                    years = [s['year'] for s in studies if s['year'] != 9999]
                    if years:
                        journal_years[journal] = min(years)
                    else:
                        journal_years[journal] = 9999
                
                sorted_journals = sorted(journals.keys(), key=lambda j: journal_years[j])
                
                for journal in sorted_journals:
                    studies_in_journal = sorted(journals[journal], key=lambda s: s['year'])
                    sorted_studies.extend([s['study_id'] for s in studies_in_journal])
            
            print(f"✓ Chronologically ordered {len(sorted_studies)} studies")
            
            study_labels = []
            for study_id in sorted_studies:
                first_author = study_id.split(' ')[0].split(',')[0]
                
                if 'Year' in analyzer.gold_standard_df.columns:
                    year_row = analyzer.gold_standard_df[
                        analyzer.gold_standard_df['Study ID'] == study_id
                    ]
                    if not year_row.empty:
                        year = year_row['Year'].iloc[0]
                        if pd.notna(year) and str(year) != 'N/A':
                            try:
                                label = f"{first_author} ({int(year)})"
                            except (ValueError, TypeError):
                                label = first_author
                        else:
                            label = first_author
                    else:
                        label = first_author
                else:
                    label = first_author
                
                study_labels.append(label)
            
            return sorted_studies, study_labels, study_mapping
        
        except Exception as e:
            print(f"❌ Error in extract_study_data: {e}")
            raise
    
    def build_combination_data_with_runs(analyzer, study_mapping):
        """Build combination data structure with run-based confidence"""
        try:
            combinations_data = []
            
            for llm in CONFIG['llms']:
                for strategy in CONFIG['strategies']:
                    for variation in CONFIG['variations']:
                        studies_by_run = {}
                        
                        for filename, info in analyzer.recalled_files.items():
                            if (info['llm'] == llm and
                                info['strategy'] == strategy and
                                info['variation'] == variation):
                                
                                try:
                                    df = analyzer.load_csv_file_content(info['path'])
                                    if not df.empty:
                                        raw_matches, _ = analyzer.extract_studies_from_recalled_files(df)
                                        
                                        run_matches = set()
                                        for match in raw_matches:
                                            normalized = analyzer.normalize_identifier_consistent(match)
                                            if normalized:
                                                run_matches.add(normalized)
                                        
                                        studies_by_run[info['run']] = run_matches
                                
                                except Exception as e:
                                    print(f"⚠ Warning: Error processing {filename}: {e}")
                                    continue
                        
                        strategy_name = CONFIG['strategy_names'].get(strategy, f"S{strategy}")
                        
                        combinations_data.append({
                            'LLM': llm,
                            'Strategy': strategy,
                            'StrategyName': strategy_name,
                            'Variation': variation,
                            'studies_by_run': studies_by_run
                        })
            
            print(f"✓ Built {len(combinations_data)} combinations")
            return combinations_data
        
        except Exception as e:
            print(f"❌ Error in build_combination_data_with_runs: {e}")
            raise
    
    def analyze_detailed_retrieval_patterns(sorted_studies, study_mapping, combinations_data):
        """Analyze retrieval patterns at multiple granularity levels"""
        try:
            study_retrieval_patterns = {}
            
            for study_id in sorted_studies:
                study_info = study_mapping[study_id]
                
                retrieving_llms = set()
                retrieving_strategies = set()
                retrieving_variations = set()
                
                for combo in combinations_data:
                    combo_retrieved = False
                    
                    for run, run_studies in combo['studies_by_run'].items():
                        for identifier in study_info['identifiers']:
                            if identifier in run_studies:
                                combo_retrieved = True
                                break
                        if combo_retrieved:
                            break
                    
                    if combo_retrieved:
                        retrieving_llms.add(combo['LLM'])
                        retrieving_strategies.add(f"{combo['LLM']}-{combo['Strategy']}")
                        retrieving_variations.add(f"{combo['LLM']}-{combo['Strategy']}{combo['Variation']}")
                
                study_retrieval_patterns[study_id] = {
                    'retrieving_llms': retrieving_llms,
                    'retrieving_strategies': retrieving_strategies,
                    'retrieving_variations': retrieving_variations,
                    'num_retrieving_llms': len(retrieving_llms),
                    'num_retrieving_strategies': len(retrieving_strategies),
                    'num_retrieving_variations': len(retrieving_variations)
                }
            
            print(f"✓ Analyzed patterns for {len(study_retrieval_patterns)} studies")
            return study_retrieval_patterns
        
        except Exception as e:
            print(f"❌ Error in analyze_detailed_retrieval_patterns: {e}")
            raise
    
    def create_matrix_with_new_logic(sorted_studies, study_mapping, combinations_data, study_patterns):
        """Create matrix with SINGLE SCALE coloring logic and calculate consistency scores"""
        try:
            matrix = []
            consistency_scores = []
            
            for combo_idx, combo in enumerate(combinations_data):
                row = []
                
                current_llm = combo['LLM']
                current_strategy = f"{combo['LLM']}-{combo['Strategy']}"
                current_variation = f"{combo['LLM']}-{combo['Strategy']}{combo['Variation']}"
                
                total_retrieved = 0
                total_runs_sum = 0
                
                for study_idx, study_id in enumerate(sorted_studies):
                    try:
                        study_info = study_mapping[study_id]
                        
                        run_matches = 0
                        for run, run_studies in combo['studies_by_run'].items():
                            for identifier in study_info['identifiers']:
                                if identifier in run_studies:
                                    run_matches += 1
                                    break
                        
                        pattern = study_patterns[study_id]
                        
                        if run_matches > 0:
                            total_retrieved += 1
                            total_runs_sum += run_matches
                            value = 7
                        else:
                            if current_strategy in pattern['retrieving_strategies']:
                                value = 6
                            elif current_llm in pattern['retrieving_llms']:
                                value = 5
                            else:
                                num_other_llms = pattern['num_retrieving_llms']
                                if num_other_llms == 0:
                                    value = 0
                                elif num_other_llms == 1:
                                    value = 1
                                elif num_other_llms == 2:
                                    value = 2
                                elif num_other_llms == 3:
                                    value = 3
                                elif num_other_llms >= 4:
                                    value = 4
                        
                        row.append(value)
                    
                    except Exception as e:
                        print(f"⚠ Warning: Error processing study {study_id} for combo {combo_idx}: {e}")
                        row.append(0)
                        continue
                
                matrix.append(row)
                
                if total_retrieved > 0:
                    avg_runs = total_runs_sum / total_retrieved
                    consistency_pct = (avg_runs / 3.0) * 100
                else:
                    consistency_pct = 0
                
                consistency_scores.append(consistency_pct)
            
            matrix_array = np.array(matrix)
            print(f"✓ Created matrix with shape {matrix_array.shape}")
            
            return matrix_array, consistency_scores
        
        except Exception as e:
            print(f"❌ Error in create_matrix_with_new_logic: {e}")
            raise
    
    def create_controlled_distance_matrix(combinations_data, config):
        """Create distance matrix with configurable parameters for rows"""
        n = len(combinations_data)
        distances = np.zeros((n, n))
        
        for i in range(n):
            for j in range(n):
                combo_i = combinations_data[i]
                combo_j = combinations_data[j]
                
                if config['force_perfect_hierarchy']:
                    if combo_i['LLM'] != combo_j['LLM']:
                        distances[i, j] = config['llm_distance']
                    elif combo_i['Strategy'] != combo_j['Strategy']:
                        distances[i, j] = config['strategy_distance']
                    elif combo_i['Variation'] != combo_j['Variation']:
                        distances[i, j] = config['variation_distance']
                    else:
                        distances[i, j] = 0.0
        
        return distances
    
    def create_forced_uniform_linkage(sorted_studies, analyzer, config):
        """Create a FORCED uniform linkage matrix that strictly follows Publisher → Journal → Study hierarchy"""
        try:
            print("🔨 Creating FORCED uniform linkage matrix for columns...")
            
            n = len(sorted_studies)
            
            study_hierarchy = []
            for idx, study_id in enumerate(sorted_studies):
                study_row = analyzer.gold_standard_df[
                    analyzer.gold_standard_df['Study ID'] == study_id
                ]
                
                if not study_row.empty:
                    publisher = study_row['Publisher'].iloc[0] if 'Publisher' in study_row.columns else 'Unknown'
                    journal = study_row['Journal/Conference'].iloc[0] if 'Journal/Conference' in study_row.columns else 'Unknown'
                    year = get_study_year(study_row)
                    
                    publisher = standardize_publisher_name(str(publisher))
                    
                    study_hierarchy.append({
                        'idx': idx,
                        'publisher': publisher,
                        'journal': journal,
                        'study_id': study_id,
                        'year': year
                    })
            
            publisher_data = defaultdict(lambda: defaultdict(list))
            publisher_order = []
            for item in study_hierarchy:
                if item['publisher'] not in publisher_order:
                    publisher_order.append(item['publisher'])
                publisher_data[item['publisher']][item['journal']].append(item['idx'])
            
            linkage_list = []
            next_cluster_id = n
            cluster_representatives = {}
            for i in range(n):
                cluster_representatives[i] = i
            
            # LEVEL 1: Merge studies within same journal
            print("  Level 1: Merging studies within journals...")
            for publisher in publisher_order:
                journals = publisher_data[publisher]
                journal_order = []
                for item in study_hierarchy:
                    if item['publisher'] == publisher and item['journal'] not in journal_order:
                        journal_order.append(item['journal'])
                
                for journal in journal_order:
                    study_indices = journals[journal]
                    
                    if len(study_indices) > 1:
                        current_cluster = cluster_representatives[study_indices[0]]
                        
                        for i in range(1, len(study_indices)):
                            next_idx = study_indices[i]
                            next_cluster = cluster_representatives[next_idx]
                            
                            linkage_list.append([
                                current_cluster,
                                next_cluster,
                                config['study_height'],
                                i + 1
                            ])
                            
                            current_cluster = next_cluster_id
                            cluster_representatives[next_idx] = next_cluster_id
                            next_cluster_id += 1
                        
                        for idx in study_indices:
                            cluster_representatives[idx] = current_cluster
            
            # LEVEL 2: Merge journals within same publisher
            print("  Level 2: Merging journals within publishers...")
            publisher_representatives = {}
            for publisher in publisher_order:
                journals = publisher_data[publisher]
                journal_order = []
                for item in study_hierarchy:
                    if item['publisher'] == publisher and item['journal'] not in journal_order:
                        journal_order.append(item['journal'])
                
                if len(journal_order) > 1:
                    first_journal_studies = journals[journal_order[0]]
                    current_cluster = cluster_representatives[first_journal_studies[-1]]
                    total_size = len(first_journal_studies)
                    
                    for journal in journal_order[1:]:
                        journal_studies = journals[journal]
                        journal_cluster = cluster_representatives[journal_studies[-1]]
                        total_size += len(journal_studies)
                        
                        linkage_list.append([
                            current_cluster,
                            journal_cluster,
                            config['journal_height'],
                            total_size
                        ])
                        
                        current_cluster = next_cluster_id
                        next_cluster_id += 1
                    
                    publisher_representatives[publisher] = current_cluster
                else:
                    single_journal_studies = journals[journal_order[0]]
                    publisher_representatives[publisher] = cluster_representatives[single_journal_studies[-1]]
            
            # LEVEL 3: Merge publishers
            print("  Level 3: Merging publishers...")
            if len(publisher_order) > 1:
                current_cluster = publisher_representatives[publisher_order[0]]
                total_size = sum(len(studies) for studies in publisher_data[publisher_order[0]].values())
                
                for publisher in publisher_order[1:]:
                    pub_cluster = publisher_representatives[publisher]
                    pub_size = sum(len(studies) for studies in publisher_data[publisher].values())
                    total_size += pub_size
                    
                    linkage_list.append([
                        current_cluster,
                        pub_cluster,
                        config['publisher_height'],
                        total_size
                    ])
                    
                    current_cluster = next_cluster_id
                    next_cluster_id += 1
            
            linkage_array = np.array(linkage_list)
            print(f"✓ Created forced uniform linkage with {len(linkage_list)} merges")
            
            return linkage_array
        
        except Exception as e:
            print(f"❌ Error creating forced linkage: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def create_hierarchical_labels(combinations_data):
        """Create hierarchical labels for rows"""
        labels = []
        for combo in combinations_data:
            label = f"{combo['LLM']} | {combo['StrategyName']} | V{combo['Variation']}"
            labels.append(label)
        return labels
    
    def create_row_colors(combinations_data):
        """Create row colors for LLM identification - Academic muted tones"""
        llm_colors = {
            'Ai2 Finder': '#9B8B7E',
            'Consensus': '#7B8FA3',
            'Claude': '#A17A74',
            'ChatGPT': '#748B75',
            'Gemini': '#6B8E9E',
        }
        
        row_colors = []
        for combo in combinations_data:
            row_colors.append(llm_colors[combo['LLM']])
        
        return row_colors, llm_colors
    
    def add_hierarchical_column_separators(ax_heatmap, reordered_studies, analyzer):
        """Add vertical separator lines at ALL levels with PDF-optimized rendering"""
        try:
            print("📐 Adding hierarchical column separators (Publisher | Journal | Study)...")
            
            # Build metadata
            study_metadata = []
            for idx, study_id in enumerate(reordered_studies):
                study_row = analyzer.gold_standard_df[
                    analyzer.gold_standard_df['Study ID'] == study_id
                ]
                
                if not study_row.empty:
                    publisher = study_row['Publisher'].iloc[0] if 'Publisher' in study_row.columns else 'Unknown'
                    journal = study_row['Journal/Conference'].iloc[0] if 'Journal/Conference' in study_row.columns else 'Unknown'
                    publisher = standardize_publisher_name(str(publisher))
                else:
                    publisher = 'Unknown'
                    journal = 'Unknown'
                
                study_metadata.append({
                    'idx': idx,
                    'study_id': study_id,
                    'publisher': publisher,
                    'journal': journal
                })
            
            # Detect ALL boundaries at three levels
            publisher_boundaries = []
            journal_boundaries = []
            study_boundaries = []
            
            for i in range(len(study_metadata) - 1):
                current = study_metadata[i]
                next_study = study_metadata[i + 1]
                
                # Level 3: Publisher change (thickest)
                if current['publisher'] != next_study['publisher']:
                    publisher_boundaries.append(i + 1)
                # Level 2: Journal change within same publisher (medium)
                elif current['journal'] != next_study['journal']:
                    journal_boundaries.append(i + 1)
                # Level 1: Study change within same journal (thinnest)
                else:
                    study_boundaries.append(i + 1)
            
            print(f"   Detected {len(study_boundaries)} study boundaries")
            print(f"   Detected {len(journal_boundaries)} journal boundaries")
            print(f"   Detected {len(publisher_boundaries)} publisher boundaries")
            
            # PDF-optimized rendering parameters
            line_params = {
                'linestyle': '-',
                'clip_on': False,
                'solid_capstyle': 'butt',      # Changed to 'butt' for thinner lines
                'solid_joinstyle': 'miter',
                'antialiased': True,
            }
            
            # Draw separators in order (thinnest to thickest)
            # Level 1: Study separators (thinnest, drawn first)
            if SEPARATOR_CONFIG['study_separator_width'] > 0:
                for x_pos in study_boundaries:
                    ax_heatmap.axvline(
                        x=x_pos,
                        color=SEPARATOR_CONFIG.get('study_separator_color', SEPARATOR_CONFIG['vertical_separator_color']),
                        linewidth=SEPARATOR_CONFIG['study_separator_width'],
                        alpha=SEPARATOR_CONFIG.get('study_separator_alpha', 0.5),
                        zorder=9999,
                        **line_params
                    )
            
            # Level 2: Journal separators (medium)
            if SEPARATOR_CONFIG['journal_separator_width'] > 0:
                for x_pos in journal_boundaries:
                    ax_heatmap.axvline(
                        x=x_pos,
                        color=SEPARATOR_CONFIG.get('journal_separator_color', SEPARATOR_CONFIG['vertical_separator_color']),
                        linewidth=SEPARATOR_CONFIG['journal_separator_width'],
                        alpha=SEPARATOR_CONFIG.get('journal_separator_alpha', 0.75),
                        zorder=10000,
                        **line_params
                    )
            
            # Level 3: Publisher separators (thickest, drawn last)
            if SEPARATOR_CONFIG['publisher_separator_width'] > 0:
                for x_pos in publisher_boundaries:
                    ax_heatmap.axvline(
                        x=x_pos,
                        color=SEPARATOR_CONFIG.get('publisher_separator_color', SEPARATOR_CONFIG['vertical_separator_color']),
                        linewidth=SEPARATOR_CONFIG['publisher_separator_width'],
                        alpha=SEPARATOR_CONFIG.get('publisher_separator_alpha', 0.9),
                        zorder=10001,
                        **line_params
                    )
            
            print(f"✓ Drew {len(study_boundaries)} study separators (thin)")
            print(f"✓ Drew {len(journal_boundaries)} journal separators (medium)")
            print(f"✓ Drew {len(publisher_boundaries)} publisher separators (thick)")
            
            total_separators = len(study_boundaries) + len(journal_boundaries) + len(publisher_boundaries)
            expected_separators = len(reordered_studies) - 1
            print(f"✓ Total separators: {total_separators} (expected: {expected_separators})")
        
        except Exception as e:
            print(f"⚠ Warning: Could not add column separators: {e}")
            import traceback
            traceback.print_exc()
    
    def add_column_dendrogram_labels(ax_col_dendro, sorted_studies, analyzer, config):
        """Add Publisher and Journal labels with MANUAL JOURNAL + MANUAL PUBLISHER offsets"""
        try:
            print("🏷️ Adding Publisher and Journal labels with manual offsets...")
            
            publisher_journals = defaultdict(lambda: defaultdict(list))
            for idx, study_id in enumerate(sorted_studies):
                study_row = analyzer.gold_standard_df[
                    analyzer.gold_standard_df['Study ID'] == study_id
                ]
                
                if not study_row.empty:
                    publisher = study_row['Publisher'].iloc[0] if 'Publisher' in study_row.columns else 'Unknown'
                    journal = study_row['Journal/Conference'].iloc[0] if 'Journal/Conference' in study_row.columns else 'Unknown'
                    year = get_study_year(study_row)
                    
                    publisher = standardize_publisher_name(str(publisher))
                    
                    publisher_journals[publisher][journal].append({
                        'idx': idx,
                        'year': year
                    })
            
            publisher_years = {}
            for publisher, journals in publisher_journals.items():
                all_years = []
                for journal, studies in journals.items():
                    all_years.extend([s['year'] for s in studies if s['year'] != 9999])
                if all_years:
                    publisher_years[publisher] = min(all_years)
                else:
                    publisher_years[publisher] = 9999
            
            sorted_publishers = sorted(publisher_journals.keys(), key=lambda p: publisher_years[p])
            
            dendro_xlim = ax_col_dendro.get_xlim()
            dendro_ylim = ax_col_dendro.get_ylim()
            
            vertical_lines = []
            for line in ax_col_dendro.get_lines():
                xdata, ydata = line.get_data()
                if len(xdata) == 2 and abs(xdata[0] - xdata[1]) < 1e-10:
                    vertical_lines.append((xdata[0], ydata[0]))
            
            vertical_lines.sort(key=lambda x: x[1])
            
            ordered_publisher_groups = {}
            ordered_journal_groups = {}
            publisher_appearance_order = []
            journal_appearance_order = []
            
            current_pos = 0
            for publisher_idx, publisher in enumerate(sorted_publishers):
                journals = publisher_journals[publisher]
                
                journal_years = {}
                for journal, studies in journals.items():
                    years = [s['year'] for s in studies if s['year'] != 9999]
                    if years:
                        journal_years[journal] = min(years)
                    else:
                        journal_years[journal] = 9999
                
                sorted_journals = sorted(journals.keys(), key=lambda j: journal_years[j])
                
                publisher_start = current_pos
                publisher_indices = []
                
                for journal_idx, journal in enumerate(sorted_journals):
                    journal_study_count = len(journals[journal])
                    journal_indices = list(range(current_pos, current_pos + journal_study_count))
                    
                    # Get MANUAL journal offset or calculate INCREMENTAL offset
                    journal_display = clean_journal_name(journal)
                    manual_journal_offset = config['journal_manual_x_offsets'].get(journal_display)
                    
                    if manual_journal_offset is None:
                        manual_journal_offset = config['journal_manual_x_offsets'].get(journal)
                    
                    if manual_journal_offset is None:
                        # Use incremental calculation as fallback
                        journal_offset = sum([i * config['journal_multi_study_x_offset']
                                            for i in range(journal_study_count)])
                    else:
                        # Use manual offset
                        journal_offset = manual_journal_offset
                    
                    journal_key = f"{publisher}_{journal}"
                    if journal_key not in ordered_journal_groups:
                        ordered_journal_groups[journal_key] = {
                            'indices': journal_indices,
                            'publisher': publisher,
                            'journal': journal,
                            'first_appearance': current_pos,
                            'num_studies': journal_study_count,
                            'offset': journal_offset,
                        }
                        journal_appearance_order.append(journal_key)
                    
                    publisher_indices.extend(journal_indices)
                    current_pos += journal_study_count
                
                total_publisher_studies = len(publisher_indices)
                num_journals = len(sorted_journals)
                
                # Get MANUAL publisher offset
                manual_offset = config['publisher_manual_x_offsets'].get(
                    publisher,
                    config['publisher_manual_x_offsets']['default']
                )
                
                if publisher not in ordered_publisher_groups:
                    ordered_publisher_groups[publisher] = {
                        'indices': publisher_indices,
                        'journals': {},
                        'first_appearance': publisher_start,
                        'num_studies': total_publisher_studies,
                        'num_journals': num_journals,
                        'manual_offset': manual_offset
                    }
                    publisher_appearance_order.append(publisher)
            
            def find_merge_point_for_group_col(group_indices, target_height):
                """Find merge point for column groups"""
                if len(group_indices) <= 1:
                    return None
                
                group_x_positions = [dendro_xlim[0] + (idx / len(sorted_studies)) * (dendro_xlim[1] - dendro_xlim[0])
                                    for idx in group_indices]
                x_center = np.mean(group_x_positions)
                x_range = max(group_x_positions) - min(group_x_positions)
                
                candidates = []
                for x_pos, y_pos in vertical_lines:
                    if abs(y_pos - target_height) < config['col_merge_point_tolerance']:
                        if (min(group_x_positions) - x_range * 0.2 <= x_pos <=
                            max(group_x_positions) + x_range * 0.2):
                            candidates.append((x_pos, y_pos, abs(x_pos - x_center)))
                
                if candidates:
                    candidates.sort(key=lambda x: x[2])
                    return candidates[0][:2]
                
                return (x_center, target_height)
            
            # Journal labels WITH MANUAL/INCREMENTAL OFFSET
            journal_group_counter = 0
            current_journal_group = None
            for journal_idx, journal_key in enumerate(journal_appearance_order):
                journal_group = ordered_journal_groups[journal_key]
                journal_indices = journal_group['indices']
                journal_offset = journal_group['offset']
                current_publisher = journal_group['publisher']
                
                if current_journal_group != current_publisher:
                    if current_journal_group is not None:
                        journal_group_counter += 1
                    current_journal_group = current_publisher
                
                if len(journal_indices) >= 1:
                    merge_point = find_merge_point_for_group_col(journal_indices, config['journal_height'])
                    
                    if merge_point:
                        base_x, base_y = merge_point
                    else:
                        base_x = np.mean([dendro_xlim[0] + (idx / len(sorted_studies)) * (dendro_xlim[1] - dendro_xlim[0])
                                         for idx in journal_indices])
                        base_y = config['journal_height']
                    
                    # Apply spacing, offset (manual or incremental), AND base x-offset
                    x_inter_spacing = journal_group_counter * config['journal_inter_x_spacing']
                    x_pos = base_x + x_inter_spacing + journal_offset + config['journal_x_offset']
                    y_pos = base_y + config['journal_y_offset']
                    
                    journal_display = clean_journal_name(journal_group['journal'])
                    
                    ax_col_dendro.text(
                        x_pos, y_pos, journal_display,
                        fontsize=config['journal_fontsize'],
                        verticalalignment='bottom',
                        horizontalalignment='center',
                        rotation=config['journal_label_rotation'],
                        clip_on=False
                    )
            
            # Publisher labels WITH MANUAL OFFSET
            for pub_idx, publisher in enumerate(publisher_appearance_order):
                pub_data = ordered_publisher_groups[publisher]
                manual_offset = pub_data['manual_offset']
                
                if len(pub_data['indices']) >= 1:
                    merge_point = find_merge_point_for_group_col(pub_data['indices'], config['publisher_height'])
                    
                    if merge_point:
                        base_x, base_y = merge_point
                    else:
                        base_x = np.mean([dendro_xlim[0] + (idx / len(sorted_studies)) * (dendro_xlim[1] - dendro_xlim[0])
                                         for idx in pub_data['indices']])
                        base_y = config['publisher_height']
                    
                    # Apply spacing, MANUAL offset, AND base x-offset
                    x_inter_spacing = pub_idx * config['publisher_inter_x_spacing']
                    x_pos = base_x + x_inter_spacing + manual_offset + config['publisher_x_offset']
                    y_pos = base_y + config['publisher_y_offset']
                    
                    ax_col_dendro.text(
                        x_pos, y_pos, publisher,
                        fontsize=config['publisher_fontsize'],
                        fontweight='bold',
                        verticalalignment='bottom',
                        horizontalalignment='center',
                        rotation=config['publisher_label_rotation'],
                        clip_on=False
                    )
            
            print(f"✓ Added Publisher and Journal labels with manual offsets")
        
        except Exception as e:
            print(f"⚠ Warning: Could not add column dendrogram labels: {e}")
            import traceback
            traceback.print_exc()
    
    def rgb_to_hex(rgb):
        """Convert RGB tuple to hex color"""
        return '#{:02x}{:02x}{:02x}'.format(
            int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255)
        )
    
    def add_bottom_legend_with_subtitles(fig, llm_color_map, cmap, color_list, config):
        """Add a structured legend with TWO SEPARATE PARTITIONS for LLMs and Retrieval Status"""
        try:
            print("📋 Adding partitioned legend with headings...")
            
            # ========== CREATE TWO SEPARATE LEGEND BOXES ==========
            
            # === LEFT PARTITION: LLM MODELS ===
            ax_legend_llm = fig.add_axes([0.01, -0.08, 0.30, 0.08])
            ax_legend_llm.axis('off')
            
            llm_elements = []
            # Add LLM color patches
            for llm, color in llm_color_map.items():
                llm_elements.append(Patch(facecolor=color, edgecolor='black',
                                         linewidth=0.5, label=llm))
            
            # Create LLM legend
            llm_legend = ax_legend_llm.legend(
                handles=llm_elements,
                title='LLM Models',
                loc='center',
                frameon=True,
                fancybox=False,
                shadow=False,
                fontsize=config['item_fontsize'],
                title_fontsize=config['title_fontsize'],
                ncol=5,
                columnspacing=config['columnspacing'],
                handlelength=config['handlelength'],
                handleheight=config['handleheight'],
                framealpha=config['box_alpha'],
                edgecolor=config['box_edgecolor'],
                facecolor=config['box_facecolor']
            )
            llm_legend.get_frame().set_linewidth(config['box_linewidth'])
            llm_legend.get_title().set_fontweight('bold')
            
            # === RIGHT PARTITION: RETRIEVAL STATUS ===
            ax_legend_retrieval = fig.add_axes([0.52, -0.08, 0.45, 0.08])
            ax_legend_retrieval.axis('off')
            
            retrieval_elements = []
            # Add retrieval status patches using actual generated colors
            retrieval_labels = [
                'Retrieved by this prompt',
                'Other prompt variation',
                'Other prompt strategy',
                '4+ other LLMs',
                '3 other LLMs',
                '2 other LLMs',
                '1 other LLM',
                'Never retrieved',
            ]
            
            for i, label in enumerate(retrieval_labels):
                color_idx = 7 - i  # Reverse order for display
                color = color_list[color_idx]
                hex_color = rgb_to_hex(color)
                retrieval_elements.append(Patch(facecolor=hex_color, edgecolor='gray',
                                              linewidth=0.3, label=label))
            
            # Create Retrieval legend
            retrieval_legend = ax_legend_retrieval.legend(
                handles=retrieval_elements,
                title='Retrieval Status',
                loc='center',
                frameon=True,
                fancybox=False,
                shadow=False,
                fontsize=config['item_fontsize'],
                title_fontsize=config['title_fontsize'],
                ncol=8,
                columnspacing=config['columnspacing'],
                handlelength=config['handlelength'],
                handleheight=config['handleheight'],
                framealpha=config['box_alpha'],
                edgecolor=config['box_edgecolor'],
                facecolor=config['box_facecolor']
            )
            retrieval_legend.get_frame().set_linewidth(config['box_linewidth'])
            retrieval_legend.get_title().set_fontweight('bold')
            
            print("✓ Partitioned legend with headings added successfully")
        
        except Exception as e:
            print(f"⚠ Warning: Could not add legend: {e}")
            import traceback
            traceback.print_exc()
    
    # ========== MAIN EXECUTION ==========
    try:
        print("🔍 Starting with asymmetric blue-to-gold gradient and THINNER separators...")
        
        analyzer = base_results['analyzer']
        sorted_studies, study_labels, study_mapping = extract_study_data()
        combinations_data = build_combination_data_with_runs(analyzer, study_mapping)
        study_patterns = analyze_detailed_retrieval_patterns(sorted_studies, study_mapping, combinations_data)
        matrix, consistency_scores = create_matrix_with_new_logic(sorted_studies, study_mapping, combinations_data, study_patterns)
        row_labels = create_hierarchical_labels(combinations_data)
        row_colors, llm_color_map = create_row_colors(combinations_data)
        
        print("🌳 Creating uniform hierarchical dendrograms...")
        
        row_distance_matrix = create_controlled_distance_matrix(combinations_data, DENDROGRAM_CONFIG)
        row_condensed_distances = squareform(row_distance_matrix)
        row_linkage_matrix = linkage(row_condensed_distances, method=DENDROGRAM_CONFIG['linkage_method'])
        
        col_linkage_matrix = create_forced_uniform_linkage(sorted_studies, analyzer, DENDROGRAM_CONFIG)
        
        if col_linkage_matrix is None:
            print("⚠ Failed to create forced linkage")
            return None, None, None, None, None
        
        df_matrix = pd.DataFrame(matrix, index=row_labels, columns=study_labels)
        
        # ========== CREATE ASYMMETRIC GRADIENT ==========
        cmap, color_list = create_strategic_gradient_colormap()
        
        print("📊 Creating clustermap...")
        
        g = sns.clustermap(
            df_matrix,
            cmap=cmap,
            vmin=0, vmax=7,
            linewidths=0,
            linecolor='white',
            figsize=(24, 22),
            row_colors=row_colors,
            col_cluster=True,
            row_cluster=True,
            row_linkage=row_linkage_matrix,
            col_linkage=col_linkage_matrix,
            dendrogram_ratio=(DENDROGRAM_CONFIG['dendrogram_ratio'], DENDROGRAM_CONFIG['col_dendrogram_ratio']),
            colors_ratio=0.03,
            tree_kws={'linewidths': DENDROGRAM_CONFIG['tree_linewidth'], 'colors': DENDROGRAM_CONFIG['tree_color']},
            yticklabels=False,
        )
        
        print("✓ Clustermap created!")
        
        # ========== GET REORDERED DATA (CRITICAL: THIS MUST HAPPEN BEFORE REPORT) ==========
        leaf_order = g.dendrogram_row.reordered_ind
        col_leaf_order = g.dendrogram_col.reordered_ind
        
        reordered_consistency = [consistency_scores[idx] for idx in leaf_order]
        reordered_combinations = [combinations_data[idx] for idx in leaf_order]
        reordered_matrix = matrix[leaf_order, :][:, col_leaf_order]
        reordered_studies = [sorted_studies[idx] for idx in col_leaf_order]
        
        ax_heatmap = g.ax_heatmap
        ax_row_colors = g.ax_row_colors
        ax_col_dendro = g.ax_col_dendrogram
        
        # Add column dendrogram labels
        if ax_col_dendro is not None and DENDROGRAM_CONFIG['show_col_dendrogram']:
            add_column_dendrogram_labels(ax_col_dendro, reordered_studies, analyzer, DENDROGRAM_CONFIG)
        
        print("📐 Adding gap between row colors and heatmap...")
        row_colors_pos = ax_row_colors.get_position()
        heatmap_pos = ax_heatmap.get_position()
        
        new_heatmap_x0 = row_colors_pos.x1 + ROW_COLORS_CONFIG['gap_after_colors']
        new_heatmap_width = heatmap_pos.x1 - new_heatmap_x0
        
        ax_heatmap.set_position([
            new_heatmap_x0,
            heatmap_pos.y0,
            new_heatmap_width,
            heatmap_pos.height
        ])
        
        print("📐 Identifying hierarchical boundaries...")
        
        prev_llm = None
        prev_strategy = None
        llm_boundaries = []
        strategy_boundaries = []
        variation_boundaries = []
        
        for i, combo in enumerate(reordered_combinations):
            current_llm = combo['LLM']
            current_strategy = combo['Strategy']
            
            if prev_llm is not None and current_llm != prev_llm:
                llm_boundaries.append(i)
            elif prev_strategy is not None and current_strategy != prev_strategy:
                strategy_boundaries.append(i)
            else:
                if i > 0:
                    variation_boundaries.append(i)
            
            prev_llm = current_llm
            prev_strategy = current_strategy
        
        # PDF-optimized rendering parameters
        line_params = {
            'linestyle': '-',
            'clip_on': False,
            'solid_capstyle': 'butt',         # Changed to 'butt' for thinner lines
            'solid_joinstyle': 'miter',
            'antialiased': True,
        }
        
        # Draw horizontal separators on heatmap
        if SEPARATOR_CONFIG['llm_separator_width'] > 0:
            for i in llm_boundaries:
                ax_heatmap.axhline(y=i, color=SEPARATOR_CONFIG['separator_color'],
                                  linewidth=SEPARATOR_CONFIG['llm_separator_width'], zorder=5, **line_params)
        
        if SEPARATOR_CONFIG['strategy_separator_width'] > 0:
            for i in strategy_boundaries:
                ax_heatmap.axhline(y=i, color=SEPARATOR_CONFIG['separator_color'],
                                  linewidth=SEPARATOR_CONFIG['strategy_separator_width'], zorder=4, **line_params)
        
        if SEPARATOR_CONFIG['variation_separator_width'] > 0:
            for i in variation_boundaries:
                ax_heatmap.axhline(y=i, color=SEPARATOR_CONFIG['separator_color'],
                                  linewidth=SEPARATOR_CONFIG['variation_separator_width'], zorder=4, **line_params)
        
        # Extend separators to row colors
        if ax_row_colors is not None:
            if SEPARATOR_CONFIG['extend_llm_separator_to_colors'] and SEPARATOR_CONFIG['llm_separator_width'] > 0:
                for boundary_y in llm_boundaries:
                    ax_row_colors.axhline(
                        y=boundary_y,
                        color=SEPARATOR_CONFIG['separator_color'],
                        linewidth=SEPARATOR_CONFIG['llm_separator_width'],
                        zorder=5,
                        xmin=0,
                        xmax=1,
                        **line_params
                    )
            
            if SEPARATOR_CONFIG['extend_strategy_separator_to_colors'] and SEPARATOR_CONFIG['strategy_separator_width'] > 0:
                for boundary_y in strategy_boundaries:
                    ax_row_colors.axhline(
                        y=boundary_y,
                        color=SEPARATOR_CONFIG['separator_color'],
                        linewidth=SEPARATOR_CONFIG['strategy_separator_width'],
                        zorder=4,
                        xmin=0,
                        xmax=1,
                        **line_params
                    )
            
            if SEPARATOR_CONFIG['extend_variation_separator_to_colors'] and SEPARATOR_CONFIG['variation_separator_width'] > 0:
                for boundary_y in variation_boundaries:
                    ax_row_colors.axhline(
                        y=boundary_y,
                        color=SEPARATOR_CONFIG['separator_color'],
                        linewidth=SEPARATOR_CONFIG['variation_separator_width'],
                        zorder=3,
                        xmin=0,
                        xmax=1,
                        **line_params
                    )
        
        print("🏷️ Adding variation labels on row color column...")
        
        if ax_row_colors is not None:
            for i, combo in enumerate(reordered_combinations):
                llm_bg_color = llm_color_map[combo['LLM']]
                text_color = get_text_color_for_background(llm_bg_color)
                
                y_pos = i + 0.5
                x_pos = 0.5
                
                ax_row_colors.text(
                    x_pos, y_pos, f"V{combo['Variation']}",
                    fontsize=DENDROGRAM_CONFIG['variation_fontsize'],
                    verticalalignment='center',
                    horizontalalignment='center',
                    color=text_color,
                    fontweight='bold',
                    clip_on=False,
                    zorder=10
                )
        
        # ========== ROW DENDROGRAM LABELING ==========
        if (hasattr(g, "ax_row_dendrogram") and
            g.ax_row_dendrogram is not None and
            DENDROGRAM_CONFIG.get('show_row_dendrogram', False)):
            
            print("🏷️ Adding hierarchical labels on row dendrogram...")
            
            ax_dendro = g.ax_row_dendrogram
            n_leaves = len(leaf_order)
            
            heatmap_ylim = ax_heatmap.get_ylim()
            row_positions = np.linspace(heatmap_ylim[0] + 0.5, heatmap_ylim[1] - 0.5, n_leaves)
            
            dendro_xlim = ax_dendro.get_xlim()
            dendro_ylim = ax_dendro.get_ylim()
            
            leaf_positions = {}
            for i in range(n_leaves):
                heatmap_y = row_positions[n_leaves - 1 - i]
                dendro_y = dendro_ylim[0] + (heatmap_y - heatmap_ylim[0]) / (heatmap_ylim[1] - heatmap_ylim[0]) * (dendro_ylim[1] - dendro_ylim[0])
                leaf_positions[i] = dendro_y
            
            horizontal_lines = []
            for line in ax_dendro.get_lines():
                xdata, ydata = line.get_data()
                if len(xdata) == 2 and abs(ydata[0] - ydata[1]) < 1e-10:
                    horizontal_lines.append((xdata[0], ydata[0]))
            
            horizontal_lines.sort(key=lambda x: x[0])
            
            ordered_llm_groups = {}
            ordered_strategy_groups = {}
            llm_appearance_order = []
            strategy_appearance_order = []
            
            for i, combo in enumerate(reordered_combinations):
                llm = combo['LLM']
                strategy = combo['StrategyName']
                llm_strategy_key = f"{llm}_{strategy}"
                
                if llm not in ordered_llm_groups:
                    ordered_llm_groups[llm] = {'indices': [], 'strategies': {}, 'first_appearance': i}
                    llm_appearance_order.append(llm)
                
                ordered_llm_groups[llm]['indices'].append(i)
                
                if strategy not in ordered_llm_groups[llm]['strategies']:
                    ordered_llm_groups[llm]['strategies'][strategy] = []
                ordered_llm_groups[llm]['strategies'][strategy].append(i)
                
                if llm_strategy_key not in ordered_strategy_groups:
                    ordered_strategy_groups[llm_strategy_key] = {
                        'indices': [],
                        'llm': llm,
                        'strategy': strategy,
                        'first_appearance': i
                    }
                    strategy_appearance_order.append(llm_strategy_key)
                
                ordered_strategy_groups[llm_strategy_key]['indices'].append(i)
            
            def find_merge_point_for_group(group_indices, target_distance):
                if len(group_indices) <= 1:
                    return None
                
                group_y_positions = [leaf_positions[idx] for idx in group_indices]
                y_center = np.mean(group_y_positions)
                y_range = max(group_y_positions) - min(group_y_positions)
                
                candidates = []
                for x_pos, y_pos in horizontal_lines:
                    if abs(x_pos - target_distance) < DENDROGRAM_CONFIG['merge_point_tolerance']:
                        if (min(group_y_positions) - y_range * 0.2 <= y_pos <=
                            max(group_y_positions) + y_range * 0.2):
                            candidates.append((x_pos, y_pos, abs(y_pos - y_center)))
                
                if candidates:
                    candidates.sort(key=lambda x: x[2])
                    return candidates[0][:2]
                
                return (target_distance, y_center)
            
            # Strategy labels
            strategy_group_counter = 0
            current_strategy_group = None
            for strategy_idx, strategy_key in enumerate(strategy_appearance_order):
                strategy_group = ordered_strategy_groups[strategy_key]
                strategy_indices = strategy_group['indices']
                current_llm = strategy_group['llm']
                
                if current_strategy_group != current_llm:
                    if current_strategy_group is not None:
                        strategy_group_counter += 1
                    current_strategy_group = current_llm
                
                if len(strategy_indices) >= 1:
                    merge_point = find_merge_point_for_group(strategy_indices, DENDROGRAM_CONFIG['variation_distance'])
                    
                    if merge_point:
                        base_x, base_y = merge_point
                    else:
                        base_y = np.mean([leaf_positions[idx] for idx in strategy_indices])
                        base_x = DENDROGRAM_CONFIG['variation_distance']
                    
                    y_inter_spacing = strategy_group_counter * DENDROGRAM_CONFIG['strategy_inter_y_spacing']
                    x_pos = base_x + DENDROGRAM_CONFIG['strategy_x_offset']
                    y_pos = base_y + y_inter_spacing + DENDROGRAM_CONFIG['strategy_y_offset']
                    
                    ax_dendro.text(
                        x_pos, y_pos, strategy_group['strategy'],
                        fontsize=DENDROGRAM_CONFIG['strategy_fontsize'],
                        verticalalignment='center',
                        horizontalalignment='center',
                        rotation=0,
                        clip_on=False
                    )
            
            # LLM labels
            for llm_idx, llm in enumerate(llm_appearance_order):
                llm_data = ordered_llm_groups[llm]
                
                if len(llm_data['indices']) >= 1:
                    merge_point = find_merge_point_for_group(llm_data['indices'], DENDROGRAM_CONFIG['strategy_distance'])
                    
                    if merge_point:
                        base_x, base_y = merge_point
                    else:
                        base_y = np.mean([leaf_positions[idx] for idx in llm_data['indices']])
                        base_x = DENDROGRAM_CONFIG['strategy_distance']
                    
                    y_inter_spacing = llm_idx * DENDROGRAM_CONFIG['llm_inter_y_spacing']
                    x_pos = base_x + DENDROGRAM_CONFIG['llm_x_offset']
                    y_pos = base_y + y_inter_spacing + DENDROGRAM_CONFIG['llm_y_offset']
                    
                    ax_dendro.text(
                        x_pos, y_pos, llm,
                        fontsize=DENDROGRAM_CONFIG['llm_fontsize'],
                        fontweight='bold',
                        verticalalignment='center',
                        horizontalalignment='center',
                        rotation=90,
                        clip_on=False
                    )
            
            print(f"✓ Placed hierarchical labels on row dendrogram successfully")
        
        # ========== ADD CONSISTENCY SIDEBAR ==========
        print("📊 Adding consistency sidebar...")
        
        heatmap_pos = ax_heatmap.get_position()
        heatmap_ylim = ax_heatmap.get_ylim()
        
        ax_sidebar = g.fig.add_axes([
            heatmap_pos.x1 + SIDEBAR_CONFIG['sidebar_gap'],
            heatmap_pos.y0,
            SIDEBAR_CONFIG['sidebar_width'],
            heatmap_pos.height
        ])
        
        n_rows = len(reordered_consistency)
        y_bar_positions = np.arange(n_rows) + 0.5
        
        if SIDEBAR_CONFIG['use_llm_colors']:
            bar_colors = [llm_color_map[combo['LLM']] for combo in reordered_combinations]
        else:
            bar_colors = SIDEBAR_CONFIG['bar_color']
        
        bars = ax_sidebar.barh(
            y_bar_positions,
            reordered_consistency,
            height=0.9,
            color=bar_colors,
            edgecolor='white',
            linewidth=0.3,
            alpha=SIDEBAR_CONFIG['bar_alpha'],
            align='center'
        )
        
        ax_sidebar.set_ylim(heatmap_ylim)
        ax_sidebar.set_xlim(0, 100)
        
        # Add MINOR vertical grid lines
        if SIDEBAR_CONFIG.get('show_minor_vertical_grid', False):
            for grid_x in SIDEBAR_CONFIG['minor_vertical_grid_positions']:
                ax_sidebar.axvline(
                    x=grid_x,
                    color=SIDEBAR_CONFIG['minor_vertical_grid_color'],
                    linewidth=SIDEBAR_CONFIG['minor_vertical_grid_linewidth'],
                    alpha=SIDEBAR_CONFIG['minor_vertical_grid_alpha'],
                    linestyle='--',
                    zorder=1
                )
        
        # Add MAJOR vertical grid lines
        if SIDEBAR_CONFIG['show_vertical_grid']:
            for grid_x in SIDEBAR_CONFIG['vertical_grid_positions']:
                ax_sidebar.axvline(
                    x=grid_x,
                    color=SIDEBAR_CONFIG['vertical_grid_color'],
                    linewidth=SIDEBAR_CONFIG['vertical_grid_linewidth'],
                    alpha=SIDEBAR_CONFIG['vertical_grid_alpha'],
                    linestyle='--',
                    zorder=2
                )
        
        if SEPARATOR_CONFIG['extend_llm_separator_to_sidebar'] and SEPARATOR_CONFIG['llm_separator_width'] > 0:
            for boundary_y in llm_boundaries:
                ax_sidebar.axhline(
                    y=boundary_y,
                    color=SEPARATOR_CONFIG['separator_color'],
                    linewidth=SEPARATOR_CONFIG['llm_separator_width'],
                    zorder=5,
                    xmin=0,
                    xmax=1,
                    **line_params
                )
        
        if SIDEBAR_CONFIG['show_labels']:
            for i, combo in enumerate(reordered_combinations):
                y_pos = i + 0.5
                x_pos = SIDEBAR_CONFIG['label_x_position']
                label_text = f"{combo['StrategyName']}-V{combo['Variation']}"
                
                ax_sidebar.text(
                    x_pos, y_pos, label_text,
                    fontsize=SIDEBAR_CONFIG['label_fontsize'],
                    verticalalignment='center',
                    horizontalalignment='left',
                    color=SIDEBAR_CONFIG['label_color'],
                    fontweight=SIDEBAR_CONFIG['label_fontweight'],
                    rotation=SIDEBAR_CONFIG['label_rotation'],
                    clip_on=False,
                    zorder=10
                )
        
        ax_sidebar.set_xlabel('Consistency (%)', fontsize=18, fontweight='bold')
        ax_sidebar.set_xticks([0, 25, 50, 75, 100])
        ax_sidebar.set_xticklabels(['0', '25', '50', '75', '100'], fontsize=10)
        ax_sidebar.set_yticks([])
        ax_sidebar.spines['top'].set_visible(False)
        ax_sidebar.spines['right'].set_visible(False)
        ax_sidebar.spines['left'].set_visible(True)
        ax_sidebar.spines['left'].set_linewidth(0.5)
        ax_sidebar.spines['left'].set_color('gray')
        ax_sidebar.spines['bottom'].set_linewidth(0.5)
        ax_sidebar.tick_params(left=False, bottom=True, labelsize=8)
        
        # Customize main plot
        ax_heatmap.set_xlabel('Relevant Studies', fontsize=22, fontweight='bold')
        plt.setp(ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=9)
        
        # Remove default colorbar
        g.cax.remove()
        
        # Add partitioned legend with headings
        if LEGEND_CONFIG['show_legend']:
            add_bottom_legend_with_subtitles(g.fig, llm_color_map, cmap, color_list, LEGEND_CONFIG)
        
        # ========== ADD VERTICAL SEPARATORS AT THE VERY END ==========
        if SEPARATOR_CONFIG.get('show_vertical_separators', False):
            print("\n📐 Drawing vertical separators (FINAL STEP)...")
            add_hierarchical_column_separators(ax_heatmap, reordered_studies, analyzer)
        
        # ========== CONFIGURE MATPLOTLIB FOR BETTER PDF RENDERING ==========
        plt.rcParams['pdf.fonttype'] = 42
        plt.rcParams['ps.fonttype'] = 42
        plt.rcParams['path.simplify'] = True
        plt.rcParams['path.simplify_threshold'] = 1.0
        
        # Save figures
        plt.savefig('LLM_Coverage_Analysis_Complete.png', format='png',
                   dpi=300, bbox_inches='tight',
                   facecolor='white', pad_inches=0.3)
        
        plt.savefig('LLM_Coverage_Analysis_Complete.pdf',
                   format='pdf', dpi=300, bbox_inches='tight',
                   facecolor='white', pad_inches=0.3,
                   backend='pdf')
        
        plt.show()
        
        # ========== GENERATE BINARY CSV (NEW!) ==========
        print("\n📄 Generating binary retrieval CSV...")
        generate_binary_retrieval_csv(
            reordered_matrix,
            reordered_combinations,
            reordered_studies,
            analyzer,
            output_file='binary_retrieval_matrix.csv'
        )
        
        # ========== GENERATE REPORT (AFTER ALL VARIABLES ARE DEFINED) ==========
        print("\n📄 Generating comprehensive heatmap report...")
        report_file = generate_heatmap_report(
            reordered_matrix,
            reordered_combinations,
            df_matrix,
            reordered_consistency,
            reordered_studies,
            analyzer,
            output_file='report_dendrogram_heatmap.txt'
        )
        
        print("\n✅ Visualization complete!")
        print(f"✅ Report saved: {report_file}")
        print(f"✅ Binary CSV saved: binary_retrieval_matrix.csv")
        
        return g, reordered_matrix, reordered_combinations, df_matrix, reordered_consistency
    
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None, None, None

# ===== FUNCTION: GENERATE BINARY RETRIEVAL CSV (unchanged) =====
def generate_binary_retrieval_csv(matrix, combos, reordered_studies, analyzer, output_file='binary_retrieval_matrix.csv'):
    """Generate a CSV file with binary (0/1) retrieval status"""
    try:
        print("\n🔄 Creating binary retrieval matrix CSV...")
        
        def get_study_metadata(study_id, analyzer):
            study_row = analyzer.gold_standard_df[
                analyzer.gold_standard_df['Study ID'] == study_id
            ]
            
            if not study_row.empty:
                first_author = study_id.split(' ')[0].split(',')[0]
                year = study_row['Year'].iloc[0] if 'Year' in study_row.columns else 'N/A'
                
                try:
                    year_int = int(year) if pd.notna(year) else 'N/A'
                    author_year = f"{first_author} {year_int}" if year_int != 'N/A' else first_author
                except:
                    author_year = first_author
                
                metadata = {
                    'Study_ID': study_id,
                    'Author_Year': author_year,
                    'Title': study_row['Title'].iloc[0] if 'Title' in study_row.columns else 'Unknown',
                    'Year': year_int if 'year_int' in locals() else year,
                    'Journal': study_row['Journal/Conference'].iloc[0] if 'Journal/Conference' in study_row.columns else 'Unknown',
                    'Publisher': study_row['Publisher'].iloc[0] if 'Publisher' in study_row.columns else 'Unknown'
                }
                return metadata
            return None
        
        def create_combo_column_name(combo):
            llm_abbrev = combo['LLM'][:3]
            strategy_abbrev = combo['StrategyName'][:2]
            variation = combo['Variation']
            return f"{llm_abbrev}-{strategy_abbrev}{variation}"
        
        rows = []
        for study_idx, study_id in enumerate(reordered_studies):
            metadata = get_study_metadata(study_id, analyzer)
            
            if metadata:
                row = {
                    'Study_ID': metadata['Study_ID'],
                    'Author_Year': metadata['Author_Year'],
                    'Title': metadata['Title'],
                    'Year': metadata['Year'],
                    'Journal': metadata['Journal'],
                    'Publisher': metadata['Publisher']
                }
                
                total_retrieved = 0
                
                for combo_idx, combo in enumerate(combos):
                    col_name = create_combo_column_name(combo)
                    matrix_value = matrix[combo_idx, study_idx]
                    binary_value = 1 if matrix_value == 7 else 0
                    row[col_name] = binary_value
                    total_retrieved += binary_value
                
                row['Total_Retrieved'] = total_retrieved
                row['Retrieval_Rate'] = f"{(total_retrieved / len(combos) * 100):.2f}%"
                
                rows.append(row)
        
        df = pd.DataFrame(rows)
        
        metadata_cols = ['Study_ID', 'Author_Year', 'Title', 'Year', 'Journal', 'Publisher']
        summary_cols = ['Total_Retrieved', 'Retrieval_Rate']
        combo_cols = [col for col in df.columns if col not in metadata_cols + summary_cols]
        combo_cols_sorted = sorted(combo_cols)
        
        final_cols = metadata_cols + summary_cols + combo_cols_sorted
        df = df[final_cols]
        
        df.to_csv(output_file, index=False)
        
        print(f"✓ Binary retrieval matrix saved to: {output_file}")
        print(f"   Dimensions: {len(df)} studies × {len(combo_cols)} combinations")
        
        return df
    
    except Exception as e:
        print(f"❌ Error generating binary CSV: {e}")
        return None

# ===== GENERATE REPORT FUNCTION (unchanged, keeping it brief) =====
def generate_heatmap_report(matrix, combos, df, consistency, reordered_studies, analyzer, output_file='report_dendrogram_heatmap.txt'):
    """Generate comprehensive text report"""
    _STRATEGY_NAMES = {'1': 'Zero-Shot', '2': 'Few-Shot', '3': 'Persona', '4': 'CoT', '5': 'PICO'}

    def get_study_metadata(study_id, analyzer):
        study_row = analyzer.gold_standard_df[
            analyzer.gold_standard_df['Study ID'] == study_id
        ]
        if not study_row.empty:
            metadata = {
                'study_id': study_id,
                'title': study_row['Title'].iloc[0] if 'Title' in study_row.columns else 'Unknown Title',
                'year': study_row['Year'].iloc[0] if 'Year' in study_row.columns else 'N/A',
                'journal': study_row['Journal/Conference'].iloc[0] if 'Journal/Conference' in study_row.columns else 'Unknown',
                'publisher': study_row['Publisher'].iloc[0] if 'Publisher' in study_row.columns else 'Unknown',
                'authors': study_row['Authors'].iloc[0] if 'Authors' in study_row.columns else 'Unknown'
            }
            return metadata
        return None

    with open(output_file, 'w') as f:
        f.write("=" * 100 + "\n")
        f.write("COMPREHENSIVE LLM COVERAGE HEATMAP ANALYSIS REPORT\n")
        f.write("=" * 100 + "\n\n")

        f.write(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total Combinations: {len(combos)}\n")
        f.write(f"Total Studies: {matrix.shape[1]}\n")
        f.write(f"Total Cells: {matrix.size}\n\n")

        # Summary statistics
        total_retrieved_cells = np.sum(matrix == 7)
        overall_rate = (total_retrieved_cells / matrix.size) * 100

        f.write(f"Overall Statistics:\n")
        f.write(f"  Total Cells:            {matrix.size:,}\n")
        f.write(f"  Retrieved Cells (=7):   {total_retrieved_cells:,} ({overall_rate:.2f}%)\n")
        f.write(f"  Never Retrieved (=0):   {np.sum(matrix == 0):,} ({np.sum(matrix == 0) / matrix.size * 100:.2f}%)\n\n")

        f.write("-" * 100 + "\n")
        f.write("RETRIEVAL RATE BY PLATFORM-FORMULATION (top and bottom 10 by studies retrieved)\n")
        f.write("-" * 100 + "\n")
        combo_retrieved = (matrix == 7).sum(axis=1)
        combo_order = np.argsort(-combo_retrieved)
        f.write(f"{'Platform':<14}{'Strategy':<12}{'Variation':<10}{'Studies Retrieved':>18}{'% of Studies':>14}\n")
        n_studies = matrix.shape[1]
        for rank, idx in enumerate(list(combo_order[:10]) + (['...'] if len(combo_order) > 20 else []) + list(combo_order[-10:])):
            if idx == '...':
                f.write("  ...\n")
                continue
            c = combos[idx]
            n = int(combo_retrieved[idx])
            strat_label = _STRATEGY_NAMES.get(str(c.get('Strategy', '?')), c.get('Strategy', '?'))
            var_label = f"V{c.get('Variation', '?')}"
            f.write(f"{c.get('LLM', '?'):<14}{strat_label:<12}{var_label:<10}"
                    f"{n:>18}{n / n_studies * 100:>13.1f}%\n")
        f.write("\n")

        f.write("-" * 100 + "\n")
        f.write("NEVER-RETRIEVED STUDIES (retrieved by 0 of the displayed combinations)\n")
        f.write("-" * 100 + "\n")
        study_retrieved = (matrix == 7).sum(axis=0)
        never_idx = np.where(study_retrieved == 0)[0]
        f.write(f"Count: {len(never_idx)} of {n_studies} studies\n\n")
        for j in never_idx:
            sid = reordered_studies[j]
            meta = get_study_metadata(sid, analyzer)
            if meta:
                f.write(f"  - {meta['title']} ({meta['year']}, {meta['journal']}, {meta['publisher']})\n")
            else:
                f.write(f"  - {sid} (metadata not found)\n")
        f.write("\n" + "=" * 100 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 100 + "\n")

    print(f"✓ Report saved to: {output_file}")
    return output_file

# Execute
if 'base_results' in globals() and base_results:
    print("🔍 Creating heatmap with THINNER separators...")
    g, matrix, combos, df, consistency = create_dendrogram_heatmap()
    if g is not None:
        print("✅ Heatmap completed with thinner separators!")
    else:
        print("❌ Failed to create heatmap")
else:
    print("❌ No base results available")

## Cell 10 — SANKEY

*Figure 6 (from derived data)*


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from collections import defaultdict
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def create_llm_study_journal_sankey_with_colored_links():
    """Create 3-column Sankey diagram with Study→Journal links colored by which LLMs retrieved them"""
    print("\n" + "="*80)
    print("  3-COLUMN SANKEY DIAGRAM: LLMs → STUDIES → JOURNALS")
    print("  Study→Journal links colored by retrieval status")
    print("="*80)
    
    # ========== CONFIGURATION ==========
    CONFIG = {
        'llms': ['Ai2 Finder', 'Consensus', 'Claude', 'ChatGPT', 'Gemini'],
        'llm_colors': {
            'Ai2 Finder': 'rgba(155, 139, 126, 0.8)',
            'Consensus': 'rgba(123, 143, 163, 0.8)',
            'Claude': 'rgba(161, 122, 116, 0.8)',
            'ChatGPT': 'rgba(116, 139, 117, 0.8)',
            'Gemini': 'rgba(107, 142, 158, 0.8)',
        },
        'study_color': 'rgba(200, 200, 200, 0.8)',
        'journal_color': 'rgba(150, 150, 150, 0.8)',
        'link_opacity_llm_study': 0.3,
        'link_opacity_study_journal': 0.5,
        # Colors for different retrieval statuses
        'status_colors': {
            1: 'rgba(231, 76, 60, 0.5)',    # 1 LLM - Red
            2: 'rgba(230, 126, 34, 0.5)',   # 2 LLMs - Orange
            3: 'rgba(241, 196, 15, 0.5)',   # 3 LLMs - Yellow
            4: 'rgba(46, 204, 113, 0.5)',   # 4 LLMs - Green
            5: 'rgba(52, 152, 219, 0.5)',   # 5 LLMs - Blue
        }
    }
    
    def clean_journal_name(journal_name):
        """Clean journal names for display"""
        import re
        journal = str(journal_name).strip()
        journal = re.sub(r'\s*\((Journal|Conference.*?)\)\s*$', '', journal)
        if len(journal) > 45:
            journal = journal[:42] + "..."
        return journal
    
    def create_study_label(study_id, study_row):
        """Create compact study label"""
        first_author = study_id.split(' ')[0].split(',')[0]
        if 'Year' in study_row.columns:
            year = study_row['Year'].iloc[0]
            if pd.notna(year) and year != 'N/A':
                try:
                    return f"{first_author} ({int(year)})"
                except:
                    return first_author
        return first_author
    
    # ========== EXTRACT DATA USING EXACT SAME LOGIC ==========
    analyzer = base_results['analyzer']
    
    print("\n📊 Step 1: Creating comprehensive study mapping...")
    # PART 1: CREATE COMPREHENSIVE STUDY MAPPING
    study_mapping = {}
    for idx, row in analyzer.gold_standard_df.iterrows():
        study_id = row.get('Study ID', f'Study_{idx}')
        identifiers = set()
        
        if 'Title' in row and pd.notna(row['Title']):
            title_normalized = analyzer.normalize_identifier_consistent(row['Title'])
            if title_normalized:
                identifiers.add(title_normalized)
        
        if 'DOI' in row and pd.notna(row['DOI']):
            doi_normalized = analyzer.normalize_identifier_consistent(row['DOI'])
            if doi_normalized:
                identifiers.add(doi_normalized)
        
        study_id_normalized = analyzer.normalize_identifier_consistent(study_id)
        if study_id_normalized:
            identifiers.add(study_id_normalized)
        
        study_mapping[study_id] = {
            'identifiers': identifiers,
            'row_data': row
        }
    
    print(f"✓ Created mapping for {len(study_mapping)} gold standard studies")
    
    # PART 2: COLLECT ALL MATCHES FROM RECALLED FILES - COUNT EVERY INSTANCE
    print("\n📊 Step 2: Collecting matches from recalled files (counting all instances)...")
    llm_names = CONFIG['llms']
    
    # Collect matches per LLM using ORIGINAL LOGIC first to compare
    llm_retrieved_studies_original = defaultdict(set)
    
    for filename, info in analyzer.recalled_files.items():
        llm = info['llm']
        parsed = analyzer.parse_filename(filename)
        if not parsed:
            continue
        
        try:
            df = pd.read_csv(info['path'])
            matches_raw = set()
            
            for col in ['DOI', 'Study name', 'Title', 'LLM Title']:
                if col in df.columns:
                    title_data = df[col].dropna().astype(str).str.strip()
                    for title in title_data:
                        normalized = analyzer.normalize_identifier_consistent(title)
                        if normalized:
                            matches_raw.add(normalized)
            
            for study_id, study_info in study_mapping.items():
                for identifier in study_info['identifiers']:
                    if identifier in matches_raw:
                        llm_retrieved_studies_original[llm].add(study_id)
                        break
        except Exception as e:
            continue
    
    # Now collect with counts
    llm_retrieved_studies = defaultdict(set)  # For unique studies
    llm_study_retrieval_counts = defaultdict(lambda: defaultdict(int))  # For counting instances
    
    for filename, info in analyzer.recalled_files.items():
        llm = info['llm']
        parsed = analyzer.parse_filename(filename)
        if not parsed:
            continue
        
        try:
            df = pd.read_csv(info['path'])
            
            # For this FILE, track which studies we've found
            file_matches = defaultdict(int)  # study_id -> count in this file
            
            for col in ['DOI', 'Study name', 'Title', 'LLM Title']:
                if col in df.columns:
                    title_data = df[col].dropna().astype(str).str.strip()
                    for title in title_data:
                        normalized = analyzer.normalize_identifier_consistent(title)
                        if normalized:
                            # Find which study this matches
                            for study_id, study_info in study_mapping.items():
                                if normalized in study_info['identifiers']:
                                    file_matches[study_id] += 1
                                    break
            
            # Add the counts from this file to the overall counts
            for study_id, count in file_matches.items():
                llm_retrieved_studies[llm].add(study_id)
                llm_study_retrieval_counts[llm][study_id] += count
                
        except Exception as e:
            continue
    
    # Compare the two approaches
    print(f"\n🔍 DEBUGGING - Comparing retrieval methods:")
    all_original = set().union(*llm_retrieved_studies_original.values())
    all_new = set().union(*llm_retrieved_studies.values())
    
    print(f"  Original method: {len(all_original)} unique studies")
    print(f"  New method: {len(all_new)} unique studies")
    
    if all_original != all_new:
        missing_in_new = all_original - all_new
        extra_in_new = all_new - all_original
        
        if missing_in_new:
            print(f"\n  ⚠ Studies in ORIGINAL but NOT in NEW ({len(missing_in_new)}):")
            for study_id in sorted(missing_in_new):
                llms_that_found = [llm for llm in llm_names if study_id in llm_retrieved_studies_original[llm]]
                print(f"    - {study_id[:70]} (found by: {', '.join(llms_that_found)})")
        
        if extra_in_new:
            print(f"\n  ⚠ Studies in NEW but NOT in ORIGINAL ({len(extra_in_new)}):")
            for study_id in sorted(extra_in_new):
                llms_that_found = [llm for llm in llm_names if study_id in llm_retrieved_studies[llm]]
                print(f"    - {study_id[:70]} (found by: {', '.join(llms_that_found)})")
    else:
        print("  ✓ Both methods found the same studies!")
    
    print(f"\n✓ Collected matches for {len(llm_names)} LLMs:")
    for llm in llm_names:
        unique_studies = len(llm_retrieved_studies[llm])
        total_retrievals = sum(llm_study_retrieval_counts[llm].values())
        original_count = len(llm_retrieved_studies_original[llm])
        
        status = "✓" if unique_studies == original_count else "⚠"
        print(f"  {status} {llm}: {unique_studies} unique studies (original: {original_count}), {total_retrievals} total retrieval instances")
    
    # ========== BUILD FLOWS WITH RETRIEVAL STATUS TRACKING ==========
    print("\n📊 Step 3: Building flows with retrieval status tracking...")
    flows = {
        'llm_to_study': defaultdict(lambda: defaultdict(int)),
        'study_to_journal': defaultdict(lambda: defaultdict(int))
    }
    all_studies_in_flow = set()
    all_journals_in_flow = set()
    study_to_journal_map = {}
    study_retrieval_status = {}  # Track which LLMs retrieved each study
    
    # Track which studies were retrieved but didn't make it to the flow
    all_retrieved_study_ids = set().union(*llm_retrieved_studies.values())
    missing_studies = []
    
    # Build LLM → Study flows and track retrieval status
    for llm in llm_names:
        for study_id in llm_retrieved_studies[llm]:
            study_row = analyzer.gold_standard_df[
                analyzer.gold_standard_df['Study ID'] == study_id
            ]
            
            if study_row.empty:
                missing_studies.append({
                    'study_id': study_id,
                    'reason': 'Study not found in gold standard DataFrame',
                    'llms': [llm]
                })
                continue
            
            # Check if journal/conference information exists
            if 'Journal/Conference' not in study_row.columns:
                missing_studies.append({
                    'study_id': study_id,
                    'reason': 'Journal/Conference column missing',
                    'llms': [llm]
                })
                continue
            
            journal = study_row['Journal/Conference'].iloc[0]
            if pd.isna(journal) or str(journal).strip() == '':
                missing_studies.append({
                    'study_id': study_id,
                    'reason': 'Journal/Conference value is empty or NaN',
                    'llms': [llm]
                })
                continue
            
            # Study has all required info - add to flow WITH ACTUAL COUNT
            study_label = create_study_label(study_id, study_row)
            journal = clean_journal_name(journal)
            
            # Use the actual retrieval count instead of just 1
            retrieval_count = llm_study_retrieval_counts[llm][study_id]
            flows['llm_to_study'][llm][study_label] = retrieval_count
            
            all_studies_in_flow.add(study_label)
            study_to_journal_map[study_label] = journal
            all_journals_in_flow.add(journal)
            
            # Track which LLMs retrieved this study
            if study_label not in study_retrieval_status:
                study_retrieval_status[study_label] = set()
            study_retrieval_status[study_label].add(llm)
    
    # Consolidate missing studies by study_id
    missing_by_id = {}
    for item in missing_studies:
        sid = item['study_id']
        if sid not in missing_by_id:
            missing_by_id[sid] = {
                'reason': item['reason'],
                'llms': set(item['llms'])
            }
        else:
            missing_by_id[sid]['llms'].update(item['llms'])
    
    print(f"  After LLM→Study: {len(all_studies_in_flow)} studies in flow")
    print(f"  Missing from flow: {len(missing_by_id)} studies")
    
    # Build Study → Journal flows - sum up all retrieval counts for each study
    for study_label, journal in study_to_journal_map.items():
        total_retrievals = sum(flows['llm_to_study'][llm].get(study_label, 0) for llm in llm_names)
        flows['study_to_journal'][study_label][journal] = total_retrievals
    
    print(f"  After Study→Journal: {len(all_journals_in_flow)} journals")
    
    studies = sorted(all_studies_in_flow)
    journals = sorted(all_journals_in_flow)
    llms = llm_names
    
    print(f"\n✓ Flow structure:")
    print(f"  - {len(llms)} LLMs")
    print(f"  - {len(studies)} studies")
    print(f"  - {len(journals)} journals")
    
    # ========== BUILD SANKEY DATA STRUCTURE ==========
    print("\n📊 Step 4: Building Sankey diagram data...")
    
    all_nodes = llms + studies + journals
    node_dict = {node: idx for idx, node in enumerate(all_nodes)}
    
    # Create node labels with different font sizes using HTML
    formatted_labels = []
    for node in all_nodes:
        if node in llms:
            # Large font for LLMs
            formatted_labels.append(f'<b><span style="font-size: 22px">{node}</span></b>')
        elif node in studies:
            # Small font for studies
            formatted_labels.append(f'<span style="font-size: 12px">{node}</span>')
        else:
            # Small font for journals
            formatted_labels.append(f'<span style="font-size: 12px">{node}</span>')
    
    # Create node colors
    node_colors = []
    for node in all_nodes:
        if node in CONFIG['llm_colors']:
            node_colors.append(CONFIG['llm_colors'][node])
        elif node in studies:
            node_colors.append(CONFIG['study_color'])
        else:
            node_colors.append(CONFIG['journal_color'])
    
    # Build links
    sources = []
    targets = []
    values = []
    link_colors = []
    link_labels = []
    
    # Links: LLM → Study (colored by LLM) - NOW WITH ACTUAL RETRIEVAL COUNTS
    print("  Creating LLM → Study links...")
    for llm in llms:
        llm_idx = node_dict[llm]
        llm_color = CONFIG['llm_colors'][llm]
        for study, count in flows['llm_to_study'][llm].items():
            if count > 0:
                study_idx = node_dict[study]
                sources.append(llm_idx)
                targets.append(study_idx)
                values.append(count)  # This is now the actual retrieval count
                link_colors.append(llm_color.replace('0.8', str(CONFIG['link_opacity_llm_study'])))
                link_labels.append(f"{llm} retrieved this study {count} time(s) across all strategies")
    
    # Links: Study → Journal (colored by number of LLMs that retrieved the study)
    print("  Creating Study → Journal links (colored by retrieval status)...")
    for study, journals_dict in flows['study_to_journal'].items():
        study_idx = node_dict[study]
        
        # Get number of LLMs that retrieved this study
        num_llms = len(study_retrieval_status[study])
        status_color = CONFIG['status_colors'].get(num_llms, 'rgba(150, 150, 150, 0.5)')
        
        for journal, count in journals_dict.items():
            journal_idx = node_dict[journal]
            sources.append(study_idx)
            targets.append(journal_idx)
            values.append(count)  # Total retrievals across all LLMs
            link_colors.append(status_color)
            
            # Create informative label
            llms_list = ', '.join(sorted(study_retrieval_status[study]))
            link_labels.append(f"Retrieved by {num_llms} LLM(s): {llms_list} (Total: {count} times)")
    
    print(f"✓ Created {len(sources)} links")
    
    # Print link thickness statistics
    print(f"\n📊 Link thickness statistics:")
    llm_study_count = sum(1 for llm in llms for study in flows['llm_to_study'][llm])
    llm_study_values = values[:llm_study_count]
    if llm_study_values:
        print(f"  LLM→Study links: min={min(llm_study_values)}, max={max(llm_study_values)}, avg={sum(llm_study_values)/len(llm_study_values):.1f}")
        print(f"  Studies retrieved 10+ times: {sum(1 for v in llm_study_values if v >= 10)}")
        print(f"  Studies retrieved 5-9 times: {sum(1 for v in llm_study_values if 5 <= v < 10)}")
        print(f"  Studies retrieved 1-4 times: {sum(1 for v in llm_study_values if 1 <= v < 5)}")
    
    # ========== CREATE SANKEY DIAGRAM ==========
    print("\n📊 Step 5: Creating Sankey visualization...")
    
    fig = go.Figure(data=[go.Sankey(
        arrangement='snap',
        node=dict(
            pad=20,
            thickness=25,
            line=dict(color="white", width=2),
            label=formatted_labels,  # Use HTML formatted labels
            color=node_colors,
            customdata=[f"{node}" for node in all_nodes],
            hovertemplate='%{customdata}<br />Total Flow: %{value}<extra></extra>',
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors,
            customdata=link_labels,
            hovertemplate='%{source.label} → %{target.label}<br />%{customdata}<extra></extra>',
        )
    )])
    
    fig.update_layout(
        font=dict(size=18, color='black', family='Arial, sans-serif'),
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=1200,
        width=1800,
        margin=dict(l=20, r=300, t=120, b=80),
        annotations=[
            dict(
                x=0.05, y=1.08,
                xref='paper', yref='paper',
                text=f'<b>LLM Models</b><br>({len(llms)})',
                showarrow=False,
                font=dict(size=22),
                align='center'
            ),
            dict(
                x=0.5, y=1.08,
                xref='paper', yref='paper',
                text=f'<b>Retrieved Studies</b><br>({len(all_original)})',  # Use original count for display
                showarrow=False,
                font=dict(size=22),
                align='center'
            ),
            dict(
                x=0.95, y=1.08,
                xref='paper', yref='paper',
                text=f'<b>Source Journals</b><br>({len(journals)})',
                showarrow=False,
                font=dict(size=22),
                align='center'
            ),
            # Legend for Study→Journal link colors
            dict(
                x=0.02, y=-0.08,
                xref='paper', yref='paper',
                text='<span style="color: #E74C3C;">■</span> Retrieved by 1 LLM  ' +
                     '<span style="color: #E67E22;">■</span> Retrieved by 2 LLMs  ' +
                     '<span style="color: #F1C40F;">■</span> Retrieved by 3 LLMs  ' +
                     '<span style="color: #2ECC71;">■</span> Retrieved by 4 LLMs  ' +
                     '<span style="color: #3498DB;">■</span> Retrieved by 5 LLMs (All)',
                showarrow=False,
                font=dict(size=26),
                align='left',
                xanchor='left'
            ),
        ]
    )
    
    # Save as HTML
    fig.write_html('LLM_Study_Journal_Sankey_Colored_Status.html')
    print("✓ Saved: LLM_Study_Journal_Sankey_Colored_Status.html")
    
    # Also save as static image
    try:
        fig.write_image('LLM_Study_Journal_Sankey_Colored_Status.png', width=1800, height=1200, scale=2)
        print("✓ Saved: LLM_Study_Journal_Sankey_Colored_Status.png")
    except:
        print("⚠ Could not save PNG (requires kaleido). HTML version saved.")
    
    fig.show()
    
    # ========== PRINT STATISTICS ==========
    print("\n" + "="*80)
    print("STATISTICS - Retrieval Status Distribution")
    print("="*80)
    
    print("\nStudies by Number of LLMs That Retrieved Them:")
    for num_llms in range(5, 0, -1):
        studies_with_n = [s for s in studies if len(study_retrieval_status[s]) == num_llms]
        percentage = len(studies_with_n) / len(studies) * 100 if len(studies) > 0 else 0
        print(f"  {num_llms} LLM(s): {len(studies_with_n):3d} studies ({percentage:5.1f}%)")
    
    print("\nJournal Coverage by Retrieval Status:")
    journal_status_breakdown = defaultdict(lambda: defaultdict(int))
    for study, journal in study_to_journal_map.items():
        num_llms = len(study_retrieval_status[study])
        journal_status_breakdown[journal][num_llms] += 1
    
    print("\nTop 10 Journals by Total Studies:")
    journal_totals = {j: sum(counts.values()) for j, counts in journal_status_breakdown.items()}
    for i, (journal, total) in enumerate(sorted(journal_totals.items(), key=lambda x: x[1], reverse=True)[:10], 1):
        print(f"  {i:2d}. {journal[:45]:45s}: {total:2d} studies")
        breakdown = journal_status_breakdown[journal]
        status_str = ', '.join([f"{n}LLM={breakdown[n]}" for n in range(5, 0, -1) if breakdown[n] > 0])
        print(f"      [{status_str}]")
    
    print("\nLLM-specific Statistics:")
    for llm in llms:
        studies_count = len(llm_retrieved_studies[llm])
        total_retrievals = sum(llm_study_retrieval_counts[llm].values())
        # Count how many of this LLM's studies are unique vs shared
        unique = sum(1 for s in flows['llm_to_study'][llm].keys() if len(study_retrieval_status[s]) == 1)
        shared = studies_count - unique
        print(f"  {llm:12s}: {studies_count:2d} unique studies, {total_retrievals:3d} total retrievals ({unique} unique, {shared} shared)")
    
    all_retrieved = len(all_original)  # Use original count
    studies_in_flow = len(studies)
    
    print(f"\n✓ Verification:")
    print(f"  Total studies in gold standard: {len(study_mapping)}")
    print(f"  Studies retrieved by at least one LLM: {all_retrieved} ({all_retrieved/len(study_mapping)*100:.1f}%)")
    print(f"  Studies displayed in Sankey diagram: {studies_in_flow} ({studies_in_flow/len(study_mapping)*100:.1f}%)")
    
    if missing_by_id:
        print(f"\n⚠ MISSING STUDIES FROM FLOW ({len(missing_by_id)} studies):")
        print("=" * 80)
        for study_id, info in sorted(missing_by_id.items()):
            llms_str = ', '.join(sorted(info['llms']))
            print(f"  • {study_id[:60]:60s}")
            print(f"    Reason: {info['reason']}")
            print(f"    Retrieved by: {llms_str}")
            
            # Get more info from gold standard
            study_row = analyzer.gold_standard_df[
                analyzer.gold_standard_df['Study ID'] == study_id
            ]
            if not study_row.empty:
                if 'Journal/Conference' in study_row.columns:
                    journal_val = study_row['Journal/Conference'].iloc[0]
                    print(f"    Journal value: {repr(journal_val)}")
                if 'Title' in study_row.columns:
                    title_val = study_row['Title'].iloc[0]
                    if pd.notna(title_val):
                        print(f"    Title: {str(title_val)[:70]}...")
            print()
    
    print("\n" + "="*80)
    print("✓ Sankey diagram with colored retrieval status completed!")
    print("="*80)
    
    return fig, study_retrieval_status, study_to_journal_map, missing_by_id

# Execute
if 'base_results' in globals() and base_results:
    print("🔍 Creating LLM → Study → Journal Sankey with colored retrieval status...")
    fig, retrieval_status, study_journal, missing = create_llm_study_journal_sankey_with_colored_links()
else:
    print("❌ No base results available")

## Cell 11 — PUBLISHER REPORT

*Venue/year non-retrieval*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from datetime import datetime
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ===== GLOBAL CONFIG =====
CONFIG = {
    'llms': ['Ai2 Finder', 'Consensus', 'Claude', 'ChatGPT', 'Gemini'],
    'strategies': ['1', '2', '3', '4', '5'],
    'variations': ['0', '1', '2'],
    'strategy_names': {
        '1': 'Zero-Shot', '2': 'Few-Shot', '3': 'Persona',
        '4': 'CoT', '5': 'PICO'
    },
}

def format_p_value(p_value, threshold=0.05):
    """Format p-value for display"""
    if pd.isna(p_value):
        return "p = NA"
    
    if p_value < 0.001:
        return "p < 0.001"
    elif p_value < threshold:
        return f"p < {threshold}"
    elif p_value < 0.01:
        return f"p = {p_value:.3f}"
    else:
        return f"p = {p_value:.2f}"

def standardize_publisher_name(publisher_name):
    """Standardize publisher names for consistency"""
    publisher = str(publisher_name)
    if 'Wolters Kluwer' in publisher: return 'Wolters Kluwer'
    if 'Taylor & Francis' in publisher or 'Dove Medical Press' in publisher: return 'Taylor & Francis'
    if 'Springer' in publisher: return 'Springer'
    if 'Elsevier' in publisher: return 'Elsevier'
    if 'IEEE' in publisher: return 'IEEE'
    if 'JMIR' in publisher: return 'JMIR'
    if 'SCITEPRESS' in publisher: return 'SCITEPRESS'
    if 'IOS Press' in publisher: return 'IOS Press'
    if 'Oxford University Press' in publisher or 'OUP' in publisher: return 'Oxford University Press'
    if 'MDPI' in publisher: return 'MDPI'
    if 'Wiley' in publisher: return 'Wiley'
    if 'Sage' in publisher: return 'Sage'
    if 'BMJ' in publisher: return 'BMJ'
    if 'AMIA' in publisher: return 'AMIA'
    if 'Frontiers' in publisher: return 'Frontiers'
    if 'Nature Publishing' in publisher: return 'Nature Publishing'
    if 'PLOS' in publisher or 'Public Library of Science' in publisher: return 'PLOS'
    if 'Cambridge University Press' in publisher: return 'Cambridge University Press'
    if 'Lippincott' in publisher: return 'Lippincott'
    if 'Karger' in publisher: return 'Karger'
    if 'Thieme' in publisher: return 'Thieme'
    if 'Mary Ann Liebert' in publisher: return 'Mary Ann Liebert'
    if 'American Medical Association' in publisher or 'AMA' in publisher: return 'American Medical Association'
    return publisher

def plot_publisher_retrieval_by_llm_separate(base_results):
    """
    Publisher retrieval analysis with properly defined statistical tests:
    (i) Overall non-retrieval (never retrieved by any platform)
    (ii) Platform-specific non-retrieval (not retrieved by a given platform)
    """
    print("\n" + "="*80)
    print("  PUBLISHER RETRIEVAL ANALYSIS WITH STATISTICAL TESTS")
    print("="*80)
    
    # ========== EXTRACT DATA ==========
    analyzer = base_results['analyzer']
    study_mapping = analyzer.create_study_identifier_mapping()
    
    print(f"✓ Analyzing {len(study_mapping)} gold standard studies")
    
    # Build combination data
    print("\n📊 Building combination data...")
    combinations_data = []
    
    for llm in CONFIG['llms']:
        for strategy in CONFIG['strategies']:
            for variation in CONFIG['variations']:
                studies_by_run = {}
                
                for filename, info in analyzer.recalled_files.items():
                    if (info['llm'] == llm and info['strategy'] == strategy and info['variation'] == variation):
                        try:
                            df = analyzer.load_csv_file_content(info['path'])
                            if not df.empty:
                                raw_matches, _= analyzer.extract_studies_from_recalled_files(df)
                                run_matches = set()
                                for match in raw_matches:
                                    normalized = analyzer.normalize_identifier_consistent(match)
                                    if normalized:
                                        run_matches.add(normalized)
                                studies_by_run[info['run']] = run_matches
                        except Exception as e:
                            continue
                
                combinations_data.append({
                    'LLM': llm,
                    'Strategy': strategy,
                    'Variation': variation,
                    'studies_by_run': studies_by_run
                })
    
    print(f"✓ Built {len(combinations_data)} combinations")
    
    # ========== CALCULATE RETRIEVAL ==========
    print("\n📊 Calculating retrieval patterns...")
    
    llm_retrieval_by_study = {study_id: set() for study_id in study_mapping.keys()}
    
    for study_id in study_mapping.keys():
        study_info = study_mapping[study_id]
        
        for combo in combinations_data:
            combo_retrieved = False
            
            for run, run_studies in combo['studies_by_run'].items():
                for identifier in study_info['identifiers']:
                    if identifier in run_studies:
                        combo_retrieved = True
                        break
                if combo_retrieved:
                    break
            
            if combo_retrieved:
                llm_retrieval_by_study[study_id].add(combo['LLM'])
    
    study_llm_counts = {study_id: len(llms) for study_id, llms in llm_retrieval_by_study.items()}
    
    # ========== CATEGORIZE BY PUBLISHER ==========
    publisher_llm_data = defaultdict(lambda: defaultdict(int))
    publisher_by_llm = {llm: defaultdict(lambda: {'retrieved': 0, 'not_retrieved': 0}) 
                        for llm in CONFIG['llms']}
    
    for study_id in study_mapping.keys():
        study_row = analyzer.gold_standard_df[analyzer.gold_standard_df['Study ID'] == study_id]
        
        if not study_row.empty:
            publisher = standardize_publisher_name(str(study_row['Publisher'].iloc[0]))
            
            # Overall categorization
            llm_count = study_llm_counts[study_id]
            publisher_llm_data[publisher][f"{llm_count} LLMs"] += 1
            
            # Per-LLM categorization
            for llm in CONFIG['llms']:
                if llm in llm_retrieval_by_study[study_id]:
                    publisher_by_llm[llm][publisher]['retrieved'] += 1
                else:
                    publisher_by_llm[llm][publisher]['not_retrieved'] += 1
    
    # ========== STATISTICAL TESTS ==========
    print("\n📊 Performing Statistical Tests...")
    
    # Test (i): Overall non-retrieval (never retrieved by ANY platform)
    print("  Test (i): Overall non-retrieval association")
    
    overall_test_data = []
    publisher_list = []
    
    for publisher in publisher_llm_data.keys():
        retrieved_by_any = sum(publisher_llm_data[publisher][f"{i} LLMs"] for i in range(1, 6))
        never_retrieved = publisher_llm_data[publisher]["0 LLMs"]
        overall_test_data.append([retrieved_by_any, never_retrieved])
        publisher_list.append(publisher)
    
    # Fisher's exact: each publisher vs all others (overall non-retrieval)
    fisher_overall = {}
    p_values_overall = []
    
    total_retrieved = sum(c[0] for c in overall_test_data)
    total_not_retrieved = sum(c[1] for c in overall_test_data)
    
    for idx, publisher in enumerate(publisher_list):
        pub_retrieved = overall_test_data[idx][0]
        pub_not_retrieved = overall_test_data[idx][1]
        
        other_retrieved = total_retrieved - pub_retrieved
        other_not_retrieved = total_not_retrieved - pub_not_retrieved
        
        fisher_table = [[pub_retrieved, pub_not_retrieved],
                       [other_retrieved, other_not_retrieved]]
        
        odds_ratio, p_value = stats.fisher_exact(fisher_table)
        
        fisher_overall[publisher] = {
            'odds_ratio': odds_ratio,
            'p_value': p_value,
            'retrieved': pub_retrieved,
            'not_retrieved': pub_not_retrieved,
            'total': pub_retrieved + pub_not_retrieved
        }
        
        p_values_overall.append(p_value)
    
    # FDR correction
    reject_overall, pvals_corrected_overall, _, _ = multipletests(p_values_overall, alpha=0.05, method='fdr_bh')
    
    for idx, publisher in enumerate(publisher_list):
        fisher_overall[publisher]['p_corrected'] = pvals_corrected_overall[idx]
        fisher_overall[publisher]['significant'] = reject_overall[idx]
    
    sig_overall = sum(reject_overall)
    print(f"    Significant publishers (overall): {sig_overall}")
    
    # Test (ii): Platform-specific non-retrieval
    print("\n  Test (ii): Platform-specific non-retrieval")
    
    fisher_by_platform = {}
    
    for llm in CONFIG['llms']:
        print(f"    Testing {llm}...")
        
        llm_test_data = []
        llm_pub_list = []
        
        for publisher in publisher_list:
            retrieved = publisher_by_llm[llm][publisher]['retrieved']
            not_retrieved = publisher_by_llm[llm][publisher]['not_retrieved']
            llm_test_data.append([retrieved, not_retrieved])
            llm_pub_list.append(publisher)
        
        # Fisher's exact for each publisher
        llm_fisher = {}
        llm_p_values = []
        
        llm_total_retrieved = sum(c[0] for c in llm_test_data)
        llm_total_not_retrieved = sum(c[1] for c in llm_test_data)
        
        for idx, publisher in enumerate(llm_pub_list):
            pub_retrieved = llm_test_data[idx][0]
            pub_not_retrieved = llm_test_data[idx][1]
            
            other_retrieved = llm_total_retrieved - pub_retrieved
            other_not_retrieved = llm_total_not_retrieved - pub_not_retrieved
            
            fisher_table = [[pub_retrieved, pub_not_retrieved],
                           [other_retrieved, other_not_retrieved]]
            
            odds_ratio, p_value = stats.fisher_exact(fisher_table)
            
            llm_fisher[publisher] = {
                'odds_ratio': odds_ratio,
                'p_value': p_value,
                'retrieved': pub_retrieved,
                'not_retrieved': pub_not_retrieved,
                'total': pub_retrieved + pub_not_retrieved
            }
            
            llm_p_values.append(p_value)
        
        # FDR correction
        reject_llm, pvals_corrected_llm, _, _ = multipletests(llm_p_values, alpha=0.05, method='fdr_bh')
        
        for idx, publisher in enumerate(llm_pub_list):
            llm_fisher[publisher]['p_corrected'] = pvals_corrected_llm[idx]
            llm_fisher[publisher]['significant'] = reject_llm[idx]
        
        fisher_by_platform[llm] = llm_fisher
        print(f"      Significant publishers: {sum(reject_llm)}")
    
    # ========== CREATE VISUALIZATION ==========
    print("\n📊 Creating visualization...")
    
    category_order = ['0 LLMs', '1 LLMs', '2 LLMs', '3 LLMs', '4 LLMs', '5 LLMs']
    legend_labels = {
        '0 LLMs': 'Never Retrieved by Any LLM',
        '1 LLMs': 'Retrieved by 1 LLM',
        '2 LLMs': 'Retrieved by 2 LLMs',
        '3 LLMs': 'Retrieved by 3 LLMs',
        '4 LLMs': 'Retrieved by 4 LLMs',
        '5 LLMs': 'Retrieved by 5 LLMs'
    }
    colors_llm = ['#E74C3C', '#EB984E', '#F7DC6F', '#52BE80', '#27AE60', '#1E8449']
    
    pub_llm_list = []
    for pub, categories in publisher_llm_data.items():
        for category in category_order:
            pub_llm_list.append({
                'Publisher': pub,
                'Category': category,
                'Count': categories.get(category, 0)
            })
    
    pub_llm_df = pd.DataFrame(pub_llm_list)
    pub_llm_df = pub_llm_df.groupby(['Publisher', 'Category'], as_index=False)['Count'].sum()
    
    publisher_totals = pub_llm_df.groupby('Publisher')['Count'].sum().sort_values(ascending=False)
    all_publishers = publisher_totals.index.tolist()
    
    pub_pivot = pub_llm_df.pivot(index='Publisher', columns='Category', values='Count').fillna(0)
    
    for cat in category_order:
        if cat not in pub_pivot.columns:
            pub_pivot[cat] = 0
    
    pub_pivot = pub_pivot[category_order]
    pub_pivot['Total'] = pub_pivot.sum(axis=1)
    pub_pivot = pub_pivot.sort_values('Total', ascending=True)
    max_total = int(pub_pivot['Total'].max())
    pub_pivot = pub_pivot.drop('Total', axis=1)
    
    num_publishers = len(pub_pivot)
    fig_height = max(12, num_publishers * 0.4)
    
    fig, ax = plt.subplots(figsize=(18, fig_height))
    
    pub_pivot.plot(kind='barh', stacked=True, ax=ax,
                  color=colors_llm, edgecolor='black', linewidth=0.5, alpha=0.85)
    
    # Add significance markers (overall non-retrieval)
    y_positions = {pub: idx for idx, pub in enumerate(pub_pivot.index)}
    
    for publisher, result in fisher_overall.items():
        if publisher in y_positions and result['significant']:
            y_pos = y_positions[publisher]
            x_pos = max_total + 0.5
            
            if result['p_corrected'] < 0.001:
                marker = '***'
            elif result['p_corrected'] < 0.01:
                marker = '**'
            else:
                marker = '*'
            
            ax.text(x_pos, y_pos, marker, fontsize=12, va='center', ha='left', 
                   color='black', fontweight='bold')
    
    ax.set_xlabel('Number of Studies', fontsize=20, fontweight='bold')
    ax.set_ylabel('Publisher', fontsize=20, fontweight='bold')
    ax.set_title("Study Retrieval Distribution by Publisher", 
                fontsize=26, fontweight='bold', pad=20)
    
    ax.set_xticks(np.arange(0, max_total + 2, 1))
    ax.set_xlim(0, max_total + 1.5)
    
    handles, labels = ax.get_legend_handles_labels()
    new_labels = [legend_labels[label] for label in labels]
    ax.legend(handles, new_labels, title='Retrieval Status',
             bbox_to_anchor=(0.55, 0.1), loc='lower left',
             fontsize=20, title_fontsize=14, frameon=True,
             edgecolor='black', fancybox=False)
    
    footnote = "* q < 0.05, ** q < 0.01, *** q < 0.001 (Fisher's exact test, FDR-corrected, overall non-retrieval)"
    fig.text(0.5, 0.02, footnote, ha='center', fontsize=10, style='italic')
    
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=12)
    
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.savefig('Publisher_Retrieval_Distribution_All_Publishers.png',
                dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.show()
    
    print("\n✓ Visualization saved")
    
    # Return comprehensive data
    return {
        'publisher_llm_data': dict(publisher_llm_data),
        'pub_pivot': pub_pivot,
        'pub_llm_df': pub_llm_df,
        'all_publishers': all_publishers,
        'publisher_totals': publisher_totals,
        'standardize_publisher_name': standardize_publisher_name,
        'combinations_data': combinations_data,
        'study_mapping': study_mapping,
        'llm_retrieval_by_study': llm_retrieval_by_study,
        'publisher_by_llm': publisher_by_llm,
        'statistical_tests': {
            'overall_nonretrieval': {
                'description': 'Never retrieved by any platform vs retrieved by ≥1 platform',
                'fisher_exact': fisher_overall,
                'correction': 'fdr_bh',
                'alpha': 0.05
            },
            'platform_specific_nonretrieval': {
                'description': 'Not retrieved by given platform vs retrieved by that platform',
                'fisher_by_platform': fisher_by_platform,
                'correction': 'fdr_bh',
                'alpha': 0.05
            }
        }
    }

# ===== COMPREHENSIVE REPORT GENERATION =====
def generate_publisher_report(publisher_results, base_results, output_file='report_publisher_retrieval.txt'):
    """Generate report with properly defined publisher-level statistical tests"""
    analyzer = base_results['analyzer']
    study_mapping = publisher_results['study_mapping']
    statistical_tests = publisher_results['statistical_tests']
    
    with open(output_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("PUBLISHER RETRIEVAL ANALYSIS WITH STATISTICAL TESTS\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total Publishers: {len(publisher_results['all_publishers'])}\n")
        f.write(f"Total Studies: {len(analyzer.gold_standard_df)}\n\n")
        
        # ===== SECTION 1: STATISTICAL TESTS =====
        f.write("=" * 80 + "\n")
        f.write("SECTION 1: STATISTICAL ANALYSES\n")
        f.write("=" * 80 + "\n\n")
        
        # Test (i): Overall non-retrieval
        f.write("1.1 Overall Non-Retrieval Association\n")
        f.write("-" * 80 + "\n\n")
        
        overall_stats = statistical_tests['overall_nonretrieval']
        fisher_overall = overall_stats['fisher_exact']
        
        f.write(f"Research Question: Are publication characteristics (publisher) associated\n")
        f.write(f"                   with overall retrieval failure?\n\n")
        f.write(f"Method: Fisher's exact test with FDR correction\n")
        f.write(f"  - Definition: Each publisher vs all other publishers\n")
        f.write(f"  - Outcome: Never retrieved by ANY platform (0 LLMs) vs\n")
        f.write(f"             Retrieved by ≥1 platform (1-5 LLMs)\n")
        f.write(f"  - Correction: FDR (Benjamini-Hochberg) across {len(fisher_overall)} publishers\n")
        f.write(f"  - α = {overall_stats['alpha']}\n\n")
        
        f.write(f"{'Publisher':<30} | {'Retrieved':<10} | {'Never':<10} | {'Total':<6} | ")
        f.write(f"{'OR':<8} | {'p-value':<12} | {'q-value':<12} | {'Sig.':<6}\n")
        f.write("-" * 105 + "\n")
        
        sorted_overall = sorted(fisher_overall.items(), key=lambda x: x[1]['p_corrected'])
        
        for publisher, result in sorted_overall:
            sig_marker = "***" if result['p_corrected'] < 0.001 else "**" if result['p_corrected'] < 0.01 else "*" if result['significant'] else "ns"
            
            pub_name = publisher[:28]
            p_text = format_p_value(result['p_value'], 0.05)
            q_text = format_p_value(result['p_corrected'], 0.05)
            
            f.write(f"{pub_name:<30} | {result['retrieved']:<10} | {result['not_retrieved']:<10} | ")
            f.write(f"{result['total']:<6} | {result['odds_ratio']:<8.2f} | ")
            f.write(f"{p_text:<12} | {q_text:<12} | {sig_marker:<6}\n")
        
        f.write("\nNote: OR > 1 = lower risk of never being retrieved; OR < 1 = higher risk\n\n")
        
        sig_publishers = [(p, r) for p, r in fisher_overall.items() if r['significant']]
        
        if sig_publishers:
            f.write(f"Significant Publishers: {len(sig_publishers)}\n\n")
            
            high_risk = [(p, r) for p, r in sig_publishers if r['odds_ratio'] < 1]
            low_risk = [(p, r) for p, r in sig_publishers if r['odds_ratio'] > 1]
            
            if high_risk:
                f.write(f"HIGHER risk of never being retrieved:\n")
                for pub, result in sorted(high_risk, key=lambda x: x[1]['odds_ratio']):
                    never_rate = (result['not_retrieved'] / result['total'] * 100) if result['total'] > 0 else 0
                    f.write(f"  {pub}: OR={result['odds_ratio']:.2f}, Never={never_rate:.1f}%, {format_p_value(result['p_corrected'], 0.05)}\n")
                f.write("\n")
            
            if low_risk:
                f.write(f"LOWER risk of never being retrieved:\n")
                for pub, result in sorted(low_risk, key=lambda x: x[1]['odds_ratio'], reverse=True):
                    never_rate = (result['not_retrieved'] / result['total'] * 100) if result['total'] > 0 else 0
                    f.write(f"  {pub}: OR={result['odds_ratio']:.2f}, Never={never_rate:.1f}%, {format_p_value(result['p_corrected'], 0.05)}\n")
                f.write("\n")
        
        # Test (ii): Platform-specific
        f.write("\n1.2 Platform-Specific Non-Retrieval Associations\n")
        f.write("-" * 80 + "\n\n")
        
        platform_stats = statistical_tests['platform_specific_nonretrieval']
        fisher_by_platform = platform_stats['fisher_by_platform']
        
        f.write(f"Research Question: Are publishers associated with platform-specific\n")
        f.write(f"                   retrieval failure?\n\n")
        f.write(f"Method: Fisher's exact test with FDR correction (per platform)\n")
        f.write(f"  - Definition: Each publisher vs all others, tested separately per platform\n")
        f.write(f"  - Outcome: Not retrieved by GIVEN platform vs retrieved by that platform\n")
        f.write(f"  - Correction: FDR within each platform\n\n")
        
        for llm in CONFIG['llms']:
            if llm not in fisher_by_platform:
                continue
            
            f.write(f"\n{llm}:\n")
            f.write("-" * 80 + "\n")
            
            llm_fisher = fisher_by_platform[llm]
            sig_pubs = [p for p, r in llm_fisher.items() if r['significant']]
            
            f.write(f"Significant publishers for {llm}: {len(sig_pubs)}\n\n")
            
            if sig_pubs:
                f.write(f"{'Publisher':<30} | {'Retrieved':<10} | {'Not Retr.':<10} | {'OR':<8} | {'q-value':<12} | {'Sig.':<6}\n")
                f.write("-" * 95 + "\n")
                
                sorted_llm = sorted(llm_fisher.items(), key=lambda x: x[1]['p_corrected'])
                
                for publisher, result in sorted_llm:
                    if result['significant']:
                        sig_marker = "***" if result['p_corrected'] < 0.001 else "**" if result['p_corrected'] < 0.01 else "*"
                        pub_name = publisher[:28]
                        q_text = format_p_value(result['p_corrected'], 0.05)
                        
                        f.write(f"{pub_name:<30} | {result['retrieved']:<10} | {result['not_retrieved']:<10} | ")
                        f.write(f"{result['odds_ratio']:<8.2f} | {q_text:<12} | {sig_marker:<6}\n")
                
                f.write("\n")
        
        # ===== SECTION 2: DESCRIPTIVE STATISTICS =====
        f.write("\n" + "=" * 80 + "\n")
        f.write("SECTION 2: DESCRIPTIVE STATISTICS\n")
        f.write("=" * 80 + "\n\n")
        
        # [Continue with descriptive statistics as before...]
        
        # ===== FOOTER =====
        f.write("\n" + "=" * 80 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 80 + "\n")
    
    print(f"✓ Publisher report saved to: {output_file}")
    return output_file

def generate_llm_nonretrieval_report(publisher_results, base_results, output_file='report_llm_nonretrieval.txt'):
    """Generate comprehensive report for non-retrieval by each LLM"""
    analyzer = base_results['analyzer']
    standardize_publisher_name = publisher_results['standardize_publisher_name']
    combinations_data = publisher_results['combinations_data']
    study_mapping = publisher_results['study_mapping']
    
    llm_list = ['Ai2 Finder', 'Consensus', 'Claude', 'ChatGPT', 'Gemini']
    
    # Calculate which studies each LLM did NOT retrieve
    llm_nonretrieval = {llm: [] for llm in llm_list}
    
    for study_id in study_mapping.keys():
        study_info = study_mapping[study_id]
        
        # For each LLM, check if it retrieved this study
        for llm in llm_list:
            llm_retrieved = False
            
            for combo in combinations_data:
                if combo['LLM'] != llm:
                    continue
                    
                for run, run_studies in combo['studies_by_run'].items():
                    for identifier in study_info['identifiers']:
                        if identifier in run_studies:
                            llm_retrieved = True
                            break
                    if llm_retrieved:
                        break
                if llm_retrieved:
                    break
            
            # If LLM did not retrieve this study, add to non-retrieval list
            if not llm_retrieved:
                # Get study details
                study_row = analyzer.gold_standard_df[
                    analyzer.gold_standard_df['Study ID'] == study_id
                ]
                if not study_row.empty:
                    publisher = standardize_publisher_name(str(study_row['Publisher'].iloc[0]))
                    journal = study_row.get('Journal/Conference', ['Unknown']).iloc[0] if 'Journal/Conference' in study_row.columns else 'Unknown'
                    year = study_row.get('Year', ['N/A']).iloc[0] if 'Year' in study_row.columns else 'N/A'
                    
                    # Count how many OTHER LLMs retrieved it
                    other_llms_count = 0
                    retrieving_llms = []
                    for other_llm in llm_list:
                        if other_llm == llm:
                            continue
                        other_retrieved = False
                        for combo in combinations_data:
                            if combo['LLM'] != other_llm:
                                continue
                            for run, run_studies in combo['studies_by_run'].items():
                                for identifier in study_info['identifiers']:
                                    if identifier in run_studies:
                                        other_retrieved = True
                                        break
                                if other_retrieved:
                                    break
                            if other_retrieved:
                                break
                        if other_retrieved:
                            other_llms_count += 1
                            retrieving_llms.append(other_llm)
                    
                    llm_nonretrieval[llm].append({
                        'study_id': study_id,
                        'publisher': publisher,
                        'journal': journal,
                        'year': year,
                        'other_llms_count': other_llms_count,
                        'retrieving_llms': retrieving_llms
                    })
    
    # Write report
    with open(output_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("NON-RETRIEVAL ANALYSIS BY LLM - COMPLETE REPORT\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total Studies Analyzed: {len(study_mapping)}\n\n")
        
        # Summary statistics
        f.write("=" * 80 + "\n")
        f.write("SUMMARY: NON-RETRIEVAL COUNTS BY LLM\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"{'LLM':<15} | {'Not Retrieved':<15} | {'Retrieval Rate':<15} | {'% Not Retrieved':<15}\n")
        f.write("-" * 70 + "\n")
        
        for llm in llm_list:
            not_retrieved = len(llm_nonretrieval[llm])
            total = len(study_mapping)
            retrieval_rate = ((total - not_retrieved) / total * 100) if total > 0 else 0
            non_retrieval_rate = (not_retrieved / total * 100) if total > 0 else 0
            f.write(f"{llm:<15} | {not_retrieved:<15} | {retrieval_rate:>14.1f}% | {non_retrieval_rate:>14.1f}%\n")
        
        # Detailed breakdown for each LLM
        for llm in llm_list:
            f.write("\n\n" + "=" * 80 + "\n")
            f.write(f"DETAILED NON-RETRIEVAL ANALYSIS: {llm}\n")
            f.write("=" * 80 + "\n\n")
            
            non_retrieved_studies = llm_nonretrieval[llm]
            f.write(f"Total Studies NOT Retrieved by {llm}: {len(non_retrieved_studies)}\n")
            f.write(f"Retrieval Rate: {((len(study_mapping) - len(non_retrieved_studies)) / len(study_mapping) * 100):.1f}%\n\n")
            
            if not non_retrieved_studies:
                f.write(f"✓ {llm} successfully retrieved all studies!\n")
                continue
            
            # Group by publisher
            publisher_breakdown = defaultdict(list)
            for study in non_retrieved_studies:
                publisher_breakdown[study['publisher']].append(study)
            
            f.write(f"Non-Retrieval Breakdown by Publisher:\n")
            f.write("-" * 80 + "\n")
            f.write(f"{'Publisher':<30} | {'Count':<7} | {'% of LLM Non-Retrieval':<25}\n")
            f.write("-" * 80 + "\n")
            
            for publisher in sorted(publisher_breakdown.keys(), key=lambda x: len(publisher_breakdown[x]), reverse=True):
                count = len(publisher_breakdown[publisher])
                pct = (count / len(non_retrieved_studies) * 100) if len(non_retrieved_studies) > 0 else 0
                f.write(f"{publisher:<30} | {count:<7} | {pct:>24.1f}%\n")
            
            # Distribution by other LLMs retrieval
            f.write(f"\n\nDistribution by Other LLMs' Success:\n")
            f.write("-" * 80 + "\n")
            f.write(f"{'Category':<40} | {'Count':<7} | {'Percentage':<10}\n")
            f.write("-" * 80 + "\n")
            
            for i in range(5):
                count = len([s for s in non_retrieved_studies if s['other_llms_count'] == i])
                pct = (count / len(non_retrieved_studies) * 100) if len(non_retrieved_studies) > 0 else 0
                category = f"Not retrieved by {llm}, retrieved by {i} other LLM(s)"
                f.write(f"{category:<40} | {count:<7} | {pct:>9.1f}%\n")
            
            # Complete list of non-retrieved studies
            f.write(f"\n\nComplete List of Studies NOT Retrieved by {llm}:\n")
            f.write("-" * 120 + "\n")
            f.write(f"{'Study ID':<40} | {'Publisher':<25} | {'Year':<6} | {'Other LLMs':<11} | {'Which LLMs Retrieved':<30}\n")
            f.write("-" * 120 + "\n")
            
            # Sort by other_llms_count (descending) then by publisher
            sorted_studies = sorted(non_retrieved_studies, 
                                   key=lambda x: (-x['other_llms_count'], x['publisher']))
            
            for study in sorted_studies:
                llms_str = ', '.join(study['retrieving_llms']) if study['retrieving_llms'] else 'None'
                if len(llms_str) > 28:
                    llms_str = llms_str[:25] + '...'
                f.write(f"{study['study_id']:<40} | {study['publisher']:<25} | "
                       f"{str(study['year']):<6} | {study['other_llms_count']:<11} | {llms_str:<30}\n")
            
            # Publisher-Journal breakdown for non-retrieved studies
            f.write(f"\n\nPublisher-Journal Breakdown for Non-Retrieved Studies ({llm}):\n")
            f.write("-" * 80 + "\n")
            
            for publisher in sorted(publisher_breakdown.keys()):
                f.write(f"\n{publisher} ({len(publisher_breakdown[publisher])} studies):\n")
                journal_groups = defaultdict(list)
                for study in publisher_breakdown[publisher]:
                    journal_groups[study['journal']].append(study)
                
                for journal in sorted(journal_groups.keys()):
                    f.write(f"  {journal} ({len(journal_groups[journal])} studies):\n")
                    for study in journal_groups[journal]:
                        llms_str = ', '.join(study['retrieving_llms']) if study['retrieving_llms'] else 'None'
                        f.write(f"    {study['study_id']:<40} | Year: {study['year']:<6} | "
                               f"Other LLMs: {study['other_llms_count']} ({llms_str})\n")
        
        # Cross-LLM comparison
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("CROSS-LLM NON-RETRIEVAL COMPARISON\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("Studies Never Retrieved by Any LLM:\n")
        f.write("-" * 80 + "\n")
        
        never_retrieved = []
        for study_id in study_mapping.keys():
            all_missed = all(
                any(s['study_id'] == study_id for s in llm_nonretrieval[llm])
                for llm in llm_list
            )
            if all_missed:
                study_row = analyzer.gold_standard_df[
                    analyzer.gold_standard_df['Study ID'] == study_id
                ]
                if not study_row.empty:
                    publisher = standardize_publisher_name(str(study_row['Publisher'].iloc[0]))
                    journal = study_row.get('Journal/Conference', ['Unknown']).iloc[0] if 'Journal/Conference' in study_row.columns else 'Unknown'
                    year = study_row.get('Year', ['N/A']).iloc[0] if 'Year' in study_row.columns else 'N/A'
                    never_retrieved.append({
                        'study_id': study_id,
                        'publisher': publisher,
                        'journal': journal,
                        'year': year
                    })
        
        f.write(f"Total: {len(never_retrieved)} studies\n\n")
        if never_retrieved:
            f.write(f"{'Study ID':<40} | {'Publisher':<25} | {'Journal':<40} | {'Year':<6}\n")
            f.write("-" * 120 + "\n")
            for study in sorted(never_retrieved, key=lambda x: (x['publisher'], x['journal'])):
                f.write(f"{study['study_id']:<40} | {study['publisher']:<25} | "
                       f"{study['journal']:<40} | {str(study['year']):<6}\n")
        
        # LLM-specific misses (missed by one LLM but retrieved by all others)
        f.write("\n\nLLM-Specific Misses (Retrieved by 4 Other LLMs):\n")
        f.write("-" * 80 + "\n")
        
        for llm in llm_list:
            specific_misses = [s for s in llm_nonretrieval[llm] if s['other_llms_count'] == 4]
            f.write(f"\n{llm} Only: {len(specific_misses)} studies\n")
            if specific_misses:
                for study in sorted(specific_misses, key=lambda x: x['publisher'])[:10]:  # Show top 10
                    f.write(f"  {study['study_id']:<40} | {study['publisher']:<25}\n")
                if len(specific_misses) > 10:
                    f.write(f"  ... and {len(specific_misses) - 10} more\n")
        
        f.write("\n" + "=" * 80 + "\n")
        f.write("END OF NON-RETRIEVAL REPORT\n")
        f.write("=" * 80 + "\n")
    
    print(f"✓ LLM non-retrieval report saved to: {output_file}")
    return output_file

# =====================================================================
# RUN ANALYSIS
# =====================================================================
if 'base_results' in globals() and base_results:
    print("\n" + "="*80)
    print("  RUNNING PUBLISHER RETRIEVAL ANALYSIS BY LLM")
    print("="*80)
    
    publisher_results = plot_publisher_retrieval_by_llm_separate(base_results)
    
    # Generate complete publisher report
    report_file = generate_publisher_report(publisher_results, base_results)
    print(f"✓ Report saved to: {report_file}")
    
    # Generate LLM non-retrieval report
    nonretrieval_report_file = generate_llm_nonretrieval_report(publisher_results, base_results)
    print(f"✓ Non-retrieval report saved to: {nonretrieval_report_file}")
    
    print("\n✓ Publisher retrieval analysis completed!")
    print(f"✓ Two reports generated:")
    print(f"  1. {report_file}")
    print(f"  2. {nonretrieval_report_file}")
else:
    print("❌ Base results not available. Please run Part 1A first.")

# ===== EXECUTION =====
if 'base_results' in globals() and base_results:
    print("\n" + "="*80)
    print("  PUBLISHER ANALYSIS WITH PROPERLY DEFINED STATISTICAL TESTS")
    print("="*80)
    
    publisher_results = plot_publisher_retrieval_by_llm_separate(base_results)
    
    report_file = generate_publisher_report(publisher_results, base_results)
    print(f"✓ Publisher report saved to: {report_file}")
    
    print("\n✓ Analysis completed with proper statistical definitions!")
    
    nonretrieval_report_file = generate_llm_nonretrieval_report(publisher_results, base_results)
    print(f"✓ Non-retrieval report saved to: {nonretrieval_report_file}")
else:
    print("❌ Base results not available.")

## Cell 12 — DOMAIN COCHRAN'S Q

*R1.2 domain zero-retrieval*


In [ ]:
# ============================================================================
#  DOMAIN-LEVEL ZERO-RETRIEVAL PROBABILITY  (Reviewer 1, comment 2 — Table C-3.1/C-3.2)
#  NEW CELL — this analysis does not exist anywhere in the original notebook
#  (verified: no cell, including the 10 cells later removed as redundant,
#  ever computed P(zero|domain,platform) or ran Cochran's Q across platforms
#  for domain-level zero-retrieval). The manuscript's Table C-3.1 (cannabinoids
#  n=4 p<0.001, tobacco p<0.01, conference p<0.05, pre-2016 p<0.001, opioids
#  p>0.05) and Table C-3.2 (Bonferroni McNemar pairwise) have no corresponding
#  source in this codebase to verify or regenerate them from. This cell builds
#  that analysis using the now-corrected Cochran's Q formula (same as cell 24)
#  and the same matched-ID derivation pattern already used in cell 25.
#
#  Definition (Appendix C1.2.4): for evidence domain D and platform p,
#  P(zero|D,p) = (# of the 15 formulations where platform p retrieved ZERO
#  domain-D studies, pooled across its 3 runs) / 15.
# ============================================================================
import re
import numpy as np
import pandas as pd
import scipy.stats as stats
from itertools import combinations as _combos

try:
    from statsmodels.stats.contingency_tables import cochrans_q as _smQ
    _HAVE_SM = True
except Exception:
    _HAVE_SM = False

SUB_CATS = ['Cannabinoids', 'Opioids', 'Tobacco', 'Alcohol', 'Polysubstance', 'Other']

def _cochrans_q(M):
    """Corrected Cochran's Q (same formula validated in cell 24)."""
    M = np.asarray(M); n, k = M.shape; R = M.sum(1)
    if (k * R.sum() - (R ** 2).sum()) == 0:
        return 0.0, 1.0, k - 1
    if _HAVE_SM:
        res = _smQ(M)
        return float(res.statistic), float(res.pvalue), k - 1
    C, N = M.sum(0), M.sum()
    Q = ((k - 1) * (k * (C ** 2).sum() - N ** 2)) / (k * R.sum() - (R ** 2).sum())
    return float(Q), float(stats.chi2.sf(Q, k - 1)), k - 1

def _mcnemar_exact_or_corrected(b, c):
    """Same branching as cell 24: exact binomial for sparse cells, else continuity-corrected."""
    nbc = b + c
    if nbc == 0:
        return 0.0, 1.0
    if nbc < 25:
        return float(abs(b - c)), float(min(2 * stats.binom.cdf(min(b, c), nbc, 0.5), 1.0))
    chi2 = ((abs(b - c) - 1) ** 2) / nbc
    return float(chi2), float(stats.chi2.sf(chi2, 1))

def monte_carlo_power_cochran(p_zero_by_platform, n_formulations=15, alpha=0.05, n_sims=4000, seed=42):
    """Monte Carlo power for THIS domain's Cochran's Q
    test specifically -- previously power was only computed for the Fisher's-exact
    family (cell 27), not for these domain-level Cochran's Q tests, which is the
    test R1.2's example (cannabinoids, n=4) is actually about. Given the OBSERVED
    per-platform zero-retrieval rates, what fraction of simulated 15-formulation
    datasets (drawn from those same rates) would reach significance? Validated:
    power -> ~1.0 for an extreme synthetic effect, power -> ~alpha for a null
    effect (sanity checks passed before use)."""
    rng = np.random.default_rng(seed)
    k = len(p_zero_by_platform)
    sig_count = 0
    for _ in range(n_sims):
        M = np.zeros((n_formulations, k), dtype=int)
        for pi, p in enumerate(p_zero_by_platform):
            M[:, pi] = rng.binomial(1, p, size=n_formulations)
        if M.sum(axis=0).std() == 0:
            continue
        try:
            if _HAVE_SM:
                res = _smQ(M)
                pv = float(res.pvalue)
            else:
                _, pv, _ = _cochrans_q(M)
            if pv < alpha:
                sig_count += 1
        except Exception:
            continue
    return sig_count / n_sims

def categorize_main_substances(substance_list):
    if not substance_list:
        return 'Other'
    substances = [s.strip() for s in substance_list]
    substances_lower = [s.lower() for s in substances]
    has_tobacco = any('tobacco' in s or 'nicotine' in s or
                      'smoking' in s or 'cigarette' in s for s in substances_lower)
    has_alcohol = any('alcohol' in s for s in substances_lower)
    has_opioids = any('opioid' in s for s in substances_lower)
    has_cannabis = any('cannabis' in s or 'marijuana' in s or
                       'cannabinoid' in s for s in substances_lower)
    n_main = sum([has_tobacco, has_alcohol, has_opioids, has_cannabis])
    if len(substances) == 1:
        s = substances[0].lower()
        if 'cannabis' in s or 'marijuana' in s: return 'Cannabinoids'
        elif 'opioid' in s: return 'Opioids'
        elif 'tobacco' in s or 'nicotine' in s or 'smoking' in s: return 'Tobacco'
        elif 'alcohol' in s: return 'Alcohol'
        else: return 'Other'
    else:
        if n_main >= 2: return 'Polysubstance'
        elif has_cannabis: return 'Cannabinoids'
        elif has_opioids: return 'Opioids'
        elif has_tobacco: return 'Tobacco'
        elif has_alcohol: return 'Alcohol'
        else: return 'Other'

def _categorize_studies(analyzer):
    """Build per-study domain membership: substance category, venue type, year bucket."""
    study_sub, study_venue, study_year = {}, {}, {}
    gold_df = analyzer.gold_standard_df
    for _, row in gold_df.iterrows():
        sid = str(row.get('Study ID', row.get('Title', '')))
        sub_raw = str(row.get('Substances covered', ''))
        cats = [s.strip() for s in sub_raw.split(';')] if sub_raw and sub_raw != 'nan' else []
        study_sub[sid] = categorize_main_substances(cats)
        venue_raw = str(row.get('Journal/Conference', ''))
        study_venue[sid] = 'Conference' if 'conference' in venue_raw.lower() or 'proceedings' in venue_raw.lower() else 'Journal'
        try:
            yr = int(float(row.get('Year', float('nan'))))
            study_year[sid] = 'Pre-2016' if yr < 2016 else '2016+'
        except (TypeError, ValueError):
            study_year[sid] = None
    return study_sub, study_venue, study_year

def _formulation_matched_ids(analyzer, llm, strat, var, study_map):
    """Pooled (across 3 runs) matched study-ID set for one platform-formulation, using the
    SAME derivation as cell 25's df_form construction, for exact consistency."""
    pooled_norm = set()
    for fname, info in analyzer.recalled_files.items():
        if info['llm'] == llm and info.get('strategy') == strat and info.get('variation') == var:
            df = analyzer.load_csv_file_content(info['path'])
            if df.empty:
                continue
            for col in ['DOI', 'Study name', 'Title', 'LLM Title']:
                if col in df.columns:
                    for val in df[col].dropna().astype(str):
                        pooled_norm.add(analyzer.normalize_identifier_consistent(val))
                    break
    matched_ids = set()
    for sid, sinfo in study_map.items():
        if any(i in pooled_norm for i in sinfo['identifiers']):
            matched_ids.add(sid)
    return matched_ids

def run_domain_zero_retrieval_analysis(base_results, alpha=0.05):
    analyzer = base_results['analyzer']
    llm_names = sorted(set(info['llm'] for info in analyzer.recalled_files.values()))
    study_map = analyzer.create_study_identifier_mapping()
    study_sub, study_venue, study_year = _categorize_studies(analyzer)

    domains = {f'Substance: {c}': (study_sub, c) for c in SUB_CATS}
    domains['Venue: Conference'] = (study_venue, 'Conference')
    domains['Temporal: Pre-2016'] = (study_year, 'Pre-2016')

    n_per_domain = {name: sum(1 for v in catdict.values() if v == val)
                    for name, (catdict, val) in domains.items()}

    print("=" * 80)
    print("  DOMAIN-LEVEL ZERO-RETRIEVAL PROBABILITY  (Table C-3.1/C-3.2 equivalent)")
    print("=" * 80)
    print(f"  Platforms: {llm_names}")
    print(f"  Domain sizes (of 83 gold-standard studies): {n_per_domain}\n")

    # cache matched-ID sets per (llm, strat, var) -- computed once, reused across all domains
    formulations = [(s, v) for s in ['1', '2', '3', '4', '5'] for v in ['1', '2', '3']]
    matched_cache = {}
    for llm in llm_names:
        for strat, var in formulations:
            matched_cache[(llm, strat, var)] = _formulation_matched_ids(analyzer, llm, strat, var, study_map)

    results_summary = []
    pairwise_all = []
    for dname, (catdict, val) in domains.items():
        M = np.zeros((len(formulations), len(llm_names)), int)
        for fi, (strat, var) in enumerate(formulations):
            for pi, llm in enumerate(llm_names):
                matched = matched_cache[(llm, strat, var)]
                n_domain_retrieved = sum(1 for sid in matched if catdict.get(sid) == val)
                M[fi, pi] = 1 if n_domain_retrieved == 0 else 0

        Q, p, df = _cochrans_q(M)
        p_zero_by_platform = M.mean(axis=0)  # P(zero) per platform across the 15 formulations
        n_dom = n_per_domain[dname]
        small_n = n_dom <= 5

        print(f"  {dname:<24} n={n_dom:<3} Q={Q:6.2f} df={df} "
              f"p={'<0.001' if p < 0.001 else f'{p:.3f}'}  "
              f"P(zero) per platform: " +
              ", ".join(f"{llm}={p_zero_by_platform[i]:.0%}" for i, llm in enumerate(llm_names)) +
              ("  [SMALL n -- interpret cautiously]" if small_n else ""))

        results_summary.append({'Domain': dname, 'N_studies': n_dom, 'Cochran_Q': Q,
                                'df': df, 'p_value': p, 'Small_n_flag': small_n,
                                **{f'P_zero_{llm}': p_zero_by_platform[i] for i, llm in enumerate(llm_names)}})

        if p < alpha:
            bonf_alpha = alpha / (len(llm_names) * (len(llm_names) - 1) // 2)
            for i, j in _combos(range(len(llm_names)), 2):
                b = int(np.sum((M[:, i] == 1) & (M[:, j] == 0)))
                c = int(np.sum((M[:, i] == 0) & (M[:, j] == 1)))
                stat, pv = _mcnemar_exact_or_corrected(b, c)
                pairwise_all.append({'Domain': dname, 'Platform_1': llm_names[i], 'Platform_2': llm_names[j],
                                     'b': b, 'c': c, 'statistic': stat, 'p_value': pv,
                                     'Significant_Bonferroni': pv < bonf_alpha})

    summary_df = pd.DataFrame(results_summary)
    pairwise_df = pd.DataFrame(pairwise_all)
    summary_df.to_csv('domain_zero_retrieval_cochran.csv', index=False)
    if not pairwise_df.empty:
        pairwise_df.to_csv('domain_zero_retrieval_mcnemar_pairwise.csv', index=False)

    sig_pairs = int(pairwise_df['Significant_Bonferroni'].sum()) if not pairwise_df.empty else 0
    print(f"\n  ✓ saved domain_zero_retrieval_cochran.csv ({len(summary_df)} domains)")
    print(f"  ✓ saved domain_zero_retrieval_mcnemar_pairwise.csv "
          f"({len(pairwise_df)} pairwise tests, {sig_pairs} significant after Bonferroni)")
    return {'summary': summary_df, 'pairwise': pairwise_df}

# ===== EXECUTION =====
if 'base_results' in globals() and base_results:
    domain_zero_retrieval_results = run_domain_zero_retrieval_analysis(base_results)
else:
    print("❌ base_results not found — run the base-analysis cell first.")

## Cell 13 — MATCH PROVENANCE

*R1.3 exact/near-exact FN rate*


In [ ]:
# ============================================================================
#  MATCH PROVENANCE & CONSENSUS-CONFLICT TAXONOMY  (Reviewer 1, comment 3)
#  Because matches were established by two-reviewer CONSENSUS (not independent
#  parallel rating), kappa is not the right statistic. This cell reports what the
#  consensus response needs: (1) exact vs near-exact proportion, and (2) a
#  taxonomy of near-exact conflicts (abbreviation / index-formatting /
#  version-wording) with examples. Reads *R.csv; no change to matching/results.
# ============================================================================
import os, re, glob, numpy as np, pandas as pd

FILES_PATH = os.path.expanduser("~/Desktop/Pooled Std Recall Runs")

ACRONYMS = {'nlp': 'natural language processing', 'ehr': 'electronic health record',
            'emr': 'electronic medical record', 'llm': 'large language model',
            'llms': 'large language models', 'ml': 'machine learning',
            'ai': 'artificial intelligence', 'sud': 'substance use disorder',
            'oud': 'opioid use disorder', 'gpt': 'generative pretrained transformer'}
PREFIXES = (r'^(technical brief|brief communication|original article|research article|'
            r'short communication|letter|editorial|review|abstract|case report)\b\s*')
STOP = {'a','an','the','of','to','for','from','with','and','in','on','using','via'}

LISTNUM = r'^\d+\.\s*'

def _low(s): return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', ' ', str(s).lower())).strip()
def _strip_listnum(s): return re.sub(LISTNUM, '', str(s)).strip()
def _dedupe_repeat(s):
    n = len(s)
    if n < 10:
        return s
    half = s[:n // 2].strip()
    if half and s.strip() == (half + ' ' + half).strip():
        return half
    return s
def _strip_prefix(s): return re.sub(PREFIXES, '', s).strip()
def _expand(s):
    for ac, full in ACRONYMS.items():
        s = re.sub(rf'\b{ac}\b', full, s)
    return s
def _toks(s): return [w for w in _expand(_strip_prefix(_low(s))).split() if w not in STOP]

def classify_conflict(llm_title, gold_title):
    L0 = _dedupe_repeat(_strip_listnum(str(llm_title)))
    G0 = _dedupe_repeat(_strip_listnum(str(gold_title)))
    L, G = _low(L0), _low(G0)
    if L == G:
        return 'exact'
    if _strip_prefix(L) == _strip_prefix(G):
        return 'index_formatting'
    ls, gs = set(_toks(L0)), set(_toks(G0))
    if ls and gs and (ls == gs or ls <= gs or gs <= ls):
        return 'abbreviation'
    return 'version_wording'

def run_provenance_audit(files_path=FILES_PATH):
    rows = []
    for fp in glob.glob(os.path.join(files_path, "*.csv")):
        if not re.search(r'R\.csv$', os.path.basename(fp)): continue
        try: df = pd.read_csv(fp)
        except Exception: continue
        if df.empty: continue
        c = {x.lower(): x for x in df.columns}
        s, g, sc = c.get('llm title'), c.get('title'), c.get('match_score')
        if not s or not g: continue
        for _, r in df.iterrows():
            if not _low(r[g]): continue
            m = re.search(r'(\d+\.?\d*)', str(r[sc])) if sc and pd.notna(r[sc]) else None
            rows.append({'llm_title': str(r[s]), 'gold_title': str(r[g]),
                         'score': float(m.group(1)) if m else np.nan,
                         'conflict_type': classify_conflict(r[s], r[g])})
    rec = pd.DataFrame(rows)
    if rec.empty:
        print("No matched rows found."); return rec
    n = len(rec); exact = int((rec['conflict_type'] == 'exact').sum())
    print("="*70); print(f"MATCH PROVENANCE  (n = {n} matched records)"); print("="*70)
    print(f"  Exact (verbatim):         {exact:5d}  ({exact/n*100:.1f}%)")
    print(f"  Near-exact (adjudicated): {n-exact:5d}  ({(n-exact)/n*100:.1f}%)")
    near = rec[rec['conflict_type'] != 'exact']
    print("\nNear-exact conflict taxonomy (resolved by consensus):")
    for t in ['abbreviation','index_formatting','version_wording']:
        sub = near[near['conflict_type'] == t]
        if not len(sub): continue
        pct = len(sub)/len(near)*100 if len(near) else 0
        ex = sub.sort_values('score').iloc[0]
        print(f"\n  {t.upper()} — {len(sub)} ({pct:.0f}% of near-exact)")
        print(f"     e.g. [{ex['score']:.1f}%] LLM : {ex['llm_title'][:82]}")
        print(f"                     Gold: {ex['gold_title'][:82]}")
    rec.to_csv('match_provenance_with_conflicts.csv', index=False)
    print("\n  ✓ saved match_provenance_with_conflicts.csv")
    return rec

if __name__ == "__main__":
    run_provenance_audit()

# ──────────────────────────────────────────────────────────────────────
# ── GAP FIX #4: FALSE-NEGATIVE RATE ESTIMATE (R1.3 third ask) ────────────
# Reviewer 1, Comment 3 asked for three things:
#   (i)  inter-rater agreement          → addressed: consensus framing (not κ)
#   (ii) exact/near-exact proportion    → addressed: 82.6% / 17.4%
#   (iii) false-negative rate estimate  → THIS BLOCK (was missing)
#
# The false-negative rate of the 60% Jaccard matching threshold is now
# empirically established by the post-hoc sensitivity analysis (see
# Supplementary Table [X]) rather than estimated from first principles.
# This block computes and reports the headline figure from that analysis.
#
# Definitions:
#   FN_confirmed   = gold studies missed by the 60% threshold that a platform
#                    DID output but with a reformatted title (40–60% band),
#                    confirmed by two human reviewers as the same publication
#   FN_borderline  = cases where reviewers could not rule out the same paper
#   Total_matches  = all confirmed matches at ≥60% across 225 runs (from *R files)
#   FN_rate        = FN_confirmed / (Total_matches + FN_confirmed)
#   CI             = Clopper-Pearson exact binomial 95% CI
# ─────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  GAP FIX #4: FALSE-NEGATIVE RATE OF THE 60% MATCHING THRESHOLD")
print("  [R1.3 — empirical estimate from post-hoc sensitivity analysis]")
print("=" * 70)

# ── Sensitivity analysis results (human-reviewed; June 2026) ──────────────
FN_confirmed  = 9    # Gold studies retrieved by ≥1 platform at 40–60%,
                     # confirmed by 2 reviewers as same published paper,
                     # all caused by title formatting artifacts only.
FN_borderline = 5    # Cases where reviewers could not rule out same paper
                     # (verb substitution, scope narrowing, prefix addition)
N_TOTAL_RUNS  = 225
N_GOLD        = 83
THRESHOLD_PCT = 60

# Total confirmed matches at ≥60% across all 225 runs
# (from Aggregate_Output_File match counts; 2,344 matched records total)
total_matches_at_60 = 2344

# ── Point estimates ───────────────────────────────────────────────────────
fn_denominator_confirmed  = total_matches_at_60 + FN_confirmed
fn_denominator_upper      = total_matches_at_60 + FN_confirmed + FN_borderline

fn_rate_confirmed = FN_confirmed / fn_denominator_confirmed
fn_rate_upper     = (FN_confirmed + FN_borderline) / fn_denominator_upper

# Clopper-Pearson exact binomial 95% CI (conservative, preferred for small counts)
from scipy.stats import beta as beta_dist
def clopper_pearson(k, n, alpha=0.05):
    if n == 0:
        return (0.0, 1.0)
    lo = beta_dist.ppf(alpha/2, k, n - k + 1) if k > 0 else 0.0
    hi = beta_dist.ppf(1 - alpha/2, k + 1, n - k) if k < n else 1.0
    return (lo, hi)

lo_conf, hi_conf = clopper_pearson(FN_confirmed, fn_denominator_confirmed)
lo_upper, hi_upper = clopper_pearson(
    FN_confirmed + FN_borderline, fn_denominator_upper
)

print(f"""
  Sensitivity analysis scope: all {N_TOTAL_RUNS} runs × {N_GOLD} gold studies × 5 platforms
  Sub-threshold candidates examined (40–60% band): 2,155
  Of which:
    Correctly excluded (different study):       1,889  (87.7%)
    Topically similar, genuinely different:       211   (9.8%)
    Warranted full manual adjudication:            55   (2.6%)
  Of the 55 adjudicated:
    Confirmed false negatives (same paper,
    format artifact only):                          {FN_confirmed}
    Confirmed non-match (content change):           1
    Borderline (cannot rule out same paper):        {FN_borderline}

  ── False-negative rate estimate ─────────────────────────────────────────
  Total confirmed matches at ≥60% threshold:  {total_matches_at_60:,}
  Confirmed false-negative rate:
    {FN_confirmed}/{fn_denominator_confirmed} = {fn_rate_confirmed*100:.2f}%
    95% Clopper-Pearson CI: ({lo_conf*100:.2f}%, {hi_conf*100:.2f}%)
  Conservative upper bound (confirmed + borderline):
    {FN_confirmed+FN_borderline}/{fn_denominator_upper} = {fn_rate_upper*100:.2f}%
    95% Clopper-Pearson CI: ({lo_upper*100:.2f}%, {hi_upper*100:.2f}%)

  ── Interpretation ───────────────────────────────────────────────────────
  All {FN_confirmed} confirmed false negatives are attributable to platform
  output formatting artifacts (numbered list prefixes, technical abbreviations,
  subtitle truncation, encoding differences) rather than threshold inadequacy.
  None correspond to any of the {10} studies never retrieved by any platform.
  Lowering the threshold to 40% yielded 1,599 additional candidates with
  zero new confirmed matches, supporting adequacy of the {THRESHOLD_PCT}% criterion.
""")

# ── Store ─────────────────────────────────────────────────────────────────
fn_rate_results = {
    'total_matches_primary_threshold': total_matches_at_60,
    'fn_confirmed': FN_confirmed,
    'fn_borderline': FN_borderline,
    'fn_rate_confirmed': fn_rate_confirmed,
    'fn_rate_confirmed_ci_95': (lo_conf, hi_conf),
    'fn_rate_upper_bound': fn_rate_upper,
    'fn_rate_upper_ci_95': (lo_upper, hi_upper),
    'sensitivity_candidates_examined': 2155,
    'correctly_excluded': 1889,
    'pct_correctly_excluded': 87.7,
    'zero_overlap_with_never_retrieved': True,
    'cause_of_all_fn': 'Platform output formatting artifacts only',
}
print(f"  False-negative rate results stored in fn_rate_results dict.")

## Cell 14 — CHARACTERISTIC STATS

*Publisher/venue/year breakdown*


In [ ]:
# ===== COMPLETE STATISTICAL ANALYSIS SUITE =====
# Aligns with methods: Cochran's Q, Friedman, and Fisher's exact tests
# for platform comparison and publication characteristics analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from datetime import datetime
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations as iter_combinations
import warnings
warnings.filterwarnings('ignore')

# ===== GLOBAL CONFIG =====
CONFIG = {
    'llms': ['Ai2 Finder', 'Consensus', 'Claude', 'ChatGPT', 'Gemini'],
    'strategies': ['1', '2', '3', '4', '5'],
    'variations': ['0', '1', '2'],
    'strategy_names': {
        '1': 'Zero-Shot', '2': 'Few-Shot', '3': 'Persona',
        '4': 'CoT', '5': 'PICO'
    },
}

# ===== UTILITY FUNCTIONS =====
def format_p_value(p_value, threshold=0.05):
    """Format p-value for display"""
    if pd.isna(p_value):
        return "p = NA"
    if p_value < 0.001:
        return "p < 0.001"
    elif p_value < threshold:
        return f"p < {threshold}"
    elif p_value < 0.01:
        return f"p = {p_value:.3f}"
    else:
        return f"p = {p_value:.2f}"

def standardize_publisher_name(publisher_name):
    """Standardize publisher names for consistency"""
    publisher = str(publisher_name)
    if 'Wolters Kluwer' in publisher: return 'Wolters Kluwer'
    if 'Taylor & Francis' in publisher or 'Dove Medical Press' in publisher: return 'Taylor & Francis'
    if 'Springer' in publisher: return 'Springer'
    if 'Elsevier' in publisher: return 'Elsevier'
    if 'IEEE' in publisher: return 'IEEE'
    if 'JMIR' in publisher: return 'JMIR'
    if 'SCITEPRESS' in publisher: return 'SCITEPRESS'
    if 'IOS Press' in publisher: return 'IOS Press'
    if 'Oxford University Press' in publisher or 'OUP' in publisher: return 'Oxford University Press'
    if 'MDPI' in publisher: return 'MDPI'
    if 'Wiley' in publisher: return 'Wiley'
    if 'Sage' in publisher: return 'Sage'
    if 'BMJ' in publisher: return 'BMJ'
    if 'AMIA' in publisher: return 'AMIA'
    if 'Frontiers' in publisher: return 'Frontiers'
    if 'Nature Publishing' in publisher: return 'Nature Publishing'
    if 'PLOS' in publisher or 'Public Library of Science' in publisher: return 'PLOS'
    if 'Cambridge University Press' in publisher: return 'Cambridge University Press'
    if 'Lippincott' in publisher: return 'Lippincott'
    if 'Karger' in publisher: return 'Karger'
    if 'Thieme' in publisher: return 'Thieme'
    if 'Mary Ann Liebert' in publisher: return 'Mary Ann Liebert'
    if 'American Medical Association' in publisher or 'AMA' in publisher: return 'American Medical Association'
    return publisher

def categorize_venue_type(study_row):
    """Categorize venue as Journal or Conference"""
    venue = str(study_row.get('Journal/Conference', '')).lower()
    
    # Conference indicators
    conference_keywords = ['conference', 'symposium', 'proceedings', 'workshop', 
                          'congress', 'meeting', 'summit']
    
    if any(keyword in venue for keyword in conference_keywords):
        return 'Conference'
    
    # IEEE and AMIA special cases
    if 'ieee' in venue or 'amia' in venue:
        return 'Conference'
    
    # Default to Journal
    return 'Journal'

def extract_publication_decade(study_row):
    """Extract publication decade from year"""
    try:
        year = int(study_row.get('Year', 0))
        if year == 0:
            return 'Unknown'
        decade = (year // 10) * 10
        return f"{decade}s"
    except:
        return 'Unknown'

# ===== CORE ANALYSIS FUNCTIONS =====

def build_retrieval_data(base_results):
    """
    Build retrieval data structure used by all characteristic analyses
    Returns: combinations_data, study_mapping, llm_retrieval_by_study
    """
    analyzer = base_results['analyzer']
    study_mapping = analyzer.create_study_identifier_mapping()
    
    print(f"Building retrieval data for {len(study_mapping)} studies...")
    
    # Build combination data
    combinations_data = []
    for llm in CONFIG['llms']:
        for strategy in CONFIG['strategies']:
            for variation in CONFIG['variations']:
                studies_by_run = {}
                for filename, info in analyzer.recalled_files.items():
                    if (info['llm'] == llm and info['strategy'] == strategy and info['variation'] == variation):
                        try:
                            df = analyzer.load_csv_file_content(info['path'])
                            if not df.empty:
                                raw_matches, _ = analyzer.extract_studies_from_recalled_files(df)
                                run_matches = set()
                                for match in raw_matches:
                                    normalized = analyzer.normalize_identifier_consistent(match)
                                    if normalized:
                                        run_matches.add(normalized)
                                studies_by_run[info['run']] = run_matches
                        except Exception as e:
                            continue
                
                combinations_data.append({
                    'LLM': llm,
                    'Strategy': strategy,
                    'Variation': variation,
                    'studies_by_run': studies_by_run
                })
    
    # Calculate retrieval by LLM
    llm_retrieval_by_study = {study_id: set() for study_id in study_mapping.keys()}
    
    for study_id in study_mapping.keys():
        study_info = study_mapping[study_id]
        for combo in combinations_data:
            combo_retrieved = False
            for run, run_studies in combo['studies_by_run'].items():
                for identifier in study_info['identifiers']:
                    if identifier in run_studies:
                        combo_retrieved = True
                        break
                if combo_retrieved:
                    break
            
            if combo_retrieved:
                llm_retrieval_by_study[study_id].add(combo['LLM'])
    
    print(f"✓ Built retrieval data for {len(combinations_data)} combinations")
    
    return combinations_data, study_mapping, llm_retrieval_by_study

def run_characteristic_analysis(base_results, characteristic_name, extract_characteristic_func, 
                                standardize_func=None, visualization_title=None):
    """
    Generic function to analyze any publication characteristic
    
    Parameters:
    - base_results: Base analysis results
    - characteristic_name: Name of characteristic (e.g., 'Publisher', 'Venue Type')
    - extract_characteristic_func: Function to extract characteristic from study row
    - standardize_func: Optional function to standardize characteristic values
    - visualization_title: Title for visualization plot
    """
    print("\n" + "="*80)
    print(f"  {characteristic_name.upper()} RETRIEVAL ANALYSIS WITH STATISTICAL TESTS")
    print("="*80)
    
    analyzer = base_results['analyzer']
    
    # Build or retrieve retrieval data
    if 'retrieval_data' not in base_results:
        combinations_data, study_mapping, llm_retrieval_by_study = build_retrieval_data(base_results)
        base_results['retrieval_data'] = {
            'combinations_data': combinations_data,
            'study_mapping': study_mapping,
            'llm_retrieval_by_study': llm_retrieval_by_study
        }
    else:
        retrieval_data = base_results['retrieval_data']
        combinations_data = retrieval_data['combinations_data']
        study_mapping = retrieval_data['study_mapping']
        llm_retrieval_by_study = retrieval_data['llm_retrieval_by_study']
    
    study_llm_counts = {study_id: len(llms) for study_id, llms in llm_retrieval_by_study.items()}
    
    # ========== CATEGORIZE BY CHARACTERISTIC ==========
    characteristic_llm_data = defaultdict(lambda: defaultdict(int))
    characteristic_by_llm = {llm: defaultdict(lambda: {'retrieved': 0, 'not_retrieved': 0})
                            for llm in CONFIG['llms']}
    
    for study_id in study_mapping.keys():
        study_row = analyzer.gold_standard_df[analyzer.gold_standard_df['Study ID'] == study_id]
        if not study_row.empty:
            # Extract and optionally standardize characteristic
            characteristic_value = extract_characteristic_func(study_row.iloc[0])
            if standardize_func:
                characteristic_value = standardize_func(characteristic_value)
            
            # Overall categorization
            llm_count = study_llm_counts[study_id]
            characteristic_llm_data[characteristic_value][f"{llm_count} LLMs"] += 1
            
            # Per-LLM categorization
            for llm in CONFIG['llms']:
                if llm in llm_retrieval_by_study[study_id]:
                    characteristic_by_llm[llm][characteristic_value]['retrieved'] += 1
                else:
                    characteristic_by_llm[llm][characteristic_value]['not_retrieved'] += 1
    
    # ========== STATISTICAL TESTS ==========
    print("\n📊 Performing Statistical Tests...")
    
    # Test (i): Overall non-retrieval
    print("  Test (i): Overall non-retrieval association")
    overall_test_data = []
    characteristic_list = []
    
    for characteristic in characteristic_llm_data.keys():
        retrieved_by_any = sum(characteristic_llm_data[characteristic][f"{i} LLMs"] for i in range(1, 6))
        never_retrieved = characteristic_llm_data[characteristic]["0 LLMs"]
        overall_test_data.append([retrieved_by_any, never_retrieved])
        characteristic_list.append(characteristic)
    
    # Fisher's exact: each characteristic vs all others
    fisher_overall = {}
    p_values_overall = []
    total_retrieved = sum(c[0] for c in overall_test_data)
    total_not_retrieved = sum(c[1] for c in overall_test_data)
    
    for idx, characteristic in enumerate(characteristic_list):
        char_retrieved = overall_test_data[idx][0]
        char_not_retrieved = overall_test_data[idx][1]
        other_retrieved = total_retrieved - char_retrieved
        other_not_retrieved = total_not_retrieved - char_not_retrieved
        
        fisher_table = [[char_retrieved, char_not_retrieved],
                       [other_retrieved, other_not_retrieved]]
        
        odds_ratio, p_value = stats.fisher_exact(fisher_table)
        
        fisher_overall[characteristic] = {
            'odds_ratio': odds_ratio,
            'p_value': p_value,
            'retrieved': char_retrieved,
            'not_retrieved': char_not_retrieved,
            'total': char_retrieved + char_not_retrieved
        }
        p_values_overall.append(p_value)
    
    # FDR correction
    reject_overall, pvals_corrected_overall, _, _ = multipletests(p_values_overall, alpha=0.05, method='fdr_bh')
    
    for idx, characteristic in enumerate(characteristic_list):
        fisher_overall[characteristic]['p_corrected'] = pvals_corrected_overall[idx]
        fisher_overall[characteristic]['significant'] = reject_overall[idx]
    
    sig_overall = sum(reject_overall)
    print(f"    Significant {characteristic_name}s (overall): {sig_overall}")
    
    # Test (ii): Platform-specific non-retrieval
    print("\n  Test (ii): Platform-specific non-retrieval")
    fisher_by_platform = {}
    
    for llm in CONFIG['llms']:
        print(f"    Testing {llm}...")
        llm_test_data = []
        llm_char_list = []
        
        for characteristic in characteristic_list:
            retrieved = characteristic_by_llm[llm][characteristic]['retrieved']
            not_retrieved = characteristic_by_llm[llm][characteristic]['not_retrieved']
            llm_test_data.append([retrieved, not_retrieved])
            llm_char_list.append(characteristic)
        
        # Fisher's exact for each characteristic
        llm_fisher = {}
        llm_p_values = []
        llm_total_retrieved = sum(c[0] for c in llm_test_data)
        llm_total_not_retrieved = sum(c[1] for c in llm_test_data)
        
        for idx, characteristic in enumerate(llm_char_list):
            char_retrieved = llm_test_data[idx][0]
            char_not_retrieved = llm_test_data[idx][1]
            other_retrieved = llm_total_retrieved - char_retrieved
            other_not_retrieved = llm_total_not_retrieved - char_not_retrieved
            
            fisher_table = [[char_retrieved, char_not_retrieved],
                           [other_retrieved, other_not_retrieved]]
            
            odds_ratio, p_value = stats.fisher_exact(fisher_table)
            
            llm_fisher[characteristic] = {
                'odds_ratio': odds_ratio,
                'p_value': p_value,
                'retrieved': char_retrieved,
                'not_retrieved': char_not_retrieved,
                'total': char_retrieved + char_not_retrieved
            }
            llm_p_values.append(p_value)
        
        # FDR correction
        reject_llm, pvals_corrected_llm, _, _ = multipletests(llm_p_values, alpha=0.05, method='fdr_bh')
        
        for idx, characteristic in enumerate(llm_char_list):
            llm_fisher[characteristic]['p_corrected'] = pvals_corrected_llm[idx]
            llm_fisher[characteristic]['significant'] = reject_llm[idx]
        
        fisher_by_platform[llm] = llm_fisher
        print(f"      Significant {characteristic_name}s: {sum(reject_llm)}")
    
    # ========== CREATE VISUALIZATION ==========
    print("\n📊 Creating visualization...")
    
    category_order = ['0 LLMs', '1 LLMs', '2 LLMs', '3 LLMs', '4 LLMs', '5 LLMs']
    legend_labels = {
        '0 LLMs': 'Never Retrieved by Any LLM',
        '1 LLMs': 'Retrieved by 1 LLM',
        '2 LLMs': 'Retrieved by 2 LLMs',
        '3 LLMs': 'Retrieved by 3 LLMs',
        '4 LLMs': 'Retrieved by 4 LLMs',
        '5 LLMs': 'Retrieved by 5 LLMs'
    }
    
    colors_llm = ['#E74C3C', '#EB984E', '#F7DC6F', '#52BE80', '#27AE60', '#1E8449']
    
    char_llm_list = []
    for char, categories in characteristic_llm_data.items():
        for category in category_order:
            char_llm_list.append({
                characteristic_name: char,
                'Category': category,
                'Count': categories.get(category, 0)
            })
    
    char_llm_df = pd.DataFrame(char_llm_list)
    char_llm_df = char_llm_df.groupby([characteristic_name, 'Category'], as_index=False)['Count'].sum()
    
    char_totals = char_llm_df.groupby(characteristic_name)['Count'].sum().sort_values(ascending=False)
    all_characteristics = char_totals.index.tolist()
    
    char_pivot = char_llm_df.pivot(index=characteristic_name, columns='Category', values='Count').fillna(0)
    
    for cat in category_order:
        if cat not in char_pivot.columns:
            char_pivot[cat] = 0
    
    char_pivot = char_pivot[category_order]
    char_pivot['Total'] = char_pivot.sum(axis=1)
    char_pivot = char_pivot.sort_values('Total', ascending=True)
    max_total = int(char_pivot['Total'].max())
    char_pivot = char_pivot.drop('Total', axis=1)
    
    num_characteristics = len(char_pivot)
    fig_height = max(8, num_characteristics * 0.5)
    
    fig, ax = plt.subplots(figsize=(16, fig_height))
    char_pivot.plot(kind='barh', stacked=True, ax=ax,
                   color=colors_llm, edgecolor='black', linewidth=0.5, alpha=0.85)
    
    # Add significance markers
    y_positions = {char: idx for idx, char in enumerate(char_pivot.index)}
    
    for characteristic, result in fisher_overall.items():
        if characteristic in y_positions and result['significant']:
            y_pos = y_positions[characteristic]
            x_pos = max_total + 0.5
            
            if result['p_corrected'] < 0.001:
                marker = '***'
            elif result['p_corrected'] < 0.01:
                marker = '**'
            else:
                marker = '*'
            
            ax.text(x_pos, y_pos, marker, fontsize=12, va='center', ha='left',
                   color='black', fontweight='bold')
    
    ax.set_xlabel('Number of Studies', fontsize=18, fontweight='bold')
    ax.set_ylabel(characteristic_name, fontsize=18, fontweight='bold')
    
    if visualization_title is None:
        visualization_title = f"Study Retrieval Distribution by {characteristic_name}"
    ax.set_title(visualization_title, fontsize=22, fontweight='bold', pad=20)
    
    ax.set_xticks(np.arange(0, max_total + 2, max(1, max_total // 10)))
    ax.set_xlim(0, max_total + 1.5)
    
    handles, labels = ax.get_legend_handles_labels()
    new_labels = [legend_labels[label] for label in labels]
    ax.legend(handles, new_labels, title='Retrieval Status',
             bbox_to_anchor=(0.55, 0.1), loc='lower left',
             fontsize=14, title_fontsize=12, frameon=True,
             edgecolor='black', fancybox=False)
    
    footnote = "* q < 0.05, ** q < 0.01, *** q < 0.001 (Fisher's exact test, FDR-corrected, overall non-retrieval)"
    fig.text(0.5, 0.02, footnote, ha='center', fontsize=9, style='italic')
    
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=11)
    
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    
    filename = f"{characteristic_name.replace(' ', '_')}_Retrieval_Distribution.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.show()
    
    print(f"\n✓ Visualization saved: {filename}")
    
    # Return results
    return {
        'characteristic_name': characteristic_name,
        'characteristic_llm_data': dict(characteristic_llm_data),
        'char_pivot': char_pivot,
        'char_llm_df': char_llm_df,
        'all_characteristics': all_characteristics,
        'char_totals': char_totals,
        'characteristic_by_llm': characteristic_by_llm,
        'statistical_tests': {
            'overall_nonretrieval': {
                'description': 'Never retrieved by any platform vs retrieved by ≥1 platform',
                'fisher_exact': fisher_overall,
                'correction': 'fdr_bh',
                'alpha': 0.05
            },
            'platform_specific_nonretrieval': {
                'description': 'Not retrieved by given platform vs retrieved by that platform',
                'fisher_by_platform': fisher_by_platform,
                'correction': 'fdr_bh',
                'alpha': 0.05
            }
        }
    }

# ===== REPORT GENERATION =====

def generate_characteristic_report(characteristic_results, base_results, output_file=None):
    """Generate comprehensive report for any characteristic analysis"""
    
    characteristic_name = characteristic_results['characteristic_name']
    if output_file is None:
        output_file = f'report_{characteristic_name.replace(" ", "_").lower()}_retrieval.txt'
    
    analyzer = base_results['analyzer']
    statistical_tests = characteristic_results['statistical_tests']
    
    with open(output_file, 'w') as f:
        # ===== HEADER =====
        f.write("=" * 80 + "\n")
        f.write(f"{characteristic_name.upper()} RETRIEVAL ANALYSIS WITH STATISTICAL TESTS\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total {characteristic_name}s: {len(characteristic_results['all_characteristics'])}\n")
        f.write(f"Total Studies: {len(analyzer.gold_standard_df)}\n\n")
        
        # ===== SECTION 1: STATISTICAL ANALYSES =====
        f.write("=" * 80 + "\n")
        f.write("SECTION 1: STATISTICAL ANALYSES\n")
        f.write("=" * 80 + "\n\n")
        
        # Test (i): Overall non-retrieval
        f.write("1.1 Overall Non-Retrieval Association\n")
        f.write("-" * 80 + "\n\n")
        
        overall_stats = statistical_tests['overall_nonretrieval']
        fisher_overall = overall_stats['fisher_exact']
        
        f.write(f"Research Question: Is {characteristic_name.lower()} associated with overall retrieval failure?\n\n")
        f.write(f"Method: Fisher's exact test with FDR correction\n")
        f.write(f"  - Definition: Each {characteristic_name.lower()} vs all others\n")
        f.write(f"  - Outcome: Never retrieved by ANY platform (0 LLMs) vs\n")
        f.write(f"             Retrieved by ≥1 platform (1-5 LLMs)\n")
        f.write(f"  - Correction: FDR (Benjamini-Hochberg) across {len(fisher_overall)} {characteristic_name.lower()}s\n")
        f.write(f"  - α = {overall_stats['alpha']}\n\n")
        
        f.write(f"{characteristic_name:<30} | {'Retrieved':<10} | {'Never':<10} | {'Total':<6} | ")
        f.write(f"{'OR':<8} | {'p-value':<12} | {'q-value':<12} | {'Sig.':<6}\n")
        f.write("-" * 105 + "\n")
        
        sorted_overall = sorted(fisher_overall.items(), key=lambda x: x[1]['p_corrected'])
        for char_value, result in sorted_overall:
            sig_marker = "***" if result['p_corrected'] < 0.001 else \
                        "**" if result['p_corrected'] < 0.01 else \
                        "*" if result['significant'] else "ns"
            char_name = str(char_value)[:28]
            p_text = format_p_value(result['p_value'], 0.05)
            q_text = format_p_value(result['p_corrected'], 0.05)
            f.write(f"{char_name:<30} | {result['retrieved']:<10} | {result['not_retrieved']:<10} | ")
            f.write(f"{result['total']:<6} | {result['odds_ratio']:<8.2f} | ")
            f.write(f"{p_text:<12} | {q_text:<12} | {sig_marker:<6}\n")
        
        f.write("\nNote: OR > 1 = higher odds of retrieval; OR < 1 = lower odds of retrieval\n\n")
        
        # Significant findings
        sig_characteristics = [(c, r) for c, r in fisher_overall.items() if r['significant']]
        if sig_characteristics:
            f.write(f"Significant {characteristic_name}s: {len(sig_characteristics)}\n\n")
            
            high_risk = [(c, r) for c, r in sig_characteristics if r['odds_ratio'] < 1]
            low_risk = [(c, r) for c, r in sig_characteristics if r['odds_ratio'] > 1]
            
            if high_risk:
                f.write(f"{characteristic_name}s with HIGHER risk of never being retrieved (OR < 1):\n")
                for char_value, result in sorted(high_risk, key=lambda x: x[1]['odds_ratio']):
                    never_rate = (result['not_retrieved'] / result['total'] * 100) if result['total'] > 0 else 0
                    f.write(f"  {char_value}:\n")
                    f.write(f"    OR = {result['odds_ratio']:.2f}\n")
                    f.write(f"    Never retrieved: {result['not_retrieved']}/{result['total']} ({never_rate:.1f}%)\n")
                    f.write(f"    q-value: {format_p_value(result['p_corrected'], 0.05)}\n\n")
            
            if low_risk:
                f.write(f"{characteristic_name}s with LOWER risk of never being retrieved (OR > 1):\n")
                for char_value, result in sorted(low_risk, key=lambda x: x[1]['odds_ratio'], reverse=True):
                    never_rate = (result['not_retrieved'] / result['total'] * 100) if result['total'] > 0 else 0
                    f.write(f"  {char_value}:\n")
                    f.write(f"    OR = {result['odds_ratio']:.2f}\n")
                    f.write(f"    Never retrieved: {result['not_retrieved']}/{result['total']} ({never_rate:.1f}%)\n")
                    f.write(f"    q-value: {format_p_value(result['p_corrected'], 0.05)}\n\n")
        else:
            f.write("No significant associations found after FDR correction.\n\n")
        
        # Test (ii): Platform-specific
        f.write("\n1.2 Platform-Specific Non-Retrieval Associations\n")
        f.write("-" * 80 + "\n\n")
        
        platform_stats = statistical_tests['platform_specific_nonretrieval']
        fisher_by_platform = platform_stats['fisher_by_platform']
        
        f.write(f"Research Question: Is {characteristic_name.lower()} associated with platform-specific\n")
        f.write(f"                   retrieval failure?\n\n")
        f.write(f"Method: Fisher's exact test with FDR correction (per platform)\n")
        f.write(f"  - Definition: Each {characteristic_name.lower()} vs all others, tested separately per platform\n")
        f.write(f"  - Outcome: Not retrieved by GIVEN platform vs retrieved by that platform\n")
        f.write(f"  - Correction: FDR within each platform\n\n")
        
        for llm in CONFIG['llms']:
            if llm not in fisher_by_platform:
                continue
            
            f.write(f"\n{llm}:\n")
            f.write("-" * 80 + "\n")
            
            llm_fisher = fisher_by_platform[llm]
            sig_chars = [c for c, r in llm_fisher.items() if r['significant']]
            
            f.write(f"Significant {characteristic_name.lower()}s: {len(sig_chars)}\n\n")
            
            if sig_chars:
                f.write(f"{characteristic_name:<30} | {'Retrieved':<10} | {'Not Retr.':<10} | ")
                f.write(f"{'OR':<8} | {'q-value':<12} | {'Sig.':<6}\n")
                f.write("-" * 95 + "\n")
                
                sorted_llm = sorted(llm_fisher.items(), key=lambda x: x[1]['p_corrected'])
                for char_value, result in sorted_llm:
                    if result['significant']:
                        sig_marker = "***" if result['p_corrected'] < 0.001 else \
                                    "**" if result['p_corrected'] < 0.01 else "*"
                        char_name = str(char_value)[:28]
                        q_text = format_p_value(result['p_corrected'], 0.05)
                        f.write(f"{char_name:<30} | {result['retrieved']:<10} | ")
                        f.write(f"{result['not_retrieved']:<10} | ")
                        f.write(f"{result['odds_ratio']:<8.2f} | {q_text:<12} | {sig_marker:<6}\n")
                
                f.write("\nNote: OR > 1 = higher odds of retrieval by this platform\n")
                f.write("      OR < 1 = lower odds of retrieval by this platform\n\n")
            else:
                f.write("No significant associations for this platform after FDR correction.\n\n")
        
        # ===== SECTION 2: DESCRIPTIVE STATISTICS =====
        f.write("\n" + "=" * 80 + "\n")
        f.write("SECTION 2: DESCRIPTIVE STATISTICS\n")
        f.write("=" * 80 + "\n\n")
        
        # Overall retrieval by characteristic
        f.write(f"2.1 Retrieval Distribution by {characteristic_name}\n")
        f.write("-" * 80 + "\n\n")
        
        characteristic_llm_data = characteristic_results['characteristic_llm_data']
        
        f.write(f"{characteristic_name:<30} | {'0 LLMs':<8} | {'1 LLM':<8} | {'2 LLMs':<8} | ")
        f.write(f"{'3 LLMs':<8} | {'4 LLMs':<8} | {'5 LLMs':<8} | {'Total':<8}\n")
        f.write("-" * 110 + "\n")
        
        for char_value in sorted(characteristic_llm_data.keys(), 
                                key=lambda x: sum(characteristic_llm_data[x].values()), 
                                reverse=True):
            char_name = str(char_value)[:28]
            counts = [characteristic_llm_data[char_value].get(f"{i} LLMs", 0) for i in range(6)]
            total = sum(counts)
            f.write(f"{char_name:<30} | {counts[0]:<8} | {counts[1]:<8} | {counts[2]:<8} | ")
            f.write(f"{counts[3]:<8} | {counts[4]:<8} | {counts[5]:<8} | {total:<8}\n")
        
        f.write("\n")
        
        # Platform-specific retrieval rates
        f.write(f"2.2 Platform-Specific Retrieval Rates by {characteristic_name}\n")
        f.write("-" * 80 + "\n\n")
        
        characteristic_by_llm = characteristic_results['characteristic_by_llm']
        
        for llm in CONFIG['llms']:
            f.write(f"\n{llm}:\n")
            f.write(f"{characteristic_name:<30} | {'Retrieved':<10} | {'Not Retrieved':<14} | {'Total':<8} | {'Rate':<8}\n")
            f.write("-" * 85 + "\n")
            
            # Sort by retrieval rate
            char_rates = []
            for char_value in characteristic_llm_data.keys():
                retrieved = characteristic_by_llm[llm][char_value]['retrieved']
                not_retrieved = characteristic_by_llm[llm][char_value]['not_retrieved']
                total = retrieved + not_retrieved
                rate = (retrieved / total * 100) if total > 0 else 0
                char_rates.append((char_value, retrieved, not_retrieved, total, rate))
            
            char_rates.sort(key=lambda x: x[4], reverse=True)
            
            for char_value, retrieved, not_retrieved, total, rate in char_rates:
                if total > 0:
                    char_name = str(char_value)[:28]
                    f.write(f"{char_name:<30} | {retrieved:<10} | {not_retrieved:<14} | ")
                    f.write(f"{total:<8} | {rate:>6.1f}%\n")
            
            f.write("\n")
        
        # Summary statistics
        f.write("2.3 Summary Statistics\n")
        f.write("-" * 80 + "\n\n")
        
        total_studies = len(analyzer.gold_standard_df)
        never_retrieved = sum(characteristic_llm_data[char].get("0 LLMs", 0) 
                             for char in characteristic_llm_data.keys())
        retrieved_by_all = sum(characteristic_llm_data[char].get("5 LLMs", 0) 
                              for char in characteristic_llm_data.keys())
        
        f.write(f"Total studies: {total_studies}\n")
        f.write(f"Never retrieved by any platform: {never_retrieved} ({never_retrieved/total_studies*100:.1f}%)\n")
        f.write(f"Retrieved by all 5 platforms: {retrieved_by_all} ({retrieved_by_all/total_studies*100:.1f}%)\n\n")
        
        # Per-platform summary
        f.write("Platform-level summary:\n")
        for llm in CONFIG['llms']:
            llm_retrieved = sum(characteristic_by_llm[llm][char]['retrieved'] 
                               for char in characteristic_llm_data.keys())
            llm_rate = (llm_retrieved / total_studies * 100) if total_studies > 0 else 0
            f.write(f"  {llm}: {llm_retrieved}/{total_studies} ({llm_rate:.1f}%)\n")
        
        f.write("\n")
        
        # Characteristic-level summary
        f.write(f"{characteristic_name}-level summary (sorted by total studies):\n")
        f.write(f"{characteristic_name:<30} | {'Total Studies':<14} | {'Never Retrieved':<16} | {'Never %':<10}\n")
        f.write("-" * 75 + "\n")
        
        for char_value in sorted(characteristic_llm_data.keys(), 
                                key=lambda x: sum(characteristic_llm_data[x].values()), 
                                reverse=True):
            char_name = str(char_value)[:28]
            total = sum(characteristic_llm_data[char_value].values())
            never = characteristic_llm_data[char_value].get("0 LLMs", 0)
            never_pct = (never / total * 100) if total > 0 else 0
            f.write(f"{char_name:<30} | {total:<14} | {never:<16} | {never_pct:>8.1f}%\n")
        
        # ===== FOOTER =====
        f.write("\n" + "=" * 80 + "\n")
        f.write("METHODOLOGICAL NOTES\n")
        f.write("=" * 80 + "\n\n")
        f.write("Statistical Approach:\n")
        f.write("1. Overall non-retrieval: Fisher's exact test comparing each characteristic\n")
        f.write("   value to all others for association with complete retrieval failure\n")
        f.write("   (never retrieved by any platform)\n\n")
        f.write("2. Platform-specific non-retrieval: Fisher's exact test comparing each\n")
        f.write("   characteristic value to all others for association with retrieval failure\n")
        f.write("   on a specific platform\n\n")
        f.write("3. Multiple testing correction: FDR (Benjamini-Hochberg) applied\n")
        f.write("   separately within each test family (overall, and per-platform)\n\n")
        f.write("4. Effect size: Odds ratios reported for all comparisons\n")
        f.write("   - OR > 1: Higher odds of retrieval\n")
        f.write("   - OR < 1: Lower odds of retrieval (higher risk of non-retrieval)\n\n")
        f.write("=" * 80 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 80 + "\n")
    
    print(f"✓ Report saved to: {output_file}")
    return output_file

# ===== CONVENIENCE WRAPPER FUNCTIONS =====

def analyze_publisher_characteristics(base_results):
    """Analyze publisher characteristics"""
    print("\n" + "="*80)
    print("  ANALYZING PUBLISHER CHARACTERISTICS")
    print("="*80)
    
    results = run_characteristic_analysis(
        base_results=base_results,
        characteristic_name='Publisher',
        extract_characteristic_func=lambda row: row.get('Publisher', 'Unknown'),
        standardize_func=standardize_publisher_name,
        visualization_title="Study Retrieval Distribution by Publisher"
    )
    
    report_file = generate_characteristic_report(results, base_results)
    
    return results, report_file

def analyze_venue_characteristics(base_results):
    """Analyze venue type characteristics"""
    print("\n" + "="*80)
    print("  ANALYZING VENUE TYPE CHARACTERISTICS")
    print("="*80)
    
    results = run_characteristic_analysis(
        base_results=base_results,
        characteristic_name='Venue Type',
        extract_characteristic_func=categorize_venue_type,
        visualization_title="Study Retrieval Distribution by Venue Type"
    )
    
    report_file = generate_characteristic_report(results, base_results)
    
    return results, report_file

def analyze_decade_characteristics(base_results):
    """Analyze publication decade characteristics"""
    print("\n" + "="*80)
    print("  ANALYZING PUBLICATION DECADE CHARACTERISTICS")
    print("="*80)
    
    results = run_characteristic_analysis(
        base_results=base_results,
        characteristic_name='Publication Decade',
        extract_characteristic_func=extract_publication_decade,
        visualization_title="Study Retrieval Distribution by Publication Decade"
    )
    
    report_file = generate_characteristic_report(results, base_results)
    
    return results, report_file

# ===== MAIN EXECUTION =====

def run_all_publication_characteristic_analyses(base_results):
    """
    Run all publication characteristic analyses
    This is the main function to call after base analysis
    """
    print("\n" + "="*80)
    print("  COMPREHENSIVE PUBLICATION CHARACTERISTICS ANALYSIS")
    print("  (Aligns with Methods: Fisher's exact tests with FDR correction)")
    print("="*80)
    
    results = {}
    
    # 1. Publisher analysis
    print("\n[1/3] Running Publisher Analysis...")
    publisher_results, publisher_report = analyze_publisher_characteristics(base_results)
    results['publisher'] = {
        'results': publisher_results,
        'report': publisher_report
    }
    
    # 2. Venue type analysis
    print("\n[2/3] Running Venue Type Analysis...")
    venue_results, venue_report = analyze_venue_characteristics(base_results)
    results['venue'] = {
        'results': venue_results,
        'report': venue_report
    }
    
    # 3. Publication decade analysis
    print("\n[3/3] Running Publication Decade Analysis...")
    decade_results, decade_report = analyze_decade_characteristics(base_results)
    results['decade'] = {
        'results': decade_results,
        'report': decade_report
    }
    
    # Summary
    print("\n" + "="*80)
    print("  ANALYSIS COMPLETE!")
    print("="*80)
    print("\nGenerated files:")
    print(f"  1. Publisher_Retrieval_Distribution.png")
    print(f"     {publisher_report}")
    print(f"  2. Venue_Type_Retrieval_Distribution.png")
    print(f"     {venue_report}")
    print(f"  3. Publication_Decade_Retrieval_Distribution.png")
    print(f"     {decade_report}")
    
    return results

# ===== EXECUTION =====
if 'base_results' in globals() and base_results:
    print("\n" + "="*80)
    print("  STARTING PUBLICATION CHARACTERISTICS ANALYSIS")
    print("  (Implements Fisher's exact tests as described in methods)")
    print("="*80)
    
    # Run all analyses
    publication_char_results = run_all_publication_characteristic_analyses(base_results)
    
    print("\n✅ All publication characteristic analyses completed successfully!")
    print("\nYou can access individual results:")
    print("  - publication_char_results['publisher']")
    print("  - publication_char_results['venue']")
    print("  - publication_char_results['decade']")
else:
    print("❌ Base results not available. Please run Part 1A first.")

## Cell 15 — MAIN STATISTICS

*PRIMARY Friedman + Cochran's Q + strategy test on adopted data*


In [ ]:
# =============================================================================
#  CELL 14 — POOLED + FORMULATION-LEVEL STATISTICS  (Figure 1 with p-values)
#
#  WHAT THIS CELL DOES (in order):
#  ─────────────────────────────────────────────────────────────────────────────
#  SETUP      Effect-size & CI helpers (Wilson CI, Kendall's W, rank-biserial r,
#             risk-difference CI, paired McNemar OR) — all validated against
#             independent library implementations before use.
#
#  FUNCTION   cochrans_q_test       — corrected numerator (R1.2/accuracy fix)
#  FUNCTION   mcnemar_pairwise_with_bonferroni — exact binomial for b+c<25
#  FUNCTION   friedman_test_formulations       — 15×5 matched-unit matrix
#  FUNCTION   wilcoxon_pairwise_with_fdr       — FDR-adjusted post-hoc
#  FUNCTION   [precision / consistency / F1 helpers, plotting helpers]
#
#  TEST 1     Cochran's Q — platform-level POOLED RECALL (83×5 binary matrix)
#             Secondary benchmarking outcome.
#             Effect size: Kendall's W.  Per-platform: Wilson 95% CI.
#             Post-hoc: Bonferroni-corrected McNemar with risk-difference CI
#             and paired OR (correct matched-pairs formula, not 2×2 table OR).
#
#  TEST 2 ★  Friedman — formulation-level RECALL  ← PRIMARY OUTCOME
#             [GAP FIX: this test was entirely missing from the original notebook;
#             Methods §48 states "platforms were compared on formulation-level
#             recall … using the Friedman test" yet only Precision/Consistency/F1
#             had this test. Results §51 claims p<0.001 and specific post-hoc
#             patterns with no corresponding code.]
#             Effect size: Kendall's W (omnibus); rank-biserial r (pairwise).
#
#  TEST 3     Friedman — formulation-level PRECISION
#             Effect size: Kendall's W; rank-biserial r per pair.
#
#  TEST 4     Friedman — formulation-level CONSISTENCY
#             Effect size: Kendall's W; rank-biserial r per pair.
#
#  TEST 5     Friedman — F1-SCORE  (additional analysis)
#             Effect size: Kendall's W; rank-biserial r per pair.
#
#  FIGURE 1   Boxplot grid with Cochran's Q / Friedman significance annotations.
#
#  OUTPUT     statistical_results dict (all tests + effect sizes)
#             df_variations DataFrame
#             df_variations_framework_aligned.csv
#             Comprehensive performance report .txt
#
#  CORRECTIONS APPLIED (all marked # MODIFIED inline):
#  ─────────────────────────────────────────────────────────────────────────────
#  R1.2  Cochran's Q numerator corrected:
#        WAS:  (k-1) * (sum(Cj²) - N²/n)   [divides by n_studies, not k]
#        IS:   (k-1) * (k·sum(Cj²) - N²)   [matches Cochran 1950 + statsmodels]
#  R1.2  McNemar: exact binomial when b+c < 25 (continuity correction otherwise)
#  R1.3  Precision capped at 1.0 (prevents values >1 from count mismatches)
#  Acc   83×5 retrieval matrix built via normalize_identifier_consistent so
#        the matrix column sums exactly equal reported pooled-recall counts
#  NEW   TEST 2 (Recall Friedman) added — see gap fix note above
#  NEW   Effect sizes and CIs throughout (see TEST descriptions above)
# =============================================================================

# ===== STATISTICAL TESTS MODULE - FRAMEWORK ALIGNED =====
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations as iter_combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------------
# EFFECT SIZE & CONFIDENCE INTERVAL HELPERS (added throughout the pipeline)
# p-values alone do not convey magnitude or precision; every reported test below
# now carries an accompanying effect size and, where applicable, a 95% CI.
# ----------------------------------------------------------------------------

def wilson_ci(successes, n, alpha=0.05):
    """Wilson score CI for a proportion. Preferred over normal approximation for
    small n / extreme proportions (e.g., n=4 strata). Validated against
    statsmodels.stats.proportion.proportion_confint(method='wilson')."""
    if n == 0:
        return (np.nan, np.nan)
    z = stats.norm.ppf(1 - alpha / 2)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return (max(0, center - margin), min(1, center + margin))

def rank_biserial_wilcoxon(x, y):
    """Matched-pairs rank-biserial correlation: effect size for Wilcoxon signed-rank,
    r=(W+ - W-)/(W+ + W-), bounded [-1,1]."""
    d = np.asarray(x) - np.asarray(y)
    d = d[d != 0]
    if len(d) == 0:
        return 0.0
    ranks = stats.rankdata(np.abs(d))
    w_pos = ranks[d > 0].sum()
    w_neg = ranks[d < 0].sum()
    return float((w_pos - w_neg) / (w_pos + w_neg)) if (w_pos + w_neg) > 0 else 0.0

def kendalls_w(chi2_stat, n_blocks, k_conditions):
    """Kendall's coefficient of concordance: effect size for Friedman/Cochran's Q
    omnibus tests. W=chi2/(n*(k-1)), bounded [0,1]."""
    denom = n_blocks * (k_conditions - 1)
    return float(chi2_stat / denom) if denom > 0 else 0.0

def risk_difference_ci(b, c, n_pair, alpha=0.05):
    """CI for the difference in paired proportions (Wald method), the standard
    companion statistic to McNemar's test."""
    if n_pair == 0:
        return (0.0, np.nan, np.nan)
    rd = (b - c) / n_pair
    se = np.sqrt((b + c) / n_pair**2 - (b - c)**2 / n_pair**3) if n_pair > 0 else np.nan
    z = stats.norm.ppf(1 - alpha / 2)
    return (rd, rd - z * se if not np.isnan(se) else np.nan, rd + z * se if not np.isnan(se) else np.nan)

def odds_ratio_ci(a, b, c, d, alpha=0.05):
    """Woolf logit-method CI for the odds ratio of an INDEPENDENT 2x2 table
    [[a,b],[c,d]] (e.g., Fisher's exact: domain membership x never-retrieved).
    Do NOT use for McNemar's matched pairs -- use paired_odds_ratio_ci instead."""
    if min(a, b, c, d) == 0:
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    OR = (a * d) / (b * c)
    se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
    z = stats.norm.ppf(1 - alpha / 2)
    log_or = np.log(OR)
    return (OR, np.exp(log_or - z * se_log_or), np.exp(log_or + z * se_log_or))

def paired_odds_ratio_ci(b, c, alpha=0.05):
    """Effect size for McNemar's test: OR_paired = b/c (ratio of discordant-pair
    counts), with log-transform CI. The correct companion to McNemar -- distinct
    from odds_ratio_ci, which assumes independent samples."""
    if b == 0 or c == 0:
        b, c = b + 0.5, c + 0.5
    OR = b / c
    se_log = np.sqrt(1/b + 1/c)
    z = stats.norm.ppf(1 - alpha / 2)
    log_or = np.log(OR)
    return (OR, np.exp(log_or - z * se_log), np.exp(log_or + z * se_log))

def cochrans_q_test(data_matrix):
    """
    Perform Cochran's Q test on binary retrieval data
    
    Framework C0.3.1: Each gold-standard study as paired binary outcome per platform
    
    Parameters:
    - data_matrix: rows=studies (n=83), columns=platforms (n=5) (1=retrieved, 0=not retrieved)
    
    Returns: Q statistic, p-value, degrees of freedom
    """
    n_studies, n_platforms = data_matrix.shape
    row_sums = data_matrix.sum(axis=1)
    col_sums = data_matrix.sum(axis=0)
    total = data_matrix.sum()
    
    # Cochran's Q formula
    numerator = (n_platforms - 1) * (n_platforms * (col_sums**2).sum() - total**2)
    denominator = (n_platforms * row_sums.sum()) - (row_sums**2).sum()
    
    if denominator == 0:
        return 0, 1.0, n_platforms - 1
    
    Q = numerator / denominator
    df = n_platforms - 1
    p_value = 1 - stats.chi2.cdf(Q, df)
    
    return Q, p_value, df

def mcnemar_pairwise_with_bonferroni(data_matrix, platform_names, alpha=0.05):
    """
    Perform pairwise McNemar tests with Bonferroni correction
    
    Framework C0.3.1: Bonferroni correction α = 0.05/10 = 0.005 for 10 comparisons
    
    Parameters:
    - data_matrix: 83×5 binary matrix from Cochran's Q
    - platform_names: list of 5 platform names
    - alpha: family-wise error rate (default 0.05)
    
    Returns: list of comparison results with Bonferroni-adjusted significance
    """
    n_platforms = len(platform_names)
    n_comparisons = n_platforms * (n_platforms - 1) // 2
    bonferroni_alpha = alpha / n_comparisons
    
    results = []
    
    for i, j in iter_combinations(range(n_platforms), 2):
        platform_i = data_matrix[:, i]
        platform_j = data_matrix[:, j]
        
        # McNemar test counts: b = only platform i, c = only platform j
        b = np.sum((platform_i == 1) & (platform_j == 0))
        c = np.sum((platform_i == 0) & (platform_j == 1))
        
        if b + c == 0:
            chi2_stat, p_value = 0.0, 1.0
        elif b + c < 25:
            chi2_stat = abs(b - c)
            p_value = min(2 * stats.binom.cdf(min(b, c), b + c, 0.5), 1.0)
        else:
            chi2_stat = ((abs(b - c) - 1)**2) / (b + c)
            p_value = 1 - stats.chi2.cdf(chi2_stat, 1)
        
        significant = p_value < bonferroni_alpha
        
        results.append({
            'platform_1': platform_names[i],
            'platform_2': platform_names[j],
            'platform_1_idx': i,
            'platform_2_idx': j,
            'retrieved_only_1': int(b),
            'retrieved_only_2': int(c),
            'chi2': chi2_stat,
            'p_value': p_value,
            'significant': significant,
            'bonferroni_alpha': bonferroni_alpha
        })
    
    return results

def friedman_test_formulations(df_variations, metric_col, platforms):
    """
    Perform Friedman test using FORMULATIONS as matched units
    
    Framework C0.3.2: 15 prompt formulations as matched repeated-measures units
    Creates 15 × 5 matrix (formulations × platforms)
    
    Parameters:
    - df_variations: DataFrame with formulation-level metrics (pooled across 3 runs)
    - metric_col: 'Precision', 'Consistency', or 'F1_Score'
    - platforms: list of 5 platform names
    
    Returns: statistic, p_value, error_message, data_matrix, n_complete_units
    """
    # Create formulation identifier (Strategy + Variation)
    if 'Formulation_ID' not in df_variations.columns:
        df_variations['Formulation_ID'] = (df_variations['Strategy_Code'].astype(str) + '_' + 
                                            df_variations['Variation'].astype(str))
    
    formulation_ids = sorted(df_variations['Formulation_ID'].unique())
    
    if len(formulation_ids) != 15:
        print(f"    Warning: Expected 15 formulations (5 strategies × 3 variations), found {len(formulation_ids)}")
    
    print(f"    Found {len(formulation_ids)} unique formulations")
    
    # Build matrix: rows = formulations (n=15), columns = platforms (n=5)
    data_matrix = []
    valid_formulations = []
    
    for formulation_id in formulation_ids:
        row = []
        complete = True
        
        for platform in platforms:
            platform_data = df_variations[
                (df_variations['LLM'] == platform) & 
                (df_variations['Formulation_ID'] == formulation_id)
            ]
            
            if len(platform_data) == 1:
                value = platform_data[metric_col].iloc[0]
                row.append(value)
            else:
                complete = False
                break
        
        if complete:
            data_matrix.append(row)
            valid_formulations.append(formulation_id)
    
    if len(data_matrix) < 3:
        return None, None, f"Insufficient complete formulations (need ≥3, have {len(data_matrix)})", None, 0
    
    data_matrix = np.array(data_matrix)
    print(f"    Matrix shape: {data_matrix.shape} (formulations × platforms)")
    
    try:
        # Friedman test: two-tailed
        statistic, p_value = stats.friedmanchisquare(*data_matrix.T)
        return statistic, p_value, None, data_matrix, len(data_matrix)
    except Exception as e:
        return None, None, str(e), None, 0

def wilcoxon_pairwise_with_fdr(data_matrix, platforms, alpha=0.05):
    """
    Perform pairwise Wilcoxon signed-rank tests with FDR correction
    
    Framework C0.3.2: 10 pairwise Wilcoxon tests with Benjamini-Hochberg FDR (q < 0.05)
    
    Parameters:
    - data_matrix: rows = matched units (formulations), columns = platforms
    - platforms: list of 5 platform names
    - alpha: FDR threshold (default 0.05)
    
    Returns: list of comparison results with FDR-adjusted significance
    """
    n_platforms = len(platforms)
    results = []
    p_values = []
    
    for i, j in iter_combinations(range(n_platforms), 2):
        platform_i_data = data_matrix[:, i]
        platform_j_data = data_matrix[:, j]
        
        try:
            # Wilcoxon signed-rank test: two-tailed
            statistic, p_value = stats.wilcoxon(platform_i_data, platform_j_data, alternative='two-sided')
        except:
            statistic, p_value = np.nan, 1.0
        
        results.append({
            'platform_1': platforms[i],
            'platform_2': platforms[j],
            'platform_1_idx': i,
            'platform_2_idx': j,
            'statistic': statistic,
            'p_value': p_value
        })
        p_values.append(p_value)
    
    # Benjamini-Hochberg FDR correction
    if p_values:
        reject, pvals_corrected, _, _ = multipletests(p_values, alpha=alpha, method='fdr_bh')
        for idx, result in enumerate(results):
            result['p_corrected'] = pvals_corrected[idx]
            result['significant'] = reject[idx]
            result['fdr_alpha'] = alpha
    
    return results

def format_p_value(p_value, threshold=0.05):
    """Format p-value for display"""
    if pd.isna(p_value):
        return "p = NA"
    if p_value < 0.001:
        return "p < 0.001"
    elif p_value < threshold:
        return f"p < {threshold}"
    elif p_value < 0.01:
        return f"p = {p_value:.3f}"
    else:
        return f"p = {p_value:.2f}"

def add_pairwise_comparison_bars(ax, data, group_col, value_col, comparisons, y_offset_start=None):
    """Add academic-style comparison bars with p-values"""
    if not comparisons:
        return
    
    groups = data[group_col].unique()
    group_to_idx = {group: idx for idx, group in enumerate(groups)}
    
    if y_offset_start is None:
        max_val = data[value_col].max()
        y_offset_start = max_val * 1.02
    
    sig_comparisons = [c for c in comparisons if c.get('significant', False)]
    
    if not sig_comparisons:
        return
    
    sig_comparisons.sort(key=lambda x: abs(group_to_idx[x['platform_1']] - group_to_idx[x['platform_2']]))
    
    y_offset = y_offset_start
    bar_height = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.03
    
    for comp in sig_comparisons:
        try:
            x1 = group_to_idx[comp['platform_1']]
            x2 = group_to_idx[comp['platform_2']]
            
            ax.plot([x1, x1], [y_offset - bar_height/2, y_offset],
                   color='black', linewidth=0.8, clip_on=False)
            ax.plot([x1, x2], [y_offset, y_offset],
                   color='black', linewidth=0.8, clip_on=False)
            ax.plot([x2, x2], [y_offset - bar_height/2, y_offset],
                   color='black', linewidth=0.8, clip_on=False)
            
            if 'p_corrected' in comp:
                p_val = comp['p_corrected']
                threshold = comp.get('fdr_alpha', 0.05)
            elif 'bonferroni_alpha' in comp:
                p_val = comp['p_value']
                threshold = comp['bonferroni_alpha']
            else:
                p_val = comp['p_value']
                threshold = 0.05
            
            p_text = format_p_value(p_val, threshold)
            mid_x = (x1 + x2) / 2
            ax.text(mid_x, y_offset + bar_height/2, p_text,
                   ha='center', va='bottom', fontsize=8, color='black')
            
            y_offset += bar_height * 2.5
        except KeyError:
            continue
    
    current_ylim = ax.get_ylim()
    new_ylim = (current_ylim[0], max(current_ylim[1], y_offset + bar_height))
    ax.set_ylim(new_ylim)

# ===== METRIC CALCULATION - FRAMEWORK ALIGNED =====

def calculate_recall_formulation_level(analyzer, recall_scores_base):
    """
    Calculate recall at FORMULATION level (pooled across 3 runs)
    
    Framework C0.2.1:
    - M^{pooled}_{p,f} = M^1_{p,f} ∪ M^2_{p,f} ∪ M^3_{p,f}
    - Recall_{p,f} = |M^{pooled}_{p,f}| / 83
    
    Parameters:
    - analyzer: LLMAnalysisBase instance
    - recall_scores_base: base recall scores from Part 1A
    
    Returns: recall_scores dict with formulation-level recall
    """
    print("\n=== Calculating Formulation-Level Recall (Framework C0.2.1) ===")
    
    recall_scores = {}
    
    # Group recalled files by formulation (LLM-Strategy-Variation)
    formulation_groups = {}
    
    for filename, info in analyzer.recalled_files.items():
        llm = info['llm']
        strategy = info['strategy']
        variation = info['variation']
        run = info['run']
        
        formulation_key = f"{llm}-{strategy}{variation}"
        
        if formulation_key not in formulation_groups:
            formulation_groups[formulation_key] = {}
        
        formulation_groups[formulation_key][run] = (filename, info)
    
    # Calculate pooled recall for each formulation
    for formulation_key, runs in formulation_groups.items():
        # Pool matched studies across all runs (union)
        pooled_matches = set()
        
        for run, (filename, info) in runs.items():
            df = analyzer.load_csv_file_content(info['path'])
            if not df.empty:
                matches, _ = analyzer.extract_studies_from_recalled_files(df)
                pooled_matches.update(matches)  # Union operation
        
        # Calculate recall: |M^pooled| / 83
        matched_count = len(pooled_matches)
        recall = matched_count / analyzer.gold_standard_count
        
        recall_scores[formulation_key] = {
            'recall': recall,
            'matched_count': matched_count,
            'matches': pooled_matches,
            'n_runs': len(runs)
        }
        
        print(f"  {formulation_key}: Runs={len(runs)}, Pooled matches={matched_count}, Recall={recall:.3f}")
    
    return recall_scores

def calculate_precision_formulation_level(analyzer, recall_scores):
    """
    Calculate precision at FORMULATION level (pooled across 3 runs)
    
    Framework C0.2.2:
    - R^{pooled}_{p,f} = R^1_{p,f} ∪ R^2_{p,f} ∪ R^3_{p,f}
    - Precision_{p,f} = |M^{pooled}_{p,f}| / |R^{pooled}_{p,f}|
    - Uses STANDARDIZED output titles for denominator
    
    Parameters:
    - analyzer: LLMAnalysisBase instance
    - recall_scores: recall scores from calculate_recall_formulation_level
    
    Returns: precision_data dict with formulation-level precision
    """
    print("\n=== Calculating Formulation-Level Precision (Framework C0.2.2) ===")
    
    precision_data = {}
    
    for var_key in recall_scores.keys():
        llm, strategy_var = var_key.split('-', 1)
        strategy = strategy_var[0]
        variation = strategy_var[1]
        
        # Get matched count from recall (numerator): |M^pooled|
        matched_count = recall_scores[var_key]['matched_count']
        
        # Pool retrieved studies across 3 runs from STANDARDIZED files (denominator): |R^pooled|
        pooled_retrieved = set()
        
        for filename, info in analyzer.standardized_files.items():
            if (info['llm'] == llm and 
                info['strategy'] == strategy and 
                info['variation'] == variation):
                df = analyzer.load_csv_file_content(info['path'])
                titles = analyzer.extract_titles_from_standardized_files(df)
                pooled_retrieved.update(titles)  # Union operation
        
        total_retrieved = len(pooled_retrieved)
        
        # Calculate precision: |M^pooled| / |R^pooled|
        if total_retrieved > 0:
            precision = min(matched_count / total_retrieved, 1.0)
        else:
            precision = 0.0
        
        precision_data[var_key] = {
            'matched_count': matched_count,
            'total_retrieved': total_retrieved,
            'precision': precision
        }
        
        print(f"  {var_key}: Retrieved={total_retrieved}, Matched={matched_count}, Precision={precision:.3f}")
    
    return precision_data

def calculate_consistency_formulation_level(analyzer):
    """
    Calculate inter-run consistency at FORMULATION level
    
    Framework C0.2.3:
    - Jaccard_{i,j} = |M^i ∩ M^j| / |M^i ∪ M^j|
    - Consistency_{p,f} = mean(Jaccard_{1,2}, Jaccard_{1,3}, Jaccard_{2,3})
    - Computed on MATCHED sets (from recalled files)
    
    Parameters:
    - analyzer: LLMAnalysisBase instance
    
    Returns: consistency_scores dict with formulation-level consistency
    """
    print("\n=== Calculating Formulation-Level Consistency (Framework C0.2.3) ===")
    
    consistency_scores = {}
    
    # Group recalled files by formulation
    formulation_groups = {}
    
    for filename, info in analyzer.recalled_files.items():
        llm = info['llm']
        strategy = info['strategy']
        variation = info['variation']
        run = info['run']
        
        formulation_key = f"{llm}-{strategy}{variation}"
        
        if formulation_key not in formulation_groups:
            formulation_groups[formulation_key] = {}
        
        formulation_groups[formulation_key][run] = (filename, info)
    
    # Calculate consistency for each formulation
    for formulation_key, runs in formulation_groups.items():
        if len(runs) < 2:
            consistency_scores[formulation_key] = 1.0
            print(f"  {formulation_key}: Only {len(runs)} run(s), consistency=1.0")
            continue
        
        # Load matched sets for each run
        matched_by_run = {}
        
        for run, (filename, info) in runs.items():
            df = analyzer.load_csv_file_content(info['path'])
            if not df.empty:
                matches, _ = analyzer.extract_studies_from_recalled_files(df)
                matched_by_run[run] = matches
        
        # Calculate pairwise Jaccard similarities
        run_ids = sorted(matched_by_run.keys())
        jaccard_scores = []
        
        for i in range(len(run_ids)):
            for j in range(i + 1, len(run_ids)):
                run_i = run_ids[i]
                run_j = run_ids[j]
                
                matches_i = matched_by_run[run_i]
                matches_j = matched_by_run[run_j]
                
                # Jaccard similarity: |M^i ∩ M^j| / |M^i ∪ M^j|
                if len(matches_i) == 0 and len(matches_j) == 0:
                    jaccard = 1.0
                elif len(matches_i) == 0 or len(matches_j) == 0:
                    jaccard = 0.0
                else:
                    intersection = len(matches_i & matches_j)
                    union = len(matches_i | matches_j)
                    jaccard = intersection / union if union > 0 else 0.0
                
                jaccard_scores.append(jaccard)
                print(f"    {formulation_key} Run{run_i} vs Run{run_j}: Jaccard={jaccard:.3f}")
        
        # Mean of pairwise Jaccard scores
        consistency = np.mean(jaccard_scores) if jaccard_scores else 1.0
        consistency_scores[formulation_key] = consistency
        
        print(f"  {formulation_key}: Mean consistency={consistency:.3f}")
    
    return consistency_scores

def calculate_f1_score(recall, precision):
    """
    Calculate F1-score from recall and precision
    
    F1 = 2 * (precision * recall) / (precision + recall)
    
    Parameters:
    - recall: recall value (0-1)
    - precision: precision value (0-1)
    
    Returns: F1-score (0-1)
    """
    if recall + precision == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_adaptive_ylim(data_series, buffer_percent=0.1):
    """Calculate y-axis limits based on actual data range with buffer"""
    min_val = data_series.min()
    max_val = data_series.max()
    data_range = max_val - min_val
    buffer = data_range * buffer_percent
    
    y_min = max(0, min_val - buffer)
    y_max = min(1, max_val + buffer)
    
    if y_max - y_min < 0.1:
        midpoint = (y_min + y_max) / 2
        y_min = max(0, midpoint - 0.05)
        y_max = min(1, midpoint + 0.05)
    
    return (y_min, y_max)

# ===== VISUALIZATION WITH FRAMEWORK-ALIGNED METRICS =====
def visualize_all_variations_with_stats(analyzer, base_recall_scores, base_variation_metadata):
    """
    Create comprehensive visualizations with framework-aligned metrics and statistics
    
    Framework alignment:
    - C0.2.1: Recall pooled across 3 runs (at-least-once coverage)
    - C0.2.2: Precision pooled across 3 runs (screening burden proxy)
    - C0.2.3: Consistency via mean pairwise Jaccard
    - C0.3.1: Cochran's Q (83×5 binary matrix) + Bonferroni McNemar
    - C0.3.2: Friedman (15×5 continuous matrix) + FDR Wilcoxon
    
    Parameters:
    - analyzer: LLMAnalysisBase instance
    - base_recall_scores: recall scores from Part 1A
    - base_variation_metadata: variation metadata from Part 1A
    
    Returns: df_variations, statistical_results
    """
    print("Creating visualizations with framework-aligned metrics...")
    
    # STEP 1: Calculate all metrics at formulation level (pooled across 3 runs)
    print("\n=== Step 1: Calculating Formulation-Level Metrics ===")
    
    recall_scores = calculate_recall_formulation_level(analyzer, base_recall_scores)
    precision_data = calculate_precision_formulation_level(analyzer, recall_scores)
    consistency_scores = calculate_consistency_formulation_level(analyzer)
    
    # STEP 2: Prepare data at FORMULATION level
    print("\n=== Step 2: Preparing Visualization Data (Formulation Level) ===")
    variation_data = []
    
    for var_key in recall_scores.keys():
        llm, strategy_var = var_key.split('-', 1)
        strategy = strategy_var[0]
        variation = strategy_var[1]
        
        # Formulation-level metrics (all pooled across 3 runs)
        recall = recall_scores[var_key]['recall']
        matched_count = recall_scores[var_key]['matched_count']
        n_runs = recall_scores[var_key]['n_runs']
        
        precision_info = precision_data.get(var_key, {})
        precision = precision_info.get('precision', 0.0)
        total_retrieved = precision_info.get('total_retrieved', 0)
        
        consistency = consistency_scores.get(var_key, 0.65)
        
        # Calculate F1-score
        f1_score = calculate_f1_score(recall, precision)
        
        variation_data.append({
            'LLM': llm,
            'Strategy': analyzer.strategy_mapping.get(strategy, f"Strategy-{strategy}"),
            'Strategy_Code': strategy,
            'Variation': variation,
            'Full_Key': var_key,
            'Consistency': consistency,
            'Recall': recall,
            'Precision': precision,
            'F1_Score': f1_score,
            'Matched_Count': matched_count,
            'Total_Retrieved': total_retrieved,
            'N_Runs': n_runs
        })
    
    df_variations = pd.DataFrame(variation_data)
    
    expected_entries = 5 * 5 * 3  # 5 LLMs × 5 strategies × 3 variations = 75
    print(f"✓ Created dataset with {len(df_variations)} entries (expected: {expected_entries})")
    print(f"   Each entry represents one formulation with metrics aggregated across 3 runs")
    
    # Check for missing formulations
    if len(df_variations) != expected_entries:
        print(f"⚠ Warning: Expected {expected_entries} formulations but found {len(df_variations)}")
    
    # STEP 3: Perform Statistical Tests
    print("\n=== Step 3: Performing Statistical Tests (Framework C0.3) ===")
    
    llm_list = sorted(df_variations['LLM'].unique())
    study_mapping = analyzer.create_study_identifier_mapping()
    n_studies = len(study_mapping)
    study_ids = list(study_mapping.keys())
    
    print(f"    Platforms: {len(llm_list)}")
    print(f"    Gold standard studies: {n_studies}")
    print(f"    Formulations: {len(df_variations['Full_Key'].unique())}")
    
    # ===== TEST 1: COCHRAN'S Q FOR POOLED RECALL (platform-level, secondary benchmarking; Framework C0.3.1) =====
    print("\n  Test 1: Cochran's Q for platform-level pooled recall (83 x 5 paired binary matrix)")
    
    # Build 83 x 5 matrix: each study x each platform (binary: ever retrieved)
    retrieval_matrix = np.zeros((n_studies, len(llm_list)))
    
    norm = analyzer.normalize_identifier_consistent
    for llm_idx, llm in enumerate(llm_list):
        llm_matches = set()
        for var_key, recall_info in recall_scores.items():
            if var_key.startswith(llm + '-'):
                llm_matches.update(norm(m) for m in recall_info['matches'])
        for study_idx, study_id in enumerate(study_ids):
            if study_mapping[study_id]['identifiers'] & llm_matches:
                retrieval_matrix[study_idx, llm_idx] = 1
    
    print(f"    Retrieval matrix shape: {retrieval_matrix.shape} (studies x platforms)")
    print(f"    Studies retrieved per platform: {retrieval_matrix.sum(axis=0).astype(int)}")
    
    Q_stat, Q_pval, Q_df = cochrans_q_test(retrieval_matrix)
    Q_W = kendalls_w(Q_stat, int(retrieval_matrix.sum()), len(llm_list))
    print(f"    Cochran's Q: Q={Q_stat:.3f}, df={Q_df}, p={Q_pval:.4f}, Kendall's W={Q_W:.3f}")
    print(f"    Pooled recall with 95% Wilson CI:")
    for li, llm in enumerate(llm_list):
        k_ = int(retrieval_matrix[:, li].sum())
        lo_, hi_ = wilson_ci(k_, n_studies)
        print(f"      {llm:14s} {k_}/{n_studies} = {k_/n_studies:.3f}  (95% CI {lo_:.3f}-{hi_:.3f})")
    
    mcnemar_results = []
    if Q_pval < 0.05:
        print("    Performing Bonferroni-adjusted McNemar pairwise tests (alpha=0.005)...")
        mcnemar_results = mcnemar_pairwise_with_bonferroni(retrieval_matrix, llm_list)
        for r in mcnemar_results:
            n_pair = retrieval_matrix.shape[0]
            rd, rd_lo, rd_hi = risk_difference_ci(r['retrieved_only_1'], r['retrieved_only_2'], n_pair)
            or_, or_lo, or_hi = paired_odds_ratio_ci(r['retrieved_only_1'], r['retrieved_only_2'])
            r['risk_difference'] = rd; r['rd_ci'] = (rd_lo, rd_hi)
            r['paired_odds_ratio'] = or_; r['or_ci'] = (or_lo, or_hi)
        sig_count = sum(1 for r in mcnemar_results if r['significant'])
        print(f"    Significant pairs (Bonferroni alpha=0.005): {sig_count}/{len(mcnemar_results)}")

    # ===== TEST 2: FRIEDMAN FOR FORMULATION-LEVEL RECALL (PRIMARY OUTCOME; Framework C0.3.2) =====
    # "Platforms were compared on formulation-level recall, precision, and consistency using the
    # Friedman test..." -- recall is the manuscript's stated PRIMARY outcome, yet only precision,
    # consistency, and F1 previously had this test. Results paragraph 51's specific claims
    # ("p<0.001; Friedman test", "academic platforms significantly outperformed conversational,
    # all FDR-adjusted p<0.05", "no significant differences among conversational platforms") had
    # no corresponding computation anywhere in the pipeline before this fix.
    print("\n  Test 2: Friedman test for formulation-level Recall (15 x 5 matrix) -- PRIMARY OUTCOME")
    
    friedman_recall_stat, friedman_recall_pval, friedman_recall_error, recall_matrix, n_recall_units = \
        friedman_test_formulations(df_variations, 'Recall', llm_list)
    
    recall_posthoc = []
    if friedman_recall_error:
        print(f"    Error: {friedman_recall_error}")
    else:
        recall_W = kendalls_w(friedman_recall_stat, n_recall_units, len(llm_list))
        print(f"    Friedman: chi2={friedman_recall_stat:.3f}, p={friedman_recall_pval:.4f}, Kendall's W={recall_W:.3f}")
        print(f"    Matched formulations: {n_recall_units}")
        if friedman_recall_pval is not None and friedman_recall_pval < 0.05:
            print("    Performing FDR-adjusted Wilcoxon pairwise tests (q<0.05)...")
            recall_posthoc = wilcoxon_pairwise_with_fdr(recall_matrix, llm_list)
            for r in recall_posthoc:
                i_, j_ = r['platform_1_idx'], r['platform_2_idx']
                r['rank_biserial_r'] = rank_biserial_wilcoxon(recall_matrix[:, i_], recall_matrix[:, j_])
            sig_count = sum(1 for r in recall_posthoc if r['significant'])
            print(f"    Significant pairs (FDR q<0.05): {sig_count}/{len(recall_posthoc)}")
            for r in recall_posthoc:
                tag = "*" if r['significant'] else " "
                print(f"      {tag} {r['platform_1']:14s} vs {r['platform_2']:14s}  "
                      f"r={r['rank_biserial_r']:+.2f}  q={r['p_corrected']:.4f}")

    # ===== TEST 3: FRIEDMAN FOR PRECISION (Framework C0.3.2) =====
    print("\n  Test 3: Friedman test for Precision (15 x 5 matrix)")
    
    friedman_precision_stat, friedman_precision_pval, friedman_precision_error, precision_matrix, n_precision_units = \
        friedman_test_formulations(df_variations, 'Precision', llm_list)
    
    if friedman_precision_error:
        print(f"    Error: {friedman_precision_error}")
    else:
        precision_W = kendalls_w(friedman_precision_stat, n_precision_units, len(llm_list))
        print(f"    Friedman: chi2={friedman_precision_stat:.3f}, p={friedman_precision_pval:.4f}, Kendall's W={precision_W:.3f}")
        print(f"    Matched formulations: {n_precision_units}")
    
    precision_posthoc = []
    if friedman_precision_pval is not None and friedman_precision_pval < 0.05:
        print("    Performing FDR-adjusted Wilcoxon pairwise tests (q<0.05)...")
        precision_posthoc = wilcoxon_pairwise_with_fdr(precision_matrix, llm_list)
        for r in precision_posthoc:
            i_, j_ = r['platform_1_idx'], r['platform_2_idx']
            r['rank_biserial_r'] = rank_biserial_wilcoxon(precision_matrix[:, i_], precision_matrix[:, j_])
        sig_count = sum(1 for r in precision_posthoc if r['significant'])
        print(f"    Significant pairs (FDR q<0.05): {sig_count}/{len(precision_posthoc)}")
    
    # ===== TEST 4: FRIEDMAN FOR CONSISTENCY (Framework C0.3.2) =====
    print("\n  Test 4: Friedman test for Consistency (15 x 5 matrix)")
    
    friedman_consistency_stat, friedman_consistency_pval, friedman_consistency_error, consistency_matrix, n_consistency_units = \
        friedman_test_formulations(df_variations, 'Consistency', llm_list)
    
    if friedman_consistency_error:
        print(f"    Error: {friedman_consistency_error}")
    else:
        consistency_W = kendalls_w(friedman_consistency_stat, n_consistency_units, len(llm_list))
        print(f"    Friedman: chi2={friedman_consistency_stat:.3f}, p={friedman_consistency_pval:.4f}, Kendall's W={consistency_W:.3f}")
        print(f"    Matched formulations: {n_consistency_units}")
    
    consistency_posthoc = []
    if friedman_consistency_pval is not None and friedman_consistency_pval < 0.05:
        print("    Performing FDR-adjusted Wilcoxon pairwise tests (q<0.05)...")
        consistency_posthoc = wilcoxon_pairwise_with_fdr(consistency_matrix, llm_list)
        for r in consistency_posthoc:
            i_, j_ = r['platform_1_idx'], r['platform_2_idx']
            r['rank_biserial_r'] = rank_biserial_wilcoxon(consistency_matrix[:, i_], consistency_matrix[:, j_])
        sig_count = sum(1 for r in consistency_posthoc if r['significant'])
        print(f"    Significant pairs (FDR q<0.05): {sig_count}/{len(consistency_posthoc)}")
    
    # ===== TEST 5: FRIEDMAN FOR F1-SCORE (Additional Analysis) =====
    print("\n  Test 5: Friedman test for F1-Score (15 x 5 matrix)")
    
    friedman_f1_stat, friedman_f1_pval, friedman_f1_error, f1_matrix, n_f1_units = \
        friedman_test_formulations(df_variations, 'F1_Score', llm_list)
    
    if friedman_f1_error:
        print(f"    Error: {friedman_f1_error}")
    else:
        f1_W = kendalls_w(friedman_f1_stat, n_f1_units, len(llm_list))
        print(f"    Friedman: chi2={friedman_f1_stat:.3f}, p={friedman_f1_pval:.4f}, Kendall's W={f1_W:.3f}")
        print(f"    Matched formulations: {n_f1_units}")
    
    f1_posthoc = []
    if friedman_f1_pval is not None and friedman_f1_pval < 0.05:
        print("    Performing FDR-adjusted Wilcoxon pairwise tests (q<0.05)...")
        f1_posthoc = wilcoxon_pairwise_with_fdr(f1_matrix, llm_list)
        for r in f1_posthoc:
            i_, j_ = r['platform_1_idx'], r['platform_2_idx']
            r['rank_biserial_r'] = rank_biserial_wilcoxon(f1_matrix[:, i_], f1_matrix[:, j_])
        sig_count = sum(1 for r in f1_posthoc if r['significant'])
        print(f"    Significant pairs (FDR q<0.05): {sig_count}/{len(f1_posthoc)}")
    
    # STEP 4: Create visualizations (UNCHANGED)
    print("\n=== Step 4: Creating Visualizations ===")
    
    fig, axes = plt.subplots(3, 3, figsize=(20, 18))
    plt.style.use('default')
    
    # ===============================================================
    # ROW 1: HEATMAPS
    # ===============================================================
    print("  Creating heatmaps...")
    
    # A. Recall
    pivot_recall = df_variations.pivot_table(
        values='Recall', index=['LLM', 'Strategy'], columns='Variation', aggfunc='first'
    ).fillna(0)
    sns.heatmap(pivot_recall, annot=True, fmt='.3f', cmap='Greens',
                ax=axes[0,0], cbar_kws={'label': 'Recall Score'})
    axes[0,0].set_title('A. Recall Scores Across All Variations\n(Pooled across 3 runs)', 
                       fontsize=18, fontweight='bold')
    axes[0,0].set_xlabel('Variation Number')
    axes[0,0].set_ylabel('LLM - Strategy')
    
    # B. Precision
    pivot_precision = df_variations.pivot_table(
        values='Precision', index=['LLM', 'Strategy'], columns='Variation', aggfunc='first'
    ).fillna(0)
    sns.heatmap(pivot_precision, annot=True, fmt='.3f', cmap='Oranges',
                ax=axes[0,1], cbar_kws={'label': 'Precision Score'})
    axes[0,1].set_title('B. Precision Scores Across All Variations\n(Pooled across 3 runs)',
                       fontsize=18, fontweight='bold')
    axes[0,1].set_xlabel('Variation Number')
    axes[0,1].set_ylabel('LLM - Strategy')
    
    # C. Consistency
    pivot_consistency = df_variations.pivot_table(
        values='Consistency', index=['LLM', 'Strategy'], columns='Variation', aggfunc='first'
    ).fillna(0)
    sns.heatmap(pivot_consistency, annot=True, fmt='.3f', cmap='Blues',
                ax=axes[0,2], cbar_kws={'label': 'Consistency Score'})
    axes[0,2].set_title('C. Consistency Scores Across All Variations\n(Mean pairwise Jaccard)',
                       fontsize=18, fontweight='bold')
    axes[0,2].set_xlabel('Variation Number')
    axes[0,2].set_ylabel('LLM - Strategy')
    
    # ===============================================================
    # ROW 2: BOX PLOTS BY LLM
    # ===============================================================
    print("  Creating boxplots by LLM...")
    
    recall_ylim = calculate_adaptive_ylim(df_variations['Recall'])
    precision_ylim = calculate_adaptive_ylim(df_variations['Precision'])
    consistency_ylim = calculate_adaptive_ylim(df_variations['Consistency'])
    
    # D. Recall by LLM
    sns.boxplot(data=df_variations, x='LLM', y='Recall', ax=axes[1,0], palette='Greens')
    axes[1,0].set_title('D. Recall Distribution by LLM', fontsize=18, fontweight='bold')
    axes[1,0].tick_params(axis='x', rotation=45)
    axes[1,0].set_xlabel('LLM', fontsize=16, fontweight='bold')
    axes[1,0].set_ylabel('Recall Score', fontsize=16, fontweight='bold')
    axes[1,0].set_ylim(recall_ylim)
    axes[1,0].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    if Q_pval < 0.05 and mcnemar_results:
        add_pairwise_comparison_bars(axes[1,0], df_variations, 'LLM', 'Recall', mcnemar_results)
    
    # E. Precision by LLM
    sns.boxplot(data=df_variations, x='LLM', y='Precision', ax=axes[1,1], palette='Oranges')
    axes[1,1].set_title('E. Precision Distribution by LLM\n(Formulation-Level)',
                       fontsize=18, fontweight='bold')
    axes[1,1].tick_params(axis='x', rotation=45)
    axes[1,1].set_xlabel('LLM', fontsize=16, fontweight='bold')
    axes[1,1].set_ylabel('Precision Score', fontsize=16, fontweight='bold')
    axes[1,1].set_ylim(precision_ylim)
    axes[1,1].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    if friedman_precision_pval is not None and friedman_precision_pval < 0.05 and precision_posthoc:
        add_pairwise_comparison_bars(axes[1,1], df_variations, 'LLM', 'Precision', precision_posthoc)
    
    # F. Consistency by LLM
    sns.boxplot(data=df_variations, x='LLM', y='Consistency', ax=axes[1,2], palette='Blues')
    axes[1,2].set_title('F. Consistency Distribution by LLM', fontsize=18, fontweight='bold')
    axes[1,2].tick_params(axis='x', rotation=45)
    axes[1,2].set_xlabel('LLM', fontsize=16, fontweight='bold')
    axes[1,2].set_ylabel('Consistency Score', fontsize=16, fontweight='bold')
    axes[1,2].set_ylim(consistency_ylim)
    axes[1,2].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    if friedman_consistency_pval is not None and friedman_consistency_pval < 0.05 and consistency_posthoc:
        add_pairwise_comparison_bars(axes[1,2], df_variations, 'LLM', 'Consistency', consistency_posthoc)
    
    # ===============================================================
    # ROW 3: BOX PLOTS BY STRATEGY
    # ===============================================================
    print("  Creating boxplots by Strategy...")
    
    # G. Recall by Strategy
    sns.boxplot(data=df_variations, x='Strategy', y='Recall', ax=axes[2,0], palette='Greens')
    axes[2,0].set_title('G. Recall Distribution by Strategy', fontsize=18, fontweight='bold')
    axes[2,0].tick_params(axis='x', rotation=45)
    axes[2,0].set_xlabel('Strategy', fontsize=16, fontweight='bold')
    axes[2,0].set_ylabel('Recall Score', fontsize=16, fontweight='bold')
    axes[2,0].set_ylim(recall_ylim)
    axes[2,0].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # H. Precision by Strategy
    sns.boxplot(data=df_variations, x='Strategy', y='Precision', ax=axes[2,1], palette='Oranges')
    axes[2,1].set_title('H. Precision Distribution by Strategy\n(Formulation-Level)',
                       fontsize=18, fontweight='bold')
    axes[2,1].tick_params(axis='x', rotation=45)
    axes[2,1].set_xlabel('Strategy', fontsize=16, fontweight='bold')
    axes[2,1].set_ylabel('Precision Score', fontsize=16, fontweight='bold')
    axes[2,1].set_ylim(precision_ylim)
    axes[2,1].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # I. Consistency by Strategy
    sns.boxplot(data=df_variations, x='Strategy', y='Consistency', ax=axes[2,2], palette='Blues')
    axes[2,2].set_title('I. Consistency Distribution by Strategy',
                       fontsize=18, fontweight='bold')
    axes[2,2].tick_params(axis='x', rotation=45)
    axes[2,2].set_xlabel('Strategy', fontsize=16, fontweight='bold')
    axes[2,2].set_ylabel('Consistency Score', fontsize=16, fontweight='bold')
    axes[2,2].set_ylim(consistency_ylim)
    axes[2,2].grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    plt.tight_layout()
    
    output_filename_1 = 'Overall_Performance_Framework_Aligned.png'
    output_filename_2 = 'Overall_Performance_Framework_Aligned.pdf'
    plt.savefig(output_filename_1, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none', format='png',
                transparent=False, pad_inches=0.2)
    plt.savefig(output_filename_2, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none', format='pdf',
                transparent=False, pad_inches=0.2)
    plt.show()
    
    # Save results
    output_file = 'df_variations_framework_aligned.csv'
    df_variations.to_csv(output_file, index=False)
    print(f"✓ Results saved to: {output_file}")
    
    # Store statistical test results
    statistical_results = {
        'recall': {  # platform-level pooled recall (Cochran's Q) -- secondary benchmarking analysis
            'test': 'cochrans_q',
            'description': 'Framework C0.3.1: Binary outcome (study-level), 83x5 paired matrix',
            'Q': Q_stat,
            'p_value': Q_pval,
            'df': Q_df,
            'kendalls_w': Q_W,
            'n_studies': n_studies,
            'posthoc': mcnemar_results,
            'posthoc_correction': 'bonferroni',
            'bonferroni_alpha': 0.005
        },
        'formulation_recall': {
            'test': 'friedman',
            'description': 'Framework C0.3.2 / Methods para. 48 (primary outcome): Continuous outcome, 15x5 formulation-matched matrix',
            'statistic': friedman_recall_stat,
            'p_value': friedman_recall_pval,
            'error': friedman_recall_error,
            'kendalls_w': recall_W if not friedman_recall_error else None,
            'n_matched_units': n_recall_units,
            'unit_definition': 'Formulations (Strategy x Variation), pooled across 3 runs',
            'posthoc': recall_posthoc,
            'posthoc_correction': 'fdr_bh',
            'fdr_alpha': 0.05
        },
        'precision': {
            'test': 'friedman',
            'description': 'Framework C0.3.2: Continuous outcome, 15x5 formulation-matched matrix',
            'statistic': friedman_precision_stat,
            'p_value': friedman_precision_pval,
            'error': friedman_precision_error,
            'kendalls_w': precision_W if not friedman_precision_error else None,
            'n_matched_units': n_precision_units,
            'unit_definition': 'Formulations (Strategy x Variation), pooled across 3 runs',
            'posthoc': precision_posthoc,
            'posthoc_correction': 'fdr_bh',
            'fdr_alpha': 0.05
        },
        'consistency': {
            'test': 'friedman',
            'description': 'Framework C0.3.2: Continuous outcome, 15x5 formulation-matched matrix',
            'statistic': friedman_consistency_stat,
            'p_value': friedman_consistency_pval,
            'error': friedman_consistency_error,
            'kendalls_w': consistency_W if not friedman_consistency_error else None,
            'n_matched_units': n_consistency_units,
            'unit_definition': 'Formulations (Strategy x Variation), mean pairwise Jaccard',
            'posthoc': consistency_posthoc,
            'posthoc_correction': 'fdr_bh',
            'fdr_alpha': 0.05
        },
        'f1_score': {
            'test': 'friedman',
            'description': 'Additional analysis: F1-score (harmonic mean of recall and precision)',
            'statistic': friedman_f1_stat,
            'p_value': friedman_f1_pval,
            'error': friedman_f1_error,
            'kendalls_w': f1_W if not friedman_f1_error else None,
            'n_matched_units': n_f1_units,
            'unit_definition': 'Formulations (Strategy x Variation), pooled across 3 runs',
            'posthoc': f1_posthoc,
            'posthoc_correction': 'fdr_bh',
            'fdr_alpha': 0.05
        },
        'retrieval_matrix': retrieval_matrix,
        'llm_list': llm_list,
        'design': {
            'n_platforms': len(llm_list),
            'n_strategies': 5,
            'n_variations': 3,
            'n_runs': 3,
            'n_formulations': 15,
            'total_experiments': len(llm_list) * 15 * 3,
            'gold_standard_studies': n_studies,
            'formulation_recall_matched_units': n_recall_units,
            'precision_matched_units': n_precision_units,
            'consistency_matched_units': n_consistency_units,
            'f1_matched_units': n_f1_units
        }
    }
    
    return df_variations, statistical_results, recall_scores

# ===== COMPREHENSIVE REPORT WITH F1 ANALYSIS =====
def generate_comprehensive_report_with_stats(df_variations, analyzer, recall_scores, statistical_results,
                                             output_filename='Performance_Report_Framework_Aligned.txt'):
    """Generate comprehensive report aligned with framework including F1-score analysis"""
    print(f"\n=== Generating Framework-Aligned Report with F1 Analysis: {output_filename} ===")
    
    with open(output_filename, 'w', encoding='utf-8') as f:
        # Header
        f.write("="*80 + "\n")
        f.write("COMPREHENSIVE PERFORMANCE ANALYSIS - FRAMEWORK ALIGNED\n")
        f.write("="*80 + "\n")
        f.write(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("FRAMEWORK ALIGNMENT:\n")
        f.write("  C0.1: Experimental Structure (5 platforms × 15 formulations × 3 runs)\n")
        f.write("  C0.2.1: Recall (pooled across runs, at-least-once coverage)\n")
        f.write("  C0.2.2: Precision (screening burden proxy, pooled across runs)\n")
        f.write("  C0.2.3: Consistency (mean pairwise Jaccard on matched sets)\n")
        f.write("  C0.3.1: Cochran's Q test for recall (83×5 binary matrix, Bonferroni)\n")
        f.write("  C0.3.2: Friedman test for precision/consistency (15×5 matrix, FDR)\n\n")
        
        f.write("REPORTING FORMAT (per methods section):\n")
        f.write("  - Recall and Precision: reported as percentages\n")
        f.write("  - F1-scores: reported as decimals (0-1)\n")
        f.write("  - Consistency: reported as Jaccard coefficients (0-1)\n\n")
        
        f.write(f"Total Formulations Analyzed: {len(df_variations)}\n")
        f.write(f"Experimental Design:\n")
        f.write(f"  - Platforms: {statistical_results['design']['n_platforms']}\n")
        f.write(f"  - Strategies: {statistical_results['design']['n_strategies']}\n")
        f.write(f"  - Variations per strategy: {statistical_results['design']['n_variations']}\n")
        f.write(f"  - Formulations tested: {statistical_results['design']['n_formulations']}\n")
        f.write(f"  - Runs per formulation: {statistical_results['design']['n_runs']}\n")
        f.write(f"  - Total experiments: {statistical_results['design']['total_experiments']}\n")
        f.write(f"  - Gold standard studies: {statistical_results['design']['gold_standard_studies']}\n")
        f.write("="*80 + "\n\n")
        
        # ===== SECTION 1: STATISTICAL ANALYSES =====
        f.write("="*80 + "\n")
        f.write("1. STATISTICAL ANALYSES (Framework C0.3)\n")
        f.write("="*80 + "\n\n")
        
        # 1.1 Cochran's Q
        f.write("1.1 Platform Recall Comparison (Cochran's Q Test - Framework C0.3.1)\n")
        f.write("-" * 80 + "\n\n")
        
        recall_stats = statistical_results['recall']
        retrieval_matrix = statistical_results['retrieval_matrix']
        llm_list = statistical_results['llm_list']
        n_studies = recall_stats['n_studies']
        
        f.write(f"Research Question: Do platforms differ in pooled recall rates?\n\n")
        f.write(f"Method: Cochran's Q test (two-tailed)\n")
        f.write(f"  - Unit of analysis: Each gold-standard study (n={n_studies})\n")
        f.write(f"  - Outcome: Binary (retrieved ≥1 time vs not retrieved)\n")
        f.write(f"  - Pooling: Across all 15 formulations × 3 runs = 45 queries per platform\n")
        f.write(f"  - Matrix structure: {n_studies} studies × {len(llm_list)} platforms\n")
        f.write(f"  - Matching: Same studies evaluated across all platforms\n")
        f.write(f"  - Rationale: Appropriate for binary repeated-measures data\n\n")
        
        f.write(f"Results:\n")
        f.write(f"  Cochran's Q statistic: {recall_stats['Q']:.3f}\n")
        f.write(f"  Degrees of freedom:    {recall_stats['df']}\n")
        f.write(f"  p-value:               {format_p_value(recall_stats['p_value'], 0.05)}\n")
        f.write(f"  Interpretation:        {'Significant difference across platforms' if recall_stats['p_value'] < 0.05 else 'No significant difference'} (α=0.05)\n\n")
        
        f.write("Platform Pooled Recall Rates:\n")
        f.write(f"{'Platform':<15} | {'Studies Retrieved':<18} | {'Total':<7} | {'Recall %':<10}\n")
        f.write("-" * 65 + "\n")
        
        for llm_idx, llm in enumerate(llm_list):
            retrieved = int(retrieval_matrix[:, llm_idx].sum())
            recall_pct = (retrieved / n_studies) * 100
            f.write(f"{llm:<15} | {retrieved:<18} | {n_studies:<7} | {recall_pct:>9.1f}%\n")
        
        f.write("\n")
        
        # McNemar post-hoc
        if recall_stats['p_value'] < 0.05:
            f.write("Post-hoc: McNemar Tests with Bonferroni Correction (Framework C0.3.1)\n")
            f.write("-" * 80 + "\n\n")
            
            mcnemar_results = recall_stats['posthoc']
            if mcnemar_results:
                bonferroni_alpha = mcnemar_results[0]['bonferroni_alpha']
                n_comparisons = len(mcnemar_results)
                f.write(f"  Bonferroni-corrected α = 0.05/{n_comparisons} = {bonferroni_alpha:.4f}\n\n")
                
                f.write(f"{'Platform 1':<15} | {'Platform 2':<15} | {'Only P1':<8} | {'Only P2':<8} | ")
                f.write(f"{'χ²':<8} | {'p-value':<15} | {'Sig.':<6}\n")
                f.write("-" * 90 + "\n")
                
                for result in mcnemar_results:
                    sig_marker = "Yes" if result['significant'] else "No"
                    p_text = format_p_value(result['p_value'], bonferroni_alpha)
                    f.write(f"{result['platform_1']:<15} | {result['platform_2']:<15} | ")
                    f.write(f"{result['retrieved_only_1']:<8} | {result['retrieved_only_2']:<8} | ")
                    f.write(f"{result['chi2']:<8.3f} | {p_text:<15} | {sig_marker:<6}\n")
                
                sig_pairs = [r for r in mcnemar_results if r['significant']]
                if sig_pairs:
                    f.write(f"\nSignificant pairwise differences (α={bonferroni_alpha:.4f}): {len(sig_pairs)}\n")
                    for pair in sig_pairs:
                        f.write(f"  {pair['platform_1']} vs {pair['platform_2']}: {format_p_value(pair['p_value'], bonferroni_alpha)}\n")
                f.write("\n")
        else:
            f.write("Post-hoc tests: Not performed (Cochran's Q not significant)\n\n")
        
        # 1.2 Friedman for Precision
        f.write("1.2 Platform Precision Comparison (Friedman Test - Framework C0.3.2)\n")
        f.write("-" * 80 + "\n\n")
        
        precision_stats = statistical_results['precision']
        
        f.write(f"Research Question: Do platforms differ in precision?\n\n")
        f.write(f"Method: Friedman test (two-tailed) with formulation-level matching\n")
        f.write(f"  - Unit of analysis: Prompt formulations (n={precision_stats['n_matched_units']})\n")
        f.write(f"  - Unit definition: {precision_stats['unit_definition']}\n")
        f.write(f"  - Outcome: Precision (continuous, 0-1) per formulation\n")
        f.write(f"  - Calculation: |M^pooled| / |R^pooled| (Framework C0.2.2)\n")
        f.write(f"  - Matrix structure: {precision_stats['n_matched_units']} formulations × {len(llm_list)} platforms\n")
        f.write(f"  - Matching: Same formulation tested across all platforms\n")
        f.write(f"  - Rationale: Non-parametric test for continuous repeated measures\n\n")
        
        if precision_stats['error']:
            f.write(f"Error: {precision_stats['error']}\n\n")
        else:
            f.write(f"Results:\n")
            f.write(f"  Friedman χ² statistic: {precision_stats['statistic']:.3f}\n")
            f.write(f"  p-value:               {format_p_value(precision_stats['p_value'], 0.05)}\n")
            f.write(f"  Matched units used:    {precision_stats['n_matched_units']}\n")
            f.write(f"  Interpretation:        {'Significant difference in precision' if precision_stats['p_value'] < 0.05 else 'No significant difference'} (α=0.05)\n\n")
            
            if precision_stats['p_value'] < 0.05 and precision_stats['posthoc']:
                f.write("Post-hoc: Wilcoxon Signed-Rank Tests with FDR Correction (Benjamini-Hochberg, q<0.05)\n")
                f.write("-" * 80 + "\n\n")
                
                posthoc = precision_stats['posthoc']
                fdr_alpha = posthoc[0]['fdr_alpha'] if posthoc else 0.05
                
                f.write(f"{'Platform 1':<15} | {'Platform 2':<15} | {'p-value':<15} | {'q-value':<15} | {'Sig.':<6}\n")
                f.write("-" * 80 + "\n")
                
                for result in posthoc:
                    sig_marker = "Yes" if result['significant'] else "No"
                    p_text = format_p_value(result['p_value'], 0.05)
                    q_text = format_p_value(result['p_corrected'], fdr_alpha)
                    f.write(f"{result['platform_1']:<15} | {result['platform_2']:<15} | ")
                    f.write(f"{p_text:<15} | {q_text:<15} | {sig_marker:<6}\n")
                
                f.write("\n")
        
        f.write("Platform Precision Summary (Formulation-Level, reported as percentages):\n")
        f.write(f"{'Platform':<15} | {'Mean':<8} | {'Median':<8} | {'SD':<8} | {'Range':<20}\n")
        f.write("-" * 75 + "\n")
        
        for llm in llm_list:
            llm_precision = df_variations[df_variations['LLM'] == llm]['Precision']
            f.write(f"{llm:<15} | {llm_precision.mean()*100:<8.1f}% | {llm_precision.median()*100:<8.1f}% | ")
            f.write(f"{llm_precision.std()*100:<8.1f}% | [{llm_precision.min()*100:.1f}%, {llm_precision.max()*100:.1f}%]\n")
        
        f.write("\n")
        
        # 1.3 Friedman for Consistency
        f.write("1.3 Platform Consistency Comparison (Friedman Test - Framework C0.3.2)\n")
        f.write("-" * 80 + "\n\n")
        
        consistency_stats = statistical_results['consistency']
        
        f.write(f"Research Question: Do platforms differ in inter-run consistency?\n\n")
        f.write(f"Method: Friedman test (two-tailed) with formulation-level matching\n")
        f.write(f"  - Unit of analysis: Prompt formulations (n={consistency_stats['n_matched_units']})\n")
        f.write(f"  - Unit definition: {consistency_stats['unit_definition']}\n")
        f.write(f"  - Outcome: Jaccard similarity coefficient (continuous, 0-1)\n")
        f.write(f"  - Calculation: mean(Jaccard₁,₂, Jaccard₁,₃, Jaccard₂,₃) (Framework C0.2.3)\n")
        f.write(f"  - Matrix structure: {consistency_stats['n_matched_units']} formulations × {len(llm_list)} platforms\n")
        f.write(f"  - Matching: Same formulation tested across all platforms\n\n")
        
        if consistency_stats['error']:
            f.write(f"Error: {consistency_stats['error']}\n\n")
        else:
            f.write(f"Results:\n")
            f.write(f"  Friedman χ² statistic: {consistency_stats['statistic']:.3f}\n")
            f.write(f"  p-value:               {format_p_value(consistency_stats['p_value'], 0.05)}\n")
            f.write(f"  Matched units used:    {consistency_stats['n_matched_units']}\n")
            f.write(f"  Interpretation:        {'Significant difference in consistency' if consistency_stats['p_value'] < 0.05 else 'No significant difference'} (α=0.05)\n\n")
            
            if consistency_stats['p_value'] < 0.05 and consistency_stats['posthoc']:
                f.write("Post-hoc: Wilcoxon Signed-Rank Tests with FDR Correction (Benjamini-Hochberg, q<0.05)\n")
                f.write("-" * 80 + "\n\n")
                
                posthoc = consistency_stats['posthoc']
                fdr_alpha = posthoc[0]['fdr_alpha'] if posthoc else 0.05
                
                f.write(f"{'Platform 1':<15} | {'Platform 2':<15} | {'p-value':<15} | {'q-value':<15} | {'Sig.':<6}\n")
                f.write("-" * 80 + "\n")
                
                for result in posthoc:
                    sig_marker = "Yes" if result['significant'] else "No"
                    p_text = format_p_value(result['p_value'], 0.05)
                    q_text = format_p_value(result['p_corrected'], fdr_alpha)
                    f.write(f"{result['platform_1']:<15} | {result['platform_2']:<15} | ")
                    f.write(f"{p_text:<15} | {q_text:<15} | {sig_marker:<6}\n")
                
                f.write("\n")
        
        f.write("Platform Consistency Summary (Jaccard Similarity Coefficients, 0-1):\n")
        f.write(f"{'Platform':<15} | {'Mean':<7} | {'Median':<7} | {'SD':<7} | {'Range':<15}\n")
        f.write("-" * 70 + "\n")
        
        for llm in llm_list:
            llm_consistency = df_variations[df_variations['LLM'] == llm]['Consistency']
            f.write(f"{llm:<15} | {llm_consistency.mean():<7.3f} | {llm_consistency.median():<7.3f} | ")
            f.write(f"{llm_consistency.std():<7.3f} | [{llm_consistency.min():.3f}, {llm_consistency.max():.3f}]\n")
        
        f.write("\n")
        
        # 1.4 Friedman for F1-Score
        f.write("1.4 Platform F1-Score Comparison (Friedman Test - Additional Analysis)\n")
        f.write("-" * 80 + "\n\n")
        
        f1_stats = statistical_results['f1_score']
        
        f.write(f"Research Question: Do platforms differ in overall retrieval quality (F1-score)?\n\n")
        f.write(f"Method: Friedman test (two-tailed) with formulation-level matching\n")
        f.write(f"  - Unit of analysis: Prompt formulations (n={f1_stats['n_matched_units']})\n")
        f.write(f"  - Unit definition: {f1_stats['unit_definition']}\n")
        f.write(f"  - Outcome: F1-score = 2×(Precision×Recall)/(Precision+Recall)\n")
        f.write(f"  - Harmonic mean balancing recall and precision trade-off\n")
        f.write(f"  - Matrix structure: {f1_stats['n_matched_units']} formulations × {len(llm_list)} platforms\n\n")
        
        if f1_stats['error']:
            f.write(f"Error: {f1_stats['error']}\n\n")
        else:
            f.write(f"Results:\n")
            f.write(f"  Friedman χ² statistic: {f1_stats['statistic']:.3f}\n")
            f.write(f"  p-value:               {format_p_value(f1_stats['p_value'], 0.05)}\n")
            f.write(f"  Matched units used:    {f1_stats['n_matched_units']}\n")
            f.write(f"  Interpretation:        {'Significant difference in F1-score' if f1_stats['p_value'] < 0.05 else 'No significant difference'} (α=0.05)\n\n")
            
            if f1_stats['p_value'] < 0.05 and f1_stats['posthoc']:
                f.write("Post-hoc: Wilcoxon Signed-Rank Tests with FDR Correction (Benjamini-Hochberg, q<0.05)\n")
                f.write("-" * 80 + "\n\n")
                
                posthoc = f1_stats['posthoc']
                fdr_alpha = posthoc[0]['fdr_alpha'] if posthoc else 0.05
                
                f.write(f"{'Platform 1':<15} | {'Platform 2':<15} | {'p-value':<15} | {'q-value':<15} | {'Sig.':<6}\n")
                f.write("-" * 80 + "\n")
                
                for result in posthoc:
                    sig_marker = "Yes" if result['significant'] else "No"
                    p_text = format_p_value(result['p_value'], 0.05)
                    q_text = format_p_value(result['p_corrected'], fdr_alpha)
                    f.write(f"{result['platform_1']:<15} | {result['platform_2']:<15} | ")
                    f.write(f"{p_text:<15} | {q_text:<15} | {sig_marker:<6}\n")
                
                f.write("\n")
        
        f.write("Platform F1-Score Summary (reported as decimals, 0-1):\n")
        f.write(f"{'Platform':<15} | {'Mean':<7} | {'Median':<7} | {'SD':<7} | {'Range':<15}\n")
        f.write("-" * 70 + "\n")
        
        for llm in llm_list:
            llm_f1 = df_variations[df_variations['LLM'] == llm]['F1_Score']
            f.write(f"{llm:<15} | {llm_f1.mean():<7.3f} | {llm_f1.median():<7.3f} | ")
            f.write(f"{llm_f1.std():<7.3f} | [{llm_f1.min():.3f}, {llm_f1.max():.3f}]\n")
        
        f.write("\n")
        
        # ===== SECTION 2: PERFORMANCE SUMMARY =====
        f.write("\n" + "="*80 + "\n")
        f.write("2. PERFORMANCE SUMMARY (Reported per Methods Section)\n")
        f.write("="*80 + "\n\n")
        
        best_recall = df_variations.loc[df_variations['Recall'].idxmax()]
        best_precision = df_variations.loc[df_variations['Precision'].idxmax()]
        best_consistency = df_variations.loc[df_variations['Consistency'].idxmax()]
        best_f1 = df_variations.loc[df_variations['F1_Score'].idxmax()]
        
        f.write("Top Formulation Performers:\n")
        f.write(f"  Best Recall:      {best_recall['LLM']} - {best_recall['Strategy']} (Var {best_recall['Variation']}) = {best_recall['Recall']*100:.1f}%\n")
        f.write(f"  Best Precision:   {best_precision['LLM']} - {best_precision['Strategy']} (Var {best_precision['Variation']}) = {best_precision['Precision']*100:.1f}%\n")
        f.write(f"  Best F1-Score:    {best_f1['LLM']} - {best_f1['Strategy']} (Var {best_f1['Variation']}) = {best_f1['F1_Score']:.3f}\n")
        f.write(f"  Best Consistency: {best_consistency['LLM']} - {best_consistency['Strategy']} (Var {best_consistency['Variation']}) = {best_consistency['Consistency']:.3f}\n\n")
        
        f.write("Overall Metrics (All Formulations):\n")
        f.write(f"  Recall:      Mean={df_variations['Recall'].mean()*100:.1f}%, Range=[{df_variations['Recall'].min()*100:.1f}%, {df_variations['Recall'].max()*100:.1f}%]\n")
        f.write(f"  Precision:   Mean={df_variations['Precision'].mean()*100:.1f}%, Range=[{df_variations['Precision'].min()*100:.1f}%, {df_variations['Precision'].max()*100:.1f}%]\n")
        f.write(f"  F1-Score:    Mean={df_variations['F1_Score'].mean():.3f}, Range=[{df_variations['F1_Score'].min():.3f}, {df_variations['F1_Score'].max():.3f}]\n")
        f.write(f"  Consistency: Mean={df_variations['Consistency'].mean():.3f}, Range=[{df_variations['Consistency'].min():.3f}, {df_variations['Consistency'].max():.3f}]\n\n")
        
        # ===== SECTION 3: PLATFORM ANALYSIS =====
        f.write("\n" + "="*80 + "\n")
        f.write("3. PLATFORM PERFORMANCE ANALYSIS\n")
        f.write("="*80 + "\n\n")
        
        for llm in sorted(df_variations['LLM'].unique()):
            llm_data = df_variations[df_variations['LLM'] == llm]
            f.write(f"{llm}:\n")
            f.write(f"  Formulations: {len(llm_data)} (5 strategies × 3 variations)\n")
            f.write(f"  Recall:       Mean={llm_data['Recall'].mean()*100:.1f}%, SD={llm_data['Recall'].std()*100:.1f}%, Range=[{llm_data['Recall'].min()*100:.1f}%, {llm_data['Recall'].max()*100:.1f}%]\n")
            f.write(f"  Precision:    Mean={llm_data['Precision'].mean()*100:.1f}%, SD={llm_data['Precision'].std()*100:.1f}%, Range=[{llm_data['Precision'].min()*100:.1f}%, {llm_data['Precision'].max()*100:.1f}%]\n")
            f.write(f"  F1-Score:     Mean={llm_data['F1_Score'].mean():.3f}, SD={llm_data['F1_Score'].std():.3f}, Range=[{llm_data['F1_Score'].min():.3f}, {llm_data['F1_Score'].max():.3f}]\n")
            f.write(f"  Consistency:  Mean={llm_data['Consistency'].mean():.3f}, SD={llm_data['Consistency'].std():.3f}, Range=[{llm_data['Consistency'].min():.3f}, {llm_data['Consistency'].max():.3f}]\n")
            
            best_var = llm_data.loc[llm_data['F1_Score'].idxmax()]
            f.write(f"  Best config:  {best_var['Strategy']} (Var {best_var['Variation']}) - F1={best_var['F1_Score']:.3f} (Recall={best_var['Recall']*100:.1f}%, Precision={best_var['Precision']*100:.1f}%)\n\n")
        
        # ===== SECTION 4: STRATEGY ANALYSIS =====
        f.write("\n" + "="*80 + "\n")
        f.write("4. STRATEGY PERFORMANCE ANALYSIS\n")
        f.write("="*80 + "\n\n")
        
        for strategy in sorted(df_variations['Strategy'].unique()):
            strat_data = df_variations[df_variations['Strategy'] == strategy]
            f.write(f"{strategy}:\n")
            f.write(f"  Formulations: {len(strat_data)} (5 platforms × 3 variations)\n")
            f.write(f"  Recall:       Mean={strat_data['Recall'].mean()*100:.1f}%, SD={strat_data['Recall'].std()*100:.1f}%\n")
            f.write(f"  Precision:    Mean={strat_data['Precision'].mean()*100:.1f}%, SD={strat_data['Precision'].std()*100:.1f}%\n")
            f.write(f"  F1-Score:     Mean={strat_data['F1_Score'].mean():.3f}, SD={strat_data['F1_Score'].std():.3f}\n")
            f.write(f"  Consistency:  Mean={strat_data['Consistency'].mean():.3f}, SD={strat_data['Consistency'].std():.3f}\n\n")
        
        # ===== SECTION 5: RECALL-PRECISION TRADE-OFF ANALYSIS =====
        f.write("\n" + "="*80 + "\n")
        f.write("5. RECALL-PRECISION TRADE-OFF ANALYSIS\n")
        f.write("="*80 + "\n\n")
        
        f.write("Platform Rankings:\n\n")
        
        # Rank by different metrics
        recall_ranking = df_variations.groupby('LLM')['Recall'].mean().sort_values(ascending=False)
        precision_ranking = df_variations.groupby('LLM')['Precision'].mean().sort_values(ascending=False)
        f1_ranking = df_variations.groupby('LLM')['F1_Score'].mean().sort_values(ascending=False)
        
        f.write("By Recall (higher is better for coverage):\n")
        for rank, (llm, score) in enumerate(recall_ranking.items(), 1):
            f.write(f"  {rank}. {llm:<15} {score*100:>6.1f}%\n")
        
        f.write("\nBy Precision (higher is better for screening burden):\n")
        for rank, (llm, score) in enumerate(precision_ranking.items(), 1):
            f.write(f"  {rank}. {llm:<15} {score*100:>6.1f}%\n")
        
        f.write("\nBy F1-Score (balanced performance):\n")
        for rank, (llm, score) in enumerate(f1_ranking.items(), 1):
            f.write(f"  {rank}. {llm:<15} {score:>6.3f}\n")
        
        f.write("\n")
        
        # ===== FOOTER =====
        f.write("\n" + "="*80 + "\n")
        f.write("FRAMEWORK ALIGNMENT SUMMARY\n")
        f.write("="*80 + "\n\n")
        f.write("This analysis fully implements the statistical framework:\n\n")
        f.write("C0.1 Experimental Structure:\n")
        f.write("  ✓ 5 platforms × 15 formulations × 3 runs = 225 query executions\n")
        f.write("  ✓ 83-study gold standard\n\n")
        f.write("C0.2 Metric Operationalization:\n")
        f.write("  ✓ C0.2.1: Recall pooled across 3 runs (at-least-once coverage)\n")
        f.write("  ✓ C0.2.2: Precision pooled across 3 runs (screening burden proxy)\n")
        f.write("  ✓ C0.2.3: Consistency via mean pairwise Jaccard on matched sets\n")
        f.write("  ✓ F1-Score: Harmonic mean of recall and precision\n\n")
        f.write("C0.3 Statistical Tests:\n")
        f.write("  ✓ C0.3.1: Cochran's Q (83×5 binary matrix) + Bonferroni McNemar (α=0.005)\n")
        f.write("  ✓ C0.3.2: Friedman (15×5 continuous matrix) + FDR Wilcoxon (q<0.05)\n")
        f.write("  ✓ Two-tailed tests throughout\n")
        f.write("  ✓ Conservative correction (Bonferroni) for binary recall outcome\n")
        f.write("  ✓ Exploratory correction (FDR) for continuous metrics\n\n")
        f.write("Reporting Format:\n")
        f.write("  ✓ Recall and Precision: reported as percentages\n")
        f.write("  ✓ F1-scores: reported as decimals (0-1)\n")
        f.write("  ✓ Consistency: reported as Jaccard coefficients (0-1)\n\n")
        f.write("="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")
    
    print(f"✓ Report saved to: {output_filename}")
    return output_filename

# ===== EXECUTION =====
print("\n" + "="*70)
print("FRAMEWORK-ALIGNED STATISTICAL ANALYSIS WITH F1-SCORE")
print("="*70)

if 'base_results' in globals() and base_results:
    print("✓ Running framework-aligned analysis with F1-score...")
    
    df_variations, statistical_results, recall_scores = visualize_all_variations_with_stats(
        base_results['analyzer'],
        base_results['recall_scores'],
        base_results['variation_metadata']
    )
    
    print("\n✓ Generating comprehensive framework-aligned report with F1 analysis...")
    report_file = generate_comprehensive_report_with_stats(
        df_variations,
        base_results['analyzer'],
        recall_scores,
        statistical_results
    )
    
    print("\n" + "="*70)
    print("FRAMEWORK ALIGNMENT VERIFICATION")
    print("="*70)
    print(f"\u2713 C0.1: Experimental structure (5 platforms x 15 formulations x 3 runs)")
    print(f"\u2713 C0.2.1: Recall pooled via union operator across runs")
    print(f"\u2713 C0.2.2: Precision uses pooled deduplicated retrieved set")
    print(f"\u2713 C0.2.3: Consistency uses mean of 3 pairwise Jaccard scores")
    print(f"\u2713 F1-Score: Harmonic mean of recall and precision")
    print(f"\u2713 C0.3.1: Cochran's Q on {statistical_results['recall']['n_studies']}x5 matrix (Bonferroni) -- pooled recall")
    print(f"\u2713 C0.3.2: Friedman on 15x5 matrices (FDR) -- formulation recall (PRIMARY), precision, consistency")
    
    print("\n" + "="*70)
    print("STATISTICAL TESTS SUMMARY")
    print("="*70)
    print(f"Pooled Recall (Cochran's Q, Bonferroni): {format_p_value(statistical_results['recall']['p_value'], 0.05)}  "
          f"W={statistical_results['recall']['kendalls_w']:.3f}")
    print(f"Formulation Recall (Friedman, FDR) [PRIMARY OUTCOME]: "
          f"{format_p_value(statistical_results['formulation_recall']['p_value'], 0.05)}  "
          f"W={statistical_results['formulation_recall']['kendalls_w']:.3f}"
          if statistical_results['formulation_recall']['kendalls_w'] is not None else "Formulation Recall: error")
    print(f"Precision (Friedman, FDR): {format_p_value(statistical_results['precision']['p_value'], 0.05)}  "
          f"W={statistical_results['precision']['kendalls_w']:.3f}")
    print(f"Consistency (Friedman, FDR): {format_p_value(statistical_results['consistency']['p_value'], 0.05)}  "
          f"W={statistical_results['consistency']['kendalls_w']:.3f}")
    print(f"F1-Score (Friedman, FDR): {format_p_value(statistical_results['f1_score']['p_value'], 0.05)}  "
          f"W={statistical_results['f1_score']['kendalls_w']:.3f}")
    
    print("\n" + "="*70)

# ── GAP FIX #3: FORMAL QUERY-STRATEGY EFFECT TEST ────────────────────────
# Results paragraph 54 describes strategy-level recall differences
# ("conversational platforms showed higher recall under few-shot prompting;
# chain-of-thought reduced recall") without any formal statistical test.
# R2.5 response letter claims "corresponding statistical tests already computed
# in our pipeline" — this adds the missing computation.
#
# Design:
#   Outcome:   Formulation-level recall (pooled across 3 runs per formulation)
#   Treatment: Prompting strategy (S1–S5), 5 levels
#   Block:     Platform (5 platforms) — Friedman treats platforms as blocks
#   Unit:      Each (strategy × variation) formulation; 3 variations per strategy
#              → 5 platforms × 5 strategies × 3 variations = 75 observations
#
# Tests:
#   a) Friedman: do strategies differ in recall across platforms? (k=5 strats, n=5 platforms)
#      Uses mean recall per (platform, strategy) as the cell value.
#   b) Kruskal-Wallis: strategy effect pooled across all platforms (n=75 formulations × 5)
#   c) Post-hoc Wilcoxon pairwise between strategies (FDR-adjusted)
#   d) Effect size: Kendall's W (Friedman), η² (Kruskal-Wallis)
# ─────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  GAP FIX #3: FORMAL QUERY-STRATEGY EFFECT ON RECALL")
print("  [Results §54 — strategy claims now backed by formal tests]")
print("=" * 70)

try:
    # ── Build strategy-level recall matrix from df_variations ─────────────
    # df_variations has columns: LLM, Strategy, Variation, Recall, Precision, ...
    strategy_col = 'Strategy'
    recall_col   = 'Recall'
    platform_col = 'LLM'

    if df_variations is None or df_variations.empty:
        raise ValueError("df_variations not available — run Cell 3 first")

    # Mean recall per (platform, strategy) — averages across variations and runs
    strat_matrix_df = (
        df_variations
        .groupby([platform_col, strategy_col])[recall_col]
        .mean()
        .reset_index()
    )
    strat_matrix = strat_matrix_df.pivot(
        index=platform_col, columns=strategy_col, values=recall_col
    )
    n_strats   = strat_matrix.shape[1]
    n_platform = strat_matrix.shape[0]
    strat_ids  = sorted(strat_matrix.columns.tolist())

    print(f"\n  Strategy × Platform mean recall matrix ({n_platform} platforms × {n_strats} strategies):")
    print(f"  {'Strategy':<12}" + "".join(f"  S{s:<5}" for s in strat_ids))
    print("  " + "-" * (12 + 8 * n_strats))
    for plat in strat_matrix.index:
        row = "".join(f"  {strat_matrix.loc[plat, s]*100:5.1f}%" for s in strat_ids)
        print(f"  {plat:<12}{row}")

    # ── a) Friedman: strategy effect across platforms ─────────────────────
    M = strat_matrix.values  # shape (n_platforms, n_strats)
    fstat, fpval = stats.friedmanchisquare(*[M[:, j] for j in range(n_strats)])
    f_W = kendalls_w(fstat, n_platform, n_strats)
    print(f"\n  a) Friedman test (strategy as treatment, platform as block):")
    print(f"     chi²={fstat:.3f}  df={n_strats-1}  p={fpval:.4f}  Kendall's W={f_W:.3f}")

    # ── b) Kruskal-Wallis: strategy effect pooled ─────────────────────────
    strat_groups = [
        df_variations[df_variations[strategy_col] == s][recall_col].values
        for s in strat_ids
    ]
    kstat, kpval = stats.kruskal(*strat_groups)
    # η² = (H - k + 1) / (N - k)
    N_total = sum(len(g) for g in strat_groups)
    eta2 = (kstat - n_strats + 1) / (N_total - n_strats) if N_total > n_strats else np.nan
    print(f"\n  b) Kruskal-Wallis (strategy effect, pooled across platforms, N={N_total}):")
    print(f"     H={kstat:.3f}  df={n_strats-1}  p={kpval:.4f}  η²={eta2:.3f}")

    print("\n  Design note: variations within a strategy embed concurrent modifications")
    print("  (Appendix B) and are not isolable; strategy effects are SECONDARY/exploratory.")

    _med = (df_variations.groupby([platform_col, strategy_col])[recall_col]
            .median().reset_index())
    _med['rank'] = _med.groupby(platform_col)[recall_col].rank(ascending=False)
    _mean_rank = _med.groupby(strategy_col)['rank'].mean().sort_values()
    print("\n  c) Platform-blocked rank-ordering of strategies (1 = best within platform):")
    for _s, _r in _mean_rank.items():
        _lbl = strategy_label_map.get(_s, str(_s)) if 'strategy_label_map' in dir() else str(_s)
        print(f"     {str(_lbl):24s} mean rank = {_r:.2f}")
    _med.to_csv('query_strategy_platform_blocked_ranks.csv', index=False)
    print("     ✓ saved query_strategy_platform_blocked_ranks.csv")

    # ── d) Post-hoc Wilcoxon pairwise between strategies (FDR) ───────────
    strategy_posthoc = []
    pairs = [(strat_ids[i], strat_ids[j])
             for i in range(n_strats) for j in range(i+1, n_strats)]
    raw_pvals, raw_stats, raw_r = [], [], []
    for s1, s2 in pairs:
        g1 = df_variations[df_variations[strategy_col] == s1][recall_col].values
        g2 = df_variations[df_variations[strategy_col] == s2][recall_col].values
        n_min = min(len(g1), len(g2))
        g1, g2 = g1[:n_min], g2[:n_min]
        try:
            wstat, wpval = stats.wilcoxon(g1, g2, alternative='two-sided')
        except Exception:
            wstat, wpval = np.nan, 1.0
        r = rank_biserial_wilcoxon(g1, g2)
        raw_pvals.append(wpval)
        raw_stats.append(wstat)
        raw_r.append(r)

    if raw_pvals:
        reject, pvals_corr, _, _ = multipletests(raw_pvals, alpha=0.05, method='fdr_bh')
        print(f"\n  c) FDR-adjusted Wilcoxon pairwise (strategy comparisons, q<0.05):")
        sig_count = 0
        for (s1, s2), wstat, praw, pcorr, rej, r in zip(
                pairs, raw_stats, raw_pvals, pvals_corr, reject, raw_r):
            tag = "*" if rej else " "
            strategy_posthoc.append({
                'strategy_1': s1, 'strategy_2': s2,
                'W_stat': wstat, 'p_raw': praw,
                'p_corrected': pcorr, 'significant': rej,
                'rank_biserial_r': r
            })
            print(f"     {tag} S{s1} vs S{s2}:  W={wstat:.0f}  "
                  f"r={r:+.2f}  q={pcorr:.4f}")
            if rej:
                sig_count += 1
        print(f"     Significant pairs: {sig_count}/{len(pairs)}")

    # ── Store results ──────────────────────────────────────────────────────
    statistical_results['strategy_effect'] = {
        'test_friedman': {
            'statistic': fstat, 'p_value': fpval,
            'df': n_strats - 1, 'kendalls_w': f_W,
            'n_platforms': n_platform, 'n_strategies': n_strats,
            'description': 'Strategy as treatment, platform as block; outcome = mean formulation recall',
        },
        'test_kruskal': {
            'statistic': kstat, 'p_value': kpval,
            'df': n_strats - 1, 'eta_squared': eta2,
            'N': N_total,
            'description': 'Strategy effect pooled across all platforms and variations',
        },
        'posthoc_wilcoxon_fdr': strategy_posthoc,
        'strategy_mean_recall': strat_matrix_df.to_dict('records'),
    }
    print(f"\n  Strategy effect stored in statistical_results['strategy_effect']")

    try:
        with open('query_strategy_recall_report.txt', 'w') as _qf:
            _qf.write("QUERY-STRATEGY EFFECT ON RECALL — SECONDARY / EXPLORATORY\n")
            _qf.write("=" * 64 + "\n\n")
            _qf.write("Design: outcome = formulation-level recall; treatment = prompting\n")
            _qf.write("strategy (5 levels); block = platform (5). Per Appendix B, the three\n")
            _qf.write("variations within a strategy embed concurrent modifications and are\n")
            _qf.write("NOT isolable single-factor manipulations; this analysis is secondary\n")
            _qf.write("and exploratory, not a single-factor causal test.\n\n")
            _qf.write("a) Friedman (strategy as treatment, platform as block):\n")
            _qf.write(f"     chi2 = {fstat:.3f}  df = {n_strats-1}  p = {fpval:.4f}  Kendall's W = {f_W:.3f}\n\n")
            _qf.write("b) Kruskal-Wallis (strategy effect pooled across platforms, "
                      f"N = {N_total}):\n")
            _qf.write(f"     H = {kstat:.3f}  df = {n_strats-1}  p = {kpval:.4f}  eta^2 = {eta2:.3f}\n\n")
            _qf.write("c) Platform-blocked rank-ordering of strategies (1 = best within platform):\n")
            for _s, _r in mean_rank.items():
                _lbl = strategy_label_map.get(_s, str(_s)) if 'strategy_label_map' in dir() else str(_s)
                _qf.write(f"     {str(_lbl):24s} mean rank = {_r:.2f}\n")
            _qf.write("\nd) FDR-adjusted Wilcoxon pairwise (strategy comparisons):\n")
            for _ph in strategy_posthoc:
                _s1 = _ph.get('strategy_1', '?'); _s2 = _ph.get('strategy_2', '?')
                _q = _ph.get('p_corrected', float('nan'))
                _sig = '*' if _ph.get('significant') else ' '
                _qf.write(f"   {_sig} {_s1} vs {_s2}: W={_ph.get('W_stat','')}  "
                          f"p_raw={_ph.get('p_raw', float('nan')):.4f}  q={_q:.4f}\n")
            _n_sig = sum(1 for _ph in strategy_posthoc if _ph.get('significant'))
            _qf.write(f"\nSignificant pairs after FDR (q<0.05): {_n_sig}/{len(strategy_posthoc)}\n")
        print("  ✓ saved query_strategy_recall_report.txt")
    except Exception as _e:
        print(f"  [SKIP] query_strategy_recall_report.txt not written: {_e}")

except Exception as e:
    print(f"\n  [SKIP] Strategy test error: {e}")
    print("  Requires df_variations from Cell 3 and statistical_results from Cell 14 body.")

# ── Final status (prints regardless of the optional strategy-test outcome) ──
print("\nANALYSIS COMPLETE - FULLY FRAMEWORK ALIGNED WITH F1!")
print("="*70)
print("✓ Visualization: Overall_Performance_Framework_Aligned.png")
print("✓ Data: df_variations_framework_aligned.csv (now includes F1_Score column)")
print("✓ F1-Score added as harmonic mean of recall and precision")
print("✓ F1-Score statistical analysis included in report")

## Cell 16 — KNOWLEDGE GAPS

*Figure 7 — routes throughed extractor (fold-in consistent)*


In [ ]:

def _panel_friedman_annotation(ax, df_form, cols, label):
    """omnibus Friedman test (platforms x formulations)
    for this panel's coverage columns; FDR applied across the 3 panels by the caller.
    Draws a compact annotation in the panel corner; does not alter the existing plot.
    df_form has no 'Formulation_ID' column — derives it from
    'Strategy' + 'Variation', which df_form actually contains."""
    import numpy as np, scipy.stats as stats
    cols = [c for c in cols if c in df_form.columns]
    if not cols or 'Strategy' not in df_form.columns or 'Variation' not in df_form.columns:
        return None
    sub = df_form[['LLM', 'Strategy', 'Variation'] + cols].copy()
    sub['_formulation_id'] = sub['Strategy'].astype(str) + '_' + sub['Variation'].astype(str)
    sub['_val'] = sub[cols].mean(axis=1, skipna=True)
    piv = sub.pivot_table(index='_formulation_id', columns='LLM', values='_val', aggfunc='mean')
    piv = piv.dropna()
    if piv.shape[0] < 3 or piv.shape[1] < 2:
        return None
    try:
        chi2, p = stats.friedmanchisquare(*[piv[c].values for c in piv.columns])
    except ValueError:
        return None
    return {'panel': label, 'chi2': float(chi2), 'p': float(p), 'n_formulations': int(piv.shape[0])}

def _annotate_panel(ax, result, p_fdr=None, corner='right'):
    """render the Friedman result as a corner annotation.
    corner='right' -> upper-right (panels B, C); corner='left' -> upper-left, placed
    below the panel legend (panel A). Larger, boxed text for readability."""
    if result is None:
        txt = "Friedman: insufficient data"
    else:
        p_show = p_fdr if p_fdr is not None else result['p']
        p_str = "p < 0.001" if p_show < 0.001 else f"p = {p_show:.3f}"
        sig = "*" if p_show < 0.05 else "ns"
        txt = f"Platform diff. (Friedman): \u03c7\u00b2={result['chi2']:.1f}, {p_str} ({sig})"
    _box = dict(boxstyle='round,pad=0.35', facecolor='white',
                edgecolor='#CCCCCC', alpha=0.9)
    if corner == 'left':
        ax.text(0.012, 0.65, txt, transform=ax.transAxes, ha='left', va='top',
                fontsize=20, style='italic', color='#333333', bbox=_box, zorder=20)
    else:
        ax.text(0.988, 0.97, txt, transform=ax.transAxes, ha='right', va='top',
                fontsize=20, style='italic', color='#333333', bbox=_box, zorder=20)

def academic_llm_knowledge_gap_analysis(base_results):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from collections import defaultdict
    from datetime import datetime
    analyzer = base_results['analyzer']
    # ====================================================================
    # ====================================================================
    print("\n" + "="*80)
    print("DIAGNOSTIC: RAW SUBSTANCE DATA")
    print("="*80)
    substance_raw_counts = defaultdict(int)
    for idx, row in analyzer.gold_standard_df.iterrows():
        if 'Substances covered' in row and pd.notna(row['Substances covered']):
            substances_str  = str(row['Substances covered'])
            substances_list = [s.strip() for s in substances_str.split(';')]
            combo_key       = ' + '.join(sorted(substances_list))
            substance_raw_counts[combo_key] += 1
    print("\n--- ALL SUBSTANCE COMBINATIONS ---")
    for sub, count in sorted(substance_raw_counts.items(),
                              key=lambda x: x[1], reverse=True):
        print(f"  {sub}: {count} studies")
    # ====================================================================
    # DATA PREPARATION
    # ====================================================================
    llm_names = sorted(set([info['llm'] for info in analyzer.recalled_files.values()]))
    llm_retrieved_titles = {llm: set() for llm in llm_names}
    for filename, info in analyzer.recalled_files.items():
        llm = info['llm']
        df  = analyzer.load_csv_file_content(info['path'])
        if not df.empty:
            # Route through the SAME matched-set extractor every other analysis uses,
            # so the accepted near-match fold-in (when ADOPT_NEARMATCHES=True) is
            # reflected here too. This keeps Figure 7's coverage panels consistent
            # with recall/precision/Friedman rather than re-deriving titles directly.
            titles, _ = analyzer.extract_studies_from_recalled_files(df)
            llm_retrieved_titles[llm].update(titles)
    llm_retrieved_studies = {}
    llm_missed_studies    = {}
    for llm in llm_names:
        retrieved, missed = [], []
        for idx, row in analyzer.gold_standard_df.iterrows():
            title = row.get('Title')
            if pd.notna(title):
                if str(title).lower().strip() in llm_retrieved_titles[llm]:
                    retrieved.append(row)
                else:
                    missed.append(row)
        llm_retrieved_studies[llm] = retrieved
        llm_missed_studies[llm]    = missed
    # ====================================================================
    # CATEGORISATION HELPERS
    # ====================================================================
    def categorize_main_substances(substance_list):
        if not substance_list:
            return 'Other'
        substances       = [s.strip() for s in substance_list]
        substances_lower = [s.lower() for s in substances]
        has_tobacco  = any('tobacco'  in s or 'nicotine' in s or
                           'smoking'  in s or 'cigarette' in s for s in substances_lower)
        has_alcohol  = any('alcohol'  in s for s in substances_lower)
        has_opioids  = any('opioid'   in s for s in substances_lower)
        has_cannabis = any('cannabis' in s or 'marijuana' in s or
                           'cannabinoid' in s for s in substances_lower)
        n_main = sum([has_tobacco, has_alcohol, has_opioids, has_cannabis])
        if len(substances) == 1:
            s = substances[0].lower()
            if   'cannabis' in s or 'marijuana'  in s: return 'Cannabinoids'
            elif 'opioid'   in s:                       return 'Opioids'
            elif 'tobacco'  in s or 'nicotine'   in s or 'smoking' in s: return 'Tobacco'
            elif 'alcohol'  in s:                       return 'Alcohol'
            else:                                       return 'Other'
        else:
            if   n_main >= 2:   return 'Polysubstance'
            elif has_cannabis:  return 'Cannabinoids'
            elif has_opioids:   return 'Opioids'
            elif has_tobacco:   return 'Tobacco'
            elif has_alcohol:   return 'Alcohol'
            else:               return 'Other'
    def categorize_nlp_methodology(method_str):
        methods  = [m.strip() for m in str(method_str).split(';')]
        category, _= analyzer.categorize_nlp_methodologies(methods)
        return category
    # ====================================================================
    # GOLD STANDARD TOTALS
    # ====================================================================
    def analyze_studies_by_categories(studies_list):
        by_year, by_method, by_substance = defaultdict(int), defaultdict(int), defaultdict(int)
        for study in studies_list:
            if 'Year' in study and pd.notna(study['Year']):
                by_year[int(study['Year'])] += 1
            if 'NLP Methodology' in study and pd.notna(study['NLP Methodology']):
                cat, _= analyzer.categorize_nlp_methodologies(
                    [m.strip() for m in str(study['NLP Methodology']).split(';')])
                by_method[cat] += 1
            if 'Substances covered' in study and pd.notna(study['Substances covered']):
                cat = categorize_main_substances(
                    [s.strip() for s in str(study['Substances covered']).split(';')])
                by_substance[cat] += 1
        return dict(by_year), dict(by_method), dict(by_substance)
    def get_total_counts():
        ty, tm, ts = defaultdict(int), defaultdict(int), defaultdict(int)
        for idx, row in analyzer.gold_standard_df.iterrows():
            if 'Year' in row and pd.notna(row['Year']):
                ty[int(row['Year'])] += 1
            if 'NLP Methodology' in row and pd.notna(row['NLP Methodology']):
                cat, _= analyzer.categorize_nlp_methodologies(
                    [m.strip() for m in str(row['NLP Methodology']).split(';')])
                tm[cat] += 1
            if 'Substances covered' in row and pd.notna(row['Substances covered']):
                cat = categorize_main_substances(
                    [s.strip() for s in str(row['Substances covered']).split(';')])
                ts[cat] += 1
        return dict(ty), dict(tm), dict(ts)
    total_year, total_method, total_substance = get_total_counts()
    print("\n=== Methodology Counts ===")
    methodology_counts = defaultdict(list)
    for idx, row in analyzer.gold_standard_df.iterrows():
        if 'NLP Methodology' in row and pd.notna(row['NLP Methodology']):
            cat, _= analyzer.categorize_nlp_methodologies(
                [m.strip() for m in str(row['NLP Methodology']).split(';')])
            methodology_counts[cat].append(row)
    for m, s in methodology_counts.items():
        print(f"  {m}: {len(s)} studies")
    # ====================================================================
    # FORMULATION-LEVEL RECALL DATA
    # ====================================================================
    print("\n📊 Building formulation-level recall data...")
    gold_df   = analyzer.gold_standard_df
    study_map = analyzer.create_study_identifier_mapping()
    SUB_CATS      = ['Tobacco', 'Opioids', 'Alcohol', 'Cannabinoids', 'Polysubstance', 'Other']
    METH_CATS_RAW = list(total_method.keys())
    METH_DISPLAY  = {
        'Rule-based':      next((k for k in METH_CATS_RAW if 'Rule'        in k), 'Rule-based'),
        'Conventional ML': next((k for k in METH_CATS_RAW if 'Conventional' in k
                                  or 'Machine' in k), 'Conventional ML'),
        'Deep Learning':   next((k for k in METH_CATS_RAW if 'Deep'        in k), 'Deep Learning'),
        'LLMs':            next((k for k in METH_CATS_RAW if 'Language'    in k
                                  or 'LLM' in k), 'LLMs'),
        'Other':           next((k for k in METH_CATS_RAW if 'Other'       in k), 'Other'),
    }
    METH_DISPLAY  = {d: r for d, r in METH_DISPLAY.items() if r in total_method}
    METH_LABELS   = list(METH_DISPLAY.keys())
    STRAT_NAMES   = {'1':'Zero-Shot','2':'Few-Shot','3':'Persona','4':'CoT','5':'PICO'}
    llm_order     = ['Ai2 Finder','Consensus','ChatGPT','Claude','Gemini']
    study_substance_cat = {}
    study_method_cat    = {}
    study_year_cat      = {}
    all_gold_ids = set()
    for _, row in gold_df.iterrows():
        sid = str(row.get('Study ID', row.get('Title', '')))
        all_gold_ids.add(sid)
        study_substance_cat[sid] = (
            categorize_main_substances(
                [s.strip() for s in str(row['Substances covered']).split(';')])
            if 'Substances covered' in row and pd.notna(row['Substances covered'])
            else 'Other'
        )
        study_method_cat[sid] = (
            categorize_nlp_methodology(str(row['NLP Methodology']))
            if 'NLP Methodology' in row and pd.notna(row['NLP Methodology'])
            else 'Other'
        )
        try:
            _yr = int(float(row.get('Year', float('nan'))))
            study_year_cat[sid] = 'Pre_2016' if _yr < 2016 else 'Y2016_plus'
        except (TypeError, ValueError):
            study_year_cat[sid] = None
    sub_counts_gs  = {c: sum(1 for v in study_substance_cat.values() if v == c)
                      for c in SUB_CATS}
    meth_counts_gs = {r: total_method.get(r, 0) for r in METH_DISPLAY.values()}
    platform_ever = {llm: set() for llm in llm_names}
    formulation_records = []
    for llm in llm_order:
        if llm not in llm_names:
            continue
        for strat in ['1','2','3','4','5']:
            for var in ['1','2','3']:
                pooled_norm = set()
                for fname, info in analyzer.recalled_files.items():
                    if (info['llm'] == llm and
                            info.get('strategy') == strat and
                            info.get('variation') == var):
                        df = analyzer.load_csv_file_content(info['path'])
                        if df.empty:
                            continue
                        for col in ['DOI','Study name','Title','LLM Title']:
                            if col in df.columns:
                                for val in df[col].dropna().astype(str):
                                    pooled_norm.add(
                                        analyzer.normalize_identifier_consistent(val))
                                break
                matched_ids = set()
                for sid, sinfo in study_map.items():
                    if any(i in pooled_norm for i in sinfo['identifiers']):
                        matched_ids.add(sid)
                platform_ever[llm] |= matched_ids
                rec = {'LLM': llm, 'Strategy': STRAT_NAMES[strat], 'Variation': var}
                for cat in SUB_CATS:
                    d = sub_counts_gs[cat]
                    n = sum(1 for s in matched_ids if study_substance_cat.get(s) == cat)
                    rec[f'sub_{cat}'] = n / d if d > 0 else np.nan
                for ycat in ('Pre_2016', 'Y2016_plus'):
                    d_y = sum(1 for v in study_year_cat.values() if v == ycat)
                    n_y = sum(1 for s in matched_ids if study_year_cat.get(s) == ycat)
                    rec[f'year_{ycat}'] = n_y / d_y if d_y > 0 else np.nan
                for disp, raw in METH_DISPLAY.items():
                    d = meth_counts_gs[raw]
                    n = sum(1 for s in matched_ids if study_method_cat.get(s) == raw)
                    rec[f'meth_{disp}'] = n / d if d > 0 else np.nan
                formulation_records.append(rec)
    df_form = pd.DataFrame(formulation_records)
    print(f"  ✓ {len(df_form)} formulation records "
          f"({df_form['LLM'].nunique()} LLMs × 5 strategies × 3 variations)")
    # ====================================================================
    # COLOUR PALETTE
    # ====================================================================
    colors = {
        'Gold Standard': '#2C3E50',
        'Ai2 Finder':    '#27AE60',
        'Consensus':     '#E67E22',
        'ChatGPT':       '#2980B9',
        'Claude':        '#8E44AD',
        'Gemini':        '#C0392B',
    }
    # ====================================================================
    # FIGURE  —  3 × 1  (A, B, C each full-width row)
    # ====================================================================
    plt.style.use('default')
    fig = plt.figure(figsize=(24, 28))
    gs_layout = fig.add_gridspec(
        3, 1,
        height_ratios=[1, 1, 1],
        hspace=0.4,
    )
    # ====================================================================
    # SUBPLOT A  — Temporal Coverage
    # ====================================================================
    print("  Creating Subplot A …")
    ax_temporal = fig.add_subplot(gs_layout[0])
    ax_temporal.set_facecolor('#FFFFFF')
    all_years        = sorted(total_year.keys())
    gold_year_counts = [total_year.get(y, 0) for y in all_years]
    max_y            = max(gold_year_counts) if gold_year_counts else 15
    for i, yr in enumerate(all_years):
        if i % 2 == 0:
            ax_temporal.axvspan(yr - 0.5, yr + 0.5,
                                facecolor='#EEF0F3', alpha=0.55, zorder=0)
    ax_temporal.grid(True, axis='y', alpha=0.4, linestyle='--',
                     linewidth=0.8, color='gray', zorder=1)
    ax_temporal.set_axisbelow(True)
    ax_temporal.plot(all_years, gold_year_counts,
                     linestyle='--', color=colors['Gold Standard'],
                     linewidth=4.0, marker='o', markersize=10,
                     label='Gold Standard', alpha=0.95, zorder=10)

    _sid_year = {}
    for _, _yr_row in gold_df.iterrows():
        _sid = str(_yr_row.get('Study ID', _yr_row.get('Title', '')))
        if 'Year' in _yr_row and pd.notna(_yr_row['Year']):
            try:
                _sid_year[_sid] = int(_yr_row['Year'])
            except (ValueError, TypeError):
                pass

    def _platform_year_counts(_llm):
        _c = defaultdict(int)
        for _sid in platform_ever.get(_llm, set()):
            _y = _sid_year.get(_sid)
            if _y is not None:
                _c[_y] += 1
        return _c

    for llm in llm_order:
        if llm in llm_names:
            yr_llm = _platform_year_counts(llm)
            ax_temporal.plot(all_years, [yr_llm.get(y, 0) for y in all_years],
                             linestyle='-', color=colors[llm],
                             linewidth=3.2, marker='o', markersize=8,
                             label=llm, alpha=0.9,
                             zorder=9 if llm == 'Ai2 Finder' else 8)  # Ai2 on top
    ax_temporal.set_ylim(0, max_y + 2)
    ax_temporal.set_xlim(min(all_years) - 0.5, max(all_years) + 0.5)
    ax_temporal.set_yticks(range(0, max_y + 3, 2))
    _res_a = _panel_friedman_annotation(
        ax_temporal, df_form,
        [c for c in df_form.columns if c.startswith('year_')], 'A')
    ax_temporal.set_title(
        'A. Temporal Coverage: Studies Retrieved by Each LLM Over Time',
        fontsize=28, fontweight='bold', pad=18, color='#1A1A2E')
    ax_temporal.set_xlabel('Publication Year',           fontsize=28, fontweight='bold')
    ax_temporal.set_ylabel('Number of Studies Retrieved',fontsize=28, fontweight='bold')
    ax_temporal.set_xticks(all_years)
    ax_temporal.tick_params(axis='x', labelsize=22)
    ax_temporal.tick_params(axis='y',               labelsize=22)
    ax_temporal.legend(loc='upper left', fontsize=28, frameon=True, ncols=3,
                       edgecolor='#CCCCCC', fancybox=False, shadow=False,
                       framealpha=0.9)
    ax_temporal.spines['top'].set_visible(False)
    ax_temporal.spines['right'].set_visible(False)
    for sp in ax_temporal.spines.values():
        sp.set_color('#BBBBBB'); sp.set_linewidth(1.2)
    # ====================================================================
    # BOXPLOT HELPER
    # ====================================================================
    def draw_boxplot_panel(ax, cats, col_prefix, title, panel_label,
                           gs_counts, legend=True):
        ax.set_facecolor('#FFFFFF')
        active_llms = [ln for ln in llm_order if ln in llm_names]
        n_llms      = len(active_llms)
        w           = 0.14
        spacing     = w + 0.032
        gap         = 0.26
        for ci in range(len(cats)):
            x_left  = ci * (n_llms * spacing + gap) - spacing
            x_right = ci * (n_llms * spacing + gap) + n_llms * spacing
            if ci % 2 == 0:
                ax.axvspan(x_left, x_right,
                           facecolor='#EEF0F3', alpha=0.55, zorder=0)
        ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.9,
                color='gray', zorder=1)
        ax.set_axisbelow(True)
        xtick_positions, xtick_labels = [], []
        for ci, cat in enumerate(cats):
            group_center = ci * (n_llms * spacing + gap)
            xtick_positions.append(group_center + (n_llms - 1) * spacing / 2)
            xtick_labels.append(cat)
            col = f'{col_prefix}{cat}'
            if col not in df_form.columns:
                continue
            for li, llm in enumerate(active_llms):
                x_pos  = group_center + li * spacing
                values = df_form[df_form['LLM'] == llm][col].dropna().values * 100
                if len(values) == 0:
                    continue
                bp = ax.boxplot(
                    values,
                    positions=[x_pos],
                    widths=w,
                    patch_artist=True,
                    showfliers=True,
                    medianprops  =dict(color='white',      linewidth=3.2),
                    whiskerprops =dict(color=colors[llm],  linewidth=2.2),
                    capprops     =dict(color=colors[llm],  linewidth=2.2),
                    flierprops   =dict(marker='o',
                                       markerfacecolor=colors[llm],
                                       markeredgecolor='white',
                                       markersize=6, alpha=0.6,
                                       linestyle='none'),
                    boxprops     =dict(facecolor=colors[llm], alpha=0.80,
                                       edgecolor=colors[llm],  linewidth=1.8),
                )
        ax.set_xticks(xtick_positions)
        ax.set_xticklabels(xtick_labels, 
                           fontsize=20, fontweight='bold', ha='center')
        ax.set_ylabel('Recall (%)', fontsize=28, fontweight='bold')
        ax.set_title(f'{panel_label}. {title}',
                     fontsize=30, fontweight='bold', pad=18, color='#1A1A2E')
        ax.set_ylim(-16, 100)
        ax.yaxis.set_major_formatter(
            plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
        ax.tick_params(axis='y', labelsize=18)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        for sp in ax.spines.values():
            sp.set_color('#BBBBBB'); sp.set_linewidth(1)
        for ci, cat in enumerate(cats):
            n        = gs_counts.get(cat, 0)
            x_center = ci * (n_llms * spacing + gap) + (n_llms - 1) * spacing / 2
            ax.text(x_center, -9, f'n={n}',
                    ha='center', va='top',
                    fontsize=26, color='#444444', style='italic')
        ax.axhline(100, color='#999999', linewidth=1.2,
                   linestyle=':', alpha=0.7, zorder=2)
        if legend:
            handles = [mpatches.Patch(facecolor=colors[ln],
                                       edgecolor='#888888',
                                       linewidth=0.8, label=ln)
                       for ln in active_llms]
            ax.legend(handles=handles, title='Platform',
                      fontsize=18, title_fontsize=20,
                      loc='upper left', frameon=True,
                      edgecolor='#CCCCCC', fancybox=False,
                      framealpha=0.92,
                      ncol=3)
    # ====================================================================
    # SUBPLOT B  — NLP Methodology
    # ====================================================================
    print("  Creating Subplot B …")
    ax_method = fig.add_subplot(gs_layout[1])
    draw_boxplot_panel(
        ax          = ax_method,
        cats        = METH_LABELS,
        col_prefix  = 'meth_',
        title       = 'NLP Methodology Coverage\n(15 formulations per platform)',
        panel_label = 'B',
        gs_counts   = {d: total_method.get(r, 0) for d, r in METH_DISPLAY.items()},
        legend      = True,
    )
    _res_b = _panel_friedman_annotation(ax_method, df_form,
        [c for c in df_form.columns if c.startswith('meth_')], 'B')
    # ====================================================================
    # SUBPLOT C  — Substance
    # ====================================================================
    print("  Creating Subplot C …")
    ax_substance = fig.add_subplot(gs_layout[2])
    draw_boxplot_panel(
        ax          = ax_substance,
        cats        = SUB_CATS,
        col_prefix  = 'sub_',
        title       = 'Substance Coverage\n(15 formulations per platform)',
        panel_label = 'C',
        gs_counts   = sub_counts_gs,
        legend      = True,
    )
    _res_c = _panel_friedman_annotation(ax_substance, df_form,
        [c for c in df_form.columns if c.startswith('sub_')], 'C')
    from statsmodels.stats.multitest import multipletests
    _panel_results = [r for r in (_res_a, _res_b, _res_c) if r is not None]
    _qmap = {}
    if _panel_results:
        _, _qvals, _, _ = multipletests([r['p'] for r in _panel_results],
                                        alpha=0.05, method='fdr_bh')
        _qmap = {r['panel']: q for r, q in zip(_panel_results, _qvals)}
    for _ax, _res, _corner in ((ax_temporal,  _res_a, 'left'),
                               (ax_method,    _res_b, 'right'),
                               (ax_substance, _res_c, 'right')):
        _annotate_panel(_ax, _res,
                        p_fdr=_qmap.get(_res['panel']) if _res else None,
                        corner=_corner)
    print("  Finalizing …")
    plt.tight_layout()
    for ext in ('png', 'pdf'):
        plt.savefig(f'LLM_Retrieval_Performance_Main_Substances.{ext}',
                    dpi=300, bbox_inches='tight',
                    facecolor=fig.get_facecolor(), edgecolor='none',
                    pad_inches=0.25)
    print("✓ Saved: LLM_Retrieval_Performance_Main_Substances.png / .pdf")
    plt.show()
    # ====================================================================
    # TEXT REPORT
    # ====================================================================
    print("\n📄 Generating report …")
    report_filename = 'report_knowledge_gap_analysis_main_substances.txt'
    preferred_order = ['Cannabinoids','Opioids','Tobacco','Alcohol','Polysubstance','Other']
    all_substances  = [c for c in preferred_order if c in total_substance]
    with open(report_filename, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("  LLM KNOWLEDGE GAP ANALYSIS — COMPLETE DETAILED REPORT\n")
        f.write(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        f.write("EXECUTIVE SUMMARY\n" + "-"*80 + "\n")
        f.write(f"Gold Standard Total: {len(analyzer.gold_standard_df)} studies\n\n")
        for llm in llm_order:
            if llm in llm_names:
                r   = len(llm_retrieved_studies[llm])
                pct = r / len(analyzer.gold_standard_df) * 100
                f.write(f"  {llm:12s}: {r:2d}/{len(analyzer.gold_standard_df)} "
                        f"({pct:5.1f}%) | {len(llm_missed_studies[llm]):2d} missed\n")
        f.write("\nSubstance Category Distribution:\n")
        for sub in all_substances:
            t   = total_substance.get(sub, 0)
            pct = t / len(analyzer.gold_standard_df) * 100
            f.write(f"  {sub:20s}: {t:2d} studies ({pct:4.1f}%)\n")
        f.write("\n" + "="*80 + "\n")
        f.write("FORMULATION-LEVEL MEDIAN RECALL (%)\n" + "="*80 + "\n")
        header = f"{'Category':<25}" + "".join(
            f"  {ln:>12}" for ln in llm_order if ln in llm_names) + "\n"
        f.write(header + "-"*80 + "\n--- SUBSTANCE ---\n")
        for cat in SUB_CATS:
            col = f'sub_{cat}'
            if col not in df_form.columns: continue
            f.write(f"{cat:<25}")
            for llm in [ln for ln in llm_order if ln in llm_names]:
                vals = df_form[df_form['LLM'] == llm][col].dropna().values * 100
                f.write(f"  {np.median(vals) if len(vals) else float('nan'):>11.1f}%")
            f.write("\n")
        f.write("\n--- NLP METHODOLOGY ---\n")
        for disp in METH_LABELS:
            col = f'meth_{disp}'
            if col not in df_form.columns: continue
            f.write(f"{disp:<25}")
            for llm in [ln for ln in llm_order if ln in llm_names]:
                vals = df_form[df_form['LLM'] == llm][col].dropna().values * 100
                f.write(f"  {np.median(vals) if len(vals) else float('nan'):>11.1f}%")
            f.write("\n")
        f.write("\n" + "="*80 + "\nEND OF REPORT\n" + "="*80 + "\n")
    print(f"✓ Report saved: {report_filename}")
    print(f"\n{'='*80}\n  RETRIEVAL PERFORMANCE — MAIN SUBSTANCES\n{'='*80}")
    for llm in llm_order:
        if llm in llm_names:
            r   = len(llm_retrieved_studies[llm])
            pct = r / len(analyzer.gold_standard_df) * 100
            print(f"  {llm:12s}: {r:2d}/{len(analyzer.gold_standard_df)} ({pct:4.1f}%)")
    print(f"\n{'='*80}\n")
    never_retrieved_ids = all_gold_ids - set().union(*platform_ever.values())
    return {
        'total_substance':                total_substance,
        'all_substances':                 all_substances,
        'llm_retrieved_studies':          llm_retrieved_studies,
        'llm_missed_studies':             llm_missed_studies,
        'categorize_substance_for_study': categorize_main_substances,
        'methodology_counts':             methodology_counts,
        'year_data':                      analyzer.gold_standard_df['Year'].value_counts().sort_index(),
        'report_filename':                report_filename,
        'df_formulations':                df_form,
        'platform_ever':       platform_ever,
        'never_retrieved_ids': never_retrieved_ids,
        'study_substance_cat': study_substance_cat,
        'study_method_cat':    study_method_cat,
        'sub_gs_counts':       sub_counts_gs,
        'meth_gs_counts':      meth_counts_gs,
    }

def academic_llm_knowledge_gap_fiveyear_bars(base_results, gap_results, bin_width=5):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from collections import defaultdict

    analyzer = base_results['analyzer']
    gold_df  = analyzer.gold_standard_df
    df_form  = gap_results['df_formulations']
    platform_ever = gap_results['platform_ever']

    llm_order   = ['Ai2 Finder', 'Consensus', 'ChatGPT', 'Claude', 'Gemini']
    llm_names   = sorted(set(info['llm'] for info in analyzer.recalled_files.values()))
    active_llms = [ln for ln in llm_order if ln in llm_names]

    colors = {
        'Gold Standard': '#2C3E50', 'Ai2 Finder': '#27AE60', 'Consensus': '#E67E22',
        'ChatGPT': '#2980B9', 'Claude': '#8E44AD', 'Gemini': '#C0392B',
    }

    SUB_CATS    = [c[len('sub_'):]  for c in df_form.columns if c.startswith('sub_')]
    METH_LABELS = [c[len('meth_'):] for c in df_form.columns if c.startswith('meth_')]

    sub_counts_gs = gap_results.get('sub_gs_counts', {}) or {c: 0 for c in SUB_CATS}

    total_method = {}
    for _, row in gold_df.iterrows():
        if 'NLP Methodology' in row and pd.notna(row['NLP Methodology']):
            cat, _ = analyzer.categorize_nlp_methodologies(
                [m.strip() for m in str(row['NLP Methodology']).split(';')])
            total_method[cat] = total_method.get(cat, 0) + 1
    METH_RAW_LOOKUP = {
        'Rule-based':      next((k for k in total_method if 'Rule'        in k), None),
        'Conventional ML': next((k for k in total_method if 'Conventional' in k or 'Machine' in k), None),
        'Deep Learning':   next((k for k in total_method if 'Deep'        in k), None),
        'LLMs':            next((k for k in total_method if 'Language'    in k or 'LLM' in k), None),
        'Other':           next((k for k in total_method if 'Other'       in k), None),
    }
    meth_counts_gs_disp = {disp: total_method.get(raw, 0)
                           for disp, raw in METH_RAW_LOOKUP.items() if disp in METH_LABELS}

    # ---- panel A data: 5-year-interval counts ----
    sid_year, all_year_vals = {}, []
    for _, row in gold_df.iterrows():
        sid = str(row.get('Study ID', row.get('Title', '')))
        if 'Year' in row and pd.notna(row['Year']):
            try:
                y = int(row['Year']); sid_year[sid] = y; all_year_vals.append(y)
            except (ValueError, TypeError):
                pass
    if not all_year_vals:
        print("  [SKIP five-year bars] no usable Year values."); return None
    ymin, ymax = min(all_year_vals), max(all_year_vals)
    start, bins, b = (ymin // bin_width) * bin_width, [], (ymin // bin_width) * bin_width
    while b <= ymax:
        bins.append((b, b + bin_width - 1)); b += bin_width
    bin_labels = [f"{lo}\u2013{hi}" for lo, hi in bins]
    def _bin_of(y):
        for i, (lo, hi) in enumerate(bins):
            if lo <= y <= hi: return i
        return None
    gs_bin = [0] * len(bins)
    for y in all_year_vals:
        bi = _bin_of(y)
        if bi is not None: gs_bin[bi] += 1
    plat_bin = {ln: [0] * len(bins) for ln in active_llms}
    for ln in active_llms:
        for sid in platform_ever.get(ln, set()):
            y = sid_year.get(sid)
            if y is None: continue
            bi = _bin_of(y)
            if bi is not None: plat_bin[ln][bi] += 1

    plt.style.use('default')
    fig = plt.figure(figsize=(24, 28))
    gs_layout = fig.add_gridspec(3, 1, height_ratios=[1, 1, 1], hspace=0.4)

    # panel A — grouped bars
    print("  Creating Subplot A (five-year bars) …")
    axA = fig.add_subplot(gs_layout[0]); axA.set_facecolor('#FFFFFF')
    series  = ['Gold Standard'] + active_llms
    n_ser   = len(series); group_w = 0.84; bar_w = group_w / n_ser
    x = np.arange(len(bins))
    axA.grid(True, axis='y', alpha=0.4, linestyle='--', linewidth=0.8, color='gray', zorder=1)
    axA.set_axisbelow(True)
    for si, s in enumerate(series):
        vals = gs_bin if s == 'Gold Standard' else plat_bin[s]
        offs = (si - (n_ser - 1) / 2) * bar_w
        axA.bar(x + offs, vals, width=bar_w, color=colors[s],
                edgecolor='white', linewidth=0.7, label=s, alpha=0.9, zorder=3)
    axA.set_xticks(x); axA.set_xticklabels(bin_labels, fontsize=22, fontweight='bold')
    axA.set_title('A. Temporal Coverage: Studies Retrieved per 5-Year Interval',
                  fontsize=28, fontweight='bold', pad=18, color='#1A1A2E')
    axA.set_xlabel('Publication Interval', fontsize=28, fontweight='bold')
    axA.set_ylabel('Number of Studies Retrieved', fontsize=28, fontweight='bold')
    axA.tick_params(axis='y', labelsize=22)
    axA.legend(loc='upper left', fontsize=24, frameon=True, ncols=3,
               edgecolor='#CCCCCC', fancybox=False, framealpha=0.9)
    axA.spines['top'].set_visible(False); axA.spines['right'].set_visible(False)
    for sp in axA.spines.values(): sp.set_color('#BBBBBB'); sp.set_linewidth(1.2)

    def draw_boxplot_panel(ax, cats, col_prefix, title, panel_label, gs_counts):
        ax.set_facecolor('#FFFFFF')
        n_llms, w, spacing, gap = len(active_llms), 0.14, 0.14 + 0.032, 0.26
        for ci in range(len(cats)):
            x_left  = ci * (n_llms * spacing + gap) - spacing
            x_right = ci * (n_llms * spacing + gap) + n_llms * spacing
            if ci % 2 == 0:
                ax.axvspan(x_left, x_right, facecolor='#EEF0F3', alpha=0.55, zorder=0)
        ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.9, color='gray', zorder=1)
        ax.set_axisbelow(True)
        xt_pos, xt_lab = [], []
        for ci, cat in enumerate(cats):
            gc = ci * (n_llms * spacing + gap)
            xt_pos.append(gc + (n_llms - 1) * spacing / 2); xt_lab.append(cat)
            col = f'{col_prefix}{cat}'
            if col not in df_form.columns: continue
            for li, llm in enumerate(active_llms):
                xp = gc + li * spacing
                vals = df_form[df_form['LLM'] == llm][col].dropna().values * 100
                if len(vals) == 0: continue
                ax.boxplot(vals, positions=[xp], widths=w, patch_artist=True, showfliers=True,
                    medianprops=dict(color='white', linewidth=3.2),
                    whiskerprops=dict(color=colors[llm], linewidth=2.2),
                    capprops=dict(color=colors[llm], linewidth=2.2),
                    flierprops=dict(marker='o', markerfacecolor=colors[llm], markeredgecolor='white',
                                    markersize=6, alpha=0.6, linestyle='none'),
                    boxprops=dict(facecolor=colors[llm], alpha=0.80, edgecolor=colors[llm], linewidth=1.8))
        ax.set_xticks(xt_pos); ax.set_xticklabels(xt_lab, fontsize=20, fontweight='bold', ha='center')
        ax.set_ylabel('Recall (%)', fontsize=28, fontweight='bold')
        ax.set_title(f'{panel_label}. {title}', fontsize=30, fontweight='bold', pad=18, color='#1A1A2E')
        ax.set_ylim(-16, 100)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
        ax.tick_params(axis='y', labelsize=18)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        for sp in ax.spines.values(): sp.set_color('#BBBBBB'); sp.set_linewidth(1)
        for ci, cat in enumerate(cats):
            n = gs_counts.get(cat, 0)
            xc = ci * (n_llms * spacing + gap) + (n_llms - 1) * spacing / 2
            ax.text(xc, -9, f'n={n}', ha='center', va='top', fontsize=26, color='#444444', style='italic')
        ax.axhline(100, color='#999999', linewidth=1.2, linestyle=':', alpha=0.7, zorder=2)
        handles = [mpatches.Patch(facecolor=colors[ln], edgecolor='#888888', linewidth=0.8, label=ln)
                   for ln in active_llms]
        ax.legend(handles=handles, title='Platform', fontsize=18, title_fontsize=20,
                  loc='upper left', frameon=True, edgecolor='#CCCCCC', fancybox=False,
                  framealpha=0.92, ncol=3)

    print("  Creating Subplot B …")
    draw_boxplot_panel(fig.add_subplot(gs_layout[1]), METH_LABELS, 'meth_',
                       'NLP Methodology Coverage\n(15 formulations per platform)',
                       'B', meth_counts_gs_disp)
    print("  Creating Subplot C …")
    draw_boxplot_panel(fig.add_subplot(gs_layout[2]), SUB_CATS, 'sub_',
                       'Substance Coverage\n(15 formulations per platform)', 'C', sub_counts_gs)

    plt.tight_layout()
    for ext in ('png', 'pdf'):
        plt.savefig(f'LLM_Retrieval_Performance_FiveYearBars.{ext}', dpi=300, bbox_inches='tight',
                    facecolor=fig.get_facecolor(), edgecolor='none', pad_inches=0.25)
    print("✓ Saved: LLM_Retrieval_Performance_FiveYearBars.png / .pdf")
    plt.show()
    return {'bins': bins, 'gs_bin_counts': gs_bin, 'platform_bin_counts': plat_bin}

# ===== EXECUTION =====
print("\n" + "="*80)
print("RUNNING KNOWLEDGE GAP ANALYSIS — MAIN SUBSTANCES (BOXPLOT VERSION)")
print("="*80)
if 'base_results' in globals() and base_results:
    print(f"✓ Gold standard studies : {len(base_results['analyzer'].gold_standard_df)}")
    print(f"✓ Recalled files        : {len(base_results['analyzer'].recalled_files)}")
    try:
        gap_results = academic_llm_knowledge_gap_analysis(base_results)
        academic_llm_knowledge_gap_fiveyear_bars(base_results, gap_results)
        print("\n✅ Analysis completed successfully!")
        print(f"✅ Formulation df shape : {gap_results['df_formulations'].shape}")
        print(f"✅ Report               : {gap_results['report_filename']}")
        print(f"✅ gap_results ready for cell 27 ({len(gap_results['never_retrieved_ids'])} never-retrieved studies)")
    except Exception as e:
        import traceback
        print(f"\n❌ Error: {e}")
        traceback.print_exc()
        gap_results = None
else:
    print("❌ base_results not available — run Part 1A first.")
    gap_results = None

## Cell 17 — DIMINISHING RETURNS

*Figure 8 + Table C-4 (from derived data)*


In [ ]:
# ===== PART: DIMINISHING RETURNS ANALYSIS (2x2 Layout) =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from collections import defaultdict, Counter
from itertools import combinations
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# UPSET PLOT — half-width, large dots, readable
# =====================================================================
def draw_upset_fullwidth(fig, parent_gs_cell, llm_names, llm_retrieved_studies,
                         study_mapping, llm_colors, top_n=20):
    LLMs     = llm_names
    N_LLMS   = len(LLMs)
    all_sids = list(study_mapping.keys())

    study_pattern = {
        sid: frozenset(ln for ln in LLMs if sid in llm_retrieved_studies.get(ln, set()))
        for sid in all_sids
    }
    pattern_counts = Counter(study_pattern.values())

    empty = frozenset()
    top_patterns = [p for p in sorted(pattern_counts, key=lambda p: -pattern_counts[p])
                    if p != empty][:top_n]
    if empty in pattern_counts:
        top_patterns.append(empty)
    N_COLS = len(top_patterns)

    set_sizes = {ln: len(llm_retrieved_studies.get(ln, set())) for ln in LLMs}

    inner = gridspec.GridSpecFromSubplotSpec(
        2, 2,
        subplot_spec=parent_gs_cell,
        height_ratios=[0.42, 0.58],
        width_ratios=[0.10, 0.90],
        hspace=0.06,
        wspace=0.03,
    )
    ax_bar   = fig.add_subplot(inner[0, 1])
    ax_mat   = fig.add_subplot(inner[1, 1])
    ax_set   = fig.add_subplot(inner[1, 0])
    ax_blank = fig.add_subplot(inner[0, 0])
    ax_blank.axis('off')

    # ── Dot matrix ──
    ax_mat.set_xlim(-0.5, N_COLS - 0.5)
    ax_mat.set_ylim(-0.6, N_LLMS - 0.4)
    ax_mat.axis('off')

    for yi in range(N_LLMS):
        if yi % 2 == 0:
            ax_mat.axhspan(yi - 0.42, yi + 0.42,
                           facecolor='#F2F2F2', zorder=0, linewidth=0)
    for yi in range(N_LLMS):
        ax_mat.axhline(yi, color='#DDDDDD', linewidth=1.0, zorder=1)

    DOT_FILLED = 480
    DOT_EMPTY  = 200
    for xi, pattern in enumerate(top_patterns):
        in_rows  = [yi for yi, ln in enumerate(LLMs) if ln in     pattern]
        out_rows = [yi for yi, ln in enumerate(LLMs) if ln not in pattern]
        if out_rows:
            ax_mat.scatter([xi]*len(out_rows), out_rows,
                           s=DOT_EMPTY, color='#D5D5D5',
                           zorder=2, linewidths=0, clip_on=False)
        if len(in_rows) >= 2:
            ax_mat.plot([xi, xi], [min(in_rows), max(in_rows)],
                        color='#2C3E50', linewidth=4.5,
                        zorder=3, solid_capstyle='round')
        if in_rows:
            ax_mat.scatter([xi]*len(in_rows), in_rows,
                           s=DOT_FILLED,
                           c=[llm_colors[LLMs[yi]] for yi in in_rows],
                           zorder=4, linewidths=1.2,
                           edgecolors='white', clip_on=False)

    ax_mat.set_yticks(range(N_LLMS))
    ax_mat.set_yticklabels(LLMs, fontsize=22, fontweight='bold')
    ax_mat.tick_params(axis='y', left=False, labelleft=True, pad=6)

    # ── Intersection size bars ──
    counts = [pattern_counts[p] for p in top_patterns]
    bar_colors = []
    for p in top_patterns:
        if   len(p) == 0:      bar_colors.append('#AAAAAA')
        elif len(p) == 1:      bar_colors.append(llm_colors[next(iter(p))])
        elif len(p) == N_LLMS: bar_colors.append('#1A252F')
        else:                  bar_colors.append('#546E7A')

    ax_bar.bar(range(N_COLS), counts, color=bar_colors,
               edgecolor='white', linewidth=0.6, width=0.72, zorder=3)
    for xi, cnt in enumerate(counts):
        ax_bar.text(xi, cnt + 0.12, str(cnt),
                    ha='center', va='bottom',
                    fontsize=18, fontweight='bold', color='#222222')

    ax_bar.set_xlim(-0.5, N_COLS - 0.5)
    ax_bar.set_ylim(0, max(counts) * 1.35)
    ax_bar.set_xticks([])
    ax_bar.set_ylabel('Intersection\nSize', fontsize=20, fontweight='bold', labelpad=4)
    ax_bar.yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=5))
    ax_bar.tick_params(axis='y', labelsize=14)
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.spines['bottom'].set_visible(False)
    ax_bar.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.8)
    ax_bar.set_axisbelow(True)

    if empty in pattern_counts:
        nxi = top_patterns.index(empty)
        ax_bar.annotate(
            f'Never retrieved\n(n={pattern_counts[empty]})',
            xy=(nxi, pattern_counts[empty]),
            xytext=(nxi - 3, pattern_counts[empty] + 1.5),
            fontsize=16, color='#888888', ha='center',
            arrowprops=dict(arrowstyle='->', color='#BBBBBB', lw=1.2),
        )

    # ── Set-size bars ──
    max_set = max(set_sizes.values())
    ax_set.set_xlim(max_set * 1.4, 0)
    ax_set.set_ylim(-0.6, N_LLMS - 0.4)
    ax_set.barh(range(N_LLMS),
                [set_sizes[ln] for ln in LLMs],
                color=[llm_colors[ln] for ln in LLMs],
                edgecolor='white', linewidth=0.5,
                height=0.55, zorder=3)
    for yi, ln in enumerate(LLMs):
        ax_set.text(set_sizes[ln] + 1, yi, str(set_sizes[ln]),
                    ha='left', va='center',
                    fontsize=14, fontweight='bold', color='#333333')
    for yi in range(N_LLMS):
        if yi % 2 == 0:
            ax_set.axhspan(yi - 0.42, yi + 0.42,
                           facecolor='#F2F2F2', zorder=0, linewidth=0)

    ax_set.set_yticks([])
    ax_set.set_xlabel('Set\nSize', fontsize=18, fontweight='bold', labelpad=4)
    ax_set.xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=3))
    ax_set.tick_params(axis='x', labelsize=14)
    ax_set.spines['top'].set_visible(False)
    ax_set.spines['right'].set_visible(False)
    ax_set.spines['left'].set_visible(False)
    ax_set.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.8)
    ax_set.set_axisbelow(True)

# =====================================================================
# MAIN FUNCTION
# =====================================================================
def create_enhanced_diminishing_returns_with_pairs(base_results):
    analyzer = base_results['analyzer']
    print("="*80)
    print("  ENHANCED DIMINISHING RETURNS + UPSET PLOT  (2×2 Layout)")
    print("="*80)

    # ── Study mapping ──
    study_mapping = {}
    for idx, row in analyzer.gold_standard_df.iterrows():
        study_id = row.get('Study ID', f'Study_{idx}')
        identifiers = set()
        for col, val in [('Title', row.get('Title')), ('DOI', row.get('DOI'))]:
            if pd.notna(val):
                n = analyzer.normalize_identifier_consistent(str(val))
                if n: identifiers.add(n)
        n = analyzer.normalize_identifier_consistent(str(study_id))
        if n: identifiers.add(n)
        study_mapping[study_id] = {'identifiers': identifiers, 'row_data': row}
    print(f"✓ Mapped {len(study_mapping)} gold standard studies")

    # ── Collect matches ──
    llm_names  = ['Ai2 Finder', 'Consensus', 'ChatGPT', 'Claude', 'Gemini']
    llm_colors = {
        'Ai2 Finder': '#2E7D32',
        'Consensus':  '#FF6F00',
        'ChatGPT':    '#1565C0',
        'Claude':     '#9C27B0',
        'Gemini':     '#D32F2F',
        'All LLMs':   '#424242',
    }
    strategy_names = {'1':'Zero-Shot','2':'Few-Shot','3':'Persona','4':'CoT','5':'PICO'}

    all_combinations_matches = {}
    llm_retrieved_studies    = defaultdict(set)

    for filename, info in analyzer.recalled_files.items():
        llm    = info['llm']
        parsed = analyzer.parse_filename(filename)
        if not parsed: continue
        strat_name = strategy_names.get(parsed['strategy'], f"S{parsed['strategy']}")
        combo_key  = f"{llm}-{strat_name}-V{parsed['variation']}"
        try:
            df = pd.read_csv(info['path'])
            matches_raw = set()
            for col in ['DOI', 'Study name', 'Title', 'LLM Title']:
                if col in df.columns:
                    for val in df[col].dropna().astype(str).str.strip():
                        n = analyzer.normalize_identifier_consistent(val)
                        if n: matches_raw.add(n)
            matched_ids = {sid for sid, si in study_mapping.items()
                           if si['identifiers'] & matches_raw}
            if combo_key not in all_combinations_matches:
                all_combinations_matches[combo_key] = matched_ids
            else:
                all_combinations_matches[combo_key].update(matched_ids)
            llm_retrieved_studies[llm].update(matched_ids)
        except Exception as e:
            print(f"  Warning: {filename}: {e}")

    print(f"✓ Processed {len(all_combinations_matches)} combinations")
    for llm in llm_names:
        tot   = len(llm_retrieved_studies[llm])
        other = set().union(*(llm_retrieved_studies[o] for o in llm_names if o != llm))
        u     = len(llm_retrieved_studies[llm] - other)
        print(f"  {llm}: {tot} total ({tot-u} shared, {u} unique)")

    # ── Pairwise ──
    pairwise_data = []
    for llm1, llm2 in combinations(llm_names, 2):
        s1, s2 = llm_retrieved_studies[llm1], llm_retrieved_studies[llm2]
        union  = s1 | s2
        pairwise_data.append({
            'LLM1': llm1, 'LLM2': llm2,
            'Both_Retrieved': len(s1 & s2),
            'Only_LLM1': len(s1 - s2),
            'Only_LLM2': len(s2 - s1),
            'Union': len(union),
            'Jaccard': len(s1 & s2) / len(union) if union else 0,
        })
    pairwise_df = pd.DataFrame(pairwise_data)

    # ── Greedy selection ──
    remaining  = [{'combo_key': k, 'study_ids': v}
                  for k, v in all_combinations_matches.items()]
    cumulative_studies = set()
    selected_order, cum_counts, inc_gains = [], [], []
    while remaining:
        best, best_gain, best_idx = None, -1, -1
        for i, c in enumerate(remaining):
            g = len(c['study_ids'] - cumulative_studies)
            if g > best_gain:
                best_gain, best, best_idx = g, c, i
        if best_gain == 0: break
        cumulative_studies.update(best['study_ids'])
        selected_order.append(best['combo_key'])
        cum_counts.append(len(cumulative_studies))
        inc_gains.append(best_gain)
        remaining.pop(best_idx)

    print(f"\n✓ Greedy: {cum_counts[-1]}/{len(study_mapping)} "
          f"({cum_counts[-1]/len(study_mapping)*100:.1f}%)")

    unique_contributions, shared_contributions = {}, {}
    for llm in llm_names:
        other = set().union(*(llm_retrieved_studies[o] for o in llm_names if o != llm))
        u = llm_retrieved_studies[llm] - other
        unique_contributions[llm] = len(u)
        shared_contributions[llm] = len(llm_retrieved_studies[llm]) - len(u)

    # =====================================================================
    # FIGURE  — 2×2 grid
    #   Row 0 (formulation level): [A] Strategy-Variation  |  [C] Diminishing Returns
    #   Row 1 (platform level):    [B] UpSet Plot          |  [D] Top-10 LLM Pairs
    # =====================================================================
    plt.style.use('default')
    fig = plt.figure(figsize=(36, 28))   # wider than tall to fit 2 columns well

    outer_gs = gridspec.GridSpec(
        2, 2,
        height_ratios=[1, 1.3],          # Row 0 slightly shorter than row 1
        width_ratios=[1, 1],
        hspace=0.38,
        wspace=0.28,
        figure=fig,
    )

    # ── A (top-left): Strategy-Variation lines ──────────────────────────
    ax1 = fig.add_subplot(outer_gs[0, 0])

    sv_data = defaultdict(lambda: defaultdict(list))
    for ck, sids in all_combinations_matches.items():
        parts = ck.split('-')
        llm   = parts[0]
        sp, vp = [], None
        for p in parts[1:]:
            if p.startswith('V') and p[1:].isdigit(): vp = p; break
            sp.append(p)
        if vp:
            sv = f"{'-'.join(sp)}-{vp}"
            sv_data[sv][llm].append(len(sids))

    def sort_sv(k):
        p = k.split('-V')
        o = {'Zero-Shot':1,'Few-Shot':2,'Persona':3,'CoT':4,'PICO':5}
        try:    return (o.get(p[0], 99), int(p[1]))
        except: return (99, 0)

    sv_keys = sorted(sv_data.keys(), key=sort_sv)
    x_sv    = np.arange(len(sv_keys))

    cur_s, shade = None, True
    for i, sv in enumerate(sv_keys):
        s = sv.split('-V')[0]
        if s != cur_s: cur_s = s; shade = not shade
        if shade:
            ax1.axvspan(i-.5, i+.5, facecolor='lightgray', alpha=0.15, zorder=0)

    for llm in llm_names:
        cts = [np.mean(sv_data[sv][llm]) if llm in sv_data[sv] else 0 for sv in sv_keys]
        ax1.plot(x_sv, cts, marker='o', linewidth=3.5, markersize=8,
                 color=llm_colors[llm], label=llm, alpha=0.9)

    combined = []
    for sv in sv_keys:
        cs = set()
        for ck, sids in all_combinations_matches.items():
            if '-'.join(ck.split('-')[1:]) == sv: cs.update(sids)
        combined.append(len(cs))

    ax1.plot(x_sv, combined, marker='s', linewidth=4, markersize=9,
             color=llm_colors['All LLMs'], label='All LLMs Combined',
             alpha=0.9, linestyle='--')
    ax1.axhline(y=len(study_mapping), color='#424242',
                linestyle=':', linewidth=2, alpha=0.6)

    all_v = ([v for llm in llm_names for sv in sv_keys
               if llm in sv_data[sv] for v in sv_data[sv][llm]] + combined)
    ax1.set_ylim(max(0, min(all_v)-5), min(len(study_mapping)+5, max(all_v)+10))
    ax1.set_xticks(x_sv)
    ax1.set_xticklabels(sv_keys, rotation=45, ha='right', fontsize=16)
    ax1.set_xlabel('Prompt Strategy-Variation', fontsize=20, fontweight='bold')
    ax1.set_ylabel('Number of Retrieved Studies', fontsize=20, fontweight='bold')
    ax1.set_title('A. Study Retrieval by Strategy-Variation',
                  fontsize=26, fontweight='bold', pad=14)
    ax1.grid(True, alpha=0.3, linewidth=0.8)
    ax1.minorticks_on()
    ax1.legend(loc='upper center', fontsize=20, frameon=True,
               edgecolor='black', fancybox=False, ncol=3)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.tick_params(axis='both', labelsize=14)

    # ── C (top-right): Diminishing returns ──────────────────────────────
    ax3 = fig.add_subplot(outer_gs[0, 1])
    x_c = np.arange(len(selected_order))

    ax3.plot(x_c, cum_counts, linewidth=3, color='#1565C0', alpha=0.85,
             marker='o', markersize=5, label='Cumulative Studies Retrieved', zorder=3)
    ax3.axhline(y=len(study_mapping), color='#424242',
                linestyle=':', linewidth=2, alpha=0.6,
                label=f'Gold Standard (n={len(study_mapping)})', zorder=2)

    ax3t = ax3.twinx()
    ax3t.bar(x_c, inc_gains, alpha=0.35, color='#757575',
             width=0.8, label='Incremental Gain', zorder=1)
    ax3t.set_ylabel('Incremental Gain (New Studies)', fontsize=18, fontweight='bold')
    ax3t.set_ylim(0, max(inc_gains) * 1.3)
    ax3t.spines['top'].set_visible(False)
    ax3t.tick_params(axis='y', labelsize=14)

    def abbrev(ck):
        parts = ck.split('-')
        a = {'Zero':'ZS','Few':'FS','Persona':'PS','CoT':'CT','PICO':'PC'}
        return f"{parts[0][:3]}-{a.get(parts[1], parts[1][:2])}-{parts[-1]}"

    ax3.set_xticks(x_c)
    ax3.set_xticklabels([abbrev(c) for c in selected_order],
                        rotation=30, ha='center', fontsize=20)
    ax3.set_xlabel('LLM-Strategy-Variation Combinations\n(Ordered by Incremental Contribution)',
                   fontsize=24, fontweight='bold')
    ax3.set_ylabel('Cumulative Studies Retrieved', fontsize=18, fontweight='bold')
    ax3.set_title('C. Diminishing Returns Analysis',
                  fontsize=26, fontweight='bold', pad=14)
    ax3.grid(True, alpha=0.5, linewidth=1, axis='y')
    ax3.set_axisbelow(True)
    ax3.set_ylim(0, len(study_mapping) * 1.05)
    ax3.spines['top'].set_visible(False)
    ax3.tick_params(axis='both', labelsize=10)
    l1, lb1 = ax3.get_legend_handles_labels()
    l2, lb2 = ax3t.get_legend_handles_labels()
    ax3.legend(l1+l2, lb1+lb2, loc='upper left', fontsize=20,
               frameon=True, edgecolor='black')

    # ── B (bottom-left): UpSet plot ──────────────────────────────────────
    ax_b_title = fig.add_subplot(outer_gs[1, 0])
    ax_b_title.axis('off')
    ax_b_title.set_title(
        'B. Platform Retrieval Overlap',
        fontsize=26, fontweight='bold', pad=12
    )
    draw_upset_fullwidth(fig, outer_gs[1, 0],
                         llm_names, llm_retrieved_studies,
                         study_mapping, llm_colors,
                         top_n=20)

    # ── D (bottom-right): Top 10 LLM pairs ──────────────────────────────
    ax4 = fig.add_subplot(outer_gs[1, 1])
    top_10 = pairwise_df.sort_values('Union', ascending=False).head(10)
    pair_labels = [f"{r['LLM1']}\n+\n{r['LLM2']}" for _, r in top_10.iterrows()]
    xp = np.arange(len(pair_labels))

    ax4.bar(xp, top_10['Both_Retrieved'].values,
            label='Both Retrieved (Overlap)', color='#3498DB', alpha=0.8,
            edgecolor='black', linewidth=1)
    ax4.bar(xp, top_10['Only_LLM1'].values,
            bottom=top_10['Both_Retrieved'].values,
            label='Unique to First LLM', color='#E74C3C', alpha=0.8,
            edgecolor='black', linewidth=1)
    ax4.bar(xp, top_10['Only_LLM2'].values,
            bottom=top_10['Both_Retrieved'].values + top_10['Only_LLM1'].values,
            label='Unique to Second LLM', color='#F39C12', alpha=0.8,
            edgecolor='black', linewidth=1)

    for i, (_, row) in enumerate(top_10.iterrows()):
        ax4.text(i, row['Union']+1, f"n={row['Union']}",
                 ha='center', va='bottom', fontsize=18, fontweight='bold')

    ax4.set_xlabel('LLM Pairs', fontsize=22, fontweight='bold')
    ax4.set_ylabel('Number of Studies', fontsize=22, fontweight='bold')
    ax4.set_title('D. Top 10 LLM Pairs by Total Coverage',
                  fontsize=26, fontweight='bold', pad=14)
    ax4.set_xticks(xp)
    ax4.set_xticklabels(pair_labels, fontsize=22, fontweight='bold')
    ax4.legend(loc='upper right', fontsize=22, frameon=True, edgecolor='black')
    ax4.grid(True, axis='y', alpha=0.5, linewidth=1)
    ax4.set_axisbelow(True)
    ax4.spines['top'].set_visible(False)
    ax4.spines['right'].set_visible(False)
    ax4.tick_params(axis='both', labelsize=11)

    # ── Save ────────────────────────────────────────────────────────────
    plt.savefig('Enhanced_Diminishing_Returns_With_Pairs.png',
                format='png', dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig('Enhanced_Diminishing_Returns_With_Pairs.pdf',
                format='pdf', dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()

    all_retrieved   = len(set().union(*llm_retrieved_studies.values()))
    never_retrieved = len(study_mapping) - all_retrieved
    print(f"\nFinal: {cum_counts[-1]}/{len(study_mapping)} | Never retrieved: {never_retrieved}")

    return {
        'selected_order': selected_order,
        'cumulative_counts': cum_counts,
        'incremental_gains': inc_gains,
        'unique_contributions': unique_contributions,
        'shared_contributions': shared_contributions,
        'never_retrieved_count': never_retrieved,
        'pairwise_df': pairwise_df,
        'top_10_pairs': top_10,
        'study_mapping': study_mapping,
        'llm_retrieved_studies': llm_retrieved_studies,
        'all_combinations_matches': all_combinations_matches,
    }

def generate_diminishing_returns_report(enhanced_results, base_results,
                                         output_file='report_diminishing_returns.txt'):
    n_gold = len(enhanced_results['study_mapping'])
    selected = enhanced_results['selected_order']
    cum_counts = enhanced_results['cumulative_counts']
    inc_gains = enhanced_results['incremental_gains']

    with open(output_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("DIMINISHING RETURNS ANALYSIS REPORT\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Combinations evaluated: {len(selected)}\n")
        f.write(f"Gold Standard studies: {n_gold}\n\n")

        f.write("-" * 80 + "\n")
        f.write("GREEDY SEQUENTIAL COMBINATION STRATEGY\n")
        f.write("-" * 80 + "\n")
        f.write(f"{'Step':<6}{'Platform-Formulation':<28}{'New':>6}{'Cumulative':>12}"
                f"{'Coverage %':>12}{'Marginal Gain':>16}\n")
        prev_pct = 0.0
        for step, (combo, cum, gain) in enumerate(zip(selected, cum_counts, inc_gains), start=1):
            pct = cum / n_gold * 100 if n_gold else 0.0
            marginal = pct - prev_pct if step > 1 else 100.0
            f.write(f"{step:<6}{combo:<28}{gain:>6}{cum:>12}{pct:>11.1f}%{marginal:>15.1f}%\n")
            prev_pct = pct
        f.write(f"\nFinal coverage: {cum_counts[-1]}/{n_gold} "
                f"({cum_counts[-1] / n_gold * 100:.1f}%)\n" if cum_counts else "\nNo combinations selected.\n")
        f.write(f"Never retrieved by any combination: {enhanced_results['never_retrieved_count']}\n\n")

        f.write("-" * 80 + "\n")
        f.write("UNIQUE AND SHARED CONTRIBUTIONS BY PLATFORM\n")
        f.write("-" * 80 + "\n")
        f.write(f"{'Platform':<16}{'Unique':>10}{'Shared':>10}{'Total':>10}\n")
        uniq = enhanced_results['unique_contributions']
        shar = enhanced_results['shared_contributions']
        for llm in sorted(uniq.keys()):
            total = uniq[llm] + shar[llm]
            f.write(f"{llm:<16}{uniq[llm]:>10}{shar[llm]:>10}{total:>10}\n")
        f.write("\n")

        f.write("-" * 80 + "\n")
        f.write("TOP PAIRWISE COMBINATIONS BY UNION RECALL\n")
        f.write("-" * 80 + "\n")
        top_10 = enhanced_results['top_10_pairs']
        f.write(f"{'Pair':<32}{'Union':>8}{'Coverage %':>12}{'Jaccard':>10}\n")
        for _, row in top_10.iterrows():
            pct = row['Union'] / n_gold * 100 if n_gold else 0.0
            f.write(f"{row['LLM1']} + {row['LLM2']:<18}{row['Union']:>8}{pct:>11.1f}%{row['Jaccard']:>10.3f}\n")
        f.write("\n" + "=" * 80 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 80 + "\n")

    print(f"✓ Report saved: {output_file}")
    return output_file

if 'base_results' in globals() and base_results:
    print("\n" + "="*80)
    print("  RUNNING ENHANCED DIMINISHING RETURNS — 2×2 LAYOUT")
    print("="*80)
    enhanced_results = create_enhanced_diminishing_returns_with_pairs(base_results)
    report_file = generate_diminishing_returns_report(enhanced_results, base_results)
    print("\n✓ Done. Saved: Enhanced_Diminishing_Returns_With_Pairs.png/.pdf")
else:
    print("❌ base_results not available. Run Part 1A first.")

## Cell 18 — FISHER + POWER

*Fisher + power*


In [ ]:
# ===== FISHER'S EXACT TEST: NEVER-RETRIEVED SUBSTANCE AND METHODOLOGY ASSOCIATIONS =====
from scipy.stats import fisher_exact

def _fisher_power(n1, n0, p1, p0, n_sim=4000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed); sig = 0
    for _ in range(n_sim):
        a = rng.binomial(n1, p1); c = rng.binomial(n0, p0)
        _, pv = fisher_exact([[a, n1 - a], [c, n0 - c]]); sig += pv < alpha
    return sig / n_sim
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np

def test_never_retrieved_associations(gap_results):
    """
    Fisher's exact tests for associations between publication characteristics
    and never-retrieved status, with FDR correction (Benjamini-Hochberg).
    Consistent with Statistical Analysis section of manuscript.
    """
    print("\n" + "="*80)
    print("FISHER'S EXACT TESTS — NEVER-RETRIEVED ASSOCIATIONS")
    print("="*80)

    never_ids  = gap_results['never_retrieved_ids']
    ever_ids   = (gap_results['platform_ever']['Ai2 Finder'] |
                  gap_results['platform_ever']['Consensus']  |
                  gap_results['platform_ever']['ChatGPT']    |
                  gap_results['platform_ever']['Claude']     |
                  gap_results['platform_ever']['Gemini'])
    total_gs   = len(never_ids) + len(ever_ids)

    study_sub  = gap_results['study_substance_cat']
    study_meth = gap_results['study_method_cat']
    sub_gs     = gap_results['sub_gs_counts']
    meth_gs    = gap_results['meth_gs_counts']

    results = []

    # ── Substance categories ───────────────────────────────────────────
    print("\n--- Substance Categories ---")
    SUB_CATS = ['Cannabinoids', 'Opioids', 'Tobacco',
                'Alcohol', 'Polysubstance', 'Other']

    for cat in SUB_CATS:
        # 2x2 contingency table
        # Rows: never-retrieved vs retrieved
        # Cols: category vs not-category
        nr_cat     = sum(1 for s in never_ids if study_sub.get(s) == cat)
        nr_not     = len(never_ids) - nr_cat
        ret_cat    = sub_gs[cat] - nr_cat
        ret_not    = len(ever_ids) - ret_cat

        table = [[nr_cat, nr_not],
                 [ret_cat, ret_not]]

        odds, p = fisher_exact(table, alternative='two-sided')
        n_in = nr_cat + ret_cat; n_out = nr_not + ret_not
        p_in = nr_cat / n_in if n_in else 0.0; p_out = nr_not / n_out if n_out else 0.0
        power = _fisher_power(n_in, n_out, p_in, p_out)
        results.append({
            'Characteristic': cat,
            'Type': 'Substance',
            'Never_Retrieved': nr_cat,
            'Total_Never': len(never_ids),
            'Retrieved': ret_cat,
            'Total_Retrieved': len(ever_ids),
            'Pct_Never': nr_cat / len(never_ids) * 100 if len(never_ids) > 0 else 0,
            'Pct_GS':    sub_gs[cat] / total_gs * 100,
            'OddsRatio': odds,
            'p_raw':     p,
            'Power':     round(power, 3),
            'N_in_category': n_in,
            'Exploratory_small_n': n_in <= 5,
        })
        print(f"  {cat:20s}: {nr_cat}/{len(never_ids)} never-retrieved"
              f" vs {sub_gs[cat]}/{total_gs} GS | OR={odds:.2f} | p={p:.4f}")

    # ── NLP Methodology categories ────────────────────────────────────
    print("\n--- NLP Methodology Categories ---")
    METH_CATS = ['Rule-based',
                 'Conventional Machine Learning',
                 'Deep Learning (Non-Transformer)',
                 'Large Language Models (LLMs)']

    for cat in METH_CATS:
        nr_cat  = sum(1 for s in never_ids if study_meth.get(s) == cat)
        nr_not  = len(never_ids) - nr_cat
        ret_cat = meth_gs[cat] - nr_cat
        ret_not = len(ever_ids) - ret_cat

        table = [[nr_cat, nr_not],
                 [ret_cat, ret_not]]

        odds, p = fisher_exact(table, alternative='two-sided')
        n_in = nr_cat + ret_cat; n_out = nr_not + ret_not
        p_in = nr_cat / n_in if n_in else 0.0; p_out = nr_not / n_out if n_out else 0.0
        power = _fisher_power(n_in, n_out, p_in, p_out)
        results.append({
            'Characteristic': cat,
            'Type': 'NLP Methodology',
            'Never_Retrieved': nr_cat,
            'Total_Never': len(never_ids),
            'Retrieved': ret_cat,
            'Total_Retrieved': len(ever_ids),
            'Pct_Never': nr_cat / len(never_ids) * 100 if len(never_ids) > 0 else 0,
            'Pct_GS':    meth_gs[cat] / total_gs * 100,
            'OddsRatio': odds,
            'p_raw':     p,
            'Power':     round(power, 3),
            'N_in_category': n_in,
            'Exploratory_small_n': n_in <= 5,
        })
        print(f"  {cat:35s}: {nr_cat}/{len(never_ids)} never-retrieved"
              f" vs {meth_gs[cat]}/{total_gs} GS | OR={odds:.2f} | p={p:.4f}")

    # ── FDR Correction (Benjamini-Hochberg) ───────────────────────────
    df = pd.DataFrame(results)
    _, p_fdr, _, _ = multipletests(df['p_raw'].values,
                                    alpha=0.05, method='fdr_bh')
    df['p_fdr'] = p_fdr
    df['Significant_FDR'] = df['p_fdr'] < 0.05

    print("\n" + "="*80)
    print("FDR-CORRECTED RESULTS (Benjamini-Hochberg, α = 0.05)")
    print("="*80)
    print(f"\n{'Characteristic':35s} {'Type':15s} {'%Never':>7s} "
          f"{'%GS':>6s} {'OR':>6s} {'p_raw':>8s} {'p_fdr':>8s} {'Sig':>5s}")
    print("-"*100)
    for _, row in df.iterrows():
        sig = "✓" if row['Significant_FDR'] else ""
        print(f"  {row['Characteristic']:33s} {row['Type']:15s} "
              f"{row['Pct_Never']:>6.1f}% {row['Pct_GS']:>5.1f}% "
              f"{row['OddsRatio']:>6.2f} {row['p_raw']:>8.4f} "
              f"{row['p_fdr']:>8.4f} {sig:>5s}")

    # ── Save ──────────────────────────────────────────────────────────
    out_file = 'fisher_never_retrieved_associations.csv'
    df.to_csv(out_file, index=False)
    print(f"\n✓ Results saved: {out_file}")

    # ── Manuscript-ready summary ──────────────────────────────────────
    print("\n" + "="*80)
    print("MANUSCRIPT-READY SUMMARY")
    print("="*80)
    sig_chars = df[df['Significant_FDR']]['Characteristic'].tolist()
    if sig_chars:
        print(f"Significant associations (FDR p < 0.05): {', '.join(sig_chars)}")
    else:
        print("No characteristics reached significance after FDR correction.")
    nonsig = df[~df['Significant_FDR']][['Characteristic', 'p_fdr']]
    if len(nonsig) > 0:
        print(f"Non-significant after FDR correction "
              f"(all FDR p ≥ {nonsig['p_fdr'].min():.2f}):")
        for _, r in nonsig.iterrows():
            print(f"  {r['Characteristic']:35s}: FDR p = {r['p_fdr']:.3f}")

    return df

# ===== EXECUTION =====
print("\n" + "="*80)
print("RUNNING FISHER'S EXACT TESTS — NEVER-RETRIEVED ASSOCIATIONS")
print("="*80)

if 'gap_results' in globals() and gap_results:
    fisher_results = test_never_retrieved_associations(gap_results)
    print("\n✅ Tests completed successfully!")
else:
    print("❌ gap_results not available — run corrected_knowledge_gap_analysis first.")

## Cell 19 — AI2 TOP-100

*R2.2 top-100 + pooled capped recall*


In [ ]:
# ============================================================================
#  AI2 FINDER TOP-100 SENSITIVITY ANALYSIS  (Reviewer 2, comment 2)
#  Supplementary robustness check, NOT a change to the primary analysis.
#
#  Rationale (principled asymmetry, not selective treatment):
#  Consensus is already capped at the first 100 rank-ordered results because its
#  "load more" option is functionally unlimited — the cap standardizes it to a
#  bounded, clinician-realistic effort. Ai2 Finder is the other platform whose
#  ranked output regularly exceeds 100 (median 151, max 271 across the 225 runs;
#  >100 in 35/45 runs), so the same cap is applied here for comparability.
#  Conversational platforms (ChatGPT, Claude, Gemini) are NOT capped: their raw
#  output medians are 10-20 and their observed maxima are 55 / 82 / 109 across
#  all 225 runs (never approaching even a 100 cap in any run). Capping them would
#  not change a single retrieval — it would only suggest comparability that the
#  data does not support. This study evaluates real-world retrieval exposure
#  (what a clinician would or would not see), not benchmark-normalized ranking;
#  capping a platform that is not exhibiting unbounded output would substitute an
#  artificial ceiling for the platform's actual, observed capability. The primary
#  analysis therefore preserves each platform's full output; this cell reports
#  Ai2's metrics under a 100-cap as a supplementary check only.
# ============================================================================
import numpy as np
import pandas as pd

TOPK = 100

def ai2_topk_sensitivity(base_results, k=TOPK):
    a = base_results['analyzer']
    rs = base_results['recall_scores']
    norm = a.normalize_identifier_consistent
    rows = []
    pooled_full_matched_gold = set()
    pooled_capped_matched_gold = set()

    for var_key, info in rs.items():
        llm, sv = var_key.split('-', 1)
        if llm != 'Ai2 Finder':
            continue
        strat, var = sv[0], sv[1]

        std_by_run = {s['run']: s for fn, s in a.standardized_files.items()
                      if s['llm'] == llm and s['strategy'] == strat and s['variation'] == var}
        rec_by_run = {s['run']: s for fn, s in a.recalled_files.items()
                      if s['llm'] == llm and s['strategy'] == strat and s['variation'] == var}

        full_output_sizes, capped_sizes = [], []
        capped_matched_gold = set()
        full_matched_gold = set()
        capped_retrieved_norm = set()
        full_retrieved_norm = set()

        for run, std_info in std_by_run.items():
            raw_df = a.load_csv_file_content(std_info['path'])
            title_col = next((c for c in ['Title', 'title', 'Study Title', 'Article Title', 'Study name']
                              if c in raw_df.columns), None)
            if title_col is None or raw_df.empty:
                continue
            ranked_titles = raw_df[title_col].dropna().astype(str).str.strip().tolist()
            full_output_sizes.append(len(ranked_titles))
            full_retrieved_norm.update(norm(t) for t in ranked_titles)

            capped_titles = ranked_titles[:k]
            capped_sizes.append(len(capped_titles))
            capped_norm_this_run = {norm(t) for t in capped_titles}
            capped_retrieved_norm.update(capped_norm_this_run)

            rec_info = rec_by_run.get(run)
            if rec_info is None:
                continue
            rec_df = a.load_csv_file_content(rec_info['path'])
            if rec_df.empty or 'LLM Title' not in rec_df.columns or 'Title' not in rec_df.columns:
                continue
            for _, row in rec_df.iterrows():
                llm_title_norm = norm(row['LLM Title'])
                gold_title_norm = norm(row['Title'])
                if not gold_title_norm:
                    continue
                full_matched_gold.add(gold_title_norm)
                if llm_title_norm in capped_norm_this_run:
                    capped_matched_gold.add(gold_title_norm)

        if not full_output_sizes:
            continue

        pooled_full_matched_gold.update(full_matched_gold)
        pooled_capped_matched_gold.update(capped_matched_gold)

        recall_full = len(full_matched_gold) / a.gold_standard_count
        recall_capped = len(capped_matched_gold) / a.gold_standard_count
        prec_full = min(len(full_matched_gold) / len(full_retrieved_norm), 1.0) if full_retrieved_norm else 0.0
        prec_capped = min(len(capped_matched_gold) / len(capped_retrieved_norm), 1.0) if capped_retrieved_norm else 0.0

        rows.append({'Formulation_ID': f"{strat}_{var}", 'Full_Key': var_key,
                     'Full_output_size': int(np.mean(full_output_sizes)),
                     'Capped_size': int(np.mean(capped_sizes)),
                     'Recall_full': recall_full, 'Recall_topk': recall_capped,
                     'Precision_full': prec_full, 'Precision_topk': prec_capped,
                     'Cap_bound': any(s > k for s in full_output_sizes)})

    df = pd.DataFrame(rows)
    if df.empty:
        print("No Ai2 Finder formulations found."); return df

    print("=" * 78)
    print(f"  AI2 FINDER TOP-{k} SENSITIVITY  (supplementary; primary analysis unchanged)")
    print("=" * 78)
    n_bound = int(df['Cap_bound'].sum())
    print(f"  Formulations where the cap actually binds: {n_bound}/{len(df)}")
    print(f"\n  {'Metric':12s} {'Full (primary)':>16s} {'Top-{k} (sensitivity)':>22s} {'Delta':>8s}")
    for metric in ['Recall', 'Precision']:
        full_mean = df[f'{metric}_full'].mean()
        topk_mean = df[f'{metric}_topk'].mean()
        print(f"  {metric:12s} {full_mean:16.3f} {topk_mean:22.3f} {topk_mean-full_mean:8.3f}")

    gsc = base_results['analyzer'].gold_standard_count
    pooled_full = len(pooled_full_matched_gold) / gsc
    pooled_capped = len(pooled_capped_matched_gold) / gsc
    print(f"\n  Pooled (platform-level) recall — union across all formulations and runs:")
    print(f"    {'Full output':16s} {len(pooled_full_matched_gold):2d}/{gsc} = {pooled_full*100:.1f}%")
    print(f"    {'Top-' + str(k) + ' capped':16s} {len(pooled_capped_matched_gold):2d}/{gsc} = {pooled_capped*100:.1f}%")
    print(f"    {'Studies dropped by cap':16s} {len(pooled_full_matched_gold) - len(pooled_capped_matched_gold)}")
    df.attrs['pooled_recall_full'] = pooled_full
    df.attrs['pooled_recall_capped'] = pooled_capped

    df.to_csv('ai2_topk_sensitivity.csv', index=False)
    pd.DataFrame([{'Level': 'Pooled (platform-level)',
                   'Recall_full': pooled_full, 'Recall_topk': pooled_capped,
                   'N_full': len(pooled_full_matched_gold),
                   'N_capped': len(pooled_capped_matched_gold),
                   'Gold_standard_count': gsc}]).to_csv('ai2_topk_pooled_recall.csv', index=False)
    print(f"\n  ✓ saved ai2_topk_sensitivity.csv ({len(df)} formulations)")
    print(f"  ✓ saved ai2_topk_pooled_recall.csv (pooled full vs capped)")
    return df

if 'base_results' in globals() and base_results:
    ai2_topk_results = ai2_topk_sensitivity(base_results, k=TOPK)
else:
    print("❌ base_results not found — run the base-analysis cell first.")

## Cell 20 — FEW-SHOT SIDE

*Few-shot exclusion*


In [ ]:
# ============================================================================
#  FEW-SHOT EXAMPLE-EXCLUSION — SIDE ANALYSIS (not part of primary results)
#  No reviewer requested this. It tests a self-claim in Table B-3 / C-1.2.1 that
#  in-context example studies were excluded from recall. Run separately, alongside
#  the primary (uncorrected-for-this) results, to confirm whether ignoring the
#  exclusion changes anything meaningfully. Does NOT modify df_variations,
#  statistical_results, or any primary output file.
# ============================================================================
import numpy as np
import pandas as pd

# Few-Shot in-context example studies (Table B-3), matched by gold title:
EXAMPLE_GOLD_TITLES = [
    "Characterizing Documentation of Cannabis Use Across a Large Health System Using Natural Language Processing",
    "Using Implicit Information to Identify Smoking Status in Smoke-Blind Medical Discharge Summaries",
    "Detecting Opioid-Related Aberrant Behavior using Natural Language Processing",
    "An Automated System for Identifying Alcohol Use Status from Clinical Text",
]
EXAMPLE_STRATEGY = '2'  # Few-Shot only

def _example_id_set(analyzer):
    norm = analyzer.normalize_identifier_consistent
    ids = set()
    gs = analyzer.gold_standard_df
    for t in EXAMPLE_GOLD_TITLES:
        nt = norm(t); ids.add(nt)
        hit = gs[gs['Title'].astype(str).map(norm) == nt]
        for _, row in hit.iterrows():
            for col in ('Title', 'DOI', 'Study ID'):
                if col in row and pd.notna(row[col]):
                    ids.add(norm(row[col]))
    return ids

def run_fewshot_exclusion_sideanalysis(base_results):
    a = base_results['analyzer']
    rs = base_results['recall_scores']
    norm = a.normalize_identifier_consistent
    ex_ids = _example_id_set(a)
    print(f"Example studies resolved to {len(ex_ids)} normalized identifiers "
          f"(expect ~{len(EXAMPLE_GOLD_TITLES)} studies, possibly fewer "
          f"if not all are in the 83-study gold standard).")

    rows = []
    for var_key, info in rs.items():
        llm, sv = var_key.split('-', 1); strat, var = sv[0], sv[1]
        matched_with = set(norm(m) for m in info['matches'])
        if strat == EXAMPLE_STRATEGY:
            matched_without = matched_with - ex_ids
        else:
            matched_without = matched_with
        rows.append({'Full_Key': var_key, 'LLM': llm, 'Strategy_Code': strat,
                     'Formulation_ID': f"{strat}_{var}",
                     'Recall_with_examples': len(matched_with) / a.gold_standard_count,
                     'Recall_without_examples': len(matched_without) / a.gold_standard_count,
                     'N_example_studies_matched': len(matched_with) - len(matched_without)})
    df = pd.DataFrame(rows)
    fs = df[df['Strategy_Code'] == EXAMPLE_STRATEGY]

    print("\n" + "=" * 78)
    print("  FEW-SHOT EXAMPLE-EXCLUSION SIDE ANALYSIS  (Few-Shot formulations only)")
    print("=" * 78)
    print(f"  Formulations affected: {len(fs)}")
    print(f"  Example studies matched (sum across formulations): {fs['N_example_studies_matched'].sum()}")
    delta = fs['Recall_without_examples'] - fs['Recall_with_examples']
    print(f"\n  Mean recall WITH examples:    {fs['Recall_with_examples'].mean():.4f}")
    print(f"  Mean recall WITHOUT examples: {fs['Recall_without_examples'].mean():.4f}")
    print(f"  Mean delta: {delta.mean():+.4f}  (max |delta| = {delta.abs().max():.4f})")
    print(f"\n  → {'Materially changes recall' if delta.abs().max() > 0.02 else 'Negligible effect (≤2 pp recall)'}"
          f" — {'recommend implementing the exclusion' if delta.abs().max() > 0.02 else 'safe to either implement or remove the manuscript claim'}.")

    df.to_csv('fewshot_exclusion_side_analysis.csv', index=False)
    print("\n  ✓ saved fewshot_exclusion_side_analysis.csv (side analysis only; primary results untouched)")
    return df

if 'base_results' in globals() and base_results:
    fewshot_side_results = run_fewshot_exclusion_sideanalysis(base_results)
else:
    print("❌ base_results not found — run the base-analysis cell first.")

## Cell 21 — THRESHOLD BAND + SUB-60 MATCHES

*Confirmed-FN band + threshold_sensitivity_report.txt*


In [ ]:
# ============================================================================
# REVIEWER-RESPONSE: THRESHOLD BAND ANALYSIS + SUB-60% MATCH ASSIGNMENT
#   (4) Are confirmed false negatives limited to [50,60), or also in [40,50)?
#   (3) Assign accepted sub-60% near-exact matches to their LLM-formulations.
# Reads the adjudicated sensitivity file (02_Sensitivity_Analysis_40to60.csv).
# These matches are REDUNDANT (each study is already retrieved >=60% by the same
# platform elsewhere), so platform/formulation recall is UNCHANGED; this block
# documents them for a fully auditable matched dataset and the threshold defense.
# ============================================================================
import os
import pandas as pd

try:
    # Locate the adjudicated sensitivity file: check the Desktop (cwd) and the
    # pooled-run data folder. Put 02_Sensitivity_Analysis_40to60.csv in either.
    from pathlib import Path as _P
    _names = ['02_Sensitivity_Analysis_20to60.csv', '02_Sensitivity_Analysis_40to60.csv',
              'Sensitivity_Analysis_Match_Threshold.csv']
    _dirs  = [_P.cwd(),
              _P('~/Desktop').expanduser(),
              _P('~/Desktop/Pooled Std Recall Runs').expanduser()]
    _sens_path = next((str(d / n) for d in _dirs for n in _names if (d / n).exists()), None)
    if _sens_path is None:
        raise FileNotFoundError(
            "Could not find 02_Sensitivity_Analysis_40to60.csv. Place it on ~/Desktop "
            "(or in ~/Desktop/Pooled Std Recall Runs) and re-run.")
    sens = pd.read_csv(_sens_path)
    print(f"  Reading adjudicated sensitivity file: {_sens_path}")
    fn = sens[sens['Sensitivity_Class'] == 'Confirmed false negative'].copy()
    fn['Score'] = fn['Match_Score'].astype(str).str.rstrip('%').astype(float)

    print("=" * 72)
    print("  (4) CONFIRMED FALSE NEGATIVES — BAND DISTRIBUTION")
    print("=" * 72)
    print(fn['Band'].value_counts().to_string())
    print(f"\n  Occurrences: {len(fn)} | Distinct studies: {fn['Gold_Title'].nunique()}")
    print(f"  Score range: {fn['Score'].min():.2f}% – {fn['Score'].max():.2f}%")

    band_by_study = (fn.groupby('Gold_Title')['Band']
                       .agg(lambda s: tuple(sorted(set(s)))))
    only_4050 = band_by_study[band_by_study.apply(lambda b: b == ('[40,50)',))]
    spans_both = band_by_study[band_by_study.apply(lambda b: len(b) == 2)]
    only_5060 = band_by_study[band_by_study.apply(lambda b: b == ('[50,60)',))]
    print(f"\n  Studies only in [50,60): {len(only_5060)}")
    print(f"  Studies only in [40,50): {len(only_4050)}")
    print(f"  Studies spanning BOTH bands: {len(spans_both)}")
    print("\n  => Confirmed false negatives are NOT limited to [50,60); genuine")
    print("     format-artifact matches occur down to {:.2f}%. Lowering the threshold".format(fn['Score'].min()))
    print("     therefore recovers only REDUNDANT occurrences of studies already")
    print("     retrieved >=60% by the same platform (study-level recall unchanged).")

    # Verify the two facts that make this strengthen the threshold defense
    try:
        _pm = next((str(d / 'Primary_Matches_60plus_FINAL.csv') for d in _dirs
                    if (d / 'Primary_Matches_60plus_FINAL.csv').exists()),
                   'Primary_Matches_60plus_FINAL.csv')
        primary = pd.read_csv(_pm)
        def _n(t): return ''.join(str(t).lower().split())
        gcol = [c for c in primary.columns if 'gold' in c.lower() and 'title' in c.lower()][0]
        prim_titles = {_n(t) for t in primary[gcol]}
        fn_titles = {_n(t) for t in fn['Gold_Title']}
        in_primary = sum(1 for t in fn_titles if t in prim_titles)
        print(f"\n  Verification: {in_primary}/{len(fn_titles)} FN studies are matched >=60% "
              f"in the primary file (recall unchanged).")
    except FileNotFoundError:
        print("\n  (Primary_Matches_60plus_FINAL.csv not in working dir — skipping >=60% cross-check.)")

    print("\n" + "=" * 72)
    print("  (3) ACCEPTED SUB-60% MATCHES BY PLATFORM-FORMULATION")
    print("=" * 72)
    fn['Formulation'] = 'S' + fn['Strategy'].astype(str) + 'V' + fn['Variation'].astype(str)
    add_map = (fn.groupby(['Platform', 'Formulation'])['Gold_Title']
                 .apply(lambda s: sorted(set(s))).reset_index())
    for _, r in add_map.iterrows():
        print(f"  {r['Platform']:18s} {r['Formulation']:5s} += {len(r['Gold_Title'])} study(ies)")
    out = fn[['Platform', 'Strategy', 'Variation', 'Run', 'LLM_Title',
              'Gold_Title', 'Match_Score', 'Band']].copy()
    out['Match_Type'] = 'Sub-threshold confirmed false negative (format artifact)'
    out['Recall_Impact'] = 'None (study already retrieved >=60% by same platform)'
    out.to_csv('Accepted_SubThreshold_Matches.csv', index=False)
    print(f"\n  ✓ saved Accepted_SubThreshold_Matches.csv ({len(out)} occurrences, "
          f"{fn['Gold_Title'].nunique()} distinct studies)")

    try:
        _band_counts = sens['Band'].value_counts().to_dict()
        _total_candidates = len(sens)
        _fn_band = fn['Band'].value_counts().to_dict()
        with open('threshold_sensitivity_report.txt', 'w') as _tf:
            _tf.write("THRESHOLD SENSITIVITY ANALYSIS — SUB-60% MATCH ADJUDICATION\n")
            _tf.write("=" * 64 + "\n\n")
            _tf.write(f"Primary matching threshold: 60% Jaccard word-overlap.\n")
            _tf.write(f"Sub-threshold candidates examined: {_total_candidates}\n")
            for _b in sorted(_band_counts):
                _tf.write(f"  {_b}: {_band_counts[_b]} candidates\n")
            _tf.write("\nConfirmed false negatives (same study, format artifact only):\n")
            _tf.write(f"  Total occurrences: {len(fn)}\n")
            _tf.write(f"  Distinct studies:  {fn['Gold_Title'].nunique()}\n")
            for _b in sorted(_fn_band):
                _tf.write(f"  {_b}: {_fn_band[_b]} occurrences\n")
            _tf.write(f"  Score range of confirmed FNs: {fn['Score'].min():.2f}% – "
                      f"{fn['Score'].max():.2f}%\n\n")
            _tf.write("Key findings:\n")
            _tf.write("  - Confirmed false negatives are NOT limited to [50,60); genuine\n")
            _tf.write(f"    format-artifact matches occur down to {fn['Score'].min():.2f}%.\n")
            _tf.write("  - Every confirmed false-negative study is already retrieved at\n")
            _tf.write("    >=60% by the same platform elsewhere, so study-level recall is\n")
            _tf.write("    unchanged; lowering the threshold recovers only redundant\n")
            _tf.write("    duplicate occurrences, not any new study.\n")
            _tf.write("  - Zero overlap between confirmed false negatives and the\n")
            _tf.write("    never-retrieved studies; the never-retrieved finding is\n")
            _tf.write("    independent of threshold choice.\n")
        print("  ✓ saved threshold_sensitivity_report.txt")
    except Exception as _e:
        print(f"  [SKIP] threshold_sensitivity_report.txt not written: {_e}")

except FileNotFoundError:
    print("02_Sensitivity_Analysis_40to60.csv not found in working dir — "
          "place the adjudicated sensitivity file alongside the notebook and re-run.")

## Cell 22 — THRESHOLD SWEEP DIGEST + TREND FIGURE

*4-band sweep digest + monotonic-trend figure*


In [ ]:
# ============================================================================
# THRESHOLD SENSITIVITY — ACADEMIC TWO-PANEL FIGURE  (Cell 22 — replacement)
#   Grounded in the record-linkage / entity-matching convention for justifying a
#   similarity cutoff (e.g. Fellegi-Sunter score-distribution plots; precision/
#   recall/F1-vs-threshold sweeps):
#     Panel A  Distribution of Jaccard match scores for SAME-STUDY (accepted) vs
#              DIFFERENT-STUDY (non-match) candidate pairs, with the 60% primary
#              threshold marked. The separation/overlap shows why 60% is defensible.
#     Panel B  Precision, Recall and F1 as the acceptance threshold is swept from
#              20% to 100%. The F1 peak coincides with the adopted 60% cutoff, and
#              precision collapses below it -> lowering the threshold only adds
#              review burden, not genuine matches.
#   Same-study = Review_Decision contains "Accept" (exact + accepted near-exact +
#   sub-threshold confirmed). Different-study = everything else (rejected + non-match).
#   Per-pair scores read from Primary_Matches_60plus_FINAL.csv and
#   02_Sensitivity_Analysis_20to60.csv at runtime -- nothing hard-coded.
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pathlib import Path as _P

PRIMARY_THRESHOLD = 60.0
C_SAME = '#2A7A4B'   # same-study (true match)   — green
C_DIFF = '#B0463C'   # different-study (non-match) — muted red
C_PREC = '#2F6F9F'   # precision — blue
C_REC  = '#C77A30'   # recall    — amber
C_F1   = '#1A1A2E'   # F1        — near-black
GRID   = '#D9D9D9'

_dirs = [_P.cwd(), _P('~/Desktop').expanduser(),
         _P('~/Desktop/Pooled Std Recall Runs').expanduser()]
def _find(names):
    for d in _dirs:
        for n in names:
            if (d / n).exists():
                return str(d / n)
    return None
def _pct(s):
    return pd.to_numeric(s.astype(str).str.replace('%', '', regex=False).str.strip(),
                         errors='coerce')

_pm_path = _find(['Primary_Matches_60plus_FINAL.csv',
                  'Primary_Matches_60plus_with_rejected.csv',
                  'Primary_Matches_60plus_reviewed.csv',
                  'Primary_Matches_60plus_ACCEPTED_only.csv'])
_sa_path = _find(['02_Sensitivity_Analysis_20to60.csv', '02_Sensitivity_Analysis_40to60.csv'])

if _pm_path is None and _sa_path is None:
    print("[SKIP] No primary-match or sensitivity file found on Desktop / data folder.")
else:
    frames = []
    if _pm_path:
        pm = pd.read_csv(_pm_path, encoding='utf-8-sig')
        pm['ms'] = _pct(pm['Match_Score'])
        frames.append(pm[['ms', 'Review_Decision']])
        print(f"Reading primary matches  : {_pm_path}  (rows={len(pm)})")
    if _sa_path:
        sa = pd.read_csv(_sa_path, encoding='utf-8-sig')
        sa['ms'] = _pct(sa['Match_Score'])
        frames.append(sa[['ms', 'Review_Decision']])
        print(f"Reading sensitivity sweep: {_sa_path}  (rows={len(sa)})")

    d = pd.concat(frames, ignore_index=True).dropna(subset=['ms'])
    d['same_study'] = d['Review_Decision'].astype(str).str.contains('Accept', case=False, na=False)
    pos = d.loc[d['same_study'], 'ms'].to_numpy()
    neg = d.loc[~d['same_study'], 'ms'].to_numpy()
    P = len(pos)
    print(f"\nCandidate pairs: {len(d)}  (same-study={P}, different-study={len(neg)})")
    last_match  = float(np.nanmin(pos)) if len(pos) else PRIMARY_THRESHOLD  # lowest confirmed match
    sweep_floor = float(np.nanmin(d['ms']))                                 # how far the sweep looked
    print(f"Lowest confirmed same-study match: {last_match:.2f}%   |   sweep floor: {sweep_floor:.0f}%")

    # ---- precision / recall / F1 sweep ----
    T = np.arange(20.0, 100.001, 0.5)
    TP = np.array([(pos >= t).sum() for t in T], dtype=float)
    FP = np.array([(neg >= t).sum() for t in T], dtype=float)
    prec = np.divide(TP, TP + FP, out=np.ones_like(TP), where=(TP + FP) > 0)
    rec  = TP / P
    f1   = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(TP), where=(prec + rec) > 0)
    i_peak = int(np.argmax(f1))
    t_peak, f1_peak = T[i_peak], f1[i_peak]
    # metrics at the adopted threshold
    j = int(np.argmin(np.abs(T - PRIMARY_THRESHOLD)))
    print(f"F1 peak at {t_peak:.1f}%  (F1={f1_peak:.3f}); "
          f"at {PRIMARY_THRESHOLD:.0f}%  P={prec[j]:.3f} R={rec[j]:.3f} F1={f1[j]:.3f}")
    pd.DataFrame({'Threshold_pct': T, 'TP': TP.astype(int), 'FP': FP.astype(int),
                  'Precision': prec.round(4), 'Recall': rec.round(4), 'F1': f1.round(4)}
                 ).to_csv('threshold_sweep_metrics.csv', index=False)
    print("  ✓ saved threshold_sweep_metrics.csv")

    # ---- figure ----
    try:
        plt.rcParams.update({'font.size': 11, 'axes.edgecolor': '#888888'})
        fig, (axA, axB) = plt.subplots(1, 2, figsize=(14, 5.6))

        # ---------- Panel A : score distribution ----------
        bins = np.arange(20, 102.5, 2.5)
        axA.hist(neg, bins=bins, color=C_DIFF, alpha=0.55, label=f'Different study (n={len(neg):,})',
                 edgecolor='white', linewidth=0.4, zorder=2)
        axA.hist(pos, bins=bins, color=C_SAME, alpha=0.75, label=f'Same study (n={P:,})',
                 edgecolor='white', linewidth=0.4, zorder=3)
        axA.set_yscale('log')
        # region of the distribution that was EXAMINED below the primary cutoff
        axA.axvspan(sweep_floor, PRIMARY_THRESHOLD, color='#EDF0F3', alpha=0.85, zorder=0,
                    label=f'examined in sweep ({sweep_floor:.0f}–60%)')
        axA.axvspan(50, 77, color='#F2E6C9', alpha=0.30, zorder=1)  # overlap / review zone
        axA.axvline(PRIMARY_THRESHOLD, color='#222222', linestyle='-', linewidth=1.8, zorder=6)
        axA.axvline(last_match, color=C_SAME, linestyle='--', linewidth=1.7, zorder=6)
        axA.text(PRIMARY_THRESHOLD - 0.7, 1.35, 'primary threshold (60%)', rotation=90,
                 va='bottom', ha='right', fontsize=8.5, color='#222222')
        axA.text(last_match - 0.7, 1.35, f'lowest confirmed match ({last_match:.1f}%)',
                 rotation=90, va='bottom', ha='right', fontsize=8.2, color=C_SAME)
        axA.text(63.5, 0.55, 'overlap zone', fontsize=8.5, style='italic', color='#9a7d2e')
        axA.annotate('exact matches\n(score = 100)', xy=(99, 1990), xytext=(83, 250),
                     fontsize=8.5, color=C_SAME, ha='center',
                     arrowprops=dict(arrowstyle='->', color=C_SAME, lw=1))
        axA.set_xlabel('Jaccard match score (%)', fontsize=11)
        axA.set_ylabel('Candidate pairs (log scale)', fontsize=11)
        axA.set_title('A. Match-score distribution: same- vs different-study pairs',
                      fontsize=12, fontweight='bold')
        axA.set_xlim(20, 101)
        axA.legend(loc='upper center', fontsize=9, framealpha=0.9)
        axA.grid(axis='y', color=GRID, linestyle=':', linewidth=0.7, zorder=0)
        for s in ('top', 'right'):
            axA.spines[s].set_visible(False)

        # ---------- Panel B : precision / recall / F1 vs threshold ----------
        axB.plot(T, prec, color=C_PREC, lw=2.2, label='Precision')
        axB.plot(T, rec,  color=C_REC,  lw=2.2, label='Recall')
        axB.plot(T, f1,   color=C_F1,   lw=2.6, label='F1')
        axB.axvspan(sweep_floor, PRIMARY_THRESHOLD, color='#EDF0F3', alpha=0.85, zorder=0)
        axB.axvline(PRIMARY_THRESHOLD, color='#222222', linestyle='-', linewidth=1.8, zorder=4)
        axB.axvline(last_match, color=C_SAME, linestyle='--', linewidth=1.7, zorder=4)
        axB.scatter([t_peak], [f1_peak], color=C_F1, s=55, zorder=6)
        axB.annotate(f'F1 peak\n{t_peak:.0f}% (F1={f1_peak:.2f})',
                     xy=(t_peak, f1_peak), xytext=(t_peak - 22, f1_peak - 0.18),
                     fontsize=9, color=C_F1,
                     arrowprops=dict(arrowstyle='->', color=C_F1, lw=1))
        axB.text(PRIMARY_THRESHOLD + 1.2, 0.13, 'matches declared\nhere (60%)', fontsize=8.7,
                 color='#222222')
        axB.text(last_match - 1.2, 0.34, f'last genuine\nmatch ({last_match:.1f}%)', fontsize=8.5,
                 color=C_SAME, ha='right')
        # make explicit: the sweep did NOT stop at 60% -- it continued to the floor
        axB.annotate('', xy=(sweep_floor + 0.6, 0.05), xytext=(PRIMARY_THRESHOLD - 0.6, 0.05),
                     arrowprops=dict(arrowstyle='->', color='#5f6b7a', lw=1.5))
        axB.text((sweep_floor + PRIMARY_THRESHOLD) / 2, 0.08,
                 f'sensitivity sweep — examined to {sweep_floor:.0f}%',
                 ha='center', va='bottom', fontsize=8.2, color='#5f6b7a', style='italic')
        axB.set_xlabel('Acceptance threshold (Jaccard %)', fontsize=11)
        axB.set_ylabel('Score', fontsize=11)
        axB.set_title('B. Precision, recall and F1 vs. acceptance threshold',
                      fontsize=12, fontweight='bold')
        axB.set_xlim(20, 100)
        axB.set_ylim(0, 1.04)
        axB.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
        axB.legend(loc='lower right', fontsize=9, framealpha=0.9)
        axB.grid(color=GRID, linestyle=':', linewidth=0.7, zorder=0)
        for s in ('top', 'right'):
            axB.spines[s].set_visible(False)

        fig.suptitle('Threshold sensitivity of the 60% Jaccard matching cutoff',
                     fontsize=13.5, fontweight='bold', y=1.01)
        fig.tight_layout()
        fig.savefig('Threshold_Sensitivity_Trend.png', dpi=300, bbox_inches='tight')
        fig.savefig('Threshold_Sensitivity_Trend.pdf', bbox_inches='tight')
        plt.show()
        print("  ✓ saved Threshold_Sensitivity_Trend.png / .pdf")
    except Exception as _e:
        import traceback
        print(f"  [SKIP] figure not generated: {_e}")
        traceback.print_exc()

## Cell 23 — PLATFORM RECALL VERIFICATION

*Independent platform-recall + Cochran check from binary_retrieval_matrix.csv; cochran_foldin_adjusted.txt*


In [ ]:
# ============================================================================
# REVIEWER-RESPONSE: FOLD ACCEPTED 40-60% NEAR-MATCHES INTO THE RECALL ANALYSIS
#   The accepted sub-threshold near-matches are GENUINE same-study retrievals
#   that a platform output with a reformatted title (subtitle truncation,
#   abbreviation, numbered-list prefix). Folding them in makes recall concrete
#   at BOTH levels:
#     (A) PLATFORM level  -> augments the 83x5 binary matrix, recomputes
#         Cochran's Q. Verified effect: ChatGPT 49.4%->51.8% (+2 studies),
#         Claude 44.6%->45.8% (+1 study); Ai2/Consensus/Gemini unchanged;
#         never-retrieved set (10) unchanged.
#     (B) FORMULATION level -> credits each accepted near-match to its exact
#         (platform, strategy, variation) matched set and recomputes
#         formulation-level recall (the PRIMARY Friedman outcome). This part
#         re-derives matched sets from the raw recalled *R.csv files so the
#         base is internally consistent; run it in the data folder.
# ============================================================================
import os
import numpy as np
import pandas as pd
from pathlib import Path as _P

# ---- locate the adjudicated sweep file ----
_dirs = [_P.cwd(), _P('~/Desktop').expanduser(),
         _P('~/Desktop/Pooled Std Recall Runs').expanduser()]
_names = ['02_Sensitivity_Analysis_20to60.csv', '02_Sensitivity_Analysis_40to60.csv']
_sa_path = next((str(d / n) for d in _dirs for n in _names if (d / n).exists()), None)

if _sa_path is None:
    print("[SKIP] sensitivity file not found; cannot fold in near-matches.")
elif 'base_results' not in globals() or not base_results:
    print("[SKIP] base_results not in memory — run the BASE CLASS / MAIN cells first.")
else:
    sa = pd.read_csv(_sa_path)
    acc = sa[sa['Sensitivity_Class'] == 'Confirmed false negative'].copy()
    analyzer = base_results.get('analyzer') if isinstance(base_results, dict) else None

    def _norm(t):
        return ' '.join(str(t).lower().split())

    # ---------------------------------------------------------------
    # (A) PLATFORM-LEVEL: augment the per-platform retrieved-study sets
    #     and recompute Cochran's Q on the 83x5 binary matrix.
    # ---------------------------------------------------------------
    # Build the baseline 83x5 platform matrix from the authoritative
    # binary_retrieval_matrix.csv that the heatmap cell writes (its column sums
    # reproduce canonical platform recall: Ai2 60, ChatGPT 41, Claude 37,
    # Consensus 44, Gemini 42). This avoids undercounting from a recall_scores union.
    import glob as _glob
    _bm_path = next((p for p in ['binary_retrieval_matrix.csv',
                                 str(_P('~/Desktop/binary_retrieval_matrix.csv').expanduser())]
                     if os.path.exists(p)), None)
    plat_codes = ['Ai2', 'Cha', 'Cla', 'Con', 'Gem']
    plat_disp  = {'Ai2': 'Ai2 Finder', 'Cha': 'ChatGPT', 'Cla': 'Claude',
                  'Con': 'Consensus', 'Gem': 'Gemini'}
    _sa_to_code = {'Ai2 Paper Finder': 'Ai2', 'ChatGPT-4.1': 'Cha',
                   'Claude-Sonnet-4.0': 'Cla', 'Consensus': 'Con',
                   'Gemini-2.5-Pro': 'Gem'}

    if _bm_path is None:
        print("[SKIP] binary_retrieval_matrix.csv not found — run the DENDROGRAM HEATMAP "
              "cell first so the authoritative matrix is written, then re-run this cell.")
    else:
        bm = pd.read_csv(_bm_path)
        # Identify the title column and the platform-formulation columns (e.g. Ai2-Co0).
        _title_col = next((c for c in bm.columns
                           if c.lower() in ('title', 'study title', 'study_title', 'gold_title')),
                          bm.columns[1] if len(bm.columns) > 1 else bm.columns[0])
        _pf_cols = {pc: [c for c in bm.columns if c.startswith(pc + '-')] for pc in plat_codes}
        # Baseline per-platform retrieved-study set: any formulation column == 1.
        plat_studies = {}
        gold_titles = [_norm(t) for t in bm[_title_col].tolist()]
        for pc in plat_codes:
            cols = _pf_cols[pc]
            retr = set()
            for _, r in bm.iterrows():
                if any(int(r[c]) == 1 for c in cols if pd.notna(r[c]) and str(r[c]) in ('0', '1')):
                    retr.add(_norm(r[_title_col]))
            plat_studies[pc] = retr

        # Which accepted near-match studies are NEW to each platform (not already >=60% anywhere)?
        foldin_platform = {}
        for sa_plat, pc in _sa_to_code.items():
            folded = set(_norm(t) for t in acc[acc['Platform'] == sa_plat]['Gold_Title'])
            foldin_platform[pc] = folded - plat_studies.get(pc, set())

        print("=== (A) PLATFORM-LEVEL FOLD-IN (baseline from binary_retrieval_matrix.csv) ===")
        for pc in ['Ai2', 'Con', 'Gem', 'Cha', 'Cla']:
            b = len(plat_studies[pc]); a = b + len(foldin_platform[pc])
            flag = '' if a == b else f'  (+{a-b})'
            print(f"  {plat_disp[pc]:12} {b:2}/83={b/83*100:4.1f}%  ->  {a:2}/83={a/83*100:4.1f}%{flag}")

        # Cochran's Q before/after.
        try:
            from statsmodels.stats.contingency_tables import cochrans_q
            Mb = np.zeros((len(gold_titles), 5), dtype=int)
            for ti, g in enumerate(gold_titles):
                for pi, pc in enumerate(plat_codes):
                    Mb[ti, pi] = 1 if g in plat_studies[pc] else 0
            Ma = Mb.copy()
            for pi, pc in enumerate(plat_codes):
                for g in foldin_platform[pc]:
                    if g in gold_titles:
                        Ma[gold_titles.index(g), pi] = 1
            qb, qa = cochrans_q(Mb), cochrans_q(Ma)
            N_runs = 225
            print(f"\n  Cochran's Q (platform pooled recall):")
            print(f"    before: Q={qb.statistic:.3f}  p={qb.pvalue:.3g}  W={qb.statistic/(N_runs*4):.3f}")
            print(f"    after:  Q={qa.statistic:.3f}  p={qa.pvalue:.3g}  W={qa.statistic/(N_runs*4):.3f}")
            pd.DataFrame({'Platform': [plat_disp[pc] for pc in plat_codes],
                          'Recall_before': [int(Mb[:, i].sum()) for i in range(5)],
                          'Recall_after':  [int(Ma[:, i].sum()) for i in range(5)],
                          'Recall_before_pct': [round(Mb[:, i].sum()/83*100, 1) for i in range(5)],
                          'Recall_after_pct':  [round(Ma[:, i].sum()/83*100, 1) for i in range(5)]}
                         ).to_csv('platform_recall_with_nearmatch_foldin.csv', index=False)
            print("    ✓ saved platform_recall_with_nearmatch_foldin.csv")

            # Persist the adjusted Cochran's Q to a file so it appears in the
            # merged report (which aggregates saved files, not console output).
            with open('cochran_foldin_adjusted.txt', 'w') as _cf:
                _cf.write("PLATFORM POOLED RECALL — NEAR-MATCH FOLD-IN ADJUSTMENT\n")
                _cf.write("=" * 56 + "\n\n")
                _cf.write("The accepted sub-threshold near-matches are genuine same-study\n")
                _cf.write("retrievals output with a reformatted title. Folding them into the\n")
                _cf.write("platform pooled-recall matrix gives:\n\n")
                _cf.write(f"  {'Platform':18}{'before':>10}{'after':>10}\n")
                for pi, pc in enumerate(plat_codes):
                    _b = int(Mb[:, pi].sum()); _a = int(Ma[:, pi].sum())
                    _flag = '' if _a == _b else f'  (+{_a-_b})'
                    _cf.write(f"  {plat_disp[pc]:18}{_b:>3}/83={_b/83*100:4.1f}%"
                              f"  {_a:>3}/83={_a/83*100:4.1f}%{_flag}\n")
                _cf.write("\nCochran's Q (platform pooled recall, df = 4):\n")
                _cf.write(f"  before fold-in: Q = {qb.statistic:.3f}, p = {qb.pvalue:.3g}, "
                          f"W = {qb.statistic/(N_runs*4):.3f}\n")
                _cf.write(f"  after  fold-in: Q = {qa.statistic:.3f}, p = {qa.pvalue:.3g}, "
                          f"W = {qa.statistic/(N_runs*4):.3f}\n")
                _cf.write("\nOnly ChatGPT-4.1 (+2 studies) and Claude-Sonnet-4.0 (+1) change;\n")
                _cf.write("Ai2 Finder, Consensus, and Gemini-2.5-Pro are unchanged, the\n")
                _cf.write("never-retrieved set (10 studies) is unchanged, and the omnibus\n")
                _cf.write("test remains significant (p < 0.001).\n")
            print("    ✓ saved cochran_foldin_adjusted.txt")
        except Exception as _e:
            print(f"  [SKIP] Cochran recompute: {_e}")

    # ---------------------------------------------------------------
    # (B) FORMULATION-LEVEL: re-derive matched sets from raw *R.csv,
    #     credit accepted near-matches to their exact cell, recompute
    #     formulation recall, and (if df_variations present) refresh it.
    # ---------------------------------------------------------------
    print("\n=== (B) FORMULATION-LEVEL FOLD-IN ===")
    # Accepted near-matches keyed by (platform, strategy, variation).
    acc['_k'] = list(zip(acc['Platform'], acc['Strategy'].astype(int),
                         acc['Variation'].astype(int)))
    cell_adds = (acc.groupby(['Platform', 'Strategy', 'Variation'])['Gold_Title']
                    .apply(lambda s: set(_norm(x) for x in s)).to_dict())

    if 'df_variations' in globals():
        dfv = df_variations.copy()
        scol = 'Strategy_Code' if 'Strategy_Code' in dfv.columns else 'Strategy'

        # The formulation-level recompute needs the per-cell matched set from the
        # raw recalled files. base_results['recall_scores'] holds matched studies
        # per (LLM, strategy, variation); we augment each cell's set with the
        # accepted near-matches for that exact cell and recompute recall = |set|/83.
        # Strategy/variation in recall_scores keys are the numeric S,V used in filenames.
        def _augmented_recall(row):
            llm = row['LLM']
            # map df_variations LLM label to SA Platform label
            sa_plat = {'Ai2 Finder': 'Ai2 Paper Finder', 'ChatGPT': 'ChatGPT-4.1',
                       'Claude': 'Claude-Sonnet-4.0', 'Consensus': 'Consensus',
                       'Gemini': 'Gemini-2.5-Pro'}.get(llm, llm)
            s = int(row[scol]) if str(row.get(scol, '')).isdigit() else None
            # df_variations Variation is 0-based (0-2); the SA file is 1-based (1-3).
            v0 = int(row['Variation']) if str(row['Variation']).isdigit() else None
            v_sa = v0 + 1 if v0 is not None else None
            adds = cell_adds.get((sa_plat, s, v_sa), set()) if (s and v_sa) else set()
            return len(adds)

        dfv['near_matches_added'] = dfv.apply(_augmented_recall, axis=1)
        n_cells = int((dfv['near_matches_added'] > 0).sum())
        print(f"  Accepted near-matches credited to {n_cells} (platform,formulation) cells "
              f"({len(acc)} occurrences, {acc['Gold_Title'].nunique()} distinct studies).")
        print("  NOTE: to fold these into formulation RECALL and re-run the Friedman")
        print("  primary outcome, recompute |matched set| per cell from the raw *R.csv")
        print("  files in this folder (the per-cell matched sets live in")
        print("  base_results['recall_scores']); add the cell's near-match studies to")
        print("  that set, then recall = |augmented set| / 83. The annotation column")
        print("  'near_matches_added' marks exactly which cells change and by how much.")
        dfv.to_csv('df_variations_with_nearmatch_foldin.csv', index=False)
        print("  ✓ saved df_variations_with_nearmatch_foldin.csv")
    else:
        print("  [SKIP] df_variations not in memory.")

    print("\n  Never-retrieval (overall and by platform) is UNCHANGED: every folded-in")
    print("  study is retrieved by some platform, so none leaves the never-retrieved set.")

## Cell 24 — ADOPTED-PRIMARY SUMMARY

*Confirms adopted platform recall + strict-comparison toggle guide; adopted_primary_platform_recall.csv*


In [ ]:
# ============================================================================
# PRIMARY ANALYSIS — NEAR-MATCHES ADOPTED: SUMMARY & STRICT COMPARISON GUIDE
#   With ADOPT_NEARMATCHES=True (set in the BASE CLASS cell), the accepted
#   near-matches are folded into the matched sets at the source, so EVERY cell
#   above (MAIN STATISTICS, precision/consistency/F1, Cochran's Q, post-hocs,
#   figures) already reports the corrected primary numbers. This cell summarises
#   the adopted platform recall and records how to reproduce the strict-60%
#   comparison for the robustness statement.
# ============================================================================
import os
import pandas as pd
from pathlib import Path as _P

if 'base_results' not in globals() or not base_results:
    print("[SKIP] base_results not in memory.")
else:
    import pandas as pd
    from pathlib import Path as _P
    def _norm(t): return ' '.join(str(t).lower().split())

    DISPLAY    = ['Ai2 Finder', 'ChatGPT', 'Claude', 'Consensus', 'Gemini']
    _PF        = {'Ai2 Finder': 'Ai2', 'ChatGPT': 'Cha', 'Claude': 'Cla',
                  'Consensus': 'Con', 'Gemini': 'Gem'}
    _bm_path = next((p for p in ['binary_retrieval_matrix.csv',
                                 str(_P('~/Desktop/binary_retrieval_matrix.csv').expanduser())]
                     if _P(p).exists()), None)

    pooled = {}
    if _bm_path is not None:
        bm = pd.read_csv(_bm_path)
        _title_col = next((c for c in bm.columns if c.strip().lower() == 'title'), 'Title')
        for llm, pf in _PF.items():
            cols = [c for c in bm.columns if c.startswith(pf + '-')]
            sub  = bm[cols].apply(pd.to_numeric, errors='coerce').fillna(0)
            pooled[llm] = int((sub == 1).any(axis=1).sum())
        print("=== ADOPTED PRIMARY — platform pooled recall (from binary_retrieval_matrix.csv) ===")
        for llm in DISPLAY:
            n = pooled.get(llm, 0)
            print(f"  {llm:12} {n:2}/83 = {n/83*100:4.1f}%")
        print()
        print("  These match the canonical near-match-folded recall used by the")
        print("  non-retrieval report and the manuscript headline (Ai2 60/72.3%,")
        print("  ChatGPT 43/51.8%, Claude 38/45.8%, Consensus 44/53.0%, Gemini 42/50.6%).")
    else:
        # Fallback only if the binary matrix is absent (run the DENDROGRAM HEATMAP cell
        # first). Recomputing from recall_scores is known to diverge from canonical.
        print("  [WARN] binary_retrieval_matrix.csv not found; falling back to")
        print("         base_results['recall_scores'], which may DIVERGE from the")
        print("         canonical fold-in recall. Run the DENDROGRAM HEATMAP cell first.")
        rs = base_results['recall_scores']
        plat_pool = {}
        for vkey, info in rs.items():
            llm = vkey.split('-', 1)[0]
            plat_pool.setdefault(llm, set()).update(_norm(m) for m in info.get('matches', []))
        pooled = {llm: len(plat_pool.get(llm, set())) for llm in DISPLAY}
        for llm in DISPLAY:
            print(f"  {llm:12} {pooled[llm]:2}/83 = {pooled[llm]/83*100:4.1f}%")

    print()
    print("  STRICT-60% COMPARISON (for the robustness sentence): set")
    print("  ADOPT_NEARMATCHES=False in the BASE CLASS cell and re-run the notebook;")
    print("  recall_scores then reproduces the strict-60% primary (ChatGPT 41/49.4%,")
    print("  Claude 37/44.6%). The two runs give the strict-vs-adopted contrast that")
    print("  documents robustness (the omnibus conclusions are unchanged either way).")

    pd.DataFrame([{'Platform': k, 'Pooled_matched': v,
                   'Pooled_recall_pct': round(v / 83 * 100, 1)}
                  for k, v in sorted(pooled.items())]
                 ).to_csv('adopted_primary_platform_recall.csv', index=False)
    print("\n  \u2713 saved adopted_primary_platform_recall.csv")

## Cell 25 — APPENDIX C TABLE GENERATOR

*Emits the formulation-combinations-by-F1 table in exact Appendix C markdown from adopted df_variations; appendix_c_formulation_table.md/.csv*


In [ ]:
# ============================================================================
# APPENDIX C TABLE GENERATOR — formulation combinations ranked by F1
#   Emits the "strategy-formulation combinations, ranked by F1-score" table in
#   the EXACT Appendix C markdown format, from the adopted-primary df_variations
#   (i.e. with ADOPT_NEARMATCHES=True so the recovered matches are already in the
#   matched sets). Run after MAIN STATISTICS. Writes appendix_c_formulation_table.md
#   so the fold-in-corrected rows can be dropped straight into Appendix C without
#   hand-mapping any variation numbering.
#
#   Columns (exact): Rank | Platform | Strategy | Variation | Recall | Precision |
#                    F1 | Consistency | Matched studies | Total retrieved | Level
#   Conventions matched to the existing table:
#     - Platform short names: Ai2, ChatGPT, Claude, Consensus, Gemini
#     - Strategy labels: Zero-Shot, Few-Shot, Persona, CoT, PICO
#     - Variation labels: V0/V1/V2 (the df_variations 0-2 numbering)
#     - Recall/Precision/F1/Consistency to 3 decimals; counts as integers
#     - Sorted by F1 descending (ties broken by Recall, then Precision)
# ============================================================================
import os
import pandas as pd
from pathlib import Path as _P

if 'df_variations' not in globals():
    print("[SKIP] df_variations not in memory — run MAIN STATISTICS first.")
else:
    dfv = df_variations.copy()

    # --- normalise platform short names to match the existing table ---
    _plat_short = {'Ai2 Finder': 'Ai2', 'Ai2 Paper Finder': 'Ai2', 'Ai2': 'Ai2',
                   'ChatGPT': 'ChatGPT', 'ChatGPT-4.1': 'ChatGPT',
                   'Claude': 'Claude', 'Claude-Sonnet-4.0': 'Claude',
                   'Consensus': 'Consensus',
                   'Gemini': 'Gemini', 'Gemini-2.5-Pro': 'Gemini'}
    # --- normalise strategy labels to match the existing table ---
    _strat_lbl = {'Zero-Shot': 'Zero-Shot', 'Few-Shot': 'Few-Shot',
                  'Persona-based': 'Persona', 'Persona': 'Persona',
                  'Chain-of-Thought': 'CoT', 'CoT': 'CoT',
                  'PICO Framework': 'PICO', 'PICO': 'PICO'}

    def _platf(x): return _plat_short.get(str(x), str(x))
    def _strat(row):
        for col in ('Strategy', 'Strategy_Code'):
            if col in row and str(row[col]) in _strat_lbl:
                return _strat_lbl[str(row[col])]
        # numeric Strategy_Code fallback (1=Zero,2=Few,3=Persona,4=CoT,5=PICO)
        _num = {'1': 'Zero-Shot', '2': 'Few-Shot', '3': 'Persona', '4': 'CoT', '5': 'PICO'}
        return _num.get(str(row.get('Strategy_Code', '')), str(row.get('Strategy', '')))

    # column resolver (be tolerant of naming)
    def _col(*names):
        for n in names:
            if n in dfv.columns:
                return n
        return None
    c_rec = _col('Recall'); c_pre = _col('Precision'); c_f1 = _col('F1_Score', 'F1')
    c_con = _col('Consistency'); c_mat = _col('Matched_Count', 'Matched')
    c_tot = _col('Total_Retrieved', 'Total'); c_var = _col('Variation')

    rows = []
    for _, r in dfv.iterrows():
        try:
            var = int(r[c_var])
        except (TypeError, ValueError):
            var = r[c_var]
        rows.append({
            'Platform': _platf(r['LLM']),
            'Strategy': _strat(r),
            'Variation': f"V{var}",
            'Recall': round(float(r[c_rec]), 3),
            'Precision': round(float(r[c_pre]), 3),
            'F1': round(float(r[c_f1]), 3),
            'Consistency': round(float(r[c_con]), 3),
            'Matched studies': int(round(float(r[c_mat]))) if c_mat else '',
            'Total retrieved': int(round(float(r[c_tot]))) if c_tot else '',
        })
    out = pd.DataFrame(rows)
    # rank by F1 desc, then Recall desc, then Precision desc (stable, matches existing ordering)
    out = out.sort_values(['F1', 'Recall', 'Precision'],
                          ascending=[False, False, False]).reset_index(drop=True)
    out.insert(0, 'Rank', range(1, len(out) + 1))
    out['Level'] = 'Form.'

    # --- emit the exact Appendix C markdown table ---
    cols = ['Rank', 'Platform', 'Strategy', 'Variation', 'Recall', 'Precision',
            'F1', 'Consistency', 'Matched studies', 'Total retrieved', 'Level']
    lines = ['| ' + ' | '.join(f'**{c}**' for c in cols) + ' |',
             '| ' + ' | '.join('---' for _ in cols) + ' |']
    for _, r in out.iterrows():
        cells = [str(r['Rank']), r['Platform'], r['Strategy'], r['Variation'],
                 f"{r['Recall']:.3f}", f"{r['Precision']:.3f}", f"{r['F1']:.3f}",
                 f"{r['Consistency']:.3f}", str(r['Matched studies']),
                 str(r['Total retrieved']), r['Level']]
        lines.append('| ' + ' | '.join(cells) + ' |')
    table_md = '\n'.join(lines)

    with open('appendix_c_formulation_table.md', 'w') as _f:
        _f.write("# Appendix C — Strategy-formulation combinations, ranked by F1-score "
                 "(ADOPTED PRIMARY: near-matches folded in)\n\n")
        _f.write(table_md + "\n")
    out.to_csv('appendix_c_formulation_table.csv', index=False)

    print("=== APPENDIX C FORMULATION TABLE (adopted primary) ===")
    print(f"  {len(out)} rows written.")
    print("  ✓ saved appendix_c_formulation_table.md (drop-in markdown)")
    print("  ✓ saved appendix_c_formulation_table.csv")
    print("\n  Rows whose Matched studies / metrics shift under the fold-in (ChatGPT/Claude")
    print("  cells that received a recovered near-match) are now reflected automatically.")
    print("  Compare these rows to the current Appendix C table to confirm the deltas before")
    print("  replacing. Top of table for a quick check:")
    print(table_md.split('\n')[0])
    for ln in table_md.split('\n')[2:7]:
        print("   ", ln)